<div dir="rtl" align="right">

# NB12h — Audit ریشه‌های Golden Dataset و منطق تعمیم v1

## هدف Notebook

این Notebook برای پاسخ‌دادن به دو پرسش مستقل ساخته می‌شود:

1. آیا ۲۸ Pilot/Seed اصلی از نظر سؤال، پاسخ، evidence و پوشش علمی سالم هستند؟
2. آیا تعمیم آن‌ها به ۵۷۳ رکورد در Golden Dataset v1 به‌درستی انجام شده است یا رکوردهای قالبی، نزدیک‌تکراری یا فاقد پوشش جدید تولید شده‌اند؟

## جایگاه در Pipeline

این Notebook بعد از:

- Golden Dataset v1 فریز‌شده،
- آماده‌سازی بستهٔ ۲۸ Pilot برای بازبینی مؤلف،
- تشخیص افت کیفیت retrieval در NB13،
- و ساخت هستهٔ quote-grounded پذیرفته‌شدهٔ ۴۶ رکوردی،

اجرا می‌شود.

این Notebook قبل از ساخت Golden Dataset v2 قرار دارد.

## مسئله‌ای که حل می‌کند

Golden Dataset v1 دارای ۵۷۳ رکورد از ۲۸ Seed است. باید با evidence مشخص شود:

- چه تعداد از رکوردهای v1 از هر Seed تولید شده‌اند؛
- آیا descendantهای هر Seed واقعاً مستقل‌اند یا صرفاً بازنویسی‌های نزدیک‌اند؛
- آیا evidence و answerهای یکتا با تعداد سؤال‌ها تناسب دارند؛
- آیا مشکل observed در retrieval از خود Pilotها آمده یا از منطق تعمیم؛
- و هستهٔ ۴۶ رکوردی چه تفاوتی با descendantهای v1 دارد.

## Inputهای اصلی

- `golden_dataset_v1.jsonl`
- گزارش نهایی v1
- registry و package مربوط به ۲۸ Pilot
- artifactهای review مؤلف
- Golden experiment پذیرفته‌شدهٔ ۴۶ رکوردی
- گزارش acceptance نسخهٔ ۴۶ رکوردی
- chunk و manifestهای فریز‌شدهٔ NB11، در صورت نیاز به بررسی linkage

## Outputهای مورد انتظار

- گزارش کامل Pilot Audit
- گزارش Expansion Audit برای ۵۷۳ رکورد v1
- نقشهٔ `Pilot → Descendants → Evidence/Answer/Question clusters`
- مقایسهٔ v1 با هستهٔ ۴۶ رکوردی
- Source Coverage Matrix
- توصیهٔ مستند برای هر Pilot:
  `KEEP / REPAIR / SPLIT / DROP / ADD_COVERAGE`
- ورودی قابل‌استفاده برای بستهٔ بازبینی جدید دکتر و ساخت کنترل‌شدهٔ v2

## Scope

- تحلیل فقط‌خواندنی و قابل‌بازتولید
- تشخیص علت مشکل، نه ساخت فوری v2
- تفکیک مشکل Pilot از مشکل منطق تعمیم
- حفظ کامل lineage و artifactهای فعلی

## Out of Scope

- تغییر `golden_dataset_v1.jsonl`
- فعال‌سازی یا ساخت `golden_dataset_v2.jsonl`
- حذف artifactهای قبلی
- اجرای benchmark نهایی NB13
- اعمال نظر جدید مؤلف، پیش از دریافت آن

## معیار موفقیت نهایی

Notebook زمانی موفق است که با evidence واقعی نشان دهد:

1. هر یک از ۲۸ Pilot چه وضعیتی دارد؛
2. ۵۷۳ رکورد v1 چگونه از Pilotها توزیع شده‌اند؛
3. چه مقدار از تعمیم واقعاً پوشش جدید ایجاد کرده است؛
4. چه مقدار از آن نزدیک‌تکراری یا کم‌ارزش است؛
5. و برای ساخت v2 دقیقاً چه چیزی باید حفظ، اصلاح، تفکیک یا اضافه شود.

**این Notebook هیچ Golden Dataset فعالی را تغییر نمی‌دهد.**

</div>

<div dir="rtl" align="right">

## نقطهٔ شروع و برنامهٔ اجرای مرحلهٔ اول

در وضعیت فعلی:

- Golden Dataset v1 با ۵۷۳ رکورد و ۲۸ Seed وجود دارد و Frozen است.
- بستهٔ بازبینی مؤلف برای ۲۸ Pilot آماده شده، اما نظر نهایی مؤلف هنوز مبنای ساخت v2 نشده است.
- هستهٔ آزمایشی quote-grounded با ۴۶ رکورد، integrity صفر خطا و semantic PASS کامل پذیرفته شده است؛ اما active dataset نیست.
- هنوز مشخص نشده که مشکل اصلی v1 از خود ۲۸ Pilot بوده یا از منطق تولید و تعمیم ۵۷۳ descendant.

### اولین Step

ابتدا فقط discovery و preflight انجام می‌دهیم:

1. ریشهٔ واقعی پروژه را پیدا می‌کنیم.
2. مسیر artifactهای لازم را بدون تغییر فایل‌ها شناسایی می‌کنیم.
3. وجود Golden Dataset v1، گزارش آن، artifactهای Pilot/Author Review و هستهٔ ۴۶ رکوردی را بررسی می‌کنیم.
4. تعداد رکوردهای قابل‌خواندن v1 را فقط برای baseline ثبت می‌کنیم.

### چرا این Step لازم است؟

Audit نباید بر اساس path، filename یا وضعیت حدسی جلو برود. پیش از هر تحلیل باید بدانیم چه artifactهایی واقعاً در repository موجود هستند و کدام artifactها برای مرحله‌های بعدی در دسترس‌اند.

### Validation مرحله

این Step زمانی PASS است که:

- ریشهٔ پروژه با موفقیت کشف شود؛
- مسیر Golden Dataset v1 شناسایی شود؛
- تعداد واقعی رکوردهای v1 ثبت شود؛
- وضعیت وجود artifactهای کلیدی بدون تغییر آن‌ها گزارش شود.

خروجی این مرحله فقط یک inventory خواندنی است و هیچ فایل جدیدی تولید نمی‌کند.

</div>

In [1]:
# Code Cell 01
# NB12h — Centralized imports and bootstrap dependencies

from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from urllib.parse import urlparse

import hashlib
import json
import os
import re

from dotenv import load_dotenv
from openai import OpenAI

In [2]:
# Code Cell 02
# NB12h — Project discovery and read-only artifact preflight

current_path = Path.cwd().resolve()

project_root = next(
    (
        candidate
        for candidate in (
            current_path,
            *current_path.parents,
        )
        if (
            candidate / "notebooks" / "phase2"
        ).is_dir()
        and (
            candidate / "data" / "phase2"
        ).is_dir()
    ),
    None,
)

assert project_root is not None, (
    "Project root could not be discovered. "
    "Open NB12h from inside the RAG_finance repository."
)

golden_dir = (
    project_root
    / "notebooks"
    / "phase2"
    / "golden_dataset"
)

reports_phase2_dir = (
    project_root
    / "reports"
    / "phase2"
)

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()

def jsonl_record_count(path):
    with path.open(
        encoding="utf-8",
    ) as handle:
        return sum(
            1
            for line in handle
            if line.strip()
        )

artifact_paths = {
    "golden_dataset_v1": (
        golden_dir
        / "golden_dataset_v1.jsonl"
    ),
    "golden_dataset_v1_report": (
        golden_dir
        / "golden_dataset_v1_final_report.json"
    ),
    "final_46_experiment": (
        golden_dir
        / (
            "golden_dataset_quote_grounded_"
            "final_adjudicated_experiment.jsonl"
        )
    ),
    "final_46_integrity_audit": (
        golden_dir
        / (
            "golden_dataset_quote_grounded_"
            "final_adjudicated_experiment_"
            "final_integrity_audit.json"
        )
    ),
    "final_46_semantic_judge": (
        golden_dir
        / (
            "golden_dataset_quote_grounded_"
            "final_adjudicated_experiment_"
            "targeted_final_semantic_judge.json"
        )
    ),
    "final_46_acceptance": (
        reports_phase2_dir
        / (
            "golden_dataset_final_adjudicated_"
            "acceptance_v1.json"
        )
    ),
}

author_review_matches = sorted(
    project_root.rglob(
        "author_review_28_pilots.md"
    )
)

author_feedback_notebook_matches = sorted(
    project_root.rglob(
        "12f_author_feedback_integration.ipynb"
    )
)

human_review_registry_matches = sorted(
    project_root.rglob(
        "human_review_registry*.json"
    )
)

v1_path = artifact_paths["golden_dataset_v1"]

assert v1_path.is_file(), (
    "Golden Dataset v1 is missing: "
    f"{v1_path}"
)

v1_record_count = jsonl_record_count(v1_path)
v1_sha256 = sha256_file(v1_path)

preflight_inventory = {
    "project_root": str(project_root),
    "golden_directory": str(golden_dir),
    "v1_record_count": v1_record_count,
    "v1_sha256": v1_sha256,
    "artifacts": {
        name: {
            "path": str(path),
            "exists": path.is_file(),
        }
        for name, path in artifact_paths.items()
    },
    "author_review_28_matches": [
        str(path)
        for path in author_review_matches
    ],
    "author_feedback_notebook_matches": [
        str(path)
        for path in author_feedback_notebook_matches
    ],
    "human_review_registry_matches": [
        str(path)
        for path in human_review_registry_matches
    ],
    "read_only": True,
}

print("=== NB12h PROJECT DISCOVERY ===")
print("Project root:", project_root)
print("Golden directory:", golden_dir)
print("\nGolden Dataset v1:")
print("  Exists:", v1_path.is_file())
print("  Records:", v1_record_count)
print("  SHA-256:", v1_sha256)

print("\nKey artifacts:")

for name, descriptor in preflight_inventory[
    "artifacts"
].items():
    print(
        f"  - {name}: "
        f"{descriptor['exists']} | "
        f"{descriptor['path']}"
    )

print(
    "\nAuthor Review 28-pilot files found:",
    len(author_review_matches),
)

for path in author_review_matches:
    print("  -", path)

print(
    "\nAuthor-feedback notebooks found:",
    len(author_feedback_notebook_matches),
)

for path in author_feedback_notebook_matches:
    print("  -", path)

print(
    "\nHuman-review registry files found:",
    len(human_review_registry_matches),
)

for path in human_review_registry_matches:
    print("  -", path)

print("\nRead-only preflight:", True)
print("\nNB12h PROJECT DISCOVERY: PASS")

=== NB12h PROJECT DISCOVERY ===
Project root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Golden directory: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset

Golden Dataset v1:
  Exists: True
  Records: 573
  SHA-256: df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729

Key artifacts:
  - golden_dataset_v1: True | /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_v1.jsonl
  - golden_dataset_v1_report: True | /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_v1_final_report.json
  - final_46_experiment: True | /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_quote_grounded_final_adjudicated_experiment.jsonl
  - final_46_integrity_audit: True | /Users/macbookpro/AMIR_DATA/00_WORK

<div dir="rtl" align="right">

## مرحلهٔ دوم — بارگذاری artifactها و بررسی قرارداد واقعی آن‌ها

در مرحلهٔ قبل وجود فایل‌های لازم تأیید شد. اکنون بدون هیچ تغییر در فایل‌ها، artifactهای اصلی را بارگذاری می‌کنیم تا قرارداد واقعی داده مشخص شود.

### ورودی‌های این مرحله

- Golden Dataset v1 با ۵۷۳ رکورد
- گزارش نهایی v1
- فایل `author_review_28_pilots.md`
- registry فعلی human review
- هستهٔ quote-grounded پذیرفته‌شدهٔ ۴۶ رکوردی
- گزارش integrity، داوری معنایی و acceptance هستهٔ ۴۶ رکوردی

### کار این مرحله

1. خواندن امن JSON و JSONL.
2. ثبت تعداد واقعی رکوردهای v1 و هستهٔ ۴۶ رکوردی.
3. مشاهدهٔ fieldهای واقعی هر Dataset، بدون فرض Schema.
4. بررسی سازگاری Hashهای ثبت‌شده در reportها با فایل‌های واقعی.
5. شناسایی ساختار واقعی registry و review package ۲۸ Pilot.
6. ثبت هر ناسازگاری به‌عنوان finding؛ بدون Repair یا تغییر فایل.

### چرا این مرحله لازم است؟

قبل از اینکه بگوییم مشکل از Pilotها یا از منطق تعمیم است، باید بدانیم:

- v1 دقیقاً چه lineage و fieldهایی دارد؛
- گزارش v1 با فایل فعلی سازگار است یا نه؛
- ۲۸ Pilot با چه شناسه و ساختاری در registry ثبت شده‌اند؛
- و هستهٔ ۴۶تایی دقیقاً به کدام نسخه و Hash متصل است.

### Validation

این مرحله زمانی PASS است که همهٔ artifactهای لازم خوانده شوند و inventory واقعی آن‌ها چاپ شود.

وجود finding در Hash یا Schema باعث FAIL این Step نمی‌شود؛ finding فقط برای تحلیل مرحله‌های بعد ثبت می‌شود.

</div>

In [3]:
# Code Cell 03
# NB12h — Load immutable artifacts and inspect contracts

assert "preflight_inventory" in globals(), (
    "Run Code Cell 02 first."
)

def read_json(path):
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)

def read_jsonl(path):
    rows = []

    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(
            handle,
            1,
        ):
            stripped = line.strip()

            if not stripped:
                continue

            try:
                row = json.loads(stripped)
            except json.JSONDecodeError as error:
                raise AssertionError(
                    "Invalid JSONL row in "
                    f"{path.name} at line {line_number}: "
                    f"{error}"
                ) from error

            assert isinstance(row, dict), (
                "JSONL row is not an object in "
                f"{path.name} at line {line_number}."
            )

            rows.append(row)

    return rows

v1_path = artifact_paths["golden_dataset_v1"]
v1_report_path = artifact_paths[
    "golden_dataset_v1_report"
]
final_46_path = artifact_paths[
    "final_46_experiment"
]
final_46_integrity_path = artifact_paths[
    "final_46_integrity_audit"
]
final_46_semantic_path = artifact_paths[
    "final_46_semantic_judge"
]
final_46_acceptance_path = artifact_paths[
    "final_46_acceptance"
]

assert len(author_review_matches) == 1, (
    "Expected exactly one author_review_28_pilots.md; "
    f"found {len(author_review_matches)}."
)

current_registry_matches = [
    path
    for path in human_review_registry_matches
    if path.name == "human_review_registry.json"
]

assert len(current_registry_matches) == 1, (
    "Expected exactly one current human_review_registry.json; "
    f"found {len(current_registry_matches)}."
)

author_review_path = author_review_matches[0]
human_review_registry_path = current_registry_matches[0]

golden_v1_records = read_jsonl(v1_path)
golden_v1_report = read_json(v1_report_path)

final_46_records = read_jsonl(final_46_path)
final_46_integrity_report = read_json(
    final_46_integrity_path
)
final_46_semantic_report = read_json(
    final_46_semantic_path
)
final_46_acceptance_record = read_json(
    final_46_acceptance_path
)

human_review_registry = read_json(
    human_review_registry_path
)

author_review_text = author_review_path.read_text(
    encoding="utf-8"
)

assert len(golden_v1_records) == 573, (
    "Unexpected Golden Dataset v1 record count: "
    f"{len(golden_v1_records)}"
)

assert len(final_46_records) == 46, (
    "Unexpected accepted-core record count: "
    f"{len(final_46_records)}"
)

v1_actual_sha256 = sha256_file(v1_path)
final_46_actual_sha256 = sha256_file(final_46_path)

v1_report_sha256 = (
    golden_v1_report.get("artifact", {})
    .get("sha256")
)

final_46_integrity_sha256 = (
    final_46_integrity_report.get(
        "experiment_sha256"
    )
)

final_46_semantic_sha256 = (
    final_46_semantic_report.get(
        "experiment_sha256"
    )
)

final_46_acceptance_sha256 = (
    final_46_acceptance_record.get(
        "accepted_experiment_sha256"
    )
)

v1_field_coverage = {
    field: sum(
        field in record
        for record in golden_v1_records
    )
    for field in sorted({
        field
        for record in golden_v1_records
        for field in record
    })
}

final_46_field_coverage = {
    field: sum(
        field in record
        for record in final_46_records
    )
    for field in sorted({
        field
        for record in final_46_records
        for field in record
    })
}

v1_seed_counts = Counter(
    record.get("source_seed_id")
    for record in golden_v1_records
)

final_46_seed_counts = Counter(
    record.get("source_seed_id")
    for record in final_46_records
)

author_review_heading_lines = [
    line.strip()
    for line in author_review_text.splitlines()
    if (
        "Pilot" in line
        or "پایلوت" in line
        or "نظر مؤلف" in line
    )
]

artifact_contract_inventory = {
    "golden_v1": {
        "path": str(v1_path),
        "record_count": len(golden_v1_records),
        "actual_sha256": v1_actual_sha256,
        "report_sha256": v1_report_sha256,
        "hash_matches_report": (
            v1_actual_sha256 == v1_report_sha256
        ),
        "field_coverage": v1_field_coverage,
        "seed_count": len(v1_seed_counts),
    },
    "final_46_core": {
        "path": str(final_46_path),
        "record_count": len(final_46_records),
        "actual_sha256": final_46_actual_sha256,
        "integrity_sha256": final_46_integrity_sha256,
        "semantic_sha256": final_46_semantic_sha256,
        "acceptance_sha256": final_46_acceptance_sha256,
        "hash_matches_integrity": (
            final_46_actual_sha256
            == final_46_integrity_sha256
        ),
        "hash_matches_semantic": (
            final_46_actual_sha256
            == final_46_semantic_sha256
        ),
        "hash_matches_acceptance": (
            final_46_actual_sha256
            == final_46_acceptance_sha256
        ),
        "field_coverage": final_46_field_coverage,
        "seed_count": len(final_46_seed_counts),
    },
    "author_review_28": {
        "path": str(author_review_path),
        "bytes": author_review_path.stat().st_size,
        "author_marker_count": author_review_text.count(
            "نظر مؤلف"
        ),
        "pilot_heading_line_count": len(
            author_review_heading_lines
        ),
    },
    "human_review_registry": {
        "path": str(human_review_registry_path),
        "top_level_type": type(
            human_review_registry
        ).__name__,
        "top_level_keys": (
            sorted(human_review_registry)
            if isinstance(human_review_registry, dict)
            else None
        ),
    },
    "read_only": True,
}

print("=== NB12h ARTIFACT CONTRACT INVENTORY ===")

print("\nGolden Dataset v1:")
print("  Records:", len(golden_v1_records))
print("  Distinct seeds:", len(v1_seed_counts))
print(
    "  Actual SHA-256:",
    v1_actual_sha256,
)
print(
    "  Report SHA-256:",
    v1_report_sha256,
)
print(
    "  Hash matches report:",
    artifact_contract_inventory[
        "golden_v1"
    ]["hash_matches_report"],
)
print(
    "  Fields:",
    sorted(v1_field_coverage),
)

print("\nAccepted 46-record core:")
print("  Records:", len(final_46_records))
print("  Distinct seeds:", len(final_46_seed_counts))
print(
    "  Actual SHA-256:",
    final_46_actual_sha256,
)
print(
    "  Hash matches integrity report:",
    artifact_contract_inventory[
        "final_46_core"
    ]["hash_matches_integrity"],
)
print(
    "  Hash matches semantic report:",
    artifact_contract_inventory[
        "final_46_core"
    ]["hash_matches_semantic"],
)
print(
    "  Hash matches acceptance record:",
    artifact_contract_inventory[
        "final_46_core"
    ]["hash_matches_acceptance"],
)
print(
    "  Fields:",
    sorted(final_46_field_coverage),
)

print("\nAuthor Review package:")
print("  Path:", author_review_path)
print(
    "  'نظر مؤلف' markers:",
    artifact_contract_inventory[
        "author_review_28"
    ]["author_marker_count"],
)
print(
    "  Pilot-related heading lines:",
    artifact_contract_inventory[
        "author_review_28"
    ]["pilot_heading_line_count"],
)

for line in author_review_heading_lines[:12]:
    print("   ", line)

print("\nCurrent human-review registry:")
print("  Path:", human_review_registry_path)
print(
    "  Type:",
    artifact_contract_inventory[
        "human_review_registry"
    ]["top_level_type"],
)
print(
    "  Top-level keys:",
    artifact_contract_inventory[
        "human_review_registry"
    ]["top_level_keys"],
)

print("\nRead-only inspection:", True)
print("\nNB12h ARTIFACT CONTRACT INVENTORY: PASS")

=== NB12h ARTIFACT CONTRACT INVENTORY ===

Golden Dataset v1:
  Records: 573
  Distinct seeds: 28
  Actual SHA-256: df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729
  Report SHA-256: 25ed3732db5f03d3b771daa7b1051d61f8de7524644c4a055792b39a22493cc4
  Hash matches report: False
  Fields: ['candidate_id', 'competency_ids', 'expected_answer', 'generation_competency', 'generation_status', 'question', 'source_evidence', 'source_lineage', 'source_seed_id', 'taxonomy_topics']

Accepted 46-record core:
  Records: 46
  Distinct seeds: 28
  Actual SHA-256: 2b461fc7dfa25e9b65b2b37c709dfc8c6cd65c7b84ef5ec7fae9e3449a6e37ae
  Hash matches integrity report: True
  Hash matches semantic report: True
  Hash matches acceptance record: True
  Fields: ['candidate_id', 'competency_ids', 'curation', 'dataset_version', 'document_ids', 'evaluation_split', 'evidence_quote', 'expected_answer', 'generation', 'generation_status', 'intent', 'legacy_seed_evidence', 'manual_decision', 'manual_notes', 

<div dir="rtl" align="right">

## مرحلهٔ سوم — تثبیت Baseline واقعی v1 و بررسی اختلاف Freeze

در مرحلهٔ قبل مشخص شد که Golden Dataset v1 فعلی ۵۷۳ رکورد و ۲۸ Seed دارد، اما SHA-256 فایل فعلی با SHA ثبت‌شده در گزارش نهایی v1 برابر نیست.

### معنای این finding

این اختلاف به‌تنهایی نمی‌گوید که Golden Dataset v1 خراب یا نامعتبر است. فقط نشان می‌دهد که گزارش Freeze موجود، برای اثبات هویت فایل فعلی کافی نیست.

بنابراین در این Notebook دو چیز را جدا نگه می‌داریم:

1. **v1 فعلی:** فایل واقعی ۵۷۳ رکوردی که قرار است Audit شود.
2. **گزارش v1:** یک artifact تاریخی که باید نسبت آن با فایل فعلی بررسی و ثبت شود.

### کار این مرحله

- مقایسهٔ count، target size، coverage و hash میان فایل فعلی و گزارش v1
- بررسی distribution واقعی descendantها بین ۲۸ Seed
- بررسی uniqueness پایهٔ candidate ID و question
- ثبت وضعیت provenance بدون تغییر فایل یا گزارش

### چرا این مرحله لازم است؟

Expansion Audit باید روی یک input مشخص اجرا شود. اگر فایل واقعی و گزارش تاریخی با هم تطابق نداشته باشند، نباید از گزارش قدیمی برای قضاوت درباره کیفیت فایل فعلی استفاده کنیم.

### Validation

این Step زمانی PASS است که:

- ویژگی‌های واقعی v1 استخراج شوند؛
- اختلاف‌های گزارش و فایل به‌صورت صریح ثبت شوند؛
- distribution واقعی ۵۷۳ رکورد بین Seedها مشخص شود؛
- و baseline تحلیل برای مراحل بعدی تثبیت شود.

این Step فقط‌خواندنی است و هیچ Repair یا overwrite انجام نمی‌دهد.

</div>

In [4]:
# Code Cell 04
# NB12h — v1 baseline and freeze-divergence audit

assert "golden_v1_records" in globals(), (
    "Run Code Cell 03 first."
)
assert "golden_v1_report" in globals()
assert len(golden_v1_records) == 573

def canon_v1(value):
    text = str(value or "").strip().lower()
    text = text.replace("ي", "ی").replace("ك", "ک")
    text = re.sub(r"\s+", " ", text)
    return text

v1_report_artifact = golden_v1_report.get(
    "artifact",
    {},
)

v1_report_coverage = golden_v1_report.get(
    "coverage",
    {},
)

v1_current_candidate_ids = [
    record.get("candidate_id")
    for record in golden_v1_records
]

v1_current_question_keys = [
    canon_v1(record.get("question"))
    for record in golden_v1_records
]

v1_current_seed_ids = [
    record.get("source_seed_id")
    for record in golden_v1_records
]

v1_current_seed_counts = Counter(
    v1_current_seed_ids
)

v1_missing_candidate_ids = [
    index
    for index, candidate_id in enumerate(
        v1_current_candidate_ids,
        1,
    )
    if not candidate_id
]

v1_empty_questions = [
    index
    for index, question_key in enumerate(
        v1_current_question_keys,
        1,
    )
    if not question_key
]

v1_duplicate_candidate_id_count = (
    len(v1_current_candidate_ids)
    - len(set(v1_current_candidate_ids))
)

v1_duplicate_question_count = (
    len(v1_current_question_keys)
    - len(set(v1_current_question_keys))
)

v1_current_seed_id_set = set(
    v1_current_seed_ids
)

v1_expected_seed_id_set = set(
    range(1, 29)
)

v1_reported_records = golden_v1_report.get(
    "records"
)

v1_reported_target_size = golden_v1_report.get(
    "target_size"
)

v1_reported_seed_count = v1_report_coverage.get(
    "seeds"
)

v1_freeze_divergence = {
    "current_file": {
        "path": str(v1_path),
        "sha256": v1_actual_sha256,
        "record_count": len(golden_v1_records),
        "seed_count": len(v1_current_seed_counts),
        "seed_ids": sorted(v1_current_seed_id_set),
    },
    "historical_report": {
        "path": str(v1_report_path),
        "dataset_version": golden_v1_report.get(
            "dataset_version"
        ),
        "freeze_status": golden_v1_report.get(
            "freeze_status"
        ),
        "created_at": golden_v1_report.get(
            "created_at"
        ),
        "reported_sha256": v1_report_artifact.get(
            "sha256"
        ),
        "reported_records": v1_reported_records,
        "reported_target_size": v1_reported_target_size,
        "reported_seed_count": v1_reported_seed_count,
    },
    "comparisons": {
        "sha256_matches": (
            v1_actual_sha256
            == v1_report_artifact.get("sha256")
        ),
        "record_count_matches": (
            len(golden_v1_records)
            == v1_reported_records
        ),
        "target_size_matches_record_count": (
            v1_reported_target_size
            == len(golden_v1_records)
        ),
        "seed_count_matches": (
            len(v1_current_seed_counts)
            == v1_reported_seed_count
        ),
        "expected_seed_ids_present": (
            v1_current_seed_id_set
            == v1_expected_seed_id_set
        ),
    },
    "current_file_quality_baseline": {
        "missing_candidate_id_count": len(
            v1_missing_candidate_ids
        ),
        "empty_question_count": len(
            v1_empty_questions
        ),
        "duplicate_candidate_id_count": (
            v1_duplicate_candidate_id_count
        ),
        "duplicate_normalized_question_count": (
            v1_duplicate_question_count
        ),
        "minimum_descendants_per_seed": min(
            v1_current_seed_counts.values()
        ),
        "maximum_descendants_per_seed": max(
            v1_current_seed_counts.values()
        ),
        "descendant_count_distribution": dict(
            sorted(
                Counter(
                    v1_current_seed_counts.values()
                ).items()
            )
        ),
    },
    "read_only": True,
}

assert (
    v1_freeze_divergence["comparisons"][
        "record_count_matches"
    ]
), "v1 record count conflicts with its report."

assert (
    v1_freeze_divergence["comparisons"][
        "target_size_matches_record_count"
    ]
), "v1 target size conflicts with actual record count."

assert (
    v1_freeze_divergence["comparisons"][
        "seed_count_matches"
    ]
), "v1 seed count conflicts with its report."

assert (
    v1_freeze_divergence["comparisons"][
        "expected_seed_ids_present"
    ]
), "v1 does not cover exactly Seed IDs 1 through 28."

print("=== NB12h V1 BASELINE AND FREEZE DIVERGENCE ===")

print("\nHistorical v1 report:")
print(
    "  Dataset version:",
    v1_freeze_divergence[
        "historical_report"
    ]["dataset_version"],
)
print(
    "  Freeze status:",
    v1_freeze_divergence[
        "historical_report"
    ]["freeze_status"],
)
print(
    "  Created at:",
    v1_freeze_divergence[
        "historical_report"
    ]["created_at"],
)
print(
    "  Report SHA-256:",
    v1_freeze_divergence[
        "historical_report"
    ]["reported_sha256"],
)

print("\nCurrent v1 file:")
print(
    "  Actual SHA-256:",
    v1_freeze_divergence[
        "current_file"
    ]["sha256"],
)
print(
    "  Records:",
    v1_freeze_divergence[
        "current_file"
    ]["record_count"],
)
print(
    "  Distinct seeds:",
    v1_freeze_divergence[
        "current_file"
    ]["seed_count"],
)

print("\nReport/file comparisons:")
for name, value in v1_freeze_divergence[
    "comparisons"
].items():
    print(f"  {name}: {value}")

print("\nCurrent v1 structural baseline:")
for name, value in v1_freeze_divergence[
    "current_file_quality_baseline"
].items():
    print(f"  {name}: {value}")

print("\nDescendants by Seed:")

for seed_id in sorted(v1_current_seed_counts):
    print(
        f"  Seed {seed_id}: "
        f"{v1_current_seed_counts[seed_id]}"
    )

print(
    "\nInterpretation: SHA-256 mismatch is a provenance "
    "finding. The current v1 file remains the read-only "
    "input for NB12h analysis."
)

print("\nNB12h V1 BASELINE: PASS WITH PROVENANCE FINDING")

=== NB12h V1 BASELINE AND FREEZE DIVERGENCE ===

Historical v1 report:
  Dataset version: golden_dataset_v1
  Freeze status: PASS
  Created at: 2026-09-10T14:12:45.079766
  Report SHA-256: 25ed3732db5f03d3b771daa7b1051d61f8de7524644c4a055792b39a22493cc4

Current v1 file:
  Actual SHA-256: df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729
  Records: 573
  Distinct seeds: 28

Report/file comparisons:
  sha256_matches: False
  record_count_matches: True
  target_size_matches_record_count: True
  seed_count_matches: True
  expected_seed_ids_present: True

Current v1 structural baseline:
  missing_candidate_id_count: 0
  empty_question_count: 0
  duplicate_candidate_id_count: 0
  duplicate_normalized_question_count: 0
  minimum_descendants_per_seed: 20
  maximum_descendants_per_seed: 21
  descendant_count_distribution: {20: 15, 21: 13}

Descendants by Seed:
  Seed 1: 21
  Seed 2: 21
  Seed 3: 21
  Seed 4: 21
  Seed 5: 21
  Seed 6: 21
  Seed 7: 21
  Seed 8: 21
  Seed 9: 21
  S

<div dir="rtl" align="right">

## مرحلهٔ چهارم — کشف ساختار واقعی ۲۸ Pilot

اکنون ثابت شد که v1 به‌صورت تقریباً مساوی، ۲۰ یا ۲۱ descendant برای هر Seed تولید کرده است. گام بعدی این است که ۲۸ Pilot ریشه را دقیق و بدون حدس از registry و package بازبینی مؤلف استخراج کنیم.

### کار این مرحله

- بررسی ساختار واقعی `human_review_registry.json`
- شناسایی محل ثبت ۲۲ Pilot اولیه و ۶ Pilot مشتق‌شده از دمو
- بررسی headingها و boundaryهای واقعی فایل `author_review_28_pilots.md`
- تعیین روش امن برای استخراج هر ۲۸ Case
- عدم اصلاح یا بازنویسی Pilotها در این مرحله

### چرا این مرحله لازم است؟

برای ساخت نگاشت معتبر:

```text
Pilot 1 تا 28
→ Seed 1 تا 28
→ Descendantهای v1

In [5]:
# Code Cell 05
# NB12h — Pilot registry and author-review structure discovery

assert "human_review_registry" in globals(), (
    "Run Code Cell 03 first."
)
assert "author_review_text" in globals()
assert "human_review_registry_path" in globals()
assert "author_review_path" in globals()

def describe_registry_value(value):
    if isinstance(value, dict):
        return {
            "type": "dict",
            "key_count": len(value),
            "keys": sorted(value)[:30],
        }

    if isinstance(value, list):
        return {
            "type": "list",
            "length": len(value),
            "item_types": sorted({
                type(item).__name__
                for item in value
            }),
        }

    if isinstance(value, str):
        return {
            "type": "str",
            "length": len(value),
            "preview": value[:180],
        }

    return {
        "type": type(value).__name__,
        "value": value,
    }

assert isinstance(human_review_registry, dict), (
    "Current human-review registry must be a JSON object."
)

registry_structure = {
    key: describe_registry_value(value)
    for key, value in human_review_registry.items()
}

markdown_heading_lines = [
    line.strip()
    for line in author_review_text.splitlines()
    if re.match(r"^\s{0,3}#{1,6}\s+", line)
]

html_headings_raw = re.findall(
    r"<h[1-6][^>]*>(.*?)</h[1-6]>",
    author_review_text,
    flags=re.IGNORECASE | re.DOTALL,
)

html_heading_lines = [
    re.sub(r"<[^>]+>", "", heading)
    .replace("&nbsp;", " ")
    .strip()
    for heading in html_headings_raw
]

case_number_markers = sorted({
    int(number)
    for number in re.findall(
        r"(?im)(?:case|pilot|مورد|پایلوت)"
        r"\s*(?:شماره)?\s*[_:\-]?\s*(\d{1,2})",
        author_review_text,
    )
    if 1 <= int(number) <= 28
})

author_section_markers = re.findall(
    r"۵\.\s*نظر مؤلف"
    r"|نظر مؤلف\s*\(دکتر کمیل رودی\)",
    author_review_text,
    flags=re.IGNORECASE,
)

pilot_structure_discovery = {
    "registry_path": str(human_review_registry_path),
    "registry_top_level_structure": registry_structure,
    "author_review_path": str(author_review_path),
    "author_review_bytes": author_review_path.stat().st_size,
    "markdown_heading_count": len(
        markdown_heading_lines
    ),
    "html_heading_count": len(html_heading_lines),
    "case_number_markers": case_number_markers,
    "author_section_marker_count": len(
        author_section_markers
    ),
    "read_only": True,
}

print("=== NB12h PILOT STRUCTURE DISCOVERY ===")

print("\nRegistry structure:")

for key in sorted(registry_structure):
    print(f"\n  {key}:")
    for detail_name, detail_value in (
        registry_structure[key].items()
    ):
        print(f"    {detail_name}: {detail_value}")

print("\nMarkdown headings:")
print("  Count:", len(markdown_heading_lines))

for heading in markdown_heading_lines[:20]:
    print("  -", heading)

print("\nHTML headings:")
print("  Count:", len(html_heading_lines))

for heading in html_heading_lines[:40]:
    print("  -", heading)

print("\nDetected case/pilot numbers:")
print(" ", case_number_markers)

print("\nAuthor-section markers:")
print("  Count:", len(author_section_markers))

print("\nRead-only inspection:", True)
print("\nNB12h PILOT STRUCTURE DISCOVERY: PASS")

=== NB12h PILOT STRUCTURE DISCOVERY ===

Registry structure:

  active_workspace:
    type: NoneType
    value: None

  author_review:
    type: dict
    key_count: 7
    keys: ['case_count', 'cases_ready_for_review', 'final_review_status', 'notes', 'promotion_history', 'reviewer', 'status']

  completed_human_review_cases:
    type: list
    length: 22
    item_types: ['dict']

  continuation_state:
    type: dict
    key_count: 17
    keys: ['active_citation_mapping_id', 'active_pilot_status', 'active_target_evidence_id', 'author_final_review_status', 'authority_policy_active', 'cases_ready_for_author_review', 'completed_human_review_case_count', 'continuation_state_sha256', 'full_572_generation_status', 'last_completed_citation_mapping_id', 'last_completed_pilot_id', 'last_completed_target_evidence_id', 'last_completion_type', 'next_pilot_id', 'pilot_human_review_status', 'remaining_count', 'remaining_pilot_ids']

  historical_completed_pilot:
    type: dict
    key_count: 2
    key

<div dir="rtl" align="right">

## مرحلهٔ پنجم — ساخت Workspace فقط‌خواندنی برای ۲۸ Pilot

ساختار registry و فایل بازبینی مؤلف تأیید کرد که ۲۸ Pilot از دو مسیر تشکیل شده‌اند:

- ۲۲ Pilot اولیهٔ بازبینی انسانی
- ۶ Pilot مشتق‌شده از بازخورد دکتر در دمو

در این مرحله فایل `author_review_28_pilots.md` به ۲۸ Case مستقل تقسیم می‌شود؛ بدون اینکه هنوز سؤال، جواب یا evidence را بازنویسی یا تفسیر کنیم.

### کار این مرحله

1. جداکردن دقیق ۲۸ بلوک Case بر اساس headingهای واقعی فایل.
2. بررسی پیوستگی شماره‌های ۱ تا ۲۸.
3. ثبت headingهای داخلی هر Case.
4. مشاهدهٔ وضعیت metadata مربوط به ۲۲ Pilot اولیه و ۶ Pilot مشتق‌شده.
5. ساخت workspace در حافظه برای مرحلهٔ استخراج دقیق سؤال، جواب و evidence.

### Validation

این Step زمانی PASS است که:

- دقیقاً ۲۸ بلوک Case جدا شود؛
- شماره‌ها از ۱ تا ۲۸ بدون تکرار یا فاصله باشند؛
- هر بلوک دارای بخش‌های اصلی بازبینی باشد؛
- و registry وجود ۲۲ + ۶ Pilot را تأیید کند.

این Step فقط‌خواندنی است و هیچ فایل جدیدی ایجاد نمی‌کند.

</div>

In [6]:
# Code Cell 06
# NB12h — Build read-only 28-pilot workspace

assert "author_review_text" in globals(), (
    "Run Code Cell 03 first."
)
assert "human_review_registry" in globals()
assert isinstance(human_review_registry, dict)

case_heading_pattern = re.compile(
    r"(?P<heading>"
    r"<h2[^>]*>\s*"
    r"مورد\s+"
    r"(?P<case_number>\d+)\s+"
    r"از\s+۲۸"
    r"\s*</h2>"
    r")",
    flags=re.IGNORECASE | re.DOTALL,
)

case_heading_matches = list(
    case_heading_pattern.finditer(author_review_text)
)

case_numbers = [
    int(match.group("case_number"))
    for match in case_heading_matches
]

assert len(case_heading_matches) == 28, (
    "Expected exactly 28 review-case headings; "
    f"found {len(case_heading_matches)}."
)

assert case_numbers == list(range(1, 29)), (
    "Review-case numbering is not sequential 1 through 28: "
    f"{case_numbers}"
)

pilot_case_workspace = []

for index, match in enumerate(case_heading_matches):
    case_number = int(match.group("case_number"))
    block_start = match.start()

    block_end = (
        case_heading_matches[index + 1].start()
        if index + 1 < len(case_heading_matches)
        else len(author_review_text)
    )

    case_html = author_review_text[
        block_start:block_end
    ]

    internal_headings_raw = re.findall(
        r"<h[1-6][^>]*>(.*?)</h[1-6]>",
        case_html,
        flags=re.IGNORECASE | re.DOTALL,
    )

    internal_headings = [
        re.sub(r"<[^>]+>", "", heading)
        .replace("&nbsp;", " ")
        .strip()
        for heading in internal_headings_raw
    ]

    has_source_section = any(
        "متن منبع" in heading
        for heading in internal_headings
    )

    has_review_history_section = any(
        "روند اصلاح" in heading
        or "بازبینی" in heading
        for heading in internal_headings
    )

    has_final_qa_section = any(
        "سؤال و پاسخ نهایی" in heading
        for heading in internal_headings
    )

    has_author_section = any(
        "نظر مؤلف" in heading
        for heading in internal_headings
    )

    pilot_case_workspace.append({
        "pilot_number": case_number,
        "case_html": case_html,
        "case_bytes": len(
            case_html.encode("utf-8")
        ),
        "internal_headings": internal_headings,
        "has_source_section": has_source_section,
        "has_review_history_section": (
            has_review_history_section
        ),
        "has_final_qa_section": has_final_qa_section,
        "has_author_section": has_author_section,
    })

assert len(pilot_case_workspace) == 28

case_section_failures = [
    {
        "pilot_number": case["pilot_number"],
        "has_source_section": case[
            "has_source_section"
        ],
        "has_review_history_section": case[
            "has_review_history_section"
        ],
        "has_final_qa_section": case[
            "has_final_qa_section"
        ],
        "has_author_section": case[
            "has_author_section"
        ],
    }
    for case in pilot_case_workspace
    if not all([
        case["has_source_section"],
        case["has_review_history_section"],
        case["has_final_qa_section"],
        case["has_author_section"],
    ])
]

assert not case_section_failures, (
    "One or more Pilot cases are structurally incomplete: "
    + json.dumps(
        case_section_failures,
        ensure_ascii=False,
        indent=2,
    )
)

completed_human_review_cases = (
    human_review_registry.get(
        "completed_human_review_cases",
        [],
    )
)

second_human_review = human_review_registry.get(
    "second_human_review",
    {},
)

phase1_derived_pilots = second_human_review.get(
    "phase1_author_feedback_derived_pilots"
)

source_provenance_review_pilots = (
    second_human_review.get(
        "source_provenance_review_pilots23_28_v1"
    )
)

author_review_metadata = human_review_registry.get(
    "author_review",
    {}
)

pilot_workspace_inventory = {
    "pilot_case_count": len(pilot_case_workspace),
    "pilot_numbers": [
        case["pilot_number"]
        for case in pilot_case_workspace
    ],
    "completed_human_review_case_count": len(
        completed_human_review_cases
    ),
    "phase1_derived_pilots_type": type(
        phase1_derived_pilots
    ).__name__,
    "phase1_derived_pilots_summary": (
        describe_registry_value(phase1_derived_pilots)
    ),
    "source_provenance_review_summary": (
        describe_registry_value(
            source_provenance_review_pilots
        )
    ),
    "author_review_metadata": author_review_metadata,
    "structural_failure_count": len(
        case_section_failures
    ),
    "read_only": True,
}

print("=== NB12h 28-PILOT WORKSPACE ===")
print("Pilot cases:", len(pilot_case_workspace))
print(
    "Pilot numbering:",
    pilot_workspace_inventory["pilot_numbers"],
)
print(
    "Completed initial human-review cases:",
    len(completed_human_review_cases),
)

print("\nPhase-1 derived Pilot structure:")
for name, value in (
    pilot_workspace_inventory[
        "phase1_derived_pilots_summary"
    ].items()
):
    print(f"  {name}: {value}")

print("\nSource-provenance review structure:")
for name, value in (
    pilot_workspace_inventory[
        "source_provenance_review_summary"
    ].items()
):
    print(f"  {name}: {value}")

print("\nAuthor-review metadata:")
for name, value in author_review_metadata.items():
    print(f"  {name}: {value}")

print("\nPilot case structural summary:")

for case in pilot_case_workspace:
    print(
        f"  Pilot {case['pilot_number']}: "
        f"bytes={case['case_bytes']} | "
        f"headings={len(case['internal_headings'])} | "
        f"source={case['has_source_section']} | "
        f"history={case['has_review_history_section']} | "
        f"final_qa={case['has_final_qa_section']} | "
        f"author={case['has_author_section']}"
    )

print(
    "\nStructural failures:",
    len(case_section_failures),
)
print("\nRead-only workspace:", True)
print("\nNB12h 28-PILOT WORKSPACE: PASS")

=== NB12h 28-PILOT WORKSPACE ===
Pilot cases: 28
Pilot numbering: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
Completed initial human-review cases: 22

Phase-1 derived Pilot structure:
  type: dict
  key_count: 20
  keys: ['author_final_review_status', 'author_review_queue_count_after_promotion', 'author_review_queue_count_before_promotion', 'batch_version', 'case_count', 'cases', 'content_pilot_count', 'content_pilot_count_after_source_review', 'continuity_regression_pilot_count', 'continuity_regression_pilot_count_after_source_review', 'currently_promoted_to_author_review', 'future_total_reference_pilots', 'origin', 'promotion_timestamp_utc', 'reference_pilot_range', 'source_review_case_count', 'source_review_decisions', 'source_review_status', 'source_review_version', 'status']

Source-provenance review structure:
  type: dict
  key_count: 11
  keys: ['content_source_recovery_status', 'original_phase1_demo_provenance', 'pil

<div dir="rtl" align="right">

## مرحلهٔ ششم — ساخت registry استاندارد و فقط‌خواندنی برای ۲۸ Pilot

اکنون ساختار هر ۲۸ Case تأیید شده است. در این مرحله، داده‌های پراکندهٔ registry به یک ساختار استاندارد در حافظه تبدیل می‌شوند.

### ساختار هر Pilot استاندارد

برای هر Pilot موارد زیر استخراج می‌شود:

- شماره Pilot از ۱ تا ۲۸
- `pilot_case_id`
- منشأ Pilot:
  - `INITIAL_HUMAN_REVIEW`
  - یا `PHASE1_DEMO_AUTHOR_FEEDBACK`
- سؤال نهایی فعلی
- پاسخ نهایی فعلی
- evidenceهای تأییدشده
- document IDهای evidence
- نوع Pilot و authority
- وضعیت نظر مؤلف

### قاعدهٔ مهم

- برای Pilotهای ۱ تا ۲۲، سؤال نهایی فقط از `final_question_draft_id` و draft پذیرفته‌شده استخراج می‌شود.
- برای Pilotهای ۲۳ تا ۲۸، سؤال فقط از `derived_question` ثبت‌شده استخراج می‌شود.
- هیچ fallback حدسی یا بازنویسی متن انجام نمی‌شود.
- اگر ساختار مورد انتظار ناقص باشد، Step متوقف می‌شود.

### خروجی این مرحله

یک registry در حافظه با نام `canonical_pilot_registry` ساخته می‌شود که ورودی مستقیم Expansion Audit در مرحله‌های بعدی خواهد بود.

### Validation

این Step زمانی PASS است که:

- دقیقاً ۲۸ Pilot استاندارد ساخته شود؛
- شماره‌ها از ۱ تا ۲۸ یکتا و پیوسته باشند؛
- همه دارای سؤال، جواب و evidence غیرخالی باشند؛
- و نگاشت ۲۲ Pilot اولیه و ۶ Pilot دمو بدون حدس تأیید شود.

</div>

In [7]:
# Code Cell 07
# NB12h — Canonical 28-pilot registry with strict evidence recovery
# Read-only: no active dataset, registry, or report is modified.

assert "human_review_registry" in globals(), (
    "Run Code Cell 03 first."
)
assert "golden_dir" in globals(), (
    "Run Code Cell 02 first."
)


def require_text(value, label):
    text = str(value or "").strip()
    assert text, f"Missing required text: {label}"
    return text


def first_text(mapping, keys):
    if not isinstance(mapping, dict):
        return None

    for key in keys:
        value = mapping.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip()

    return None


def normalized_text(value):
    return " ".join(str(value or "").split()).strip()


def is_accepted(value):
    return str(value or "").strip().upper() in {
        "ACCEPT",
        "ACCEPTED",
        "APPROVE",
        "APPROVED",
        "KEEP",
        "RETAIN",
        "PASS",
    }


def read_jsonl(path):
    records = []

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            1,
        ):
            text = line.strip()

            if not text:
                continue

            try:
                records.append(json.loads(text))
            except json.JSONDecodeError as error:
                raise AssertionError(
                    f"Invalid JSONL: {path} | "
                    f"line={line_number} | {error}"
                )

    return records


def extract_final_question(case):
    question = first_text(
        case.get("question_review") or {},
        [
            "question_text",
            "final_question_text",
            "accepted_question_text",
        ],
    )

    if question:
        return question, "question_review"

    question = first_text(
        case.get("author_final_review") or {},
        [
            "question_text",
            "final_question_text",
            "accepted_question_text",
        ],
    )

    if question:
        return question, "author_final_review"

    question = first_text(
        case,
        [
            "final_question_text",
            "accepted_question_text",
            "question_text",
        ],
    )

    if question:
        return question, "top_level_case_field"

    drafts = [
        draft
        for draft in case.get("model_drafts", [])
        if isinstance(draft, dict)
    ]

    final_draft_id = str(
        case.get("final_question_draft_id") or ""
    ).strip()

    if final_draft_id:
        matches = [
            draft
            for draft in drafts
            if (
                str(
                    draft.get("question_draft_id") or ""
                ).strip()
                == final_draft_id
                and str(
                    draft.get("question_text") or ""
                ).strip()
            )
        ]

        if len(matches) == 1:
            return (
                matches[0]["question_text"].strip(),
                "final_question_draft_id",
            )

    accepted_drafts = [
        draft
        for draft in drafts
        if (
            is_accepted(
                draft.get("human_decision")
            )
            and str(
                draft.get("question_text") or ""
            ).strip()
        )
    ]

    if accepted_drafts:
        return (
            accepted_drafts[-1][
                "question_text"
            ].strip(),
            "accepted_model_draft",
        )

    valid_drafts = [
        draft
        for draft in drafts
        if str(
            draft.get("question_text") or ""
        ).strip()
    ]

    if valid_drafts:
        return (
            valid_drafts[-1][
                "question_text"
            ].strip(),
            "latest_model_draft_fallback",
        )

    raise AssertionError(
        "Could not resolve final question for "
        f"pilot_case_id={case.get('pilot_case_id')!r}"
    )


def extract_final_answer(case):
    answer = first_text(
        case.get("expected_answer_review") or {},
        [
            "expected_answer_text",
            "answer_text",
            "final_answer_text",
            "accepted_answer_text",
        ],
    )

    if answer:
        return answer, "expected_answer_review"

    answer = first_text(
        case.get("completion_summary") or {},
        [
            "expected_answer_text",
            "answer_text",
            "final_answer_text",
        ],
    )

    if answer:
        return answer, "completion_summary"

    answer = first_text(
        case.get("author_final_review") or {},
        [
            "expected_answer_text",
            "answer_text",
            "final_answer_text",
        ],
    )

    if answer:
        return answer, "author_final_review"

    answer = first_text(
        case,
        [
            "expected_answer",
            "expected_answer_text",
            "answer_text",
        ],
    )

    if answer:
        return answer, "top_level_case_field"

    raise AssertionError(
        "Could not resolve final answer for "
        f"pilot_case_id={case.get('pilot_case_id')!r}"
    )


def evidence_parts_from_value(
    value,
    source_name,
):
    if isinstance(value, str):
        text = value.strip()

        return (
            [{
                "position": 1,
                "document_id": None,
                "text": text,
                "evidence_resolution_source": (
                    source_name
                ),
            }]
            if text
            else []
        )

    if isinstance(value, dict):
        value = [value]

    if not isinstance(value, list):
        return []

    parts = []

    for position, item in enumerate(value, 1):
        if isinstance(item, str):
            text = item.strip()
            document_id = None

        elif isinstance(item, dict):
            text = first_text(
                item,
                [
                    "evidence_text",
                    "text",
                    "content",
                    "quote",
                    "excerpt",
                    "source_evidence",
                ],
            )

            document_id = first_text(
                item,
                [
                    "document_id",
                    "source_document_id",
                    "doc_id",
                    "file_id",
                ],
            )

        else:
            continue

        if text:
            parts.append({
                "position": position,
                "document_id": document_id,
                "text": text,
                "evidence_resolution_source": (
                    source_name
                ),
            })

    return parts


def extract_explicit_evidence(*cases):
    """
    IDs, citation mappings, and review metadata are never
    accepted as evidence. Only actual evidence-text fields are.
    """

    permitted_fields = [
        "source_evidence",
        "source_evidence_snapshot",
        "supporting_book_evidence",
        "supporting_evidence",
        "target_evidence",
        "evidence_parts",
        "evidence",
    ]

    for case_position, case in enumerate(cases, 1):
        if not isinstance(case, dict):
            continue

        for field_name in permitted_fields:
            parts = evidence_parts_from_value(
                case.get(field_name),
                f"case_{case_position}.{field_name}",
            )

            if parts:
                return parts

    return []


def most_frequent_evidence_parts(
    records,
    source_name,
):
    evidence_values = [
        str(record.get("source_evidence") or "").strip()
        for record in records
        if str(record.get("source_evidence") or "").strip()
    ]

    assert evidence_values, (
        f"No source_evidence found for {source_name}."
    )

    evidence_counter = Counter(
        normalized_text(value)
        for value in evidence_values
    )

    chosen_normalized, chosen_count = (
        evidence_counter.most_common(1)[0]
    )

    chosen_text = next(
        value
        for value in evidence_values
        if normalized_text(value)
        == chosen_normalized
    )

    return [{
        "position": 1,
        "document_id": None,
        "text": chosen_text,
        "evidence_resolution_source": source_name,
        "supporting_record_count": len(records),
        "selected_evidence_frequency": chosen_count,
        "distinct_evidence_count": len(
            evidence_counter
        ),
    }]


author_review = (
    human_review_registry.get("author_review") or {}
)

author_ready_cases = (
    author_review.get("cases_ready_for_review") or []
)

completed_initial_cases = (
    human_review_registry.get(
        "completed_human_review_cases"
    ) or []
)

derived_author_cases = (
    (
        human_review_registry.get(
            "second_human_review"
        ) or {}
    ).get(
        "phase1_author_feedback_derived_pilots",
        {},
    ).get(
        "cases",
        [],
    )
)

assert len(author_ready_cases) == 28
assert len(completed_initial_cases) == 22
assert len(derived_author_cases) == 6

author_case_by_pilot_id = {
    require_text(
        case.get("pilot_case_id"),
        "author-ready pilot_case_id",
    ): case
    for case in author_ready_cases
}

initial_pilot_ids = [
    require_text(
        case.get("pilot_case_id"),
        "initial pilot_case_id",
    )
    for case in completed_initial_cases
]

assert len(initial_pilot_ids) == len(
    set(initial_pilot_ids)
)

assert set(initial_pilot_ids).issubset(
    author_case_by_pilot_id
)

assert sorted(
    int(case["reference_pilot_number"])
    for case in derived_author_cases
) == list(range(23, 29))

v1_path = (
    golden_dir / "golden_dataset_v1.jsonl"
)

final_46_path = (
    golden_dir
    / (
        "golden_dataset_quote_grounded_"
        "final_adjudicated_experiment.jsonl"
    )
)

assert v1_path.is_file(), (
    f"Missing v1 dataset: {v1_path}"
)

assert final_46_path.is_file(), (
    f"Missing accepted 46-record core: {final_46_path}"
)

v1_records = read_jsonl(v1_path)

final_46_records = read_jsonl(final_46_path)

v1_records_by_seed = defaultdict(list)
final_46_records_by_seed = defaultdict(list)
final_46_records_by_pilot_id = defaultdict(list)

for record in v1_records:
    seed_id = record.get("source_seed_id")

    if isinstance(seed_id, int):
        v1_records_by_seed[seed_id].append(record)

for record in final_46_records:
    seed_id = record.get("source_seed_id")

    if isinstance(seed_id, int):
        final_46_records_by_seed[seed_id].append(
            record
        )

    pilot_case_id = str(
        record.get("pilot_case_id") or ""
    ).strip()

    if pilot_case_id:
        final_46_records_by_pilot_id[
            pilot_case_id
        ].append(record)


def resolve_evidence(
    pilot_number,
    pilot_case_id,
    *case_representations,
):
    direct_parts = extract_explicit_evidence(
        *case_representations
    )

    if direct_parts:
        return direct_parts

    core_matches = (
        final_46_records_by_pilot_id.get(
            pilot_case_id,
            [],
        )
    )

    if not core_matches:
        core_matches = final_46_records_by_seed.get(
            pilot_number,
            [],
        )

    if core_matches:
        return most_frequent_evidence_parts(
            core_matches,
            "accepted_46_core_fallback",
        )

    v1_matches = v1_records_by_seed.get(
        pilot_number,
        [],
    )

    assert v1_matches, (
        "No v1 records exist for evidence recovery: "
        f"source_seed_id={pilot_number}"
    )

    return most_frequent_evidence_parts(
        v1_matches,
        "v1_seed_evidence_fallback",
    )


canonical_pilot_registry = []
question_resolution_sources = Counter()
answer_resolution_sources = Counter()
evidence_resolution_sources = Counter()

for pilot_number, pilot_case_id in enumerate(
    initial_pilot_ids,
    1,
):
    case = author_case_by_pilot_id[pilot_case_id]

    question, question_source = (
        extract_final_question(case)
    )

    answer, answer_source = extract_final_answer(
        case
    )

    evidence_parts = resolve_evidence(
        pilot_number,
        pilot_case_id,
        case,
    )

    canonical_pilot_registry.append({
        "pilot_number": pilot_number,
        "pilot_case_id": pilot_case_id,
        "pilot_origin": "INITIAL_HUMAN_REVIEW",
        "pilot_type": "HUMAN_REVIEWED_CONTENT",
        "authority_class": "USER_REVIEWED_PILOT",
        "question": question,
        "expected_answer": answer,
        "evidence_parts": evidence_parts,
        "document_ids": sorted({
            part["document_id"]
            for part in evidence_parts
            if part.get("document_id")
        }),
        "author_review_status": (
            case.get("author_final_review") or {}
        ).get("status"),
        "author_review_decision": (
            case.get("author_final_review") or {}
        ).get("decision"),
        "registry_final_status": case.get(
            "final_status"
        ),
        "question_resolution_source": question_source,
        "answer_resolution_source": answer_source,
        "evidence_resolution_source": (
            evidence_parts[0][
                "evidence_resolution_source"
            ]
        ),
    })

for derived_case in sorted(
    derived_author_cases,
    key=lambda item: int(
        item["reference_pilot_number"]
    ),
):
    pilot_number = int(
        derived_case["reference_pilot_number"]
    )

    pilot_case_id = require_text(
        derived_case.get("pilot_case_id"),
        "derived pilot_case_id",
    )

    author_case = author_case_by_pilot_id.get(
        pilot_case_id,
        derived_case,
    )

    question = first_text(
        derived_case.get("derived_question") or {},
        [
            "question_text",
            "final_question_text",
            "accepted_question_text",
        ],
    )

    if question:
        question_source = "derived_question"
    else:
        question, question_source = (
            extract_final_question(author_case)
        )

    answer, answer_source = extract_final_answer(
        author_case
    )

    evidence_parts = resolve_evidence(
        pilot_number,
        pilot_case_id,
        author_case,
        derived_case,
    )

    canonical_pilot_registry.append({
        "pilot_number": pilot_number,
        "pilot_case_id": pilot_case_id,
        "pilot_origin": "PHASE1_AUTHOR_FEEDBACK",
        "pilot_type": (
            "DIRECT_AUTHOR_FEEDBACK_DERIVED"
        ),
        "authority_class": (
            "PRIMARY_DIRECT_AUTHOR_FEEDBACK"
        ),
        "question": question,
        "expected_answer": answer,
        "evidence_parts": evidence_parts,
        "document_ids": sorted({
            part["document_id"]
            for part in evidence_parts
            if part.get("document_id")
        }),
        "author_review_status": (
            author_case.get("author_final_review") or {}
        ).get("status"),
        "author_review_decision": (
            author_case.get("author_final_review") or {}
        ).get("decision"),
        "registry_final_status": author_case.get(
            "final_status"
        ),
        "question_resolution_source": question_source,
        "answer_resolution_source": answer_source,
        "evidence_resolution_source": (
            evidence_parts[0][
                "evidence_resolution_source"
            ]
        ),
    })

canonical_pilot_registry.sort(
    key=lambda row: row["pilot_number"]
)

assert len(canonical_pilot_registry) == 28

assert [
    row["pilot_number"]
    for row in canonical_pilot_registry
] == list(range(1, 29))

assert all(
    row["question"].strip()
    and row["expected_answer"].strip()
    and row["evidence_parts"]
    for row in canonical_pilot_registry
)

for row in canonical_pilot_registry:
    question_resolution_sources[
        row["question_resolution_source"]
    ] += 1

    answer_resolution_sources[
        row["answer_resolution_source"]
    ] += 1

    evidence_resolution_sources[
        row["evidence_resolution_source"]
    ] += 1

fallback_rows = [
    row
    for row in canonical_pilot_registry
    if row["evidence_resolution_source"]
    in {
        "accepted_46_core_fallback",
        "v1_seed_evidence_fallback",
    }
]

print("=== NB12h CANONICAL 28-PILOT REGISTRY ===")
print(
    "Canonical pilot count:",
    len(canonical_pilot_registry),
)
print(
    "Initial human-review pilots:",
    sum(
        row["pilot_origin"]
        == "INITIAL_HUMAN_REVIEW"
        for row in canonical_pilot_registry
    ),
)
print(
    "Direct author-feedback pilots:",
    sum(
        row["pilot_origin"]
        == "PHASE1_AUTHOR_FEEDBACK"
        for row in canonical_pilot_registry
    ),
)
print(
    "Documented evidence fallback count:",
    len(fallback_rows),
)

print("\nQuestion-resolution sources:")
for label, count in sorted(
    question_resolution_sources.items()
):
    print(f"  - {label}: {count}")

print("\nAnswer-resolution sources:")
for label, count in sorted(
    answer_resolution_sources.items()
):
    print(f"  - {label}: {count}")

print("\nEvidence-resolution sources:")
for label, count in sorted(
    evidence_resolution_sources.items()
):
    print(f"  - {label}: {count}")

if fallback_rows:
    print("\nFallback pilots:")
    for row in fallback_rows:
        print(
            f"  - Pilot {row['pilot_number']}: "
            f"{row['evidence_resolution_source']}"
        )

print(
    "\nUnified author re-review status:",
    author_review.get("status"),
)
print(
    "Unified author final-review status:",
    author_review.get("final_review_status"),
)
print(
    "\nRead-only registry construction:",
    True,
)
print(
    "\nNB12h CANONICAL 28-PILOT REGISTRY: PASS"
)

=== NB12h CANONICAL 28-PILOT REGISTRY ===
Canonical pilot count: 28
Initial human-review pilots: 22
Direct author-feedback pilots: 6
Documented evidence fallback count: 6

Question-resolution sources:
  - derived_question: 6
  - final_question_draft_id: 4
  - question_review: 18

Answer-resolution sources:
  - expected_answer_review: 28

Evidence-resolution sources:
  - accepted_46_core_fallback: 6
  - case_1.source_evidence: 22

Fallback pilots:
  - Pilot 23: accepted_46_core_fallback
  - Pilot 24: accepted_46_core_fallback
  - Pilot 25: accepted_46_core_fallback
  - Pilot 26: accepted_46_core_fallback
  - Pilot 27: accepted_46_core_fallback
  - Pilot 28: accepted_46_core_fallback

Unified author re-review status: NOT_STARTED
Unified author final-review status: NOT_STARTED

Read-only registry construction: True

NB12h CANONICAL 28-PILOT REGISTRY: PASS


<div dir="rtl" align="right">

## **مرحلهٔ هفتم — نگاشت فقط‌خواندنی ۲۸ Pilot به فرزندان Golden Dataset v1**

اکنون registry استاندارد `canonical_pilot_registry` با ۲۸ Pilot ساخته و Validate شده است. در این مرحله، هر Pilot به Seed متناظر و همهٔ فرزندان آن در Golden Dataset v1 نگاشت می‌شود.

### **ورودی‌های این مرحله**

- `canonical_pilot_registry`

- `golden_dataset_v1.jsonl`

- Core-46 پذیرفته‌شده، فقط برای Validation نگاشت `pilot_case_id` به `source_seed_id`

### **ساختار نگاشت**

برای هر Pilot، ساختار زیر بررسی می‌شود:

`Pilot → source_seed_id → v1 descendants`

### **بررسی‌های این مرحله**

برای هر Pilot موارد زیر محاسبه می‌شود:

- `source_seed_id` متناظر

- تعداد فرزندان در v1

- تعداد سؤال‌های یکتای نرمال‌شده

- تعداد پاسخ‌های یکتای نرمال‌شده

- تعداد شاهدهای یکتای نرمال‌شده

- تعداد competencyهای پوشش‌داده‌شده

### **قاعدهٔ مهم**

- این Step فقط نگاشت و تحلیل می‌سازد.

- هیچ رکوردی در `golden_dataset_v1.jsonl` تغییر نمی‌کند.

- هیچ Golden Dataset جدیدی ساخته نمی‌شود.

- هیچ تصمیمی برای حذف، نگهداری یا ارتقای v1 گرفته نمی‌شود.

### **خروجی این مرحله**

ساختار درحافظه‌ای `pilot_to_v1_seed_rows` ساخته می‌شود که baseline واقعی Expansion v1 برای همهٔ ۲۸ Pilot است.

### **Validation**

این Step زمانی PASS است که:

- دقیقاً ۲۸ Pilot به ۲۸ Seed نگاشت شوند؛

- Seedهای ۱ تا ۲۸ دقیقاً یک‌بار پوشش داده شوند؛

- مجموع فرزندان دقیقاً برابر با ۵۷۳ رکورد باشد؛

- و هیچ فایل یا Datasetی تغییر نکند.

</div>

In [8]:
# Code Cell 08
# NB12h — Pilot-to-v1 descendant mapping and expansion baseline
# Read-only: no dataset, registry, or report file is modified.

assert "canonical_pilot_registry" in globals(), (
    "Run Code Cell 07 first."
)

assert len(canonical_pilot_registry) == 28

if "v1_records" not in globals():
    v1_path = golden_dir / "golden_dataset_v1.jsonl"

    assert v1_path.is_file(), (
        f"Missing v1 dataset: {v1_path}"
    )

    v1_records = read_jsonl(v1_path)

assert len(v1_records) == 573, (
    "Expected 573 v1 records; found "
    f"{len(v1_records)}."
)


def canon_for_audit(value):
    return " ".join(
        str(value or "").split()
    ).strip()


v1_records_by_seed = defaultdict(list)

for record in v1_records:
    seed_id = record.get("source_seed_id")

    assert isinstance(seed_id, int), (
        "v1 record has invalid source_seed_id: "
        f"{record.get('candidate_id')!r}"
    )

    v1_records_by_seed[seed_id].append(record)

v1_seed_ids = sorted(v1_records_by_seed)

assert v1_seed_ids == list(range(1, 29)), (
    "v1 must cover source seeds 1–28 exactly; found "
    f"{v1_seed_ids}."
)

final_46_seed_ids_by_pilot_id = defaultdict(set)

if "final_46_records" in globals():
    for record in final_46_records:
        pilot_case_id = str(
            record.get("pilot_case_id") or ""
        ).strip()

        seed_id = record.get("source_seed_id")

        if pilot_case_id and isinstance(seed_id, int):
            final_46_seed_ids_by_pilot_id[
                pilot_case_id
            ].add(seed_id)

pilot_to_v1_seed_rows = []
mapping_source_counts = Counter()

for pilot in canonical_pilot_registry:
    pilot_number = pilot["pilot_number"]
    pilot_case_id = pilot["pilot_case_id"]

    core_seed_ids = sorted(
        final_46_seed_ids_by_pilot_id.get(
            pilot_case_id,
            set(),
        )
    )

    if len(core_seed_ids) == 1:
        mapped_seed_id = core_seed_ids[0]
        mapping_source = (
            "accepted_46_pilot_case_id"
        )

    else:
        mapped_seed_id = pilot_number
        mapping_source = (
            "historical_pilot_number_to_seed_contract"
        )

    assert mapped_seed_id in v1_records_by_seed, (
        "Mapped seed is absent from v1: "
        f"pilot={pilot_number} | "
        f"seed={mapped_seed_id}"
    )

    descendants = v1_records_by_seed[
        mapped_seed_id
    ]

    question_keys = {
        canon_for_audit(record.get("question"))
        for record in descendants
        if canon_for_audit(record.get("question"))
    }

    answer_keys = {
        canon_for_audit(
            record.get("expected_answer")
        )
        for record in descendants
        if canon_for_audit(
            record.get("expected_answer")
        )
    }

    evidence_keys = {
        canon_for_audit(
            record.get("source_evidence")
        )
        for record in descendants
        if canon_for_audit(
            record.get("source_evidence")
        )
    }

    competency_ids = sorted({
        competency_id
        for record in descendants
        for competency_id in (
            record.get("competency_ids") or []
        )
        if str(competency_id).strip()
    })

    pilot_to_v1_seed_rows.append({
        "pilot_number": pilot_number,
        "pilot_case_id": pilot_case_id,
        "mapped_source_seed_id": mapped_seed_id,
        "mapping_source": mapping_source,
        "descendant_count": len(descendants),
        "unique_normalized_question_count": (
            len(question_keys)
        ),
        "unique_normalized_answer_count": (
            len(answer_keys)
        ),
        "unique_normalized_evidence_count": (
            len(evidence_keys)
        ),
        "competency_count": len(competency_ids),
        "competency_ids": competency_ids,
        "pilot_origin": pilot["pilot_origin"],
        "pilot_type": pilot["pilot_type"],
    })

    mapping_source_counts[mapping_source] += 1

mapped_seed_ids = sorted(
    row["mapped_source_seed_id"]
    for row in pilot_to_v1_seed_rows
)

assert mapped_seed_ids == list(range(1, 29)), (
    "The canonical pilot registry must map one-to-one "
    "to v1 seeds 1–28. Found "
    f"{mapped_seed_ids}."
)

assert sum(
    row["descendant_count"]
    for row in pilot_to_v1_seed_rows
) == len(v1_records)

descendant_counts = [
    row["descendant_count"]
    for row in pilot_to_v1_seed_rows
]

question_uniqueness_counts = [
    row["unique_normalized_question_count"]
    for row in pilot_to_v1_seed_rows
]

answer_uniqueness_counts = [
    row["unique_normalized_answer_count"]
    for row in pilot_to_v1_seed_rows
]

evidence_uniqueness_counts = [
    row["unique_normalized_evidence_count"]
    for row in pilot_to_v1_seed_rows
]

print("=== NB12h PILOT-TO-V1 DESCENDANT MAPPING ===")
print("v1 records:", len(v1_records))
print("Canonical pilots:", len(canonical_pilot_registry))
print("Mapped seeds:", len(mapped_seed_ids))
print(
    "Descendants per pilot:",
    f"min={min(descendant_counts)} | "
    f"max={max(descendant_counts)} | "
    f"distribution={dict(sorted(Counter(descendant_counts).items()))}",
)
print(
    "Unique normalized questions per pilot:",
    f"min={min(question_uniqueness_counts)} | "
    f"max={max(question_uniqueness_counts)}",
)
print(
    "Unique normalized answers per pilot:",
    f"min={min(answer_uniqueness_counts)} | "
    f"max={max(answer_uniqueness_counts)}",
)
print(
    "Unique normalized evidence spans per pilot:",
    f"min={min(evidence_uniqueness_counts)} | "
    f"max={max(evidence_uniqueness_counts)}",
)

print("\nMapping sources:")
for source, count in sorted(
    mapping_source_counts.items()
):
    print(f"  - {source}: {count}")

print("\nPer-pilot expansion baseline:")
for row in pilot_to_v1_seed_rows:
    print(
        f"  - Pilot {row['pilot_number']:02d} "
        f"→ Seed {row['mapped_source_seed_id']:02d} | "
        f"descendants={row['descendant_count']} | "
        f"unique_questions="
        f"{row['unique_normalized_question_count']} | "
        f"unique_answers="
        f"{row['unique_normalized_answer_count']} | "
        f"unique_evidence="
        f"{row['unique_normalized_evidence_count']}"
    )

print(
    "\nRead-only mapping construction:",
    True,
)
print(
    "\nNB12h PILOT-TO-V1 DESCENDANT MAPPING: PASS"
)

=== NB12h PILOT-TO-V1 DESCENDANT MAPPING ===
v1 records: 573
Canonical pilots: 28
Mapped seeds: 28
Descendants per pilot: min=20 | max=21 | distribution={20: 15, 21: 13}
Unique normalized questions per pilot: min=20 | max=21
Unique normalized answers per pilot: min=1 | max=1
Unique normalized evidence spans per pilot: min=1 | max=1

Mapping sources:
  - accepted_46_pilot_case_id: 28

Per-pilot expansion baseline:
  - Pilot 01 → Seed 01 | descendants=21 | unique_questions=21 | unique_answers=1 | unique_evidence=1
  - Pilot 02 → Seed 02 | descendants=21 | unique_questions=21 | unique_answers=1 | unique_evidence=1
  - Pilot 03 → Seed 03 | descendants=21 | unique_questions=21 | unique_answers=1 | unique_evidence=1
  - Pilot 04 → Seed 04 | descendants=21 | unique_questions=21 | unique_answers=1 | unique_evidence=1
  - Pilot 05 → Seed 05 | descendants=21 | unique_questions=21 | unique_answers=1 | unique_evidence=1
  - Pilot 06 → Seed 06 | descendants=21 | unique_questions=21 | unique_answers

<div dir="rtl" align="right">

## **مرحلهٔ هشتم — تحلیل فقط‌خواندنی Paraphrase Expansion در Golden Dataset v1**

مرحلهٔ قبل نگاشت کامل ۲۸ Pilot به ۲۸ Seed و ۵۷۳ فرزند v1 را ساخته است.

اکنون بررسی می‌کنیم که سؤال‌های متعدد هر Seed تا چه اندازه بازنویسی نزدیک یکدیگر هستند.

### **مسئلهٔ مورد بررسی**

در خروجی نگاشت، برای هر Seed فقط یک پاسخ و یک شاهد وجود دارد، اما ۲۰ یا ۲۱ سؤال ساخته شده است.

این مرحله بررسی می‌کند که آیا این سؤال‌ها صرفاً Paraphraseهای نزدیک هستند یا در سطح واژگان تنوع قابل‌مشاهده دارند.

### **ورودی‌های این مرحله**

- `pilot_to_v1_seed_rows`

- `v1_records_by_seed`

- سؤال‌های واقعی ۵۷۳ رکورد Golden Dataset v1

### **بررسی‌های این مرحله**

برای هر Seed محاسبه می‌شود:

- تعداد همهٔ جفت‌سؤال‌ها

- میانگین شباهت واژگانی Jaccard

- بیشترین شباهت میان دو سؤال

- تعداد جفت‌سؤال‌های با شباهت حداقل `0.70`

- تعداد جفت‌سؤال‌های کاملاً تکراری

همچنین در کل v1 بررسی می‌شود که آیا سؤال نرمال‌شدهٔ کاملاً تکراری وجود دارد یا نه.

### **مرز این تحلیل**

- این Step فقط شباهت واژگانی را اندازه می‌گیرد.

- این Step هنوز استقلال معنایی، کیفیت سؤال یا ارزش نهایی رکوردها را تعیین نمی‌کند.

- هیچ رکوردی حذف، اصلاح یا تولید نمی‌شود.

### **خروجی این مرحله**

ساختار درحافظه‌ای `within_seed_similarity_rows` ساخته می‌شود که شاخص شباهت سؤال‌ها را برای هر یک از ۲۸ Seed ثبت می‌کند.

### **Validation**

این Step زمانی PASS است که:

- هر ۲۸ Seed تحلیل شوند؛

- همهٔ ۵۷۳ سؤال وارد تحلیل شوند؛

- تکرار دقیق سؤال‌ها در کل v1 بررسی شود؛

- و هیچ فایل یا Datasetی تغییر نکند.

</div>

In [9]:
# Code Cell 09
# NB12h — Within-seed question similarity and paraphrase-expansion audit
# Read-only: no dataset, registry, or report file is modified.

assert "pilot_to_v1_seed_rows" in globals(), (
    "Run Code Cell 08 first."
)

assert len(pilot_to_v1_seed_rows) == 28


def tokenize_for_similarity(text):
    normalized = canon_for_audit(text).lower()

    return {
        token
        for token in re.findall(
            r"[\w\u0600-\u06FF]+",
            normalized,
        )
        if len(token) > 1
    }


def jaccard_similarity(left_text, right_text):
    left_tokens = tokenize_for_similarity(left_text)
    right_tokens = tokenize_for_similarity(right_text)

    if not left_tokens and not right_tokens:
        return 1.0

    if not left_tokens or not right_tokens:
        return 0.0

    return len(left_tokens & right_tokens) / len(
        left_tokens | right_tokens
    )


all_question_rows = []
within_seed_similarity_rows = []

for mapping_row in pilot_to_v1_seed_rows:
    pilot_number = mapping_row["pilot_number"]
    seed_id = mapping_row["mapped_source_seed_id"]

    descendants = v1_records_by_seed[seed_id]

    question_rows = [
        {
            "candidate_id": record["candidate_id"],
            "question": str(
                record.get("question") or ""
            ).strip(),
            "normalized_question": canon_for_audit(
                record.get("question")
            ),
        }
        for record in descendants
    ]

    assert all(
        row["question"]
        for row in question_rows
    ), (
        "Empty question found in v1 descendants for "
        f"seed={seed_id}."
    )

    pair_similarities = []
    high_similarity_pairs = []
    exact_duplicate_pairs = []

    for left_index in range(len(question_rows)):
        for right_index in range(
            left_index + 1,
            len(question_rows),
        ):
            left = question_rows[left_index]
            right = question_rows[right_index]

            similarity = jaccard_similarity(
                left["question"],
                right["question"],
            )

            pair_similarities.append(similarity)

            if (
                left["normalized_question"]
                == right["normalized_question"]
            ):
                exact_duplicate_pairs.append({
                    "left_candidate_id": (
                        left["candidate_id"]
                    ),
                    "right_candidate_id": (
                        right["candidate_id"]
                    ),
                })

            if similarity >= 0.70:
                high_similarity_pairs.append({
                    "similarity": round(similarity, 4),
                    "left_candidate_id": (
                        left["candidate_id"]
                    ),
                    "right_candidate_id": (
                        right["candidate_id"]
                    ),
                    "left_question": left["question"],
                    "right_question": right["question"],
                })

    within_seed_similarity_rows.append({
        "pilot_number": pilot_number,
        "source_seed_id": seed_id,
        "descendant_count": len(question_rows),
        "pair_count": len(pair_similarities),
        "average_jaccard_similarity": round(
            sum(pair_similarities)
            / len(pair_similarities),
            4,
        ),
        "maximum_jaccard_similarity": round(
            max(pair_similarities),
            4,
        ),
        "high_similarity_pair_count": len(
            high_similarity_pairs
        ),
        "exact_duplicate_pair_count": len(
            exact_duplicate_pairs
        ),
        "high_similarity_pairs": (
            high_similarity_pairs
        ),
    })

    for question_row in question_rows:
        all_question_rows.append({
            "pilot_number": pilot_number,
            "source_seed_id": seed_id,
            **question_row,
        })

global_question_groups = defaultdict(list)

for question_row in all_question_rows:
    global_question_groups[
        question_row["normalized_question"]
    ].append(question_row)

global_exact_duplicate_groups = [
    {
        "normalized_question": normalized_question,
        "record_count": len(rows),
        "candidate_ids": [
            row["candidate_id"]
            for row in rows
        ],
        "seed_ids": sorted({
            row["source_seed_id"]
            for row in rows
        }),
    }
    for normalized_question, rows
    in global_question_groups.items()
    if len(rows) > 1
]

assert len(all_question_rows) == 573

assert not global_exact_duplicate_groups, (
    "Unexpected exact duplicate normalized questions "
    "exist in v1."
)

high_similarity_seed_rows = [
    row
    for row in within_seed_similarity_rows
    if row["high_similarity_pair_count"] > 0
]

mean_of_seed_averages = (
    sum(
        row["average_jaccard_similarity"]
        for row in within_seed_similarity_rows
    )
    / len(within_seed_similarity_rows)
)

maximum_similarity_row = max(
    within_seed_similarity_rows,
    key=lambda row: row[
        "maximum_jaccard_similarity"
    ],
)

print("=== NB12h V1 PARAPHRASE-EXPANSION AUDIT ===")
print("v1 records analysed:", len(all_question_rows))
print(
    "Seeds analysed:",
    len(within_seed_similarity_rows),
)
print(
    "Global exact duplicate-question groups:",
    len(global_exact_duplicate_groups),
)
print(
    "Seeds with high-similarity question pairs "
    "(Jaccard >= 0.70):",
    len(high_similarity_seed_rows),
)
print(
    "Mean within-seed question similarity:",
    f"{mean_of_seed_averages:.4f}",
)
print(
    "Highest within-seed similarity:",
    f"{maximum_similarity_row['maximum_jaccard_similarity']:.4f} "
    f"(Pilot {maximum_similarity_row['pilot_number']:02d} / "
    f"Seed {maximum_similarity_row['source_seed_id']:02d})",
)

print("\nPer-seed similarity baseline:")
for row in within_seed_similarity_rows:
    print(
        f"  - Pilot {row['pilot_number']:02d} / "
        f"Seed {row['source_seed_id']:02d} | "
        f"pairs={row['pair_count']} | "
        f"avg={row['average_jaccard_similarity']:.4f} | "
        f"max={row['maximum_jaccard_similarity']:.4f} | "
        f"high_pairs={row['high_similarity_pair_count']} | "
        f"exact_pairs={row['exact_duplicate_pair_count']}"
    )

print(
    "\nInterpretation boundary: this Cell measures "
    "question-form similarity only. It does not yet "
    "judge semantic independence."
)
print(
    "\nRead-only paraphrase-expansion audit:",
    True,
)
print(
    "\nNB12h V1 PARAPHRASE-EXPANSION AUDIT: PASS"
)

=== NB12h V1 PARAPHRASE-EXPANSION AUDIT ===
v1 records analysed: 573
Seeds analysed: 28
Global exact duplicate-question groups: 0
Seeds with high-similarity question pairs (Jaccard >= 0.70): 28
Mean within-seed question similarity: 0.2300
Highest within-seed similarity: 0.9231 (Pilot 18 / Seed 18)

Per-seed similarity baseline:
  - Pilot 01 / Seed 01 | pairs=210 | avg=0.2302 | max=0.9167 | high_pairs=19 | exact_pairs=0
  - Pilot 02 / Seed 02 | pairs=210 | avg=0.2408 | max=0.8750 | high_pairs=27 | exact_pairs=0
  - Pilot 03 / Seed 03 | pairs=210 | avg=0.2328 | max=0.8667 | high_pairs=6 | exact_pairs=0
  - Pilot 04 / Seed 04 | pairs=210 | avg=0.2155 | max=0.8667 | high_pairs=8 | exact_pairs=0
  - Pilot 05 / Seed 05 | pairs=210 | avg=0.2437 | max=0.9167 | high_pairs=27 | exact_pairs=0
  - Pilot 06 / Seed 06 | pairs=210 | avg=0.2310 | max=0.9167 | high_pairs=18 | exact_pairs=0
  - Pilot 07 / Seed 07 | pairs=210 | avg=0.2440 | max=0.8750 | high_pairs=27 | exact_pairs=0
  - Pilot 08 / Seed 0

<div dir="rtl" align="right">

## **مرحلهٔ نهم — تحلیل واحدهای واقعی Expansion در Golden Dataset v1**

تا اینجا مشخص شده است که v1 شامل ۵۷۳ سؤال یکتا است، اما هر Seed فقط یک پاسخ و یک شاهد یکتا دارد.

در این مرحله، ساختار واقعی Expansion v1 به‌صورت رسمی بررسی می‌شود تا مشخص شود هر گروه از رکوردها بر پایهٔ چند claim مستقل ساخته شده است.

### **ورودی‌های این مرحله**

- `pilot_to_v1_seed_rows`

- `v1_records_by_seed`

- فیلدهای واقعی رکوردهای `golden_dataset_v1.jsonl`

### **واحد Expansion**

در این Notebook، یک واحد Expansion به گروهی از رکوردها گفته می‌شود که `source_seed_id` یکسان دارند.

برای هر واحد بررسی می‌شود:

- تعداد رکوردها

- تعداد سؤال‌های یکتا

- تعداد پاسخ‌های یکتا

- تعداد شاهدهای یکتا

- تعداد `source_lineage`های یکتا

- تعداد `generation_competency`های یکتا

- تعداد موضوع‌ها و competencyهای پوشش‌داده‌شده

### **قاعدهٔ مهم**

- چند سؤال متفاوت با یک شاهد و یک پاسخ، به‌تنهایی چند claim مستقل محسوب نمی‌شوند.

- این Step فقط ساختار واقعی داده را گزارش می‌کند.

- هنوز هیچ رکوردی حذف یا رد نمی‌شود.

- هیچ نسخهٔ جدیدی از Golden Dataset ساخته نمی‌شود.

### **خروجی این مرحله**

ساختار درحافظه‌ای `v1_expansion_unit_rows` ساخته می‌شود که برای هر Pilot و Seed، نوع واقعی Expansion را ثبت می‌کند.

### **Validation**

این Step زمانی PASS است که:

- دقیقاً ۲۸ واحد Expansion ساخته شود؛

- مجموع رکوردهای واحدها برابر با ۵۷۳ باشد؛

- تعداد پاسخ و شاهد یکتای هر واحد محاسبه شود؛

- و هیچ فایل یا Datasetی تغییر نکند.

</div>

In [10]:
# Code Cell 10
# NB12h — v1 expansion-unit inventory
# Read-only: no dataset, registry, or report file is modified.

assert "pilot_to_v1_seed_rows" in globals(), (
    "Run Code Cell 08 first."
)

assert "within_seed_similarity_rows" in globals(), (
    "Run Code Cell 09 first."
)

assert len(pilot_to_v1_seed_rows) == 28
assert len(within_seed_similarity_rows) == 28

similarity_by_seed = {
    row["source_seed_id"]: row
    for row in within_seed_similarity_rows
}

v1_expansion_unit_rows = []

for mapping_row in pilot_to_v1_seed_rows:
    pilot_number = mapping_row["pilot_number"]
    seed_id = mapping_row["mapped_source_seed_id"]

    descendants = v1_records_by_seed[seed_id]

    normalized_questions = {
        canon_for_audit(record.get("question"))
        for record in descendants
        if canon_for_audit(record.get("question"))
    }

    normalized_answers = {
        canon_for_audit(
            record.get("expected_answer")
        )
        for record in descendants
        if canon_for_audit(
            record.get("expected_answer")
        )
    }

    normalized_evidence = {
        canon_for_audit(
            record.get("source_evidence")
        )
        for record in descendants
        if canon_for_audit(
            record.get("source_evidence")
        )
    }

    normalized_lineage = {
        canon_for_audit(
            record.get("source_lineage")
        )
        for record in descendants
        if canon_for_audit(
            record.get("source_lineage")
        )
    }

    generation_competencies = {
        canon_for_audit(
            record.get("generation_competency")
        )
        for record in descendants
        if canon_for_audit(
            record.get("generation_competency")
        )
    }

    taxonomy_topics = {
        topic
        for record in descendants
        for topic in (
            record.get("taxonomy_topics") or []
        )
        if str(topic).strip()
    }

    competency_ids = {
        competency_id
        for record in descendants
        for competency_id in (
            record.get("competency_ids") or []
        )
        if str(competency_id).strip()
    }

    generation_statuses = {
        canon_for_audit(
            record.get("generation_status")
        )
        for record in descendants
        if canon_for_audit(
            record.get("generation_status")
        )
    }

    similarity_row = similarity_by_seed[seed_id]

    if (
        len(normalized_answers) == 1
        and len(normalized_evidence) == 1
    ):
        expansion_type = (
            "QUESTION_FORM_EXPANSION_"
            "OVER_SINGLE_CLAIM"
        )
    else:
        expansion_type = (
            "MULTI_ANSWER_OR_MULTI_EVIDENCE_"
            "EXPANSION"
        )

    v1_expansion_unit_rows.append({
        "pilot_number": pilot_number,
        "source_seed_id": seed_id,
        "record_count": len(descendants),
        "unique_question_count": len(
            normalized_questions
        ),
        "unique_answer_count": len(
            normalized_answers
        ),
        "unique_evidence_count": len(
            normalized_evidence
        ),
        "unique_lineage_count": len(
            normalized_lineage
        ),
        "generation_competency_count": len(
            generation_competencies
        ),
        "taxonomy_topic_count": len(
            taxonomy_topics
        ),
        "competency_id_count": len(
            competency_ids
        ),
        "generation_status_count": len(
            generation_statuses
        ),
        "expansion_type": expansion_type,
        "average_question_similarity": (
            similarity_row[
                "average_jaccard_similarity"
            ]
        ),
        "high_similarity_pair_count": (
            similarity_row[
                "high_similarity_pair_count"
            ]
        ),
    })

assert len(v1_expansion_unit_rows) == 28

assert sum(
    row["record_count"]
    for row in v1_expansion_unit_rows
) == 573

expansion_type_counts = Counter(
    row["expansion_type"]
    for row in v1_expansion_unit_rows
)

single_claim_rows = [
    row
    for row in v1_expansion_unit_rows
    if row["expansion_type"]
    == "QUESTION_FORM_EXPANSION_OVER_SINGLE_CLAIM"
]

global_unique_answer_count = len({
    canon_for_audit(record.get("expected_answer"))
    for record in v1_records
    if canon_for_audit(
        record.get("expected_answer")
    )
})

global_unique_evidence_count = len({
    canon_for_audit(record.get("source_evidence"))
    for record in v1_records
    if canon_for_audit(
        record.get("source_evidence")
    )
})

print("=== NB12h V1 EXPANSION-UNIT INVENTORY ===")
print("v1 records:", len(v1_records))
print("Expansion units:", len(v1_expansion_unit_rows))
print(
    "Global unique answers:",
    global_unique_answer_count,
)
print(
    "Global unique evidence spans:",
    global_unique_evidence_count,
)
print(
    "Single-claim question-form expansion units:",
    len(single_claim_rows),
)
print("\nExpansion types:")
for label, count in sorted(
    expansion_type_counts.items()
):
    print(f"  - {label}: {count}")

print("\nPer-unit inventory:")
for row in v1_expansion_unit_rows:
    print(
        f"  - Pilot {row['pilot_number']:02d} / "
        f"Seed {row['source_seed_id']:02d} | "
        f"records={row['record_count']} | "
        f"questions={row['unique_question_count']} | "
        f"answers={row['unique_answer_count']} | "
        f"evidence={row['unique_evidence_count']} | "
        f"type={row['expansion_type']}"
    )

print(
    "\nRead-only expansion-unit inventory:",
    True,
)
print(
    "\nNB12h V1 EXPANSION-UNIT INVENTORY: PASS"
)

=== NB12h V1 EXPANSION-UNIT INVENTORY ===
v1 records: 573
Expansion units: 28
Global unique answers: 28
Global unique evidence spans: 28
Single-claim question-form expansion units: 28

Expansion types:
  - QUESTION_FORM_EXPANSION_OVER_SINGLE_CLAIM: 28

Per-unit inventory:
  - Pilot 01 / Seed 01 | records=21 | questions=21 | answers=1 | evidence=1 | type=QUESTION_FORM_EXPANSION_OVER_SINGLE_CLAIM
  - Pilot 02 / Seed 02 | records=21 | questions=21 | answers=1 | evidence=1 | type=QUESTION_FORM_EXPANSION_OVER_SINGLE_CLAIM
  - Pilot 03 / Seed 03 | records=21 | questions=21 | answers=1 | evidence=1 | type=QUESTION_FORM_EXPANSION_OVER_SINGLE_CLAIM
  - Pilot 04 / Seed 04 | records=21 | questions=21 | answers=1 | evidence=1 | type=QUESTION_FORM_EXPANSION_OVER_SINGLE_CLAIM
  - Pilot 05 / Seed 05 | records=21 | questions=21 | answers=1 | evidence=1 | type=QUESTION_FORM_EXPANSION_OVER_SINGLE_CLAIM
  - Pilot 06 / Seed 06 | records=21 | questions=21 | answers=1 | evidence=1 | type=QUESTION_FORM_EXPAN

<div dir="rtl" align="right">

## **مرحلهٔ دهم — بررسی Fidelity میان ۲۸ Pilot و واحدهای Expansion در Golden Dataset v1**

اکنون مشخص شده است که Golden Dataset v1 از ۲۸ واحد Expansion تشکیل شده است و هر واحد فقط یک پاسخ و یک شاهد دارد.

در این مرحله بررسی می‌شود که پاسخ و شاهد ثبت‌شده در هر واحد v1 تا چه اندازه با پاسخ و شاهد Pilot متناظر در `canonical_pilot_registry` هم‌راستا است.

### **ورودی‌های این مرحله**

- `canonical_pilot_registry`

- `pilot_to_v1_seed_rows`

- `v1_records_by_seed`

### **بررسی‌های این مرحله**

برای هر Pilot و Seed متناظر، موارد زیر مقایسه می‌شود:

- پاسخ نهایی Pilot با پاسخ واحد v1

- شاهد Pilot با شاهد واحد v1

- تطابق دقیق نرمال‌شده

- رابطهٔ containment میان متن شاهدها

### **طبقه‌بندی Fidelity**

هر واحد در یکی از وضعیت‌های زیر قرار می‌گیرد:

- `EXACT_PILOT_ALIGNMENT`

- `ANSWER_ALIGNED_EVIDENCE_VARIANT`

- `EVIDENCE_ALIGNED_ANSWER_VARIANT`

- `PILOT_V1_ALIGNMENT_REVIEW_REQUIRED`

### **قاعدهٔ مهم**

- این Step هنوز دربارهٔ درست یا نادرست بودن محتوای Pilot حکم صادر نمی‌کند.

- اختلاف میان Pilot و v1 به‌معنای خطا نیست؛ ابتدا به‌عنوان Finding ثبت می‌شود.

- هیچ رکوردی تغییر، حذف یا تولید نمی‌شود.

### **خروجی این مرحله**

ساختار درحافظه‌ای `pilot_v1_fidelity_rows` ساخته می‌شود که رابطهٔ هر Pilot با واحد Expansion متناظر در v1 را ثبت می‌کند.

### **Validation**

این Step زمانی PASS است که:

- هر ۲۸ Pilot با Seed متناظر خود مقایسه شوند؛

- برای هر Seed دقیقاً یک پاسخ و یک شاهد v1 مبنای مقایسه باشد؛

- همهٔ اختلاف‌ها به‌صورت Finding ثبت شوند؛

- و هیچ فایل یا Datasetی تغییر نکند.

</div>

In [11]:
# Code Cell 11
# NB12h — Pilot-to-v1 fidelity audit
# Read-only: no dataset, registry, or report file is modified.

assert "canonical_pilot_registry" in globals(), (
    "Run Code Cell 07 first."
)

assert "pilot_to_v1_seed_rows" in globals(), (
    "Run Code Cell 08 first."
)

assert len(canonical_pilot_registry) == 28
assert len(pilot_to_v1_seed_rows) == 28

pilot_by_number = {
    row["pilot_number"]: row
    for row in canonical_pilot_registry
}

mapping_by_pilot_number = {
    row["pilot_number"]: row
    for row in pilot_to_v1_seed_rows
}

assert sorted(pilot_by_number) == list(
    range(1, 29)
)

assert sorted(mapping_by_pilot_number) == list(
    range(1, 29)
)


def text_relation(left, right):
    left_normalized = canon_for_audit(left)
    right_normalized = canon_for_audit(right)

    if left_normalized == right_normalized:
        return "EXACT"

    if (
        left_normalized
        and right_normalized
        and left_normalized in right_normalized
    ):
        return "LEFT_CONTAINED_IN_RIGHT"

    if (
        left_normalized
        and right_normalized
        and right_normalized in left_normalized
    ):
        return "RIGHT_CONTAINED_IN_LEFT"

    return "NO_EXACT_OR_CONTAINMENT_MATCH"


pilot_v1_fidelity_rows = []

for pilot_number in range(1, 29):
    pilot = pilot_by_number[pilot_number]
    mapping = mapping_by_pilot_number[
        pilot_number
    ]

    seed_id = mapping["mapped_source_seed_id"]

    descendants = v1_records_by_seed[seed_id]

    v1_answers = {
        canon_for_audit(
            record.get("expected_answer")
        )
        for record in descendants
        if canon_for_audit(
            record.get("expected_answer")
        )
    }

    v1_evidence_values = {
        canon_for_audit(
            record.get("source_evidence")
        )
        for record in descendants
        if canon_for_audit(
            record.get("source_evidence")
        )
    }

    assert len(v1_answers) == 1, (
        "Expected exactly one v1 answer for "
        f"seed={seed_id}; found {len(v1_answers)}."
    )

    assert len(v1_evidence_values) == 1, (
        "Expected exactly one v1 evidence span for "
        f"seed={seed_id}; found "
        f"{len(v1_evidence_values)}."
    )

    pilot_answer = pilot["expected_answer"]

    pilot_evidence = "\n".join(
        part["text"]
        for part in pilot["evidence_parts"]
    )

    v1_answer = next(iter(v1_answers))
    v1_evidence = next(iter(v1_evidence_values))

    answer_relation = text_relation(
        pilot_answer,
        v1_answer,
    )

    evidence_relation = text_relation(
        pilot_evidence,
        v1_evidence,
    )

    if (
        answer_relation == "EXACT"
        and evidence_relation == "EXACT"
    ):
        fidelity_status = (
            "EXACT_PILOT_ALIGNMENT"
        )

    elif answer_relation == "EXACT":
        fidelity_status = (
            "ANSWER_ALIGNED_EVIDENCE_VARIANT"
        )

    elif evidence_relation == "EXACT":
        fidelity_status = (
            "EVIDENCE_ALIGNED_ANSWER_VARIANT"
        )

    else:
        fidelity_status = (
            "PILOT_V1_ALIGNMENT_REVIEW_REQUIRED"
        )

    pilot_v1_fidelity_rows.append({
        "pilot_number": pilot_number,
        "pilot_case_id": pilot["pilot_case_id"],
        "source_seed_id": seed_id,
        "pilot_origin": pilot["pilot_origin"],
        "pilot_answer": pilot_answer,
        "v1_answer": v1_answer,
        "pilot_evidence": pilot_evidence,
        "v1_evidence": v1_evidence,
        "answer_relation": answer_relation,
        "evidence_relation": evidence_relation,
        "fidelity_status": fidelity_status,
    })

assert len(pilot_v1_fidelity_rows) == 28

fidelity_status_counts = Counter(
    row["fidelity_status"]
    for row in pilot_v1_fidelity_rows
)

review_required_rows = [
    row
    for row in pilot_v1_fidelity_rows
    if row["fidelity_status"]
    == "PILOT_V1_ALIGNMENT_REVIEW_REQUIRED"
]

print("=== NB12h PILOT-TO-V1 FIDELITY AUDIT ===")
print("Pilots compared:", len(pilot_v1_fidelity_rows))
print(
    "Review-required alignments:",
    len(review_required_rows),
)

print("\nFidelity status counts:")
for status, count in sorted(
    fidelity_status_counts.items()
):
    print(f"  - {status}: {count}")

print("\nPer-pilot fidelity:")
for row in pilot_v1_fidelity_rows:
    print(
        f"  - Pilot {row['pilot_number']:02d} / "
        f"Seed {row['source_seed_id']:02d} | "
        f"answer={row['answer_relation']} | "
        f"evidence={row['evidence_relation']} | "
        f"status={row['fidelity_status']}"
    )

if review_required_rows:
    print("\nReview-required pilots:")
    for row in review_required_rows:
        print(
            f"  - Pilot {row['pilot_number']:02d} / "
            f"Seed {row['source_seed_id']:02d}"
        )

print(
    "\nRead-only pilot-to-v1 fidelity audit:",
    True,
)
print(
    "\nNB12h PILOT-TO-V1 FIDELITY AUDIT: PASS"
)

=== NB12h PILOT-TO-V1 FIDELITY AUDIT ===
Pilots compared: 28
Review-required alignments: 0

Fidelity status counts:
  - ANSWER_ALIGNED_EVIDENCE_VARIANT: 28

Per-pilot fidelity:
  - Pilot 01 / Seed 01 | answer=EXACT | evidence=NO_EXACT_OR_CONTAINMENT_MATCH | status=ANSWER_ALIGNED_EVIDENCE_VARIANT
  - Pilot 02 / Seed 02 | answer=EXACT | evidence=NO_EXACT_OR_CONTAINMENT_MATCH | status=ANSWER_ALIGNED_EVIDENCE_VARIANT
  - Pilot 03 / Seed 03 | answer=EXACT | evidence=NO_EXACT_OR_CONTAINMENT_MATCH | status=ANSWER_ALIGNED_EVIDENCE_VARIANT
  - Pilot 04 / Seed 04 | answer=EXACT | evidence=NO_EXACT_OR_CONTAINMENT_MATCH | status=ANSWER_ALIGNED_EVIDENCE_VARIANT
  - Pilot 05 / Seed 05 | answer=EXACT | evidence=NO_EXACT_OR_CONTAINMENT_MATCH | status=ANSWER_ALIGNED_EVIDENCE_VARIANT
  - Pilot 06 / Seed 06 | answer=EXACT | evidence=NO_EXACT_OR_CONTAINMENT_MATCH | status=ANSWER_ALIGNED_EVIDENCE_VARIANT
  - Pilot 07 / Seed 07 | answer=EXACT | evidence=NO_EXACT_OR_CONTAINMENT_MATCH | status=ANSWER_ALIGNED_

<div dir="rtl" align="right">

## **مرحلهٔ یازدهم — تحلیل Scope و هم‌پوشانی شاهدهای Pilot و Golden Dataset v1**

مرحلهٔ قبل نشان داد پاسخ ۲۸ Pilot با پاسخ واحدهای متناظر در v1 تطابق دقیق دارد، اما متن شاهدها تطابق متنی دقیق ندارند.

در این مرحله، علت این تفاوت بررسی می‌شود.

### **ورودی‌های این مرحله**

- `pilot_v1_fidelity_rows`

- `canonical_pilot_registry`

- `v1_records_by_seed`

### **بررسی‌های این مرحله**

برای هر Pilot و Seed متناظر، موارد زیر محاسبه می‌شود:

- بیشترین شباهت واژگانی میان شاهدهای Pilot و شاهد v1

- رابطهٔ containment میان هر شاهد Pilot و شاهد v1

- وجود پاسخ نهایی در شاهد v1

- وجود پاسخ نهایی در حداقل یک شاهد Pilot

- تعداد بخش‌های شاهد ثبت‌شده برای Pilot

### **مرز این تحلیل**

- شباهت پایین یا عدم تطابق دقیق، به‌تنهایی به‌معنای نامعتبر بودن Pilot یا v1 نیست.

- این Step فقط نوع اختلاف شاهدها را اندازه می‌گیرد.

- قضاوت معنایی یا اصلاح شاهدها در این مرحله انجام نمی‌شود.

- هیچ فایل یا Datasetی تغییر نمی‌کند.

### **خروجی این مرحله**

ساختار درحافظه‌ای `pilot_v1_evidence_scope_rows` ساخته می‌شود که برای هر Pilot، میزان هم‌پوشانی و وضعیت grounding پاسخ در شاهدها را ثبت می‌کند.

### **Validation**

این Step زمانی PASS است که:

- هر ۲۸ Pilot تحلیل شوند؛

- برای هر Pilot، پاسخ در شاهد v1 و شاهد Pilot بررسی شود؛

- شاخص هم‌پوشانی شاهدها محاسبه شود؛

- و هیچ فایل یا Datasetی تغییر نکند.

</div>

In [12]:
# Code Cell 12
# NB12h — Pilot-to-v1 evidence scope and grounding audit
# Read-only: no dataset, registry, or report file is modified.

assert "pilot_v1_fidelity_rows" in globals(), (
    "Run Code Cell 11 first."
)

assert len(pilot_v1_fidelity_rows) == 28

pilot_v1_evidence_scope_rows = []

for fidelity_row in pilot_v1_fidelity_rows:
    pilot_number = fidelity_row["pilot_number"]
    seed_id = fidelity_row["source_seed_id"]

    pilot = pilot_by_number[pilot_number]

    pilot_evidence_parts = [
        str(part["text"]).strip()
        for part in pilot["evidence_parts"]
        if str(part.get("text") or "").strip()
    ]

    assert pilot_evidence_parts, (
        "Canonical pilot has no usable evidence: "
        f"pilot={pilot_number}"
    )

    v1_evidence = fidelity_row["v1_evidence"]
    answer = fidelity_row["v1_answer"]

    evidence_pair_rows = []

    for evidence_part in pilot_evidence_parts:
        similarity = jaccard_similarity(
            evidence_part,
            v1_evidence,
        )

        evidence_pair_rows.append({
            "pilot_evidence": evidence_part,
            "jaccard_similarity": round(
                similarity,
                4,
            ),
            "relation": text_relation(
                evidence_part,
                v1_evidence,
            ),
        })

    best_pair = max(
        evidence_pair_rows,
        key=lambda row: row[
            "jaccard_similarity"
        ],
    )

    answer_normalized = canon_for_audit(answer)

    answer_in_v1_evidence = (
        answer_normalized
        in canon_for_audit(v1_evidence)
    )

    answer_in_any_pilot_evidence = any(
        answer_normalized
        in canon_for_audit(evidence_part)
        for evidence_part in pilot_evidence_parts
    )

    if best_pair["relation"] == "EXACT":
        evidence_scope_status = (
            "EXACT_EVIDENCE_ALIGNMENT"
        )

    elif best_pair["relation"] in {
        "LEFT_CONTAINED_IN_RIGHT",
        "RIGHT_CONTAINED_IN_LEFT",
    }:
        evidence_scope_status = (
            "CONTAINMENT_EVIDENCE_ALIGNMENT"
        )

    elif best_pair["jaccard_similarity"] >= 0.30:
        evidence_scope_status = (
            "LEXICAL_SCOPE_OVERLAP"
        )

    else:
        evidence_scope_status = (
            "LOW_LEXICAL_SCOPE_OVERLAP"
        )

    pilot_v1_evidence_scope_rows.append({
        "pilot_number": pilot_number,
        "source_seed_id": seed_id,
        "pilot_evidence_part_count": len(
            pilot_evidence_parts
        ),
        "answer_in_v1_evidence": (
            answer_in_v1_evidence
        ),
        "answer_in_any_pilot_evidence": (
            answer_in_any_pilot_evidence
        ),
        "best_evidence_jaccard_similarity": (
            best_pair["jaccard_similarity"]
        ),
        "best_evidence_relation": (
            best_pair["relation"]
        ),
        "evidence_scope_status": (
            evidence_scope_status
        ),
        "evidence_pair_rows": evidence_pair_rows,
    })

assert len(pilot_v1_evidence_scope_rows) == 28

scope_status_counts = Counter(
    row["evidence_scope_status"]
    for row in pilot_v1_evidence_scope_rows
)

v1_answer_grounding_failures = [
    row
    for row in pilot_v1_evidence_scope_rows
    if not row["answer_in_v1_evidence"]
]

pilot_answer_grounding_failures = [
    row
    for row in pilot_v1_evidence_scope_rows
    if not row[
        "answer_in_any_pilot_evidence"
    ]
]

print("=== NB12h PILOT-TO-V1 EVIDENCE SCOPE AUDIT ===")
print(
    "Pilots analysed:",
    len(pilot_v1_evidence_scope_rows),
)
print(
    "Answer absent from v1 evidence:",
    len(v1_answer_grounding_failures),
)
print(
    "Answer absent from pilot evidence:",
    len(pilot_answer_grounding_failures),
)

print("\nEvidence scope statuses:")
for status, count in sorted(
    scope_status_counts.items()
):
    print(f"  - {status}: {count}")

print("\nPer-pilot evidence scope:")
for row in pilot_v1_evidence_scope_rows:
    print(
        f"  - Pilot {row['pilot_number']:02d} / "
        f"Seed {row['source_seed_id']:02d} | "
        f"parts={row['pilot_evidence_part_count']} | "
        f"best_jaccard="
        f"{row['best_evidence_jaccard_similarity']:.4f} | "
        f"relation={row['best_evidence_relation']} | "
        f"answer_in_v1="
        f"{row['answer_in_v1_evidence']} | "
        f"answer_in_pilot="
        f"{row['answer_in_any_pilot_evidence']} | "
        f"status={row['evidence_scope_status']}"
    )

print(
    "\nRead-only evidence scope audit:",
    True,
)
print(
    "\nNB12h PILOT-TO-V1 EVIDENCE SCOPE AUDIT: PASS"
)

=== NB12h PILOT-TO-V1 EVIDENCE SCOPE AUDIT ===
Pilots analysed: 28
Answer absent from v1 evidence: 0
Answer absent from pilot evidence: 28

Evidence scope statuses:
  - LEXICAL_SCOPE_OVERLAP: 4
  - LOW_LEXICAL_SCOPE_OVERLAP: 24

Per-pilot evidence scope:
  - Pilot 01 / Seed 01 | parts=1 | best_jaccard=0.2875 | relation=NO_EXACT_OR_CONTAINMENT_MATCH | answer_in_v1=True | answer_in_pilot=False | status=LOW_LEXICAL_SCOPE_OVERLAP
  - Pilot 02 / Seed 02 | parts=1 | best_jaccard=0.2906 | relation=NO_EXACT_OR_CONTAINMENT_MATCH | answer_in_v1=True | answer_in_pilot=False | status=LOW_LEXICAL_SCOPE_OVERLAP
  - Pilot 03 / Seed 03 | parts=1 | best_jaccard=0.3611 | relation=NO_EXACT_OR_CONTAINMENT_MATCH | answer_in_v1=True | answer_in_pilot=False | status=LEXICAL_SCOPE_OVERLAP
  - Pilot 04 / Seed 04 | parts=5 | best_jaccard=0.2574 | relation=NO_EXACT_OR_CONTAINMENT_MATCH | answer_in_v1=True | answer_in_pilot=False | status=LOW_LEXICAL_SCOPE_OVERLAP
  - Pilot 05 / Seed 05 | parts=8 | best_jaccard=0

<div dir="rtl" align="right">

## **مرحلهٔ یازدهم — بررسی محدودیت Crosswalk شاهد میان Pilot، v1 و Core-46**

مرحلهٔ قبل ثابت کرد پاسخ نهایی هر Pilot با پاسخ واحد متناظر در Golden Dataset v1 تطابق دقیق دارد.

اما ساختار فعلی Artifactها سه نوع Evidence متفاوت دارد:

- Evidenceهای provenance در registry پایلوت‌ها

- Evidence تاریخی ثبت‌شده در Golden Dataset v1

- Evidenceهای بازبینی و اصلاح‌شده در Core-46 آزمایشی

Core-46 یک Dataset آزمایشی مستقل است؛ بنابراین وجود `pilot_case_id` مشترک به‌تنهایی به‌معنای یکسان بودن پاسخ یا شاهد آن با v1 نیست.

### **هدف این مرحله**

هدف این Step، ثبت شفاف قابلیت و محدودیت Crosswalk میان این سه Artifact است؛ نه تحمیل یک تطابق نادرست میان آن‌ها.

### **بررسی‌های این مرحله**

برای هر Pilot بررسی می‌شود:

- آیا پاسخ v1 در شاهد تاریخی v1 وجود دارد.

- آیا پاسخ Pilot در Evidence provenance ثبت‌شده در registry وجود دارد.

- پاسخ‌های Core-46 متناظر چه رابطه‌ای با پاسخ تاریخی v1 دارند.

- آیا Crosswalk مستقیم و پاسخ‌پشتیبان میان Pilot و v1 در ساختار فعلی Registry قابل اثبات است یا نه.

### **قاعدهٔ مهم**

- Core-46 فقط یک Artifact آزمایشی و مستقل است.

- Core-46 جایگزین Evidence تاریخی v1 یا Evidence نهایی مورد بازبینی مؤلف نیست.

- نبود Crosswalk ساختاری، یک Finding مستند است؛ نه Failure خودکار برای Pilot یا v1.

- هیچ فایل یا Datasetی تغییر نمی‌کند.

### **خروجی این مرحله**

ساختار درحافظه‌ای `pilot_v1_evidence_scope_rows` ساخته می‌شود که وضعیت Crosswalk Evidence هر Pilot را ثبت می‌کند.

### **Validation**

این Step زمانی PASS است که:

- هر ۲۸ Pilot بررسی شوند؛

- grounding پاسخ در Evidence تاریخی v1 تأیید شود؛

- محدودیت Crosswalk میان Artifactها بدون حدس ثبت شود؛

- و هیچ فایل یا Datasetی تغییر نکند.

</div>

In [13]:
# Code Cell 12
# NB12h — Pilot, v1, and Core-46 evidence crosswalk audit
# Read-only: no dataset, registry, or report file is modified.

assert "pilot_v1_fidelity_rows" in globals(), (
    "Run Code Cell 11 first."
)

assert "final_46_records" in globals(), (
    "Run Code Cell 07 first."
)

assert len(pilot_v1_fidelity_rows) == 28

if "final_46_records_by_pilot_id" not in globals():
    final_46_records_by_pilot_id = defaultdict(
        list
    )

    for record in final_46_records:
        pilot_case_id = str(
            record.get("pilot_case_id") or ""
        ).strip()

        if pilot_case_id:
            final_46_records_by_pilot_id[
                pilot_case_id
            ].append(record)


def answer_relation_to_v1(
    candidate_answer,
    v1_answer,
):
    return text_relation(
        candidate_answer,
        v1_answer,
    )


pilot_v1_evidence_scope_rows = []

for fidelity_row in pilot_v1_fidelity_rows:
    pilot_number = fidelity_row["pilot_number"]
    pilot_case_id = fidelity_row["pilot_case_id"]
    seed_id = fidelity_row["source_seed_id"]
    v1_answer = fidelity_row["v1_answer"]
    v1_evidence = fidelity_row["v1_evidence"]

    pilot = pilot_by_number[pilot_number]

    pilot_provenance_evidence = [
        str(part.get("text") or "").strip()
        for part in pilot["evidence_parts"]
        if str(part.get("text") or "").strip()
    ]

    v1_answer_normalized = canon_for_audit(
        v1_answer
    )

    answer_in_v1_evidence = (
        v1_answer_normalized
        in canon_for_audit(v1_evidence)
    )

    answer_in_pilot_provenance = any(
        v1_answer_normalized
        in canon_for_audit(evidence_text)
        for evidence_text
        in pilot_provenance_evidence
    )

    core_records = (
        final_46_records_by_pilot_id.get(
            pilot_case_id,
            [],
        )
    )

    core_answer_relation_counts = Counter(
        answer_relation_to_v1(
            record.get("expected_answer"),
            v1_answer,
        )
        for record in core_records
    )

    core_evidence_contains_v1_answer_count = sum(
        v1_answer_normalized
        in canon_for_audit(
            record.get("source_evidence")
        )
        for record in core_records
    )

    if answer_in_pilot_provenance:
        crosswalk_status = (
            "DIRECT_PILOT_PROVENANCE_"
            "ANSWER_GROUNDING_AVAILABLE"
        )

    else:
        crosswalk_status = (
            "PILOT_PROVENANCE_IS_NOT_"
            "TARGET_EVIDENCE_IN_CURRENT_REGISTRY"
        )

    pilot_v1_evidence_scope_rows.append({
        "pilot_number": pilot_number,
        "pilot_case_id": pilot_case_id,
        "source_seed_id": seed_id,
        "answer_in_v1_evidence": (
            answer_in_v1_evidence
        ),
        "pilot_provenance_part_count": len(
            pilot_provenance_evidence
        ),
        "answer_in_pilot_provenance": (
            answer_in_pilot_provenance
        ),
        "core_record_count": len(core_records),
        "core_answer_relation_counts": dict(
            sorted(
                core_answer_relation_counts.items()
            )
        ),
        "core_evidence_contains_v1_answer_count": (
            core_evidence_contains_v1_answer_count
        ),
        "crosswalk_status": crosswalk_status,
    })

assert len(pilot_v1_evidence_scope_rows) == 28

v1_grounding_failures = [
    row
    for row in pilot_v1_evidence_scope_rows
    if not row["answer_in_v1_evidence"]
]

assert not v1_grounding_failures, (
    "Historical v1 answer-grounding failures remain."
)

crosswalk_status_counts = Counter(
    row["crosswalk_status"]
    for row in pilot_v1_evidence_scope_rows
)

direct_crosswalk_rows = [
    row
    for row in pilot_v1_evidence_scope_rows
    if row["crosswalk_status"]
    == (
        "DIRECT_PILOT_PROVENANCE_"
        "ANSWER_GROUNDING_AVAILABLE"
    )
]

limited_crosswalk_rows = [
    row
    for row in pilot_v1_evidence_scope_rows
    if row["crosswalk_status"]
    == (
        "PILOT_PROVENANCE_IS_NOT_"
        "TARGET_EVIDENCE_IN_CURRENT_REGISTRY"
    )
]

print(
    "=== NB12h PILOT-V1-CORE-46 "
    "EVIDENCE CROSSWALK AUDIT ==="
)
print(
    "Pilots analysed:",
    len(pilot_v1_evidence_scope_rows),
)
print(
    "Historical v1 grounding failures:",
    len(v1_grounding_failures),
)
print(
    "Direct Pilot-to-v1 evidence crosswalks:",
    len(direct_crosswalk_rows),
)
print(
    "Registry-limited Pilot-to-v1 crosswalks:",
    len(limited_crosswalk_rows),
)

print("\nCrosswalk statuses:")
for status, count in sorted(
    crosswalk_status_counts.items()
):
    print(f"  - {status}: {count}")

print("\nPer-pilot crosswalk:")
for row in pilot_v1_evidence_scope_rows:
    core_relations = (
        row["core_answer_relation_counts"]
    )

    print(
        f"  - Pilot {row['pilot_number']:02d} / "
        f"Seed {row['source_seed_id']:02d} | "
        f"v1_grounded="
        f"{row['answer_in_v1_evidence']} | "
        f"pilot_provenance_grounded="
        f"{row['answer_in_pilot_provenance']} | "
        f"core_records={row['core_record_count']} | "
        f"core_answer_relations="
        f"{core_relations} | "
        f"status={row['crosswalk_status']}"
    )

print(
    "\nInterpretation: v1 grounding is validated. "
    "A direct Pilot-to-v1 evidence crosswalk requires "
    "extraction from the author-review package in a "
    "later dedicated Step."
)
print(
    "\nRead-only evidence crosswalk audit:",
    True,
)
print(
    "\nNB12h PILOT-V1-CORE-46 "
    "EVIDENCE CROSSWALK AUDIT: PASS "
    "WITH DOCUMENTED LIMITATION"
)

=== NB12h PILOT-V1-CORE-46 EVIDENCE CROSSWALK AUDIT ===
Pilots analysed: 28
Historical v1 grounding failures: 0
Direct Pilot-to-v1 evidence crosswalks: 0
Registry-limited Pilot-to-v1 crosswalks: 28

Crosswalk statuses:
  - PILOT_PROVENANCE_IS_NOT_TARGET_EVIDENCE_IN_CURRENT_REGISTRY: 28

Per-pilot crosswalk:
  - Pilot 01 / Seed 01 | v1_grounded=True | pilot_provenance_grounded=False | core_records=1 | core_answer_relations={'NO_EXACT_OR_CONTAINMENT_MATCH': 1} | status=PILOT_PROVENANCE_IS_NOT_TARGET_EVIDENCE_IN_CURRENT_REGISTRY
  - Pilot 02 / Seed 02 | v1_grounded=True | pilot_provenance_grounded=False | core_records=2 | core_answer_relations={'NO_EXACT_OR_CONTAINMENT_MATCH': 2} | status=PILOT_PROVENANCE_IS_NOT_TARGET_EVIDENCE_IN_CURRENT_REGISTRY
  - Pilot 03 / Seed 03 | v1_grounded=True | pilot_provenance_grounded=False | core_records=3 | core_answer_relations={'NO_EXACT_OR_CONTAINMENT_MATCH': 3} | status=PILOT_PROVENANCE_IS_NOT_TARGET_EVIDENCE_IN_CURRENT_REGISTRY
  - Pilot 04 / Seed 04

<div dir="rtl" align="right">

## **مرحلهٔ دوازدهم — استخراج ساختاری بستهٔ بازبینی مؤلف برای ۲۸ Pilot**

تا اینجا محدودیت Crosswalk در registry ثبت شده است. اکنون منبع اصلیِ بازبینی مؤلف، یعنی فایل `author_review_28_pilots.md`، به‌صورت فقط‌خواندنی بررسی و به ۲۸ Case مجزا تقسیم می‌شود.

### **هدف این مرحله**

هدف، آماده‌کردن متن اصلی بستهٔ بازبینی مؤلف برای استخراج دقیق Evidence، سؤال نهایی، پاسخ نهایی و نظر مؤلف در Stepهای بعدی است.

### **ورودی این مرحله**

- فایل واقعی `author_review_28_pilots.md`

### **بررسی‌های این مرحله**

برای هر Case بررسی می‌شود:

- وجود عنوان `مورد X از ۲۸`

- وجود بخش متن منبع

- وجود بخش سؤال و پاسخ نهایی

- وجود بخش نظر مؤلف

- یکتا و پیوسته بودن شمارهٔ Caseها از ۱ تا ۲۸

### **قاعدهٔ مهم**

- در این Step هیچ متن، سؤال، پاسخ یا Evidence بازنویسی نمی‌شود.

- نظر مؤلف تفسیر نمی‌شود.

- فقط ساختار فایل اصلی استخراج و Validate می‌شود.

- هیچ فایل یا Datasetی تغییر نمی‌کند.

### **خروجی این مرحله**

ساختار درحافظه‌ای `author_review_case_blocks` ساخته می‌شود که متن خام هر یک از ۲۸ Case را جداگانه نگه می‌دارد.

### **Validation**

این Step زمانی PASS است که:

- دقیقاً یک فایل بستهٔ بازبینی مؤلف پیدا شود؛

- دقیقاً ۲۸ Case استخراج شود؛

- شماره‌ها از ۱ تا ۲۸ یکتا و پیوسته باشند؛

- هر Case بخش‌های اصلی مورد انتظار را داشته باشد؛

- و هیچ فایل یا Datasetی تغییر نکند.

</div>

In [14]:
# Code Cell 13
# NB12h — Author-review package structural extraction
# Read-only: no dataset, registry, or report file is modified.

assert "project_root" in globals(), (
    "Run Code Cell 02 first."
)

author_review_package_candidates = sorted(
    project_root.rglob(
        "author_review_28_pilots.md"
    )
)

assert len(author_review_package_candidates) == 1, (
    "Expected exactly one author_review_28_pilots.md; "
    f"found {len(author_review_package_candidates)}: "
    f"{author_review_package_candidates}"
)

author_review_package_path = (
    author_review_package_candidates[0]
)

author_review_package_text = (
    author_review_package_path.read_text(
        encoding="utf-8",
    )
)

persian_digit_translation = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹",
    "0123456789",
)


def parse_pilot_number(value):
    normalized = str(value).translate(
        persian_digit_translation
    )

    return int(normalized)


case_heading_pattern = re.compile(
    r"<h[1-6][^>]*>"
    r"\s*مورد\s+([0-9۰-۹]+)\s+از\s+"
    r"(?:۲۸|28)\s*"
    r"</h[1-6]>",
    flags=re.IGNORECASE,
)

case_matches = list(
    case_heading_pattern.finditer(
        author_review_package_text
    )
)

assert len(case_matches) == 28, (
    "Expected exactly 28 HTML case headings; found "
    f"{len(case_matches)}."
)

author_review_case_blocks = {}

for index, match in enumerate(case_matches):
    pilot_number = parse_pilot_number(
        match.group(1)
    )

    block_start = match.start()

    block_end = (
        case_matches[index + 1].start()
        if index + 1 < len(case_matches)
        else len(author_review_package_text)
    )

    assert pilot_number not in author_review_case_blocks, (
        "Duplicate author-review pilot number: "
        f"{pilot_number}"
    )

    author_review_case_blocks[pilot_number] = (
        author_review_package_text[
            block_start:block_end
        ]
    )

assert sorted(author_review_case_blocks) == list(
    range(1, 29)
), (
    "Author-review package must contain case numbers "
    "1–28 exactly."
)

author_review_case_structure_rows = []

for pilot_number in range(1, 29):
    block = author_review_case_blocks[pilot_number]

    has_source_section = (
        "متن منبع" in block
    )

    has_final_qa_section = (
        "سؤال و پاسخ نهایی" in block
    )

    has_author_section = (
        "نظر مؤلف" in block
    )

    author_review_case_structure_rows.append({
        "pilot_number": pilot_number,
        "character_count": len(block),
        "has_source_section": has_source_section,
        "has_final_qa_section": (
            has_final_qa_section
        ),
        "has_author_section": has_author_section,
    })

missing_structure_rows = [
    row
    for row in author_review_case_structure_rows
    if not (
        row["has_source_section"]
        and row["has_final_qa_section"]
        and row["has_author_section"]
    )
]

assert not missing_structure_rows, (
    "Required author-review sections are missing: "
    f"{missing_structure_rows}"
)

print(
    "=== NB12h AUTHOR-REVIEW PACKAGE "
    "STRUCTURAL EXTRACTION ==="
)
print(
    "Package:",
    author_review_package_path,
)
print(
    "Extracted cases:",
    len(author_review_case_blocks),
)
print(
    "Case numbers:",
    sorted(author_review_case_blocks),
)
print(
    "Cases missing required sections:",
    len(missing_structure_rows),
)

print("\nPer-case structure:")
for row in author_review_case_structure_rows:
    print(
        f"  - Pilot {row['pilot_number']:02d} | "
        f"characters={row['character_count']} | "
        f"source={row['has_source_section']} | "
        f"final_qa={row['has_final_qa_section']} | "
        f"author={row['has_author_section']}"
    )

print(
    "\nRead-only author-review extraction:",
    True,
)
print(
    "\nNB12h AUTHOR-REVIEW PACKAGE "
    "STRUCTURAL EXTRACTION: PASS"
)

=== NB12h AUTHOR-REVIEW PACKAGE STRUCTURAL EXTRACTION ===
Package: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/archive/author_review_28_pilots.md
Extracted cases: 28
Case numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
Cases missing required sections: 0

Per-case structure:
  - Pilot 01 | characters=4343 | source=True | final_qa=True | author=True
  - Pilot 02 | characters=4250 | source=True | final_qa=True | author=True
  - Pilot 03 | characters=3402 | source=True | final_qa=True | author=True
  - Pilot 04 | characters=8716 | source=True | final_qa=True | author=True
  - Pilot 05 | characters=5585 | source=True | final_qa=True | author=True
  - Pilot 06 | characters=3723 | source=True | final_qa=True | author=True
  - Pilot 07 | characters=6587 | source=True | final_qa=True | author=True
  - Pilot 08 | characters=3133 | source=True | final_qa=True | author=True
  - Pilot 09 

<div dir="rtl" align="right">

## **مرحلهٔ سیزدهم — جداسازی بخش‌های اصلی هر Case در بستهٔ بازبینی مؤلف**

ساختار ۲۸ Case در `author_review_28_pilots.md` تأیید شده است.

در این مرحله، متن خام هر Case به سه بخش اصلی تقسیم می‌شود تا در مرحله‌های بعدی فقط از متن اصلی بستهٔ مؤلف برای استخراج Evidence، سؤال نهایی، پاسخ نهایی و نظر مؤلف استفاده شود.

### **بخش‌های استخراجی**

برای هر Pilot بخش‌های زیر جدا می‌شوند:

- متن منبع

- روند سؤال و پاسخ

- سؤال و پاسخ نهایی

- نظر مؤلف

### **قاعدهٔ مهم**

- متن‌ها فقط از فایل اصلی بازبینی مؤلف استخراج می‌شوند.

- هیچ متن یا HTMLی بازنویسی نمی‌شود.

- هیچ سؤال، پاسخ یا Evidence جدیدی ساخته نمی‌شود.

- اگر یکی از بخش‌های اصلی قابل جداسازی نباشد، Step متوقف می‌شود.

### **خروجی این مرحله**

ساختار درحافظه‌ای `author_review_extracted_sections` ساخته می‌شود که بخش‌های خام هر ۲۸ Pilot را نگه می‌دارد.

### **Validation**

این Step زمانی PASS است که:

- برای هر ۲۸ Pilot، هر چهار بخش استخراج شود؛

- هیچ بخش استخراجی خالی نباشد؛

- شمارهٔ Pilotها از ۱ تا ۲۸ پیوسته باقی بماند؛

- و هیچ فایل یا Datasetی تغییر نکند.

</div>

In [15]:
# Code Cell 14
# NB12h — Author-review case section extraction
# Read-only: no dataset, registry, or report file is modified.

assert "author_review_case_blocks" in globals(), (
    "Run Code Cell 13 first."
)

assert len(author_review_case_blocks) == 28


def strip_html_tags(value):
    text = re.sub(
        r"<[^>]+>",
        " ",
        str(value or ""),
        flags=re.DOTALL,
    )

    return " ".join(text.split()).strip()


def extract_html_headings(block):
    heading_pattern = re.compile(
        r"<h([1-6])[^>]*>(.*?)</h\1>",
        flags=re.IGNORECASE | re.DOTALL,
    )

    headings = []

    for match in heading_pattern.finditer(block):
        headings.append({
            "start": match.start(),
            "end": match.end(),
            "level": int(match.group(1)),
            "text": strip_html_tags(
                match.group(2)
            ),
        })

    return headings


def find_heading_index(
    headings,
    required_text,
    start_index=0,
):
    for index in range(
        start_index,
        len(headings),
    ):
        if required_text in headings[index]["text"]:
            return index

    raise AssertionError(
        "Could not find expected heading: "
        f"{required_text!r}"
    )


def section_between(
    block,
    headings,
    start_index,
    end_index,
):
    section_start = headings[start_index]["end"]

    section_end = (
        headings[end_index]["start"]
        if end_index is not None
        else len(block)
    )

    section = block[
        section_start:section_end
    ].strip()

    assert strip_html_tags(section), (
        "Extracted section is empty for heading: "
        f"{headings[start_index]['text']!r}"
    )

    return section


author_review_extracted_sections = {}

for pilot_number in range(1, 29):
    block = author_review_case_blocks[pilot_number]

    headings = extract_html_headings(block)

    assert headings, (
        "No HTML headings found for "
        f"Pilot {pilot_number}."
    )

    source_index = find_heading_index(
        headings,
        "متن منبع",
    )

    qa_process_index = find_heading_index(
        headings,
        "سؤال‌ها و پاسخ‌ها",
        start_index=source_index + 1,
    )

    final_qa_index = find_heading_index(
        headings,
        "سؤال و پاسخ نهایی",
        start_index=qa_process_index + 1,
    )

    author_index = find_heading_index(
        headings,
        "نظر مؤلف",
        start_index=final_qa_index + 1,
    )

    source_section = section_between(
        block,
        headings,
        source_index,
        qa_process_index,
    )

    qa_process_section = section_between(
        block,
        headings,
        qa_process_index,
        final_qa_index,
    )

    final_qa_section = section_between(
        block,
        headings,
        final_qa_index,
        author_index,
    )

    author_section = section_between(
        block,
        headings,
        author_index,
        None,
    )

    author_review_extracted_sections[
        pilot_number
    ] = {
        "pilot_number": pilot_number,
        "source_section_html": source_section,
        "qa_process_section_html": (
            qa_process_section
        ),
        "final_qa_section_html": final_qa_section,
        "author_section_html": author_section,
        "source_section_text": strip_html_tags(
            source_section
        ),
        "qa_process_section_text": strip_html_tags(
            qa_process_section
        ),
        "final_qa_section_text": strip_html_tags(
            final_qa_section
        ),
        "author_section_text": strip_html_tags(
            author_section
        ),
        "heading_count": len(headings),
    }

assert sorted(
    author_review_extracted_sections
) == list(range(1, 29))

assert all(
    row["source_section_text"]
    and row["qa_process_section_text"]
    and row["final_qa_section_text"]
    and row["author_section_text"]
    for row in author_review_extracted_sections.values()
), (
    "At least one extracted author-review section "
    "is empty."
)

print(
    "=== NB12h AUTHOR-REVIEW "
    "SECTION EXTRACTION ==="
)
print(
    "Extracted pilots:",
    len(author_review_extracted_sections),
)

print("\nPer-pilot section sizes:")
for pilot_number in range(1, 29):
    row = author_review_extracted_sections[
        pilot_number
    ]

    print(
        f"  - Pilot {pilot_number:02d} | "
        f"source={len(row['source_section_text'])} | "
        f"qa_process="
        f"{len(row['qa_process_section_text'])} | "
        f"final_qa="
        f"{len(row['final_qa_section_text'])} | "
        f"author={len(row['author_section_text'])} | "
        f"headings={row['heading_count']}"
    )

print(
    "\nRead-only author-review section extraction:",
    True,
)
print(
    "\nNB12h AUTHOR-REVIEW "
    "SECTION EXTRACTION: PASS"
)

=== NB12h AUTHOR-REVIEW SECTION EXTRACTION ===
Extracted pilots: 28

Per-pilot section sizes:
  - Pilot 01 | source=278 | qa_process=1521 | final_qa=563 | author=85 | headings=6
  - Pilot 02 | source=671 | qa_process=1169 | final_qa=609 | author=85 | headings=6
  - Pilot 03 | source=335 | qa_process=787 | final_qa=479 | author=85 | headings=6
  - Pilot 04 | source=3034 | qa_process=2270 | final_qa=723 | author=85 | headings=10
  - Pilot 05 | source=804 | qa_process=1546 | final_qa=443 | author=85 | headings=10
  - Pilot 06 | source=677 | qa_process=762 | final_qa=453 | author=85 | headings=6
  - Pilot 07 | source=2364 | qa_process=731 | final_qa=422 | author=85 | headings=6
  - Pilot 08 | source=237 | qa_process=688 | final_qa=379 | author=85 | headings=6
  - Pilot 09 | source=786 | qa_process=1449 | final_qa=494 | author=85 | headings=10
  - Pilot 10 | source=1289 | qa_process=1509 | final_qa=332 | author=85 | headings=10
  - Pilot 11 | source=413 | qa_process=779 | final_qa=470 | aut

<div dir="rtl" align="right">

## **مرحلهٔ چهاردهم — کشف ساختار واقعی سؤال و پاسخ نهایی در بستهٔ بازبینی مؤلف**

بخش «سؤال و پاسخ نهایی» برای هر ۲۸ Pilot با موفقیت جدا شده است.

در این مرحله، ساختار HTML و برچسب‌های واقعی این بخش بررسی می‌شود تا استخراج سؤال، پاسخ و Evidence نهایی در Step بعدی بر اساس ساختار واقعی فایل انجام شود.

### **ورودی‌های این مرحله**

- `author_review_extracted_sections`

### **بررسی‌های این مرحله**

برای هر Pilot بررسی می‌شود:

- تعداد سلول‌های HTML جدول

- متن برچسب‌های موجود در بخش سؤال و پاسخ نهایی

- وجود برچسب‌های مربوط به سؤال، پاسخ، Evidence و تصمیم

- پیش‌نمایش متنی محدود از بخش نهایی، فقط برای بررسی ساختار

### **قاعدهٔ مهم**

- در این Step هیچ سؤال، پاسخ یا Evidence استخراج نهایی نمی‌شود.

- هیچ متن یا HTMLی بازنویسی نمی‌شود.

- ساختار هر Case جداگانه بررسی می‌شود.

- اگر ساختار بین Caseها متفاوت باشد، اختلاف ثبت می‌شود تا Parser مرحلهٔ بعد با همان تفاوت سازگار باشد.

### **خروجی این مرحله**

ساختار درحافظه‌ای `final_qa_markup_inventory` ساخته می‌شود که schema واقعی بخش سؤال و پاسخ نهایی را برای هر ۲۸ Pilot ثبت می‌کند.

### **Validation**

این Step زمانی PASS است که:

- هر ۲۸ بخش Final QA بررسی شوند؛

- تعداد و محتوای سلول‌های HTML ثبت شود؛

- تفاوت ساختاری میان Caseها مشخص شود؛

- و هیچ فایل یا Datasetی تغییر نکند.

</div>

In [16]:
# Code Cell 15
# NB12h — Final-QA HTML markup inventory
# Read-only: no dataset, registry, or report file is modified.

assert "author_review_extracted_sections" in globals(), (
    "Run Code Cell 14 first."
)

assert len(author_review_extracted_sections) == 28


def extract_html_tag_texts(
    html_text,
    tag_name,
):
    pattern = re.compile(
        rf"<{tag_name}[^>]*>(.*?)</{tag_name}>",
        flags=re.IGNORECASE | re.DOTALL,
    )

    return [
        strip_html_tags(match.group(1))
        for match in pattern.finditer(html_text)
        if strip_html_tags(match.group(1))
    ]


field_marker_terms = [
    "سؤال",
    "پاسخ",
    "شاهد",
    "منبع",
    "evidence",
    "question",
    "answer",
    "decision",
    "تصمیم",
]

final_qa_markup_inventory = []

for pilot_number in range(1, 29):
    final_qa_html = (
        author_review_extracted_sections[
            pilot_number
        ]["final_qa_section_html"]
    )

    heading_texts = extract_html_tag_texts(
        final_qa_html,
        "h[1-6]",
    )

    table_header_texts = extract_html_tag_texts(
        final_qa_html,
        "th",
    )

    table_data_texts = extract_html_tag_texts(
        final_qa_html,
        "td",
    )

    all_markup_texts = (
        heading_texts
        + table_header_texts
        + table_data_texts
    )

    marker_texts = [
        text
        for text in all_markup_texts
        if any(
            term.lower() in text.lower()
            for term in field_marker_terms
        )
    ]

    text_preview = (
        author_review_extracted_sections[
            pilot_number
        ]["final_qa_section_text"][:240]
    )

    final_qa_markup_inventory.append({
        "pilot_number": pilot_number,
        "heading_texts": heading_texts,
        "table_header_texts": table_header_texts,
        "table_data_cell_count": len(
            table_data_texts
        ),
        "field_marker_texts": marker_texts,
        "text_preview": text_preview,
    })

assert len(final_qa_markup_inventory) == 28

table_cell_distribution = Counter(
    row["table_data_cell_count"]
    for row in final_qa_markup_inventory
)

print(
    "=== NB12h FINAL-QA MARKUP INVENTORY ==="
)
print(
    "Pilots inspected:",
    len(final_qa_markup_inventory),
)
print(
    "Table-cell-count distribution:",
    dict(sorted(table_cell_distribution.items())),
)

print("\nPer-pilot Final-QA structure:")
for row in final_qa_markup_inventory:
    marker_preview = row[
        "field_marker_texts"
    ][:8]

    print(
        f"  - Pilot {row['pilot_number']:02d} | "
        f"headings={len(row['heading_texts'])} | "
        f"table_cells="
        f"{row['table_data_cell_count']} | "
        f"markers={marker_preview}"
    )

print(
    "\nRead-only Final-QA markup inventory:",
    True,
)
print(
    "\nNB12h FINAL-QA MARKUP INVENTORY: PASS"
)

=== NB12h FINAL-QA MARKUP INVENTORY ===
Pilots inspected: 28
Table-cell-count distribution: {6: 28}

Per-pilot Final-QA structure:
  - Pilot 01 | headings=0 | table_cells=6 | markers=['منبع', 'همان منبع و شواهد پذیرفته\u200cشده در بخش ۱ بالا.', 'متن سؤال نهایی', 'چه مشکلی پیش می\u200cآید اگر بر اساس شناختی که فقط از یک بخش داریم، درباره کل موضوع تصمیم بگیریم؟', 'متن پاسخ نهایی', 'مشکل این است که ممکن است همان شناخت ناقص را به کل موضوع تعمیم بدهیم و بخواهیم بر اساس آن تصمیم بگیریم یا از کل موضوع استفاده کنیم. چنین کاری شاید در ظاهر مناسب به نظر برسد، اما بخش\u200cهای یک موضوع می\u200cتوانند به هم وابسته باشند و نمی\u200cشود یک بخش را کاملاً جدا از بقیه در نظر گرفت. در مثال متن نیز فیل یک کل به\u200cهم\u200cپیوسته است و شناخت یک قسمت از آن، به\u200cتنهایی تصویر کاملی از فیل نمی\u200cدهد.']
  - Pilot 02 | headings=0 | table_cells=6 | markers=['منبع', 'همان منبع و شواهد پذیرفته\u200cشده در بخش ۱ بالا.', 'متن سؤال نهایی', 'متن پاسخ نهایی']
  - Pilot 03 | headings=0 | table_cells=6 | markers

<div dir="rtl" align="right">

## **مرحلهٔ پانزدهم — استخراج سؤال و پاسخ نهایی واقعی از بستهٔ بازبینی مؤلف**

Step قبل ساختار یکنواخت بخش Final QA را برای هر ۲۸ Pilot تأیید کرد.

در این مرحله، سؤال نهایی و پاسخ نهایی هر Case مستقیماً از جدول Final QA استخراج می‌شود.

### **ساختار تأییدشده**

هر Final QA دقیقاً شامل شش سلول است:

- برچسب منبع

- مقدار منبع

- برچسب سؤال نهایی

- متن سؤال نهایی

- برچسب پاسخ نهایی

- متن پاسخ نهایی

### **بررسی‌های این مرحله**

برای هر Pilot استخراج و ثبت می‌شود:

- مرجع منبع نهایی

- سؤال نهایی بستهٔ مؤلف

- پاسخ نهایی بستهٔ مؤلف

- رابطهٔ متنی سؤال و پاسخ مؤلف با سؤال و پاسخ فعلی registry

### **قاعدهٔ مهم**

- سؤال و پاسخ فقط از جدول Final QA استخراج می‌شوند.

- متن registry جایگزین متن مؤلف نمی‌شود.

- اختلاف میان registry و بستهٔ مؤلف فقط Finding است.

- هیچ Dataset یا فایل موجود تغییر نمی‌کند.

### **خروجی این مرحله**

ساختار درحافظه‌ای `author_review_final_qa_rows` ساخته می‌شود که سؤال، پاسخ و مرجع نهایی هر ۲۸ Pilot را ثبت می‌کند.

### **Validation**

این Step زمانی PASS است که:

- دقیقاً ۲۸ سؤال و ۲۸ پاسخ نهایی استخراج شوند؛

- برچسب‌های جدول در همهٔ Caseها معتبر باشند؛

- هیچ سؤال یا پاسخ استخراجی خالی نباشد؛

- و هیچ فایل یا Datasetی تغییر نکند.

</div>

In [17]:
# Code Cell 16
# NB12h — Extract final QA from the author-review package
# Read-only: no dataset, registry, or report file is modified.

assert "author_review_extracted_sections" in globals(), (
    "Run Code Cell 14 first."
)

assert "canonical_pilot_registry" in globals(), (
    "Run Code Cell 07 first."
)

assert len(author_review_extracted_sections) == 28
assert len(canonical_pilot_registry) == 28

canonical_pilot_by_number = {
    row["pilot_number"]: row
    for row in canonical_pilot_registry
}

author_review_final_qa_rows = []

for pilot_number in range(1, 29):
    final_qa_html = (
        author_review_extracted_sections[
            pilot_number
        ]["final_qa_section_html"]
    )

    table_cells = extract_html_tag_texts(
        final_qa_html,
        "td",
    )

    assert len(table_cells) == 6, (
        "Expected exactly six Final-QA table cells "
        f"for Pilot {pilot_number}; found "
        f"{len(table_cells)}."
    )

    source_label = table_cells[0]
    source_reference = table_cells[1]
    question_label = table_cells[2]
    final_question = table_cells[3]
    answer_label = table_cells[4]
    final_answer = table_cells[5]

    assert "منبع" in source_label, (
        "Invalid source label for "
        f"Pilot {pilot_number}: {source_label!r}"
    )

    assert "سؤال" in question_label, (
        "Invalid final-question label for "
        f"Pilot {pilot_number}: {question_label!r}"
    )

    assert "پاسخ" in answer_label, (
        "Invalid final-answer label for "
        f"Pilot {pilot_number}: {answer_label!r}"
    )

    final_question = require_text(
        final_question,
        f"author final question for Pilot {pilot_number}",
    )

    final_answer = require_text(
        final_answer,
        f"author final answer for Pilot {pilot_number}",
    )

    canonical_pilot = canonical_pilot_by_number[
        pilot_number
    ]

    question_relation_to_registry = text_relation(
        final_question,
        canonical_pilot["question"],
    )

    answer_relation_to_registry = text_relation(
        final_answer,
        canonical_pilot["expected_answer"],
    )

    author_review_final_qa_rows.append({
        "pilot_number": pilot_number,
        "pilot_case_id": canonical_pilot[
            "pilot_case_id"
        ],
        "source_reference": source_reference,
        "final_question": final_question,
        "final_answer": final_answer,
        "question_relation_to_registry": (
            question_relation_to_registry
        ),
        "answer_relation_to_registry": (
            answer_relation_to_registry
        ),
    })

assert len(author_review_final_qa_rows) == 28

assert all(
    row["final_question"]
    and row["final_answer"]
    for row in author_review_final_qa_rows
)

question_relation_counts = Counter(
    row["question_relation_to_registry"]
    for row in author_review_final_qa_rows
)

answer_relation_counts = Counter(
    row["answer_relation_to_registry"]
    for row in author_review_final_qa_rows
)

print(
    "=== NB12h AUTHOR-REVIEW "
    "FINAL-QA EXTRACTION ==="
)
print(
    "Final-QA rows extracted:",
    len(author_review_final_qa_rows),
)

print("\nQuestion relation to current registry:")
for relation, count in sorted(
    question_relation_counts.items()
):
    print(f"  - {relation}: {count}")

print("\nAnswer relation to current registry:")
for relation, count in sorted(
    answer_relation_counts.items()
):
    print(f"  - {relation}: {count}")

print("\nPer-pilot Final QA:")
for row in author_review_final_qa_rows:
    print(
        f"  - Pilot {row['pilot_number']:02d} | "
        f"question_relation="
        f"{row['question_relation_to_registry']} | "
        f"answer_relation="
        f"{row['answer_relation_to_registry']} | "
        f"source_reference="
        f"{row['source_reference']}"
    )

print(
    "\nRead-only Final-QA extraction:",
    True,
)
print(
    "\nNB12h AUTHOR-REVIEW "
    "FINAL-QA EXTRACTION: PASS"
)

=== NB12h AUTHOR-REVIEW FINAL-QA EXTRACTION ===
Final-QA rows extracted: 28

Question relation to current registry:
  - EXACT: 28

Answer relation to current registry:
  - EXACT: 28

Per-pilot Final QA:
  - Pilot 01 | question_relation=EXACT | answer_relation=EXACT | source_reference=همان منبع و شواهد پذیرفته‌شده در بخش ۱ بالا.
  - Pilot 02 | question_relation=EXACT | answer_relation=EXACT | source_reference=همان منبع و شواهد پذیرفته‌شده در بخش ۱ بالا.
  - Pilot 03 | question_relation=EXACT | answer_relation=EXACT | source_reference=همان منبع و شواهد پذیرفته‌شده در بخش ۱ بالا.
  - Pilot 04 | question_relation=EXACT | answer_relation=EXACT | source_reference=منبع بازبینی‌شده / جایگزینِ پذیرفته‌شده در بخش ۱ بالا.
  - Pilot 05 | question_relation=EXACT | answer_relation=EXACT | source_reference=منبع بازبینی‌شده / جایگزینِ پذیرفته‌شده در بخش ۱ بالا.
  - Pilot 06 | question_relation=EXACT | answer_relation=EXACT | source_reference=همان منبع و شواهد پذیرفته‌شده در بخش ۱ بالا.
  - Pilot 07 | 

<div dir="rtl" align="right">

## **مرحلهٔ شانزدهم — بررسی وضعیت واقعی نظر مؤلف برای ۲۸ Pilot**

سؤال و پاسخ نهایی ثبت‌شده در بستهٔ بازبینی مؤلف با registry فعلی برای هر ۲۸ Pilot تطابق دقیق دارد.

در این مرحله، بخش «نظر مؤلف» بررسی می‌شود تا مشخص شود آیا دکتر برای هر Case تصمیم نهایی ثبت کرده است یا بسته هنوز در وضعیت انتظار بازخورد قرار دارد.

### **ورودی‌های این مرحله**

- `author_review_extracted_sections`

- `author_review_final_qa_rows`

- وضعیت `author_review` در human-review registry

### **بررسی‌های این مرحله**

برای هر Pilot بررسی می‌شود:

- متن نظر مؤلف

- تعداد متن‌های متمایز نظر مؤلف

- وجود یا نبود تصمیم صریح

- سازگاری متن بسته با وضعیت ثبت‌شده در registry

### **قاعدهٔ مهم**

- نبود نظر مؤلف Failure داده نیست.

- نبود نظر مؤلف به‌معنای مجاز نبودن ساخت v2 از این بسته است.

- هیچ تفسیر محتوایی از نظر مؤلف انجام نمی‌شود.

- هیچ فایل، Registry یا Datasetی تغییر نمی‌کند.

### **خروجی این مرحله**

ساختار درحافظه‌ای `author_feedback_status_rows` ساخته می‌شود که وضعیت واقعی بازخورد مؤلف را برای هر ۲۸ Pilot ثبت می‌کند.

### **Validation**

این Step زمانی PASS است که:

- هر ۲۸ بخش نظر مؤلف بررسی شوند؛

- وضعیت registry با محتوای واقعی بسته مقایسه شود؛

- آماده‌بودن یا آماده‌نبودن مرحلهٔ بازخورد بدون حدس ثبت شود؛

- و هیچ فایل یا Datasetی تغییر نکند.

</div>

In [18]:
# Code Cell 17
# NB12h — Author-feedback readiness audit
# Read-only: no dataset, registry, or report file is modified.

assert "author_review_extracted_sections" in globals(), (
    "Run Code Cell 14 first."
)

assert "author_review_final_qa_rows" in globals(), (
    "Run Code Cell 16 first."
)

assert len(author_review_extracted_sections) == 28
assert len(author_review_final_qa_rows) == 28

author_feedback_status_rows = []

for pilot_number in range(1, 29):
    author_text = (
        author_review_extracted_sections[
            pilot_number
        ]["author_section_text"]
    )

    normalized_author_text = canon_for_audit(
        author_text
    )

    author_feedback_status_rows.append({
        "pilot_number": pilot_number,
        "author_text": author_text,
        "normalized_author_text": (
            normalized_author_text
        ),
    })

assert len(author_feedback_status_rows) == 28

author_text_groups = defaultdict(list)

for row in author_feedback_status_rows:
    author_text_groups[
        row["normalized_author_text"]
    ].append(row["pilot_number"])

registry_author_review = (
    human_review_registry.get("author_review") or {}
)

registry_author_status = registry_author_review.get(
    "status"
)

registry_final_review_status = (
    registry_author_review.get(
        "final_review_status"
    )
)

print("=== NB12h AUTHOR-FEEDBACK READINESS AUDIT ===")
print(
    "Pilots inspected:",
    len(author_feedback_status_rows),
)
print(
    "Distinct author-section texts:",
    len(author_text_groups),
)
print(
    "Registry author-review status:",
    registry_author_status,
)
print(
    "Registry final-review status:",
    registry_final_review_status,
)

print("\nAuthor-section text groups:")
for position, (
    normalized_text_value,
    pilot_numbers,
) in enumerate(
    sorted(
        author_text_groups.items(),
        key=lambda item: min(item[1]),
    ),
    1,
):
    preview = normalized_text_value[:260]

    print(
        f"  - Group {position} | "
        f"pilots={pilot_numbers} | "
        f"text={preview!r}"
    )

all_author_sections_identical = (
    len(author_text_groups) == 1
)

print(
    "\nAll author sections identical:",
    all_author_sections_identical,
)
print(
    "Author feedback can be treated as final:",
    False,
)
print(
    "\nInterpretation: final-author readiness is "
    "determined by explicit author content and the "
    "registry status, not by the existence of the "
    "review-package template."
)
print(
    "\nRead-only author-feedback readiness audit:",
    True,
)
print(
    "\nNB12h AUTHOR-FEEDBACK READINESS AUDIT: PASS"
)

=== NB12h AUTHOR-FEEDBACK READINESS AUDIT ===
Pilots inspected: 28
Distinct author-section texts: 3
Registry author-review status: NOT_STARTED
Registry final-review status: NOT_STARTED

Author-section text groups:
  - Group 1 | pilots=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27] | text='تصمیم دکتر توضیحات دکتر ☐ تأیید ☐ نیاز به اصلاح ☐ رد ☐ باید دوباره سؤال ساخته شود ---'
  - Group 2 | pilots=[22] | text='تصمیم دکتر توضیحات دکتر ☐ تأیید ☐ نیاز به اصلاح ☐ رد ☐ باید دوباره سؤال ساخته شود --- ---'
  - Group 3 | pilots=[28] | text='تصمیم دکتر توضیحات دکتر ☐ تأیید ☐ نیاز به اصلاح ☐ رد ☐ باید دوباره سؤال ساخته شود'

All author sections identical: False
Author feedback can be treated as final: False

Interpretation: final-author readiness is determined by explicit author content and the registry status, not by the existence of the review-package template.

Read-only author-feedback readiness audit: True

NB12h AUTHOR-FEEDBACK READINESS AUDIT:

<div dir="rtl" align="right">

## **مرحلهٔ هفدهم — بررسی پاسخ‌پشتیبانی در متن منبع بستهٔ بازبینی مؤلف**

بخش نظر مؤلف هنوز خالی است، اما بخش «متن منبع» برای هر ۲۸ Pilot در بستهٔ بازبینی مؤلف وجود دارد.

در این مرحله بررسی می‌شود که آیا پاسخ نهایی هر Pilot در متن منبع همان Case وجود دارد یا نه.

### **ورودی‌های این مرحله**

- `author_review_extracted_sections`

- `author_review_final_qa_rows`

- شاهد تاریخی متناظر در Golden Dataset v1

### **بررسی‌های این مرحله**

برای هر Pilot بررسی می‌شود:

- وجود پاسخ نهایی در متن منبع بستهٔ مؤلف

- وجود پاسخ نهایی در شاهد تاریخی v1

- شباهت واژگانی میان متن منبع مؤلف و شاهد v1

- وضعیت اولیهٔ Crosswalk میان بستهٔ مؤلف و v1

### **قاعدهٔ مهم**

- متن منبعِ کامل یک Case ممکن است شامل تاریخچه، Evidence اولیه و Evidence بازبینی‌شده باشد.

- نبود پاسخ در متن کامل منبع، Finding برای بررسی بعدی است؛ نه Failure خودکار.

- این Step هنوز Target Evidence دقیق را از متن منبع جدا نمی‌کند.

- هیچ فایل یا Datasetی تغییر نمی‌کند.

### **خروجی این مرحله**

ساختار درحافظه‌ای `author_package_evidence_crosswalk_rows` ساخته می‌شود که وضعیت پاسخ‌پشتیبانی و هم‌پوشانی اولیهٔ Evidence را برای هر ۲۸ Pilot ثبت می‌کند.

### **Validation**

این Step زمانی PASS است که:

- هر ۲۸ Pilot بررسی شوند؛

- وجود پاسخ در متن منبع مؤلف و شاهد v1 ثبت شود؛

- شاخص هم‌پوشانی برای هر Pilot محاسبه شود؛

- و هیچ فایل یا Datasetی تغییر نکند.

</div>

In [19]:
# Code Cell 18
# NB12h — Author-package source-section answer-grounding audit
# Read-only: no dataset, registry, or report file is modified.

assert "author_review_extracted_sections" in globals(), (
    "Run Code Cell 14 first."
)

assert "author_review_final_qa_rows" in globals(), (
    "Run Code Cell 16 first."
)

assert "pilot_v1_fidelity_rows" in globals(), (
    "Run Code Cell 11 first."
)

assert len(author_review_final_qa_rows) == 28
assert len(pilot_v1_fidelity_rows) == 28

final_qa_by_pilot_number = {
    row["pilot_number"]: row
    for row in author_review_final_qa_rows
}

fidelity_by_pilot_number = {
    row["pilot_number"]: row
    for row in pilot_v1_fidelity_rows
}

author_package_evidence_crosswalk_rows = []

for pilot_number in range(1, 29):
    source_section_text = (
        author_review_extracted_sections[
            pilot_number
        ]["source_section_text"]
    )

    final_qa = final_qa_by_pilot_number[
        pilot_number
    ]

    fidelity_row = fidelity_by_pilot_number[
        pilot_number
    ]

    final_answer = final_qa["final_answer"]
    v1_evidence = fidelity_row["v1_evidence"]

    normalized_answer = canon_for_audit(
        final_answer
    )

    answer_in_author_source_section = (
        normalized_answer
        in canon_for_audit(source_section_text)
    )

    answer_in_v1_evidence = (
        normalized_answer
        in canon_for_audit(v1_evidence)
    )

    source_to_v1_similarity = jaccard_similarity(
        source_section_text,
        v1_evidence,
    )

    source_to_v1_relation = text_relation(
        source_section_text,
        v1_evidence,
    )

    if answer_in_author_source_section:
        author_source_grounding_status = (
            "ANSWER_PRESENT_IN_AUTHOR_SOURCE_SECTION"
        )
    else:
        author_source_grounding_status = (
            "ANSWER_NOT_PRESENT_IN_AUTHOR_"
            "SOURCE_SECTION_REVIEW_REQUIRED"
        )

    if (
        answer_in_author_source_section
        and answer_in_v1_evidence
    ):
        crosswalk_status = (
            "ANSWER_GROUNDED_IN_BOTH_ARTIFACTS"
        )
    elif answer_in_v1_evidence:
        crosswalk_status = (
            "GROUNDED_IN_V1_ONLY_"
            "AUTHOR_SOURCE_REVIEW_REQUIRED"
        )
    else:
        crosswalk_status = (
            "V1_GROUNDING_REVIEW_REQUIRED"
        )

    author_package_evidence_crosswalk_rows.append({
        "pilot_number": pilot_number,
        "source_seed_id": fidelity_row[
            "source_seed_id"
        ],
        "answer_in_author_source_section": (
            answer_in_author_source_section
        ),
        "answer_in_v1_evidence": (
            answer_in_v1_evidence
        ),
        "author_source_to_v1_jaccard": round(
            source_to_v1_similarity,
            4,
        ),
        "author_source_to_v1_relation": (
            source_to_v1_relation
        ),
        "author_source_grounding_status": (
            author_source_grounding_status
        ),
        "crosswalk_status": crosswalk_status,
    })

assert len(author_package_evidence_crosswalk_rows) == 28

author_source_status_counts = Counter(
    row["author_source_grounding_status"]
    for row in author_package_evidence_crosswalk_rows
)

crosswalk_status_counts = Counter(
    row["crosswalk_status"]
    for row in author_package_evidence_crosswalk_rows
)

author_source_review_rows = [
    row
    for row in author_package_evidence_crosswalk_rows
    if not row[
        "answer_in_author_source_section"
    ]
]

v1_grounding_failures = [
    row
    for row in author_package_evidence_crosswalk_rows
    if not row["answer_in_v1_evidence"]
]

assert not v1_grounding_failures, (
    "Unexpected v1 grounding failures detected."
)

print(
    "=== NB12h AUTHOR-PACKAGE-TO-V1 "
    "EVIDENCE CROSSWALK AUDIT ==="
)
print(
    "Pilots analysed:",
    len(author_package_evidence_crosswalk_rows),
)
print(
    "Answer present in author source section:",
    len(
        author_package_evidence_crosswalk_rows
    ) - len(author_source_review_rows),
)
print(
    "Answer absent from author source section:",
    len(author_source_review_rows),
)
print(
    "Answer absent from v1 evidence:",
    len(v1_grounding_failures),
)

print("\nAuthor-source grounding statuses:")
for status, count in sorted(
    author_source_status_counts.items()
):
    print(f"  - {status}: {count}")

print("\nCrosswalk statuses:")
for status, count in sorted(
    crosswalk_status_counts.items()
):
    print(f"  - {status}: {count}")

print("\nPer-pilot crosswalk:")
for row in author_package_evidence_crosswalk_rows:
    print(
        f"  - Pilot {row['pilot_number']:02d} / "
        f"Seed {row['source_seed_id']:02d} | "
        f"author_grounded="
        f"{row['answer_in_author_source_section']} | "
        f"v1_grounded="
        f"{row['answer_in_v1_evidence']} | "
        f"jaccard="
        f"{row['author_source_to_v1_jaccard']:.4f} | "
        f"relation="
        f"{row['author_source_to_v1_relation']} | "
        f"status={row['crosswalk_status']}"
    )

print(
    "\nRead-only author-package evidence crosswalk:",
    True,
)
print(
    "\nNB12h AUTHOR-PACKAGE-TO-V1 "
    "EVIDENCE CROSSWALK AUDIT: PASS"
)

=== NB12h AUTHOR-PACKAGE-TO-V1 EVIDENCE CROSSWALK AUDIT ===
Pilots analysed: 28
Answer present in author source section: 0
Answer absent from author source section: 28
Answer absent from v1 evidence: 0

Author-source grounding statuses:
  - ANSWER_NOT_PRESENT_IN_AUTHOR_SOURCE_SECTION_REVIEW_REQUIRED: 28

Crosswalk statuses:
  - GROUNDED_IN_V1_ONLY_AUTHOR_SOURCE_REVIEW_REQUIRED: 28

Per-pilot crosswalk:
  - Pilot 01 / Seed 01 | author_grounded=False | v1_grounded=True | jaccard=0.2875 | relation=NO_EXACT_OR_CONTAINMENT_MATCH | status=GROUNDED_IN_V1_ONLY_AUTHOR_SOURCE_REVIEW_REQUIRED
  - Pilot 02 / Seed 02 | author_grounded=False | v1_grounded=True | jaccard=0.2906 | relation=NO_EXACT_OR_CONTAINMENT_MATCH | status=GROUNDED_IN_V1_ONLY_AUTHOR_SOURCE_REVIEW_REQUIRED
  - Pilot 03 / Seed 03 | author_grounded=False | v1_grounded=True | jaccard=0.3611 | relation=NO_EXACT_OR_CONTAINMENT_MATCH | status=GROUNDED_IN_V1_ONLY_AUTHOR_SOURCE_REVIEW_REQUIRED
  - Pilot 04 / Seed 04 | author_grounded=Fals

<div dir="rtl" align="right">

## **مرحلهٔ هجدهم — ساخت صف داوری معنایی برای ۲۸ Pilot**

بررسی‌های واژگانی انجام شده‌اند، اما تطابق واژگانی دقیق معیار کافی برای سنجش پاسخ‌پشتیبانی نیست.

در این مرحله، متن منبع، سؤال نهایی و پاسخ نهایی هر ۲۸ Pilot در یک ساختار استاندارد برای داوری معنایی آماده می‌شود.

### **ورودی‌های این مرحله**

- `author_review_extracted_sections`

- `author_review_final_qa_rows`

- `canonical_pilot_registry`

### **ساختار هر مورد داوری**

هر مورد شامل موارد زیر است:

- شماره Pilot

- `pilot_case_id`

- متن منبع کامل از بستهٔ مؤلف

- سؤال نهایی

- پاسخ نهایی

- منشأ Pilot و authority

- وضعیت نهایی نظر مؤلف

### **قاعدهٔ مهم**

- این Step فقط صف داوری می‌سازد.

- هنوز هیچ مدل، API یا هزینه‌ای استفاده نمی‌شود.

- این Step هیچ تصمیم PASS، REVIEW یا REJECT صادر نمی‌کند.

- هیچ فایل یا Datasetی تغییر نمی‌کند.

### **خروجی این مرحله**

ساختار درحافظه‌ای `author_package_semantic_review_queue` ساخته می‌شود که ورودی مستقیم داوری معنایی کنترل‌شده در Step بعدی است.

### **Validation**

این Step زمانی PASS است که:

- دقیقاً ۲۸ مورد داوری ساخته شود؛

- هر مورد دارای منبع، سؤال و پاسخ غیرخالی باشد؛

- شماره‌ها از ۱ تا ۲۸ پیوسته باشند؛

- و هیچ فایل یا Datasetی تغییر نکند.

</div>

In [20]:
# Code Cell 19
# NB12h — Author-package semantic-review queue construction
# Read-only: no dataset, registry, or report file is modified.

assert "author_review_extracted_sections" in globals(), (
    "Run Code Cell 14 first."
)

assert "author_review_final_qa_rows" in globals(), (
    "Run Code Cell 16 first."
)

assert "canonical_pilot_registry" in globals(), (
    "Run Code Cell 07 first."
)

final_qa_by_pilot_number = {
    row["pilot_number"]: row
    for row in author_review_final_qa_rows
}

canonical_pilot_by_number = {
    row["pilot_number"]: row
    for row in canonical_pilot_registry
}

author_package_semantic_review_queue = []

for pilot_number in range(1, 29):
    source_text = (
        author_review_extracted_sections[
            pilot_number
        ]["source_section_text"]
    )

    final_qa = final_qa_by_pilot_number[
        pilot_number
    ]

    canonical_pilot = canonical_pilot_by_number[
        pilot_number
    ]

    question = require_text(
        final_qa["final_question"],
        f"semantic-review question for Pilot {pilot_number}",
    )

    answer = require_text(
        final_qa["final_answer"],
        f"semantic-review answer for Pilot {pilot_number}",
    )

    source_text = require_text(
        source_text,
        f"semantic-review source for Pilot {pilot_number}",
    )

    author_package_semantic_review_queue.append({
        "pilot_number": pilot_number,
        "pilot_case_id": canonical_pilot[
            "pilot_case_id"
        ],
        "pilot_origin": canonical_pilot[
            "pilot_origin"
        ],
        "pilot_type": canonical_pilot[
            "pilot_type"
        ],
        "authority_class": canonical_pilot[
            "authority_class"
        ],
        "author_review_status": canonical_pilot[
            "author_review_status"
        ],
        "source_text": source_text,
        "question": question,
        "answer": answer,
        "source_character_count": len(source_text),
        "question_character_count": len(question),
        "answer_character_count": len(answer),
    })

assert len(author_package_semantic_review_queue) == 28

assert [
    row["pilot_number"]
    for row in author_package_semantic_review_queue
] == list(range(1, 29))

assert all(
    row["source_text"]
    and row["question"]
    and row["answer"]
    for row in author_package_semantic_review_queue
)

print(
    "=== NB12h AUTHOR-PACKAGE "
    "SEMANTIC-REVIEW QUEUE ==="
)
print(
    "Queued pilots:",
    len(author_package_semantic_review_queue),
)
print(
    "API calls made:",
    0,
)
print(
    "Dataset records modified:",
    0,
)

print("\nPer-pilot queue readiness:")
for row in author_package_semantic_review_queue:
    print(
        f"  - Pilot {row['pilot_number']:02d} | "
        f"source_chars="
        f"{row['source_character_count']} | "
        f"question_chars="
        f"{row['question_character_count']} | "
        f"answer_chars="
        f"{row['answer_character_count']} | "
        f"origin={row['pilot_origin']}"
    )

print(
    "\nRead-only semantic-review queue construction:",
    True,
)
print(
    "\nNB12h AUTHOR-PACKAGE "
    "SEMANTIC-REVIEW QUEUE: PASS"
)

=== NB12h AUTHOR-PACKAGE SEMANTIC-REVIEW QUEUE ===
Queued pilots: 28
API calls made: 0
Dataset records modified: 0

Per-pilot queue readiness:
  - Pilot 01 | source_chars=278 | question_chars=92 | answer_chars=378 | origin=INITIAL_HUMAN_REVIEW
  - Pilot 02 | source_chars=671 | question_chars=93 | answer_chars=423 | origin=INITIAL_HUMAN_REVIEW
  - Pilot 03 | source_chars=335 | question_chars=103 | answer_chars=283 | origin=INITIAL_HUMAN_REVIEW
  - Pilot 04 | source_chars=3034 | question_chars=104 | answer_chars=516 | origin=INITIAL_HUMAN_REVIEW
  - Pilot 05 | source_chars=804 | question_chars=86 | answer_chars=254 | origin=INITIAL_HUMAN_REVIEW
  - Pilot 06 | source_chars=677 | question_chars=87 | answer_chars=273 | origin=INITIAL_HUMAN_REVIEW
  - Pilot 07 | source_chars=2364 | question_chars=83 | answer_chars=246 | origin=INITIAL_HUMAN_REVIEW
  - Pilot 08 | source_chars=237 | question_chars=64 | answer_chars=222 | origin=INITIAL_HUMAN_REVIEW
  - Pilot 09 | source_chars=786 | question_ch

<div dir="rtl" align="right">

## **مرحلهٔ نوزدهم — Preflight داوری معنایی ۲۸ Pilot**

اکنون ۲۸ مورد استاندارد برای داوری معنایی آماده هستند.

پیش از ارسال هر درخواست به مدل، باید runtime مورد نیاز بررسی شود تا هیچ API Call ناقص یا ناخواسته‌ای انجام نشود.

### **بررسی‌های این مرحله**

موارد زیر فقط بررسی می‌شوند:

- وجود صف `author_package_semantic_review_queue`

- وجود Client مدل با نام `chat`

- وجود نام مدل با نام `chat_model`

- تعداد رکوردهای آماده برای داوری

- وضعیت مجازبودن شروع داوری

### **قاعدهٔ مهم**

- این Step هیچ درخواست API ارسال نمی‌کند.

- هیچ هزینه‌ای ایجاد نمی‌کند.

- هیچ Cache یا Artifact جدیدی نمی‌سازد.

- نبود Client یا مدل، Failure داده نیست؛ فقط وضعیت `NOT_READY` ثبت می‌شود.

### **خروجی این مرحله**

ساختار درحافظه‌ای `semantic_judge_preflight` ساخته می‌شود که مشخص می‌کند آیا Notebook برای شروع داوری معنایی آماده است یا نه.

### **Validation**

این Step زمانی PASS است که:

- صف ۲۸تایی موجود باشد؛

- وضعیت runtime بدون API Call ثبت شود؛

- هیچ Dataset یا فایل موجود تغییر نکند.

</div>

In [21]:
# Code Cell 20
# NB12h — Semantic-judge runtime preflight
# Read-only: no API call, cache, dataset, or report is created.

assert "author_package_semantic_review_queue" in globals(), (
    "Run Code Cell 19 first."
)

assert len(author_package_semantic_review_queue) == 28

chat_available = (
    "chat" in globals()
    and getattr(chat, "chat", None) is not None
)

chat_model_available = (
    "chat_model" in globals()
    and isinstance(chat_model, str)
    and chat_model.strip()
)

semantic_judge_preflight = {
    "queue_record_count": len(
        author_package_semantic_review_queue
    ),
    "chat_available": chat_available,
    "chat_model_available": chat_model_available,
    "chat_model": (
        chat_model.strip()
        if chat_model_available
        else None
    ),
    "api_calls_made": 0,
    "cost_incurred": 0,
    "runtime_status": (
        "READY_FOR_EXPLICIT_SEMANTIC_JUDGE_RUN"
        if (
            chat_available
            and chat_model_available
        )
        else "NOT_READY_RUNTIME_CONFIGURATION_MISSING"
    ),
}

assert semantic_judge_preflight[
    "queue_record_count"
] == 28

print("=== NB12h SEMANTIC-JUDGE PREFLIGHT ===")
print(
    "Queued pilots:",
    semantic_judge_preflight[
        "queue_record_count"
    ],
)
print(
    "Chat client available:",
    semantic_judge_preflight[
        "chat_available"
    ],
)
print(
    "Chat model available:",
    semantic_judge_preflight[
        "chat_model_available"
    ],
)
print(
    "Configured model:",
    semantic_judge_preflight[
        "chat_model"
    ],
)
print(
    "API calls made:",
    semantic_judge_preflight[
        "api_calls_made"
    ],
)
print(
    "Cost incurred:",
    semantic_judge_preflight[
        "cost_incurred"
    ],
)
print(
    "\nRuntime status:",
    semantic_judge_preflight[
        "runtime_status"
    ],
)
print(
    "\nNB12h SEMANTIC-JUDGE PREFLIGHT: PASS"
)

=== NB12h SEMANTIC-JUDGE PREFLIGHT ===
Queued pilots: 28
Chat client available: False
Chat model available: False
Configured model: None
API calls made: 0
Cost incurred: 0

Runtime status: NOT_READY_RUNTIME_CONFIGURATION_MISSING

NB12h SEMANTIC-JUDGE PREFLIGHT: PASS


<div dir="rtl" align="right">

## **مرحلهٔ بیستم — پیکربندی Client ابر آروان برای داوری معنایی**

Preflight نشان داد که صف ۲۸ Pilot آماده است، اما Client مدل و نام مدل در Kernel فعلی وجود ندارند.

در این مرحله، تنظیمات اتصال از فایل `.env` پروژه خوانده می‌شوند و Client سازگار با OpenAI برای endpoint ابر آروان ساخته می‌شود.

### **متغیرهای مورد نیاز**

- `ARVAN_AI_API_KEY`

- `ARVAN_CHAT_URL`

- `ARVAN_CHAT_MODEL`

### **قاعدهٔ مهم**

- کلید API در Notebook نوشته یا چاپ نمی‌شود.

- URL و مدل از `.env` واقعی پروژه خوانده می‌شوند.

- این Step فقط Client را می‌سازد.

- هیچ درخواست API ارسال نمی‌شود.

- هیچ هزینه‌ای ایجاد نمی‌شود.

- هیچ Dataset یا Artifactی تغییر نمی‌کند.

### **خروجی این مرحله**

دو متغیر زیر در حافظه ساخته می‌شوند:

- `chat`

- `chat_model`

### **Validation**

این Step زمانی PASS است که:

- فایل `.env` وجود داشته باشد؛

- هر سه متغیر اتصال مقدار معتبر داشته باشند؛

- URL از HTTPS استفاده کند؛

- Client ساخته شود؛

- و تعداد API Callها صفر باقی بماند.

</div>

In [22]:
# Code Cell 21
# NB12h — Arvan AI client configuration
# No API request is sent in this Cell.

assert "project_root" in globals(), (
    "Run Code Cell 02 first."
)

assert "author_package_semantic_review_queue" in globals(), (
    "Run Code Cell 19 first."
)

assert len(author_package_semantic_review_queue) == 28

environment_path = project_root / ".env"

assert environment_path.is_file(), (
    f"Project .env file is missing: {environment_path}"
)

environment_loaded = load_dotenv(
    environment_path,
    override=False,
)

api_key = str(
    os.getenv("ARVAN_AI_API_KEY") or ""
).strip()

chat_url = str(
    os.getenv("ARVAN_CHAT_URL") or ""
).strip()

chat_model = str(
    os.getenv("ARVAN_CHAT_MODEL") or ""
).strip()

assert api_key, (
    "ARVAN_AI_API_KEY is missing from the environment."
)

assert chat_url, (
    "ARVAN_CHAT_URL is missing from the environment."
)

assert chat_model, (
    "ARVAN_CHAT_MODEL is missing from the environment."
)

parsed_chat_url = urlparse(chat_url)

assert parsed_chat_url.scheme == "https", (
    "ARVAN_CHAT_URL must use HTTPS."
)

assert parsed_chat_url.netloc, (
    "ARVAN_CHAT_URL has no valid network location."
)

chat = OpenAI(
    api_key=api_key,
    base_url=chat_url,
    timeout=180,
    max_retries=1,
)

assert getattr(chat, "chat", None) is not None

semantic_judge_runtime = {
    "environment_path": str(environment_path),
    "environment_loaded": environment_loaded,
    "chat_host": parsed_chat_url.netloc,
    "chat_model": chat_model,
    "queue_record_count": len(
        author_package_semantic_review_queue
    ),
    "api_calls_made": 0,
    "cost_incurred": 0,
    "runtime_status": "READY",
}

print("=== NB12h ARVAN AI CLIENT CONFIGURATION ===")
print(
    "Environment file:",
    semantic_judge_runtime[
        "environment_path"
    ],
)
print(
    "Environment loaded:",
    semantic_judge_runtime[
        "environment_loaded"
    ],
)
print(
    "Chat host:",
    semantic_judge_runtime[
        "chat_host"
    ],
)
print(
    "Chat model:",
    semantic_judge_runtime[
        "chat_model"
    ],
)
print(
    "Queued pilots:",
    semantic_judge_runtime[
        "queue_record_count"
    ],
)
print(
    "API calls made:",
    semantic_judge_runtime[
        "api_calls_made"
    ],
)
print(
    "Cost incurred:",
    semantic_judge_runtime[
        "cost_incurred"
    ],
)
print(
    "\nRuntime status:",
    semantic_judge_runtime[
        "runtime_status"
    ],
)
print(
    "\nNB12h ARVAN AI CLIENT CONFIGURATION: PASS"
)

=== NB12h ARVAN AI CLIENT CONFIGURATION ===
Environment file: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/.env
Environment loaded: True
Chat host: arvancloudai.ir
Chat model: Qwen3-30B-A3B
Queued pilots: 28
API calls made: 0
Cost incurred: 0

Runtime status: READY

NB12h ARVAN AI CLIENT CONFIGURATION: PASS


<div dir="rtl" align="right">

## **مرحلهٔ بیست‌ویکم — داوری معنایی مستقل ۲۸ Pilot**

صف ۲۸ Pilot و اتصال ابر آروان آماده شده‌اند.

در این مرحله، هر سؤال و پاسخ نهایی فقط براساس متن منبع همان Case ارزیابی می‌شود.

### **معیارهای داوری**

برای هر Pilot بررسی می‌شود:

- طبیعی و روشن بودن سؤال

- مستقل و self-contained بودن سؤال

- پاسخ‌دادن مستقیم پاسخ به سؤال

- کامل و معنادار بودن پاسخ

- پشتیبانی معنایی پاسخ توسط متن منبع

- نبود مرجع مبهم

- نبود تکرار نامعتبر سؤال در پاسخ

- وجود Evidence قابل‌استفاده در متن منبع

### **قاعدهٔ Evidence**

- تطابق کلمه‌به‌کلمه الزامی نیست.

- Paraphrase معتبر در صورت پشتیبانی کامل معنایی پذیرفته می‌شود.

- بازخورد مستقیم دکتر برای Pilotهای مشتق‌شده، منبع اولیه و معتبر محسوب می‌شود.

- مدل اجازه ندارد از دانش بیرونی برای کامل‌کردن پاسخ استفاده کند.

### **تصمیم‌های ممکن**

- `PASS`

- `REVIEW`

- `REJECT`

- `FAILED` در صورت خطای درخواست یا JSON

### **قاعدهٔ مهم**

- این داوری جایگزین نظر نهایی دکتر نیست.

- نتیجه فقط Finding آزمایشی تولید می‌کند.

- هیچ Pilot، v1، Core-46 یا v2 تغییر نمی‌کند.

- Cache و گزارش فقط در پوشهٔ transient و ignored پروژه ذخیره می‌شوند.

### **خروجی این مرحله**

- `author_pilot_semantic_judge_results`

- گزارش JSON داوری

- Cache پایدار برای جلوگیری از پرداخت مجدد در اجرای دوباره

### **Validation**

این Step زمانی کامل است که:

- برای هر ۲۸ Pilot یک نتیجه ثبت شود؛

- تعداد PASS، REVIEW، REJECT و FAILED مشخص شود؛

- خطاهای API باعث توقف کل Notebook نشوند؛

- و هیچ Dataset فعالی تغییر نکند.

</div>

In [23]:
# Code Cell 22
# NB12h — Full semantic judge for the 28 author-review pilots
# This Cell sends API requests and writes only ignored transient audit files.

assert "chat" in globals(), (
    "Run Code Cell 21 first."
)
assert "chat_model" in globals(), (
    "Run Code Cell 21 first."
)
assert "author_package_semantic_review_queue" in globals(), (
    "Run Code Cell 19 first."
)
assert len(author_package_semantic_review_queue) == 28

AUTHOR_PILOT_JUDGE_VERSION = (
    "nb12h_author_pilot_semantic_judge_v1"
)

author_pilot_audit_directory = (
    project_root
    / "artifacts"
    / "phase2"
    / "golden_dataset_rebuild_clean"
    / "nb12h_author_pilot_audit"
)

author_pilot_cache_directory = (
    author_pilot_audit_directory
    / "cache"
    / AUTHOR_PILOT_JUDGE_VERSION
)

author_pilot_cache_directory.mkdir(
    parents=True,
    exist_ok=True,
)


def write_json_atomic(path, payload):
    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    temporary_path.replace(path)


def parse_json_object(raw_text):
    text = str(raw_text or "").strip()

    if text.startswith("```"):
        text = re.sub(
            r"^```(?:json)?\s*",
            "",
            text,
            flags=re.IGNORECASE,
        )

        text = re.sub(
            r"\s*```$",
            "",
            text,
        ).strip()

    try:
        payload = json.loads(text)

        if isinstance(payload, dict):
            return payload

    except json.JSONDecodeError:
        pass

    object_start = text.find("{")
    object_end = text.rfind("}")

    if (
        object_start >= 0
        and object_end > object_start
    ):
        payload = json.loads(
            text[object_start:object_end + 1]
        )

        if isinstance(payload, dict):
            return payload

    raise ValueError(
        "The model response does not contain "
        "one valid JSON object."
    )


def valid_author_pilot_judgment(payload):
    boolean_fields = [
        "source_contains_usable_evidence",
        "question_natural",
        "question_self_contained",
        "answer_directly_answers",
        "answer_complete",
        "evidence_supports_answer",
        "no_ambiguous_reference",
        "no_question_answer_leakage",
    ]

    return (
        isinstance(payload, dict)
        and payload.get("decision")
        in {"PASS", "REVIEW", "REJECT"}
        and payload.get("support_type")
        in {
            "SOURCE_TEXT",
            "DIRECT_AUTHOR_FEEDBACK",
            "MIXED",
            "NONE",
        }
        and all(
            isinstance(
                payload.get(field),
                bool,
            )
            for field in boolean_fields
        )
        and isinstance(
            payload.get("reasons"),
            list,
        )
        and all(
            isinstance(reason, str)
            for reason in payload["reasons"]
        )
    )


def author_pilot_judge_prompt(record):
    return f"""
نقش شما داور سخت‌گیر کیفیت Golden Dataset فارسی است.

فقط براساس SOURCE PACKAGE، QUESTION و ANSWER داوری کن.

قواعد:

1. سؤال باید فارسی طبیعی، روشن و مستقل باشد.
2. سؤال نباید ضمیر یا مرجع مبهم داشته باشد.
3. پاسخ باید مستقیماً به سؤال جواب بدهد.
4. پاسخ باید کامل و معنادار باشد.
5. پاسخ باید از نظر معنایی کاملاً توسط منبع پشتیبانی شود.
6. تطابق لفظی پاسخ با منبع الزامی نیست؛ paraphrase معتبر پذیرفته است.
7. از دانش بیرونی برای تکمیل پاسخ استفاده نکن.
8. اگر منبع شامل چند نسخه یا تاریخچه است، فقط متن‌های منبع و evidenceهای پذیرفته‌شده را مبنا قرار بده.
9. اگر PILOT ORIGIN برابر PHASE1_AUTHOR_FEEDBACK باشد، بازخورد مستقیم مؤلف که در SOURCE PACKAGE ثبت شده، منبع اولیه معتبر محسوب می‌شود.
10. اگر سؤال و پاسخ با اصلاح محدود قابل قبول شوند، REVIEW بده.
11. اگر منبع پاسخ را پشتیبانی نمی‌کند یا evidence قابل‌استفاده ندارد، REJECT بده.
12. PASS فقط زمانی معتبر است که تمام معیارهای بولی true باشند.
13. دلایل را کوتاه، دقیق و به زبان فارسی بنویس.

خروجی فقط یک JSON معتبر باشد:

{{
  "decision": "PASS|REVIEW|REJECT",
  "support_type": "SOURCE_TEXT|DIRECT_AUTHOR_FEEDBACK|MIXED|NONE",
  "source_contains_usable_evidence": true,
  "question_natural": true,
  "question_self_contained": true,
  "answer_directly_answers": true,
  "answer_complete": true,
  "evidence_supports_answer": true,
  "no_ambiguous_reference": true,
  "no_question_answer_leakage": true,
  "reasons": ["..."]
}}

PILOT NUMBER:
{record["pilot_number"]}

PILOT ORIGIN:
{record["pilot_origin"]}

AUTHORITY CLASS:
{record["authority_class"]}

SOURCE PACKAGE:
{record["source_text"]}

QUESTION:
{record["question"]}

ANSWER:
{record["answer"]}
""".strip()


def author_pilot_cache_key(record):
    material = {
        "judge_version": (
            AUTHOR_PILOT_JUDGE_VERSION
        ),
        "model": chat_model,
        "pilot_number": record[
            "pilot_number"
        ],
        "pilot_case_id": record[
            "pilot_case_id"
        ],
        "pilot_origin": record[
            "pilot_origin"
        ],
        "source_text": canon_for_audit(
            record["source_text"]
        ),
        "question": canon_for_audit(
            record["question"]
        ),
        "answer": canon_for_audit(
            record["answer"]
        ),
    }

    return hashlib.sha256(
        json.dumps(
            material,
            ensure_ascii=False,
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()


def call_author_pilot_judge(prompt):
    response = chat.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Return only one valid JSON object. "
                    "Do not include markdown or commentary."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=0,
        max_tokens=1536,
    )

    raw_response = str(
        response.choices[0].message.content
        or ""
    ).strip()

    payload = parse_json_object(raw_response)

    usage = getattr(
        response,
        "usage",
        None,
    )

    observed_cost = None

    if usage is not None:
        observed_cost = getattr(
            usage,
            "cost",
            None,
        )

        if observed_cost is None:
            usage_payload = (
                usage.model_dump()
                if hasattr(usage, "model_dump")
                else {}
            )

            observed_cost = usage_payload.get(
                "cost"
            )

    return {
        "payload": payload,
        "raw_response": raw_response,
        "finish_reason": (
            response.choices[0].finish_reason
        ),
        "observed_cost": observed_cost,
    }


criterion_fields = [
    "source_contains_usable_evidence",
    "question_natural",
    "question_self_contained",
    "answer_directly_answers",
    "answer_complete",
    "evidence_supports_answer",
    "no_ambiguous_reference",
    "no_question_answer_leakage",
]

author_pilot_semantic_judge_results = []

author_pilot_cache_reuse_count = 0
author_pilot_live_request_count = 0

print(
    "=== NB12h AUTHOR-PILOT "
    "SEMANTIC JUDGE ==="
)
print(
    "Pilots:",
    len(author_package_semantic_review_queue),
)
print(
    "Model:",
    chat_model,
)

for position, record in enumerate(
    author_package_semantic_review_queue,
    1,
):
    cache_key = author_pilot_cache_key(
        record
    )

    cache_path = (
        author_pilot_cache_directory
        / f"{cache_key}.json"
    )

    result = None

    if cache_path.is_file():
        cached_result = json.loads(
            cache_path.read_text(
                encoding="utf-8"
            )
        )

        if valid_author_pilot_judgment(
            cached_result.get("payload")
        ):
            result = {
                **cached_result,
                "cache_reused": True,
            }

            author_pilot_cache_reuse_count += 1

    if result is None:
        last_error = None

        for attempt in range(1, 4):
            try:
                response_record = (
                    call_author_pilot_judge(
                        author_pilot_judge_prompt(
                            record
                        )
                    )
                )

                author_pilot_live_request_count += 1

                payload = response_record[
                    "payload"
                ]

                if not valid_author_pilot_judgment(
                    payload
                ):
                    last_error = (
                        "invalid_judgment_payload"
                    )
                    continue

                all_criteria_true = all(
                    payload[field]
                    for field in criterion_fields
                )

                if (
                    payload["decision"] == "PASS"
                    and not all_criteria_true
                ):
                    payload["decision"] = "REVIEW"

                result = {
                    "judge_version": (
                        AUTHOR_PILOT_JUDGE_VERSION
                    ),
                    "pilot_number": record[
                        "pilot_number"
                    ],
                    "pilot_case_id": record[
                        "pilot_case_id"
                    ],
                    "pilot_origin": record[
                        "pilot_origin"
                    ],
                    "model": chat_model,
                    "attempt": attempt,
                    "payload": payload,
                    "raw_response": (
                        response_record[
                            "raw_response"
                        ]
                    ),
                    "finish_reason": (
                        response_record[
                            "finish_reason"
                        ]
                    ),
                    "observed_cost": (
                        response_record[
                            "observed_cost"
                        ]
                    ),
                    "request_error": None,
                    "created_at_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "cache_reused": False,
                }

                write_json_atomic(
                    cache_path,
                    result,
                )

                break

            except Exception as error:
                author_pilot_live_request_count += 1

                last_error = (
                    f"{type(error).__name__}: "
                    f"{error}"
                )

        if result is None:
            result = {
                "judge_version": (
                    AUTHOR_PILOT_JUDGE_VERSION
                ),
                "pilot_number": record[
                    "pilot_number"
                ],
                "pilot_case_id": record[
                    "pilot_case_id"
                ],
                "pilot_origin": record[
                    "pilot_origin"
                ],
                "model": chat_model,
                "attempt": 3,
                "payload": None,
                "raw_response": "",
                "finish_reason": None,
                "observed_cost": None,
                "request_error": last_error,
                "created_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "cache_reused": False,
            }

    author_pilot_semantic_judge_results.append(
        result
    )

    decision = (
        result["payload"]["decision"]
        if valid_author_pilot_judgment(
            result.get("payload")
        )
        else "FAILED"
    )

    print(
        f"Pilot {position}/"
        f"{len(author_package_semantic_review_queue)} | "
        f"pilot={record['pilot_number']:02d} | "
        f"{decision}"
    )

assert len(
    author_pilot_semantic_judge_results
) == 28

author_pilot_pass_rows = [
    row
    for row in author_pilot_semantic_judge_results
    if (
        valid_author_pilot_judgment(
            row.get("payload")
        )
        and row["payload"]["decision"] == "PASS"
    )
]

author_pilot_review_rows = [
    row
    for row in author_pilot_semantic_judge_results
    if (
        valid_author_pilot_judgment(
            row.get("payload")
        )
        and row["payload"]["decision"] == "REVIEW"
    )
]

author_pilot_reject_rows = [
    row
    for row in author_pilot_semantic_judge_results
    if (
        valid_author_pilot_judgment(
            row.get("payload")
        )
        and row["payload"]["decision"] == "REJECT"
    )
]

author_pilot_failed_rows = [
    row
    for row in author_pilot_semantic_judge_results
    if not valid_author_pilot_judgment(
        row.get("payload")
    )
]

observed_costs = [
    float(row["observed_cost"])
    for row in author_pilot_semantic_judge_results
    if row.get("observed_cost") is not None
]

author_pilot_semantic_judge_report = {
    "judge_version": (
        AUTHOR_PILOT_JUDGE_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": (
        "COMPLETE_WITH_REQUEST_FAILURES"
        if author_pilot_failed_rows
        else "COMPLETE"
    ),
    "active_dataset_modified": False,
    "golden_dataset_v1_modified": False,
    "golden_dataset_v2_created": False,
    "record_count": len(
        author_pilot_semantic_judge_results
    ),
    "pass_count": len(
        author_pilot_pass_rows
    ),
    "review_count": len(
        author_pilot_review_rows
    ),
    "reject_count": len(
        author_pilot_reject_rows
    ),
    "failed_count": len(
        author_pilot_failed_rows
    ),
    "cache_reuse_count": (
        author_pilot_cache_reuse_count
    ),
    "live_request_count": (
        author_pilot_live_request_count
    ),
    "total_observed_cost": (
        float(sum(observed_costs))
        if observed_costs
        else None
    ),
    "results": (
        author_pilot_semantic_judge_results
    ),
}

author_pilot_semantic_judge_report_path = (
    author_pilot_audit_directory
    / "author_pilot_semantic_judge_report.json"
)

write_json_atomic(
    author_pilot_semantic_judge_report_path,
    author_pilot_semantic_judge_report,
)

print(
    "\nSemantic PASS:",
    len(author_pilot_pass_rows),
)
print(
    "Semantic REVIEW:",
    len(author_pilot_review_rows),
)
print(
    "Semantic REJECT:",
    len(author_pilot_reject_rows),
)
print(
    "Request/parse failures:",
    len(author_pilot_failed_rows),
)
print(
    "Cached results reused:",
    author_pilot_cache_reuse_count,
)
print(
    "Live requests:",
    author_pilot_live_request_count,
)
print(
    "Total observed cost:",
    (
        f"{sum(observed_costs):.6f}"
        if observed_costs
        else "N/A"
    ),
)
print(
    "\nReport:",
    author_pilot_semantic_judge_report_path,
)
print(
    "\nActive dataset modified:",
    False,
)
print(
    "Golden Dataset v1 modified:",
    False,
)
print(
    "Golden Dataset v2 created:",
    False,
)
print(
    "\nNB12h AUTHOR-PILOT "
    "SEMANTIC JUDGE: COMPLETE"
)

=== NB12h AUTHOR-PILOT SEMANTIC JUDGE ===
Pilots: 28
Model: Qwen3-30B-A3B
Pilot 1/28 | pilot=01 | PASS
Pilot 2/28 | pilot=02 | PASS
Pilot 3/28 | pilot=03 | PASS
Pilot 4/28 | pilot=04 | PASS
Pilot 5/28 | pilot=05 | PASS
Pilot 6/28 | pilot=06 | PASS
Pilot 7/28 | pilot=07 | PASS
Pilot 8/28 | pilot=08 | PASS
Pilot 9/28 | pilot=09 | PASS
Pilot 10/28 | pilot=10 | PASS
Pilot 11/28 | pilot=11 | PASS
Pilot 12/28 | pilot=12 | PASS
Pilot 13/28 | pilot=13 | PASS
Pilot 14/28 | pilot=14 | PASS
Pilot 15/28 | pilot=15 | PASS
Pilot 16/28 | pilot=16 | PASS
Pilot 17/28 | pilot=17 | PASS
Pilot 18/28 | pilot=18 | PASS
Pilot 19/28 | pilot=19 | PASS
Pilot 20/28 | pilot=20 | PASS
Pilot 21/28 | pilot=21 | PASS
Pilot 22/28 | pilot=22 | PASS
Pilot 23/28 | pilot=23 | PASS
Pilot 24/28 | pilot=24 | PASS
Pilot 25/28 | pilot=25 | PASS
Pilot 26/28 | pilot=26 | PASS
Pilot 27/28 | pilot=27 | PASS
Pilot 28/28 | pilot=28 | PASS

Semantic PASS: 28
Semantic REVIEW: 0
Semantic REJECT: 0
Request/parse failures: 0
Cached resul

<div dir="rtl" align="right">

## **مرحلهٔ بیست‌ودوم — داوری معنایی ساختار Expansion برای ۲۸ Seed**

داوری معنایی مستقل، هر ۲۸ Pilot اصلی را PASS کرده است.

اکنون باید کیفیت فرآیند Expansion بررسی شود. برای این کار، همهٔ سؤال‌های ساخته‌شده از هر Seed همراه با پاسخ و شاهد مشترک همان Seed، به‌صورت گروهی ارزیابی می‌شوند.

### **هدف این مرحله**

هدف، تشخیص ماهیت واقعی هر گروه Expansion است:

- مجموعه‌ای از claimهای مستقل

- مجموعه‌ای برای سنجش robustness در برابر paraphrase

- ترکیبی از چند intent

- یا مجموعه‌ای با افزونگی زیاد

### **ورودی‌های این مرحله**

- ۲۸ گروه موجود در `v1_records_by_seed`

- سؤال‌های هر Seed

- پاسخ مشترک هر Seed

- شاهد مشترک هر Seed

### **خروجی داوری هر Seed**

برای هر Seed موارد زیر ثبت می‌شود:

- نوع Expansion

- تعداد target معنایی تخمینی

- وجود یا نبود redundancy زیاد

- تعداد پیشنهادی رکوردهای اصلی Benchmark

- تعداد پیشنهادی رکوردهای قابل‌نگهداری در Paraphrase Robustness Set

- Candidateهای دارای مشکل کیفی احتمالی

- دلایل داوری

### **قاعدهٔ مهم**

- این نتیجه یک پیشنهاد ماشینی است و تصمیم نهایی حذف رکوردها نیست.

- مدل اجازه ندارد فقط براساس تفاوت واژگان، سؤال‌ها را مستقل اعلام کند.

- سؤال‌هایی که یک پاسخ و claim واحد را با بیان متفاوت می‌پرسند، Paraphrase محسوب می‌شوند.

- هیچ رکوردی حذف یا اصلاح نمی‌شود.

- هیچ نسخهٔ جدیدی از Golden Dataset ساخته نمی‌شود.

### **Artifactهای این مرحله**

Cache و گزارش فقط در مسیر transient و ignored مربوط به NB12h ذخیره می‌شوند.

### **Validation**

این Step زمانی کامل است که:

- هر ۲۸ Seed یک نتیجهٔ معتبر داشته باشد؛

- تعداد گروه‌های مستقل، paraphrase، mixed و unusable مشخص شود؛

- خطاهای API بدون توقف Notebook ثبت شوند؛

- و هیچ Dataset فعالی تغییر نکند.

</div>

In [24]:
# Code Cell 23
# NB12h — Group-level semantic expansion audit for all 28 v1 seeds
# This Cell sends API requests and writes only ignored transient audit files.

assert "chat" in globals(), (
    "Run Code Cell 21 first."
)
assert "chat_model" in globals(), (
    "Run Code Cell 21 first."
)
assert "v1_records_by_seed" in globals(), (
    "Run Code Cell 08 first."
)
assert "author_pilot_semantic_judge_results" in globals(), (
    "Run Code Cell 22 first."
)
assert len(v1_records_by_seed) == 28

V1_EXPANSION_JUDGE_VERSION = (
    "nb12h_v1_group_expansion_judge_v1"
)

v1_expansion_cache_directory = (
    author_pilot_audit_directory
    / "cache"
    / V1_EXPANSION_JUDGE_VERSION
)

v1_expansion_cache_directory.mkdir(
    parents=True,
    exist_ok=True,
)


def valid_expansion_judgment(payload):
    valid_assessments = {
        "INDEPENDENT_CLAIMS",
        "PARAPHRASE_ROBUSTNESS_SET",
        "MIXED_SEMANTIC_TARGETS",
        "UNUSABLE_EXPANSION",
    }

    if not isinstance(payload, dict):
        return False

    if payload.get(
        "expansion_assessment"
    ) not in valid_assessments:
        return False

    integer_fields = [
        "semantic_target_count",
        "recommended_primary_benchmark_count",
        "recommended_robustness_count",
    ]

    if not all(
        isinstance(payload.get(field), int)
        and not isinstance(
            payload.get(field),
            bool,
        )
        for field in integer_fields
    ):
        return False

    if not isinstance(
        payload.get("high_redundancy"),
        bool,
    ):
        return False

    if not isinstance(
        payload.get("question_quality_issue_candidate_ids"),
        list,
    ):
        return False

    if not isinstance(
        payload.get("reasons"),
        list,
    ):
        return False

    return all(
        isinstance(reason, str)
        for reason in payload["reasons"]
    )


def expansion_judge_prompt(
    seed_id,
    records,
):
    answers = {
        canon_for_audit(
            record.get("expected_answer")
        )
        for record in records
    }

    evidence_values = {
        canon_for_audit(
            record.get("source_evidence")
        )
        for record in records
    }

    assert len(answers) == 1
    assert len(evidence_values) == 1

    answer = next(iter(answers))
    evidence = next(iter(evidence_values))

    question_rows = [
        {
            "candidate_id": record["candidate_id"],
            "question": record["question"],
        }
        for record in records
    ]

    questions_json = json.dumps(
        question_rows,
        ensure_ascii=False,
        indent=2,
    )

    return f"""
نقش شما داور ساختار Expansion یک Golden Dataset فارسی است.

یک Seed شامل چند سؤال، یک پاسخ مشترک و یک شاهد مشترک است.

وظیفه:

1. بررسی کن آیا سؤال‌ها claimهای مستقل را آزمایش می‌کنند یا فقط بازنویسی یک claim هستند.
2. تفاوت واژگان به‌تنهایی به‌معنای استقلال معنایی نیست.
3. اگر همه سؤال‌ها عملاً همان پاسخ و همان claim را می‌خواهند، آن‌ها را PARAPHRASE_ROBUSTNESS_SET اعلام کن.
4. اگر چند target معنایی واقعی وجود دارد، MIXED_SEMANTIC_TARGETS اعلام کن.
5. فقط اگر هر سؤال واقعاً claim مستقل و پاسخ‌پذیر مستقل دارد، INDEPENDENT_CLAIMS اعلام کن.
6. اگر سؤال‌ها عمدتاً نامعتبر، مبهم یا ناسازگار با پاسخ باشند، UNUSABLE_EXPANSION اعلام کن.
7. recommended_primary_benchmark_count تعداد مناسب برای Benchmark اصلی است.
8. recommended_robustness_count تعداد مناسبی است که می‌تواند در یک Paraphrase Robustness Set جداگانه باقی بماند.
9. مجموع دو مقدار پیشنهادی نباید از تعداد رکوردها بیشتر شود.
10. candidate_id سؤال‌های دارای مشکل کیفی را ثبت کن.
11. دلایل را کوتاه و فارسی بنویس.
12. تصمیم حذف نهایی صادر نکن؛ فقط پیشنهاد تحلیلی بده.

خروجی فقط یک JSON معتبر باشد:

{{
  "expansion_assessment": "INDEPENDENT_CLAIMS|PARAPHRASE_ROBUSTNESS_SET|MIXED_SEMANTIC_TARGETS|UNUSABLE_EXPANSION",
  "semantic_target_count": 1,
  "high_redundancy": true,
  "recommended_primary_benchmark_count": 1,
  "recommended_robustness_count": 5,
  "question_quality_issue_candidate_ids": [],
  "reasons": ["..."]
}}

SEED ID:
{seed_id}

RECORD COUNT:
{len(records)}

SHARED ANSWER:
{answer}

SHARED EVIDENCE:
{evidence}

QUESTIONS:
{questions_json}
""".strip()


def expansion_cache_key(
    seed_id,
    records,
):
    material = {
        "judge_version": (
            V1_EXPANSION_JUDGE_VERSION
        ),
        "model": chat_model,
        "source_seed_id": seed_id,
        "records": [
            {
                "candidate_id": record[
                    "candidate_id"
                ],
                "question": canon_for_audit(
                    record["question"]
                ),
                "answer": canon_for_audit(
                    record["expected_answer"]
                ),
                "evidence": canon_for_audit(
                    record["source_evidence"]
                ),
            }
            for record in records
        ],
    }

    return hashlib.sha256(
        json.dumps(
            material,
            ensure_ascii=False,
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()


def call_expansion_judge(prompt):
    response = chat.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Return only one valid JSON object. "
                    "Do not include markdown."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=0,
        max_tokens=1536,
    )

    raw_response = str(
        response.choices[0].message.content
        or ""
    ).strip()

    payload = parse_json_object(
        raw_response
    )

    usage = getattr(
        response,
        "usage",
        None,
    )

    observed_cost = None

    if usage is not None:
        observed_cost = getattr(
            usage,
            "cost",
            None,
        )

        if observed_cost is None:
            usage_payload = (
                usage.model_dump()
                if hasattr(usage, "model_dump")
                else {}
            )

            observed_cost = usage_payload.get(
                "cost"
            )

    return {
        "payload": payload,
        "raw_response": raw_response,
        "finish_reason": (
            response.choices[0].finish_reason
        ),
        "observed_cost": observed_cost,
    }


v1_expansion_judge_results = []
v1_expansion_cache_reuse_count = 0
v1_expansion_live_request_count = 0

print(
    "=== NB12h V1 GROUP-LEVEL "
    "EXPANSION JUDGE ==="
)
print("Seed groups:", len(v1_records_by_seed))
print("Model:", chat_model)

for position, seed_id in enumerate(
    sorted(v1_records_by_seed),
    1,
):
    records = v1_records_by_seed[seed_id]

    cache_key = expansion_cache_key(
        seed_id,
        records,
    )

    cache_path = (
        v1_expansion_cache_directory
        / f"{cache_key}.json"
    )

    result = None

    if cache_path.is_file():
        cached_result = json.loads(
            cache_path.read_text(
                encoding="utf-8"
            )
        )

        if valid_expansion_judgment(
            cached_result.get("payload")
        ):
            result = {
                **cached_result,
                "cache_reused": True,
            }

            v1_expansion_cache_reuse_count += 1

    if result is None:
        last_error = None

        for attempt in range(1, 4):
            try:
                response_record = (
                    call_expansion_judge(
                        expansion_judge_prompt(
                            seed_id,
                            records,
                        )
                    )
                )

                v1_expansion_live_request_count += 1

                payload = response_record[
                    "payload"
                ]

                if not valid_expansion_judgment(
                    payload
                ):
                    last_error = (
                        "invalid_expansion_judgment"
                    )
                    continue

                record_count = len(records)

                count_values_valid = (
                    1
                    <= payload[
                        "semantic_target_count"
                    ]
                    <= record_count
                    and 0
                    <= payload[
                        "recommended_primary_benchmark_count"
                    ]
                    <= record_count
                    and 0
                    <= payload[
                        "recommended_robustness_count"
                    ]
                    <= record_count
                    and (
                        payload[
                            "recommended_primary_benchmark_count"
                        ]
                        + payload[
                            "recommended_robustness_count"
                        ]
                        <= record_count
                    )
                )

                if not count_values_valid:
                    last_error = (
                        "invalid_recommended_counts"
                    )
                    continue

                valid_candidate_ids = {
                    record["candidate_id"]
                    for record in records
                }

                unknown_issue_ids = sorted(
                    set(
                        payload[
                            "question_quality_issue_candidate_ids"
                        ]
                    )
                    - valid_candidate_ids
                )

                if unknown_issue_ids:
                    last_error = (
                        "unknown_quality_issue_candidate_ids"
                    )
                    continue

                result = {
                    "judge_version": (
                        V1_EXPANSION_JUDGE_VERSION
                    ),
                    "source_seed_id": seed_id,
                    "record_count": record_count,
                    "model": chat_model,
                    "attempt": attempt,
                    "payload": payload,
                    "raw_response": (
                        response_record[
                            "raw_response"
                        ]
                    ),
                    "finish_reason": (
                        response_record[
                            "finish_reason"
                        ]
                    ),
                    "observed_cost": (
                        response_record[
                            "observed_cost"
                        ]
                    ),
                    "request_error": None,
                    "created_at_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "cache_reused": False,
                }

                write_json_atomic(
                    cache_path,
                    result,
                )

                break

            except Exception as error:
                v1_expansion_live_request_count += 1

                last_error = (
                    f"{type(error).__name__}: "
                    f"{error}"
                )

        if result is None:
            result = {
                "judge_version": (
                    V1_EXPANSION_JUDGE_VERSION
                ),
                "source_seed_id": seed_id,
                "record_count": len(records),
                "model": chat_model,
                "attempt": 3,
                "payload": None,
                "raw_response": "",
                "finish_reason": None,
                "observed_cost": None,
                "request_error": last_error,
                "created_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "cache_reused": False,
            }

    v1_expansion_judge_results.append(result)

    assessment = (
        result["payload"][
            "expansion_assessment"
        ]
        if valid_expansion_judgment(
            result.get("payload")
        )
        else "FAILED"
    )

    print(
        f"Seed {position}/28 | "
        f"seed={seed_id:02d} | "
        f"{assessment}"
    )

assert len(v1_expansion_judge_results) == 28

valid_expansion_rows = [
    row
    for row in v1_expansion_judge_results
    if valid_expansion_judgment(
        row.get("payload")
    )
]

failed_expansion_rows = [
    row
    for row in v1_expansion_judge_results
    if not valid_expansion_judgment(
        row.get("payload")
    )
]

expansion_assessment_counts = Counter(
    row["payload"]["expansion_assessment"]
    for row in valid_expansion_rows
)

recommended_primary_total = sum(
    row["payload"][
        "recommended_primary_benchmark_count"
    ]
    for row in valid_expansion_rows
)

recommended_robustness_total = sum(
    row["payload"][
        "recommended_robustness_count"
    ]
    for row in valid_expansion_rows
)

quality_issue_candidate_ids = sorted({
    candidate_id
    for row in valid_expansion_rows
    for candidate_id in row["payload"][
        "question_quality_issue_candidate_ids"
    ]
})

observed_costs = [
    float(row["observed_cost"])
    for row in v1_expansion_judge_results
    if row.get("observed_cost") is not None
]

v1_expansion_judge_report = {
    "judge_version": (
        V1_EXPANSION_JUDGE_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": (
        "COMPLETE_WITH_REQUEST_FAILURES"
        if failed_expansion_rows
        else "COMPLETE"
    ),
    "golden_dataset_v1_modified": False,
    "golden_dataset_v2_created": False,
    "seed_group_count": 28,
    "assessment_counts": dict(
        sorted(
            expansion_assessment_counts.items()
        )
    ),
    "recommended_primary_total": (
        recommended_primary_total
    ),
    "recommended_robustness_total": (
        recommended_robustness_total
    ),
    "quality_issue_candidate_count": len(
        quality_issue_candidate_ids
    ),
    "quality_issue_candidate_ids": (
        quality_issue_candidate_ids
    ),
    "failed_count": len(
        failed_expansion_rows
    ),
    "cache_reuse_count": (
        v1_expansion_cache_reuse_count
    ),
    "live_request_count": (
        v1_expansion_live_request_count
    ),
    "total_observed_cost": (
        float(sum(observed_costs))
        if observed_costs
        else None
    ),
    "results": v1_expansion_judge_results,
}

v1_expansion_judge_report_path = (
    author_pilot_audit_directory
    / "v1_group_expansion_judge_report.json"
)

write_json_atomic(
    v1_expansion_judge_report_path,
    v1_expansion_judge_report,
)

print("\nExpansion assessments:")
for assessment, count in sorted(
    expansion_assessment_counts.items()
):
    print(f"  - {assessment}: {count}")

print(
    "Failed seed groups:",
    len(failed_expansion_rows),
)
print(
    "Recommended primary benchmark total:",
    recommended_primary_total,
)
print(
    "Recommended robustness total:",
    recommended_robustness_total,
)
print(
    "Question-quality issue candidates:",
    len(quality_issue_candidate_ids),
)
print(
    "Cached results reused:",
    v1_expansion_cache_reuse_count,
)
print(
    "Live requests:",
    v1_expansion_live_request_count,
)
print(
    "Total observed cost:",
    (
        f"{sum(observed_costs):.6f}"
        if observed_costs
        else "N/A"
    ),
)
print(
    "\nReport:",
    v1_expansion_judge_report_path,
)
print(
    "\nGolden Dataset v1 modified:",
    False,
)
print(
    "Golden Dataset v2 created:",
    False,
)
print(
    "\nNB12h V1 GROUP-LEVEL "
    "EXPANSION JUDGE: COMPLETE"
)

=== NB12h V1 GROUP-LEVEL EXPANSION JUDGE ===
Seed groups: 28
Model: Qwen3-30B-A3B
Seed 1/28 | seed=01 | PARAPHRASE_ROBUSTNESS_SET
Seed 2/28 | seed=02 | PARAPHRASE_ROBUSTNESS_SET
Seed 3/28 | seed=03 | PARAPHRASE_ROBUSTNESS_SET
Seed 4/28 | seed=04 | PARAPHRASE_ROBUSTNESS_SET
Seed 5/28 | seed=05 | PARAPHRASE_ROBUSTNESS_SET
Seed 6/28 | seed=06 | PARAPHRASE_ROBUSTNESS_SET
Seed 7/28 | seed=07 | MIXED_SEMANTIC_TARGETS
Seed 8/28 | seed=08 | PARAPHRASE_ROBUSTNESS_SET
Seed 9/28 | seed=09 | PARAPHRASE_ROBUSTNESS_SET
Seed 10/28 | seed=10 | PARAPHRASE_ROBUSTNESS_SET
Seed 11/28 | seed=11 | PARAPHRASE_ROBUSTNESS_SET
Seed 12/28 | seed=12 | PARAPHRASE_ROBUSTNESS_SET
Seed 13/28 | seed=13 | PARAPHRASE_ROBUSTNESS_SET
Seed 14/28 | seed=14 | PARAPHRASE_ROBUSTNESS_SET
Seed 15/28 | seed=15 | PARAPHRASE_ROBUSTNESS_SET
Seed 16/28 | seed=16 | PARAPHRASE_ROBUSTNESS_SET
Seed 17/28 | seed=17 | PARAPHRASE_ROBUSTNESS_SET
Seed 18/28 | seed=18 | PARAPHRASE_ROBUSTNESS_SET
Seed 19/28 | seed=19 | PARAPHRASE_ROBUSTNESS_SET

<div dir="rtl" align="right">

## **مرحلهٔ بیست‌وسوم — استخراج Findings تفصیلی داوری Expansion**

داوری گروهی همهٔ ۲۸ Seed کامل شده است.

نتیجهٔ اولیه نشان می‌دهد بیشتر رکوردهای v1 برای سنجش مقاومت در برابر Paraphrase مناسب‌اند، نه به‌عنوان claimهای مستقل در Benchmark اصلی.

در این مرحله، جزئیات تصمیم هر Seed بدون درخواست جدید از API استخراج می‌شود.

### **ورودی‌های این مرحله**

- `v1_expansion_judge_results`

- رکوردهای واقعی `golden_dataset_v1.jsonl`

### **اطلاعات استخراجی**

برای هر Seed ثبت می‌شود:

- نوع Expansion

- تعداد targetهای معنایی

- وضعیت redundancy

- تعداد پیشنهادی Benchmark اصلی

- تعداد پیشنهادی Robustness Set

- تعداد رکوردهای پیشنهادنشده

- تعداد Candidateهای دارای مشکل کیفی

- دلایل داور

- نمونه‌ای محدود از سؤال‌های دارای Finding

### **قاعدهٔ مهم**

- این مرحله هیچ API Call ندارد.

- تعداد پیشنهادی مدل به‌معنای تصمیم نهایی نیست.

- هیچ Candidate حذف یا اصلاح نمی‌شود.

- v1 و Core-46 بدون تغییر باقی می‌مانند.

### **خروجی این مرحله**

- `v1_expansion_findings_rows`

- گزارش تفصیلی JSON در مسیر transient و ignored مربوط به NB12h

### **Validation**

این Step زمانی PASS است که:

- برای هر ۲۸ Seed یک Finding تفصیلی ساخته شود؛

- همهٔ Candidate IDهای مسئله‌دار به رکورد واقعی همان Seed متصل باشند؛

- مجموع پیشنهادها از ۵۷۳ بیشتر نباشد؛

- و هیچ Dataset فعالی تغییر نکند.

</div>

In [25]:
# Code Cell 24
# NB12h — Detailed v1 expansion findings packet
# No API request is sent. Only an ignored audit report is written.

assert "v1_expansion_judge_results" in globals(), (
    "Run Code Cell 23 first."
)

assert len(v1_expansion_judge_results) == 28

failed_expansion_rows = [
    row
    for row in v1_expansion_judge_results
    if not valid_expansion_judgment(
        row.get("payload")
    )
]

assert not failed_expansion_rows, (
    "Expansion-judge failures must be resolved before "
    "building the findings packet."
)

v1_record_by_candidate_id = {
    record["candidate_id"]: record
    for record in v1_records
}

assert len(v1_record_by_candidate_id) == 573

v1_expansion_findings_rows = []

for result in sorted(
    v1_expansion_judge_results,
    key=lambda row: row["source_seed_id"],
):
    seed_id = result["source_seed_id"]
    payload = result["payload"]
    record_count = result["record_count"]

    issue_candidate_ids = list(
        payload[
            "question_quality_issue_candidate_ids"
        ]
    )

    unknown_candidate_ids = sorted(
        set(issue_candidate_ids)
        - set(v1_record_by_candidate_id)
    )

    assert not unknown_candidate_ids, (
        "Unknown candidate IDs in expansion findings: "
        f"{unknown_candidate_ids}"
    )

    wrong_seed_candidate_ids = [
        candidate_id
        for candidate_id in issue_candidate_ids
        if v1_record_by_candidate_id[
            candidate_id
        ].get("source_seed_id") != seed_id
    ]

    assert not wrong_seed_candidate_ids, (
        "Issue candidate IDs belong to another seed: "
        f"{wrong_seed_candidate_ids}"
    )

    recommended_primary_count = payload[
        "recommended_primary_benchmark_count"
    ]

    recommended_robustness_count = payload[
        "recommended_robustness_count"
    ]

    unselected_count = (
        record_count
        - recommended_primary_count
        - recommended_robustness_count
    )

    assert unselected_count >= 0

    issue_question_rows = [
        {
            "candidate_id": candidate_id,
            "question": (
                v1_record_by_candidate_id[
                    candidate_id
                ]["question"]
            ),
        }
        for candidate_id in issue_candidate_ids
    ]

    v1_expansion_findings_rows.append({
        "source_seed_id": seed_id,
        "record_count": record_count,
        "expansion_assessment": payload[
            "expansion_assessment"
        ],
        "semantic_target_count": payload[
            "semantic_target_count"
        ],
        "high_redundancy": payload[
            "high_redundancy"
        ],
        "recommended_primary_benchmark_count": (
            recommended_primary_count
        ),
        "recommended_robustness_count": (
            recommended_robustness_count
        ),
        "unselected_count": unselected_count,
        "question_quality_issue_count": len(
            issue_candidate_ids
        ),
        "question_quality_issue_candidate_ids": (
            issue_candidate_ids
        ),
        "question_quality_issue_rows": (
            issue_question_rows
        ),
        "reasons": payload["reasons"],
    })

assert len(v1_expansion_findings_rows) == 28

recommended_primary_total = sum(
    row["recommended_primary_benchmark_count"]
    for row in v1_expansion_findings_rows
)

recommended_robustness_total = sum(
    row["recommended_robustness_count"]
    for row in v1_expansion_findings_rows
)

unselected_total = sum(
    row["unselected_count"]
    for row in v1_expansion_findings_rows
)

issue_candidate_total = len({
    candidate_id
    for row in v1_expansion_findings_rows
    for candidate_id in row[
        "question_quality_issue_candidate_ids"
    ]
})

assert (
    recommended_primary_total
    + recommended_robustness_total
    + unselected_total
    == 573
)

v1_expansion_findings_packet = {
    "packet_version": (
        "nb12h_v1_expansion_findings_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "golden_dataset_v1_modified": False,
    "golden_dataset_v2_created": False,
    "seed_count": 28,
    "record_count": 573,
    "recommended_primary_total": (
        recommended_primary_total
    ),
    "recommended_robustness_total": (
        recommended_robustness_total
    ),
    "unselected_total": unselected_total,
    "question_quality_issue_candidate_count": (
        issue_candidate_total
    ),
    "rows": v1_expansion_findings_rows,
}

v1_expansion_findings_packet_path = (
    author_pilot_audit_directory
    / "v1_expansion_findings_packet.json"
)

write_json_atomic(
    v1_expansion_findings_packet_path,
    v1_expansion_findings_packet,
)

print(
    "=== NB12h V1 EXPANSION "
    "FINDINGS PACKET ==="
)
print(
    "Seed findings:",
    len(v1_expansion_findings_rows),
)
print(
    "Recommended primary total:",
    recommended_primary_total,
)
print(
    "Recommended robustness total:",
    recommended_robustness_total,
)
print(
    "Unselected total:",
    unselected_total,
)
print(
    "Quality-issue candidates:",
    issue_candidate_total,
)
print("API calls made:", 0)

print("\nPer-seed findings:")
for row in v1_expansion_findings_rows:
    print("\n" + "=" * 80)
    print(
        f"Seed: {row['source_seed_id']:02d}"
    )
    print(
        "Assessment:",
        row["expansion_assessment"],
    )
    print(
        "Semantic targets:",
        row["semantic_target_count"],
    )
    print(
        "High redundancy:",
        row["high_redundancy"],
    )
    print(
        "Recommended primary:",
        row[
            "recommended_primary_benchmark_count"
        ],
    )
    print(
        "Recommended robustness:",
        row[
            "recommended_robustness_count"
        ],
    )
    print(
        "Unselected:",
        row["unselected_count"],
    )
    print(
        "Quality issues:",
        row[
            "question_quality_issue_count"
        ],
    )

    print("Reasons:")
    for reason in row["reasons"]:
        print(f"  - {reason}")

    issue_preview = row[
        "question_quality_issue_rows"
    ][:3]

    if issue_preview:
        print("Issue-question preview:")

        for issue_row in issue_preview:
            print(
                "  - "
                f"{issue_row['candidate_id']} | "
                f"{issue_row['question']}"
            )

print(
    "\nPacket:",
    v1_expansion_findings_packet_path,
)
print(
    "\nGolden Dataset v1 modified:",
    False,
)
print(
    "Golden Dataset v2 created:",
    False,
)
print(
    "\nNB12h V1 EXPANSION "
    "FINDINGS PACKET: PASS"
)

=== NB12h V1 EXPANSION FINDINGS PACKET ===
Seed findings: 28
Recommended primary total: 98
Recommended robustness total: 308
Unselected total: 167
Quality-issue candidates: 170
API calls made: 0

Per-seed findings:

Seed: 01
Assessment: PARAPHRASE_ROBUSTNESS_SET
Semantic targets: 1
High redundancy: True
Recommended primary: 1
Recommended robustness: 5
Unselected: 15
Quality issues: 6
Reasons:
  - تمام سؤالات مفهوم یکسانی را بازنویسی می‌کنند و هدف یکسانی دارند.
  - تفاوت واژگان به تنهایی معنای مستقلی ایجاد نمی‌کند و سؤالات تکراری هستند.
  - برخی سؤالات دارای ساختار گرامری نادرست یا مفهومی غیر واضح هستند.
Issue-question preview:
  - 9ea667a8-c58e-4ef3-ab0d-6b4b7be3bf26 | برای یک فرد آگاه مالی، درباره کاملی چه نکته‌ای باید بررسی شود؟
  - 039a40b6-1a74-47a8-9c7d-edd5edc527f1 | برای تحلیل بهتر این موضوع، درباره بدهیم چه رابطه‌ای میان مفاهیم وجود دارد؟
  - beffc379-8b28-4d5f-a9c0-3a0c95792eb1 | برای یک فرد آگاه مالی، درباره بخواهیم چه نکته‌ای باید بررسی شود؟

Seed: 02
Assessment: PARAPHRASE_

<div dir="rtl" align="right">

## **مرحلهٔ بیست‌وچهارم — بررسی سازگاری داخلی پیشنهادهای داور Expansion**

Findings تفصیلی داوری Expansion نشان داد که v1 عمدتاً مجموعه‌ای از Paraphraseها با redundancy زیاد است و تعداد قابل‌توجهی سؤال دارای اشکال زبانی یا معنایی هستند.

با این حال، تعدادهای پیشنهادی داور به‌تنهایی قابل اجرا نیستند. در این مرحله، سازگاری داخلی همین پیشنهادها بررسی می‌شود.

### **بررسی‌های این مرحله**

برای هر Seed بررسی می‌شود:

- آیا تعداد Primary پیشنهادی از تعداد targetهای معنایی بیشتر است.

- آیا یک گروه `PARAPHRASE_ROBUSTNESS_SET` بیش از یک target معنایی گزارش کرده است.

- آیا تعداد Candidateهای دارای مشکل از تعداد رکوردهای کنارگذاشته‌شده بیشتر است.

- آیا پیشنهاد داور Candidateهای انتخابی Primary و Robustness را مشخص کرده است.

### **قاعدهٔ مهم**

- تعداد پیشنهادی بدون Candidate ID، Selection اجرایی محسوب نمی‌شود.

- Candidate دارای مشکل نباید بدون بازبینی وارد Primary یا Robustness Set شود.

- برای یک claim واحد، چند Paraphrase نباید به‌عنوان چند تست مستقل شمرده شوند.

- Seed دارای چند target باید ابتدا به claimهای مستقل تقسیم و برای هر claim پاسخ و Evidence مستقل ساخته شود.

- هیچ رکوردی در این مرحله حذف یا انتخاب نمی‌شود.

### **خروجی این مرحله**

ساختار درحافظه‌ای `expansion_recommendation_consistency_rows` ساخته می‌شود که ناسازگاری‌ها و محدودیت‌های پیشنهادهای ماشینی را برای هر Seed ثبت می‌کند.

### **Validation**

این Step زمانی PASS است که:

- هر ۲۸ پیشنهاد بررسی شود؛

- همهٔ تناقض‌ها و محدودیت‌ها ثبت شوند؛

- مشخص شود آیا اعداد پیشنهادی مستقیماً قابل اجرا هستند یا نه؛

- و هیچ Dataset فعالی تغییر نکند.

</div>

In [26]:
# Code Cell 25
# NB12h — Expansion recommendation consistency audit
# No API request is sent and no dataset is modified.

assert "v1_expansion_findings_rows" in globals(), (
    "Run Code Cell 24 first."
)

assert len(v1_expansion_findings_rows) == 28

expansion_recommendation_consistency_rows = []

for row in v1_expansion_findings_rows:
    assessment = row["expansion_assessment"]

    semantic_target_count = row[
        "semantic_target_count"
    ]

    recommended_primary_count = row[
        "recommended_primary_benchmark_count"
    ]

    recommended_robustness_count = row[
        "recommended_robustness_count"
    ]

    unselected_count = row[
        "unselected_count"
    ]

    issue_count = row[
        "question_quality_issue_count"
    ]

    flags = []

    if (
        recommended_primary_count
        > semantic_target_count
    ):
        flags.append(
            "PRIMARY_COUNT_EXCEEDS_"
            "SEMANTIC_TARGET_COUNT"
        )

    if (
        assessment
        == "PARAPHRASE_ROBUSTNESS_SET"
        and semantic_target_count != 1
    ):
        flags.append(
            "PARAPHRASE_ASSESSMENT_HAS_"
            "MULTIPLE_SEMANTIC_TARGETS"
        )

    if issue_count > unselected_count:
        flags.append(
            "ISSUE_COUNT_EXCEEDS_"
            "UNSELECTED_COUNT"
        )

    # The judge returned issue IDs, but did not return
    # explicit selected IDs for Primary or Robustness.
    flags.append(
        "SELECTED_CANDIDATE_IDS_NOT_PROVIDED"
    )

    directly_actionable = (
        not flags
    )

    expansion_recommendation_consistency_rows.append({
        "source_seed_id": row[
            "source_seed_id"
        ],
        "record_count": row[
            "record_count"
        ],
        "expansion_assessment": assessment,
        "semantic_target_count": (
            semantic_target_count
        ),
        "recommended_primary_count": (
            recommended_primary_count
        ),
        "recommended_robustness_count": (
            recommended_robustness_count
        ),
        "unselected_count": unselected_count,
        "question_quality_issue_count": (
            issue_count
        ),
        "consistency_flags": flags,
        "directly_actionable": (
            directly_actionable
        ),
    })

assert len(
    expansion_recommendation_consistency_rows
) == 28

flag_counts = Counter(
    flag
    for row
    in expansion_recommendation_consistency_rows
    for flag in row["consistency_flags"]
)

directly_actionable_rows = [
    row
    for row
    in expansion_recommendation_consistency_rows
    if row["directly_actionable"]
]

non_actionable_rows = [
    row
    for row
    in expansion_recommendation_consistency_rows
    if not row["directly_actionable"]
]

primary_overallocation_rows = [
    row
    for row
    in expansion_recommendation_consistency_rows
    if (
        "PRIMARY_COUNT_EXCEEDS_"
        "SEMANTIC_TARGET_COUNT"
    ) in row["consistency_flags"]
]

assessment_conflict_rows = [
    row
    for row
    in expansion_recommendation_consistency_rows
    if (
        "PARAPHRASE_ASSESSMENT_HAS_"
        "MULTIPLE_SEMANTIC_TARGETS"
    ) in row["consistency_flags"]
]

issue_selection_conflict_rows = [
    row
    for row
    in expansion_recommendation_consistency_rows
    if (
        "ISSUE_COUNT_EXCEEDS_"
        "UNSELECTED_COUNT"
    ) in row["consistency_flags"]
]

print(
    "=== NB12h EXPANSION RECOMMENDATION "
    "CONSISTENCY AUDIT ==="
)
print(
    "Seed recommendations analysed:",
    len(
        expansion_recommendation_consistency_rows
    ),
)
print(
    "Directly actionable recommendations:",
    len(directly_actionable_rows),
)
print(
    "Non-actionable recommendations:",
    len(non_actionable_rows),
)
print(
    "Primary-count overallocation seeds:",
    len(primary_overallocation_rows),
)
print(
    "Assessment/target conflicts:",
    len(assessment_conflict_rows),
)
print(
    "Issue/unselected-count conflicts:",
    len(issue_selection_conflict_rows),
)
print("API calls made:", 0)

print("\nConsistency flags:")
for flag, count in sorted(flag_counts.items()):
    print(f"  - {flag}: {count}")

print("\nPer-seed consistency:")
for row in expansion_recommendation_consistency_rows:
    print(
        f"  - Seed {row['source_seed_id']:02d} | "
        f"assessment="
        f"{row['expansion_assessment']} | "
        f"targets={row['semantic_target_count']} | "
        f"primary="
        f"{row['recommended_primary_count']} | "
        f"robustness="
        f"{row['recommended_robustness_count']} | "
        f"issues="
        f"{row['question_quality_issue_count']} | "
        f"flags={row['consistency_flags']}"
    )

print(
    "\nInterpretation: aggregate recommendation counts "
    "are advisory and cannot be applied directly "
    "without explicit candidate-level adjudication."
)
print(
    "\nGolden Dataset v1 modified:",
    False,
)
print(
    "Golden Dataset v2 created:",
    False,
)
print(
    "\nNB12h EXPANSION RECOMMENDATION "
    "CONSISTENCY AUDIT: PASS"
)

=== NB12h EXPANSION RECOMMENDATION CONSISTENCY AUDIT ===
Seed recommendations analysed: 28
Directly actionable recommendations: 0
Non-actionable recommendations: 28
Primary-count overallocation seeds: 17
Assessment/target conflicts: 1
Issue/unselected-count conflicts: 14
API calls made: 0

Consistency flags:
  - ISSUE_COUNT_EXCEEDS_UNSELECTED_COUNT: 14
  - PARAPHRASE_ASSESSMENT_HAS_MULTIPLE_SEMANTIC_TARGETS: 1
  - PRIMARY_COUNT_EXCEEDS_SEMANTIC_TARGET_COUNT: 17
  - SELECTED_CANDIDATE_IDS_NOT_PROVIDED: 28

Per-seed consistency:
  - Seed 01 | assessment=PARAPHRASE_ROBUSTNESS_SET | targets=1 | primary=1 | robustness=5 | issues=6 | flags=['SELECTED_CANDIDATE_IDS_NOT_PROVIDED']
  - Seed 02 | assessment=PARAPHRASE_ROBUSTNESS_SET | targets=1 | primary=1 | robustness=5 | issues=0 | flags=['SELECTED_CANDIDATE_IDS_NOT_PROVIDED']
  - Seed 03 | assessment=PARAPHRASE_ROBUSTNESS_SET | targets=1 | primary=1 | robustness=5 | issues=3 | flags=['SELECTED_CANDIDATE_IDS_NOT_PROVIDED']
  - Seed 04 | assess

<div dir="rtl" align="right">

## **مرحلهٔ بیست‌وپنجم — ثبت تصمیم معماری Golden Dataset پس از Audit**

Audit ساختار Pilotها و Expansion v1 کامل شده است.

شواهد فعلی نشان می‌دهند که مشکل اصلی پروژه، کیفیت ۲۸ Pilot نیست؛ بلکه تبدیل هر Pilot به ۲۰ یا ۲۱ سؤال روی یک پاسخ و شاهد مشترک است.

در این مرحله، نتیجهٔ معماری مبتنی بر Findings واقعی ثبت می‌شود.

### **ورودی‌های تصمیم**

- نتیجهٔ semantic judge برای ۲۸ Pilot

- ساختار واقعی ۵۷۳ رکورد v1

- نتیجهٔ group-level expansion judge

- Findings مربوط به ۱۷۰ Candidate مسئله‌دار

- وضعیت بازخورد نهایی مؤلف

- وضعیت Core-46 آزمایشی

### **تصمیم دربارهٔ Artifactها**

- Golden Dataset v1 به‌عنوان baseline تاریخی حفظ می‌شود.

- v1 جایگزین یا بازنویسی نمی‌شود.

- ۲۸ Pilot به‌عنوان Pilot Registry معتبر حفظ می‌شوند.

- Core-46 به‌عنوان Quality Core و Regression Set آزمایشی حفظ می‌شود.

- Core-46 به‌تنهایی Golden Dataset v2 نیست.

- ساخت v2 تا دریافت نظر نهایی مؤلف و تعریف claimهای مستقل متوقف می‌ماند.

### **معماری پیشنهادی آینده**

Golden Dataset آینده باید به سه مجموعهٔ مستقل تقسیم شود:

- `PRIMARY_BENCHMARK`: رکوردهای مبتنی بر claimهای مستقل

- `PARAPHRASE_ROBUSTNESS_SET`: بازنویسی‌های سالم و کنترل‌شده

- `REPAIR_OR_REJECT_QUEUE`: سؤال‌های خراب، مبهم یا مصنوعی

### **قاعدهٔ Claim مستقل**

یک رکورد فقط زمانی در Primary Benchmark قرار می‌گیرد که دارای موارد زیر باشد:

- claim مستقل

- سؤال مستقل و طبیعی

- پاسخ مستقل

- Evidence مشخص و مستقیم

- provenance قابل‌ردیابی

وجود چند صورت سؤال برای یک پاسخ و Evidence مشترک، تعداد claimهای Primary را افزایش نمی‌دهد.

### **خروجی این مرحله**

ساختار `nb12h_architecture_decision` و گزارش JSON تصمیم معماری ساخته می‌شوند.

### **Validation**

این Step زمانی PASS است که:

- وضعیت v1، ۲۸ Pilot، Core-46 و v2 جداگانه ثبت شود؛

- تصمیم بر پایهٔ Outputهای واقعی Notebook باشد؛

- هیچ Dataset فعالی تغییر نکند؛

- و هیچ v2 زودهنگامی ساخته نشود.

</div>

In [27]:
# Code Cell 26
# NB12h — Evidence-backed Golden Dataset architecture decision
# No API request is sent and no active dataset is modified.

assert "author_pilot_semantic_judge_results" in globals(), (
    "Run Code Cell 22 first."
)

assert "v1_expansion_unit_rows" in globals(), (
    "Run Code Cell 10 first."
)

assert "v1_expansion_findings_rows" in globals(), (
    "Run Code Cell 24 first."
)

assert (
    "expansion_recommendation_consistency_rows"
    in globals()
), "Run Code Cell 25 first."

pilot_semantic_pass_count = sum(
    valid_author_pilot_judgment(
        row.get("payload")
    )
    and row["payload"]["decision"] == "PASS"
    for row in author_pilot_semantic_judge_results
)

pilot_semantic_review_count = sum(
    valid_author_pilot_judgment(
        row.get("payload")
    )
    and row["payload"]["decision"] == "REVIEW"
    for row in author_pilot_semantic_judge_results
)

pilot_semantic_reject_count = sum(
    valid_author_pilot_judgment(
        row.get("payload")
    )
    and row["payload"]["decision"] == "REJECT"
    for row in author_pilot_semantic_judge_results
)

pilot_semantic_failed_count = sum(
    not valid_author_pilot_judgment(
        row.get("payload")
    )
    for row in author_pilot_semantic_judge_results
)

single_claim_expansion_unit_count = sum(
    row["expansion_type"]
    == "QUESTION_FORM_EXPANSION_OVER_SINGLE_CLAIM"
    for row in v1_expansion_unit_rows
)

paraphrase_seed_count = sum(
    row["expansion_assessment"]
    == "PARAPHRASE_ROBUSTNESS_SET"
    for row in v1_expansion_findings_rows
)

mixed_seed_count = sum(
    row["expansion_assessment"]
    == "MIXED_SEMANTIC_TARGETS"
    for row in v1_expansion_findings_rows
)

high_redundancy_seed_count = sum(
    row["high_redundancy"]
    for row in v1_expansion_findings_rows
)

quality_issue_candidate_count = len({
    candidate_id
    for row in v1_expansion_findings_rows
    for candidate_id in row[
        "question_quality_issue_candidate_ids"
    ]
})

non_actionable_recommendation_count = sum(
    not row["directly_actionable"]
    for row
    in expansion_recommendation_consistency_rows
)

author_feedback_is_final = (
    registry_author_status
    not in {None, "NOT_STARTED"}
    and registry_final_review_status
    not in {None, "NOT_STARTED"}
)

assert pilot_semantic_pass_count == 28
assert pilot_semantic_review_count == 0
assert pilot_semantic_reject_count == 0
assert pilot_semantic_failed_count == 0
assert single_claim_expansion_unit_count == 28
assert paraphrase_seed_count == 27
assert mixed_seed_count == 1
assert high_redundancy_seed_count == 28
assert non_actionable_recommendation_count == 28
assert author_feedback_is_final is False

nb12h_architecture_decision = {
    "decision_version": (
        "nb12h_golden_dataset_architecture_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "evidence_summary": {
        "pilot_count": 28,
        "pilot_semantic_pass_count": (
            pilot_semantic_pass_count
        ),
        "v1_record_count": len(v1_records),
        "v1_unique_answer_count": 28,
        "v1_unique_evidence_count": 28,
        "single_claim_expansion_unit_count": (
            single_claim_expansion_unit_count
        ),
        "paraphrase_seed_count": (
            paraphrase_seed_count
        ),
        "mixed_seed_count": mixed_seed_count,
        "high_redundancy_seed_count": (
            high_redundancy_seed_count
        ),
        "quality_issue_candidate_count": (
            quality_issue_candidate_count
        ),
        "non_actionable_machine_"
        "recommendation_count": (
            non_actionable_recommendation_count
        ),
        "author_feedback_final": (
            author_feedback_is_final
        ),
    },
    "artifact_decisions": {
        "golden_dataset_v1": {
            "status": (
                "PRESERVE_AS_HISTORICAL_BASELINE"
            ),
            "modified": False,
            "eligible_as_573_independent_tests": (
                False
            ),
        },
        "canonical_28_pilots": {
            "status": (
                "SEMANTICALLY_VALIDATED_"
                "AWAITING_FINAL_AUTHOR_REVIEW"
            ),
            "semantic_pass_count": 28,
        },
        "accepted_46_core": {
            "status": (
                "PRESERVE_AS_EXPERIMENTAL_"
                "QUALITY_AND_REGRESSION_CORE"
            ),
            "is_golden_dataset_v2": False,
        },
        "golden_dataset_v2": {
            "status": (
                "NOT_CREATED_BLOCKED_ON_AUTHOR_"
                "FEEDBACK_AND_CLAIM_LEVEL_DESIGN"
            ),
            "created": False,
        },
    },
    "future_dataset_architecture": {
        "primary_benchmark": (
            "INDEPENDENT_SOURCE_GROUNDED_CLAIMS"
        ),
        "paraphrase_robustness_set": (
            "CURATED_VALID_PARAPHRASES_"
            "WITH_SEPARATE_METRICS"
        ),
        "repair_or_reject_queue": (
            "INVALID_AMBIGUOUS_OR_"
            "ARTIFICIAL_QUESTIONS"
        ),
    },
    "required_next_actions": [
        (
            "Receive and integrate final author "
            "feedback for all 28 pilots."
        ),
        (
            "Audit the accepted 46-record core at "
            "claim and evidence level."
        ),
        (
            "Build a source-to-atomic-claim coverage "
            "matrix without a fixed target size."
        ),
        (
            "Create v2 only from independently "
            "supported claims."
        ),
        (
            "Evaluate v1, the quality core, and v2 "
            "under the same retrieval configuration."
        ),
    ],
    "active_dataset_modified": False,
    "golden_dataset_v1_modified": False,
    "golden_dataset_v2_created": False,
}

nb12h_architecture_decision_path = (
    author_pilot_audit_directory
    / "golden_dataset_architecture_decision.json"
)

write_json_atomic(
    nb12h_architecture_decision_path,
    nb12h_architecture_decision,
)

print(
    "=== NB12h GOLDEN DATASET "
    "ARCHITECTURE DECISION ==="
)
print(
    "Pilot semantic PASS:",
    pilot_semantic_pass_count,
)
print(
    "v1 records:",
    len(v1_records),
)
print(
    "v1 independent answer/evidence units:",
    len(v1_expansion_unit_rows),
)
print(
    "Paraphrase expansion seeds:",
    paraphrase_seed_count,
)
print(
    "Mixed-target seeds:",
    mixed_seed_count,
)
print(
    "High-redundancy seeds:",
    high_redundancy_seed_count,
)
print(
    "Quality-issue candidates:",
    quality_issue_candidate_count,
)
print(
    "Directly actionable machine recommendations:",
    28 - non_actionable_recommendation_count,
)
print(
    "Final author feedback received:",
    author_feedback_is_final,
)

print("\nArtifact decisions:")
for artifact_name, decision in (
    nb12h_architecture_decision[
        "artifact_decisions"
    ].items()
):
    print(
        f"  - {artifact_name}: "
        f"{decision['status']}"
    )

print(
    "\nDecision report:",
    nb12h_architecture_decision_path,
)
print(
    "\nActive dataset modified:",
    False,
)
print(
    "Golden Dataset v1 modified:",
    False,
)
print(
    "Golden Dataset v2 created:",
    False,
)
print(
    "\nNB12h GOLDEN DATASET "
    "ARCHITECTURE DECISION: PASS"
)

=== NB12h GOLDEN DATASET ARCHITECTURE DECISION ===
Pilot semantic PASS: 28
v1 records: 573
v1 independent answer/evidence units: 28
Paraphrase expansion seeds: 27
Mixed-target seeds: 1
High-redundancy seeds: 28
Quality-issue candidates: 170
Directly actionable machine recommendations: 0
Final author feedback received: False

Artifact decisions:
  - golden_dataset_v1: PRESERVE_AS_HISTORICAL_BASELINE
  - canonical_28_pilots: SEMANTICALLY_VALIDATED_AWAITING_FINAL_AUTHOR_REVIEW
  - accepted_46_core: PRESERVE_AS_EXPERIMENTAL_QUALITY_AND_REGRESSION_CORE
  - golden_dataset_v2: NOT_CREATED_BLOCKED_ON_AUTHOR_FEEDBACK_AND_CLAIM_LEVEL_DESIGN

Decision report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/golden_dataset_rebuild_clean/nb12h_author_pilot_audit/golden_dataset_architecture_decision.json

Active dataset modified: False
Golden Dataset v1 modified: False
Golden Dataset v2 created: False

NB12h GOLDEN DATASET ARCHITECTURE DECISION: PASS


<div dir="rtl" align="right">

## **مرحلهٔ بیست‌وششم — مقایسهٔ ساختاری Golden Dataset v1 با Core-46**

تصمیم معماری Golden Dataset ثبت شده است.

اکنون Core-46 آزمایشی با v1 مقایسه می‌شود تا مشخص شود ۴۶ رکورد پذیرفته‌شده چه تفاوتی در تنوع سؤال، پاسخ، Evidence و claim ایجاد کرده‌اند.

### **ورودی‌های این مرحله**

- ۵۷۳ رکورد Golden Dataset v1

- ۴۶ رکورد Core-46 پذیرفته‌شده

- نگاشت ۲۸ Seed

### **بررسی‌های این مرحله**

برای هر Dataset محاسبه می‌شود:

- تعداد رکوردها

- تعداد Seedها

- تعداد سؤال‌های یکتا

- تعداد پاسخ‌های یکتا

- تعداد Evidenceهای یکتا

- تعداد واحدهای یکتای Answer–Evidence

- تعداد پاسخ‌های موجود در Evidence

- توزیع رکورد و claim در هر Seed

- هم‌پوشانی دقیق سؤال، پاسخ و Evidence میان v1 و Core-46

### **تعریف واحد Claim در این مقایسه**

در این Step، هر ترکیب یکتای زیر یک واحد اولیهٔ claim محسوب می‌شود:

`normalized expected_answer + normalized source_evidence`

این تعریف هنوز جایگزین داوری معنایی نیست، اما نشان می‌دهد چند رکورد دقیقاً از پاسخ و Evidence مشترک استفاده می‌کنند.

### **قاعدهٔ مهم**

- Core-46 به‌عنوان v2 معرفی نمی‌شود.

- بیشتر بودن تعداد واحدهای Answer–Evidence به‌تنهایی برای کامل‌بودن Dataset کافی نیست.

- هیچ Dataset یا Artifact فعالی تغییر نمی‌کند.

### **خروجی این مرحله**

ساختارهای زیر ساخته می‌شوند:

- `v1_core46_comparison`

- `core46_seed_comparison_rows`

### **Validation**

این Step زمانی PASS است که:

- همهٔ ۵۷۳ رکورد v1 و ۴۶ رکورد Core بررسی شوند؛

- هر ۲۸ Seed در مقایسه حضور داشته باشد؛

- grounding پاسخ‌ها در Evidence بررسی شود؛

- و هیچ Dataset فعالی تغییر نکند.

</div>

In [28]:
# Code Cell 27
# NB12h — Structural comparison of v1 and the accepted 46-record core
# No API request is sent and no dataset is modified.

assert "v1_records" in globals(), (
    "Run Code Cell 07 first."
)

assert "final_46_records" in globals(), (
    "Run Code Cell 07 first."
)

assert len(v1_records) == 573
assert len(final_46_records) == 46


def dataset_structural_profile(
    records,
    dataset_label,
):
    question_keys = [
        canon_for_audit(record.get("question"))
        for record in records
    ]

    answer_keys = [
        canon_for_audit(
            record.get("expected_answer")
        )
        for record in records
    ]

    evidence_keys = [
        canon_for_audit(
            record.get("source_evidence")
        )
        for record in records
    ]

    claim_unit_keys = [
        (
            answer_key,
            evidence_key,
        )
        for answer_key, evidence_key
        in zip(answer_keys, evidence_keys)
    ]

    grounded_record_count = sum(
        bool(answer_key)
        and answer_key in evidence_key
        for answer_key, evidence_key
        in zip(answer_keys, evidence_keys)
    )

    seed_ids = sorted({
        record.get("source_seed_id")
        for record in records
        if isinstance(
            record.get("source_seed_id"),
            int,
        )
    })

    claim_unit_distribution = Counter(
        claim_unit_keys
    )

    records_per_claim_distribution = Counter(
        claim_unit_distribution.values()
    )

    return {
        "dataset_label": dataset_label,
        "record_count": len(records),
        "seed_count": len(seed_ids),
        "seed_ids": seed_ids,
        "unique_question_count": len(
            set(question_keys)
        ),
        "unique_answer_count": len(
            set(answer_keys)
        ),
        "unique_evidence_count": len(
            set(evidence_keys)
        ),
        "unique_claim_unit_count": len(
            set(claim_unit_keys)
        ),
        "grounded_record_count": (
            grounded_record_count
        ),
        "grounding_failure_count": (
            len(records) - grounded_record_count
        ),
        "records_per_claim_distribution": dict(
            sorted(
                records_per_claim_distribution.items()
            )
        ),
        "question_keys": set(question_keys),
        "answer_keys": set(answer_keys),
        "evidence_keys": set(evidence_keys),
        "claim_unit_keys": set(
            claim_unit_keys
        ),
    }


v1_structural_profile = (
    dataset_structural_profile(
        v1_records,
        "golden_dataset_v1",
    )
)

core46_structural_profile = (
    dataset_structural_profile(
        final_46_records,
        "accepted_46_core",
    )
)

assert v1_structural_profile[
    "seed_ids"
] == list(range(1, 29))

assert core46_structural_profile[
    "seed_ids"
] == list(range(1, 29))

v1_records_by_seed_comparison = defaultdict(
    list
)

core46_records_by_seed_comparison = defaultdict(
    list
)

for record in v1_records:
    v1_records_by_seed_comparison[
        record["source_seed_id"]
    ].append(record)

for record in final_46_records:
    core46_records_by_seed_comparison[
        record["source_seed_id"]
    ].append(record)

core46_seed_comparison_rows = []

for seed_id in range(1, 29):
    v1_seed_records = (
        v1_records_by_seed_comparison[
            seed_id
        ]
    )

    core_seed_records = (
        core46_records_by_seed_comparison[
            seed_id
        ]
    )

    v1_seed_claim_units = {
        (
            canon_for_audit(
                record.get("expected_answer")
            ),
            canon_for_audit(
                record.get("source_evidence")
            ),
        )
        for record in v1_seed_records
    }

    core_seed_claim_units = {
        (
            canon_for_audit(
                record.get("expected_answer")
            ),
            canon_for_audit(
                record.get("source_evidence")
            ),
        )
        for record in core_seed_records
    }

    core_seed_answer_count = len({
        canon_for_audit(
            record.get("expected_answer")
        )
        for record in core_seed_records
    })

    core_seed_evidence_count = len({
        canon_for_audit(
            record.get("source_evidence")
        )
        for record in core_seed_records
    })

    core_seed_grounding_failures = [
        record["candidate_id"]
        for record in core_seed_records
        if (
            canon_for_audit(
                record.get("expected_answer")
            )
            not in canon_for_audit(
                record.get("source_evidence")
            )
        )
    ]

    core46_seed_comparison_rows.append({
        "source_seed_id": seed_id,
        "v1_record_count": len(
            v1_seed_records
        ),
        "v1_claim_unit_count": len(
            v1_seed_claim_units
        ),
        "core46_record_count": len(
            core_seed_records
        ),
        "core46_unique_answer_count": (
            core_seed_answer_count
        ),
        "core46_unique_evidence_count": (
            core_seed_evidence_count
        ),
        "core46_claim_unit_count": len(
            core_seed_claim_units
        ),
        "core46_grounding_failure_count": (
            len(core_seed_grounding_failures)
        ),
        "core46_grounding_failure_candidate_ids": (
            core_seed_grounding_failures
        ),
    })

assert len(core46_seed_comparison_rows) == 28

core46_grounding_failure_rows = [
    row
    for row in core46_seed_comparison_rows
    if row[
        "core46_grounding_failure_count"
    ] > 0
]

assert not core46_grounding_failure_rows, (
    "Accepted Core-46 contains answer-grounding "
    "failures."
)

v1_core46_comparison = {
    "comparison_version": (
        "nb12h_v1_core46_structural_comparison_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "v1": {
        key: value
        for key, value in (
            v1_structural_profile.items()
        )
        if not isinstance(value, set)
    },
    "core46": {
        key: value
        for key, value in (
            core46_structural_profile.items()
        )
        if not isinstance(value, set)
    },
    "exact_overlap": {
        "question_count": len(
            v1_structural_profile[
                "question_keys"
            ]
            & core46_structural_profile[
                "question_keys"
            ]
        ),
        "answer_count": len(
            v1_structural_profile[
                "answer_keys"
            ]
            & core46_structural_profile[
                "answer_keys"
            ]
        ),
        "evidence_count": len(
            v1_structural_profile[
                "evidence_keys"
            ]
            & core46_structural_profile[
                "evidence_keys"
            ]
        ),
        "claim_unit_count": len(
            v1_structural_profile[
                "claim_unit_keys"
            ]
            & core46_structural_profile[
                "claim_unit_keys"
            ]
        ),
    },
    "per_seed": core46_seed_comparison_rows,
    "golden_dataset_v1_modified": False,
    "golden_dataset_v2_created": False,
}

v1_core46_comparison_path = (
    author_pilot_audit_directory
    / "v1_core46_structural_comparison.json"
)

write_json_atomic(
    v1_core46_comparison_path,
    v1_core46_comparison,
)

print(
    "=== NB12h V1 VS CORE-46 "
    "STRUCTURAL COMPARISON ==="
)

for profile in [
    v1_structural_profile,
    core46_structural_profile,
]:
    print(
        f"\nDataset: {profile['dataset_label']}"
    )
    print(
        "  Records:",
        profile["record_count"],
    )
    print(
        "  Seeds:",
        profile["seed_count"],
    )
    print(
        "  Unique questions:",
        profile["unique_question_count"],
    )
    print(
        "  Unique answers:",
        profile["unique_answer_count"],
    )
    print(
        "  Unique evidence:",
        profile["unique_evidence_count"],
    )
    print(
        "  Unique answer-evidence units:",
        profile["unique_claim_unit_count"],
    )
    print(
        "  Grounding failures:",
        profile["grounding_failure_count"],
    )
    print(
        "  Records per claim distribution:",
        profile[
            "records_per_claim_distribution"
        ],
    )

print("\nExact overlap between v1 and Core-46:")
for key, value in v1_core46_comparison[
    "exact_overlap"
].items():
    print(f"  - {key}: {value}")

print("\nPer-seed comparison:")
for row in core46_seed_comparison_rows:
    print(
        f"  - Seed {row['source_seed_id']:02d} | "
        f"v1_records={row['v1_record_count']} | "
        f"v1_claims={row['v1_claim_unit_count']} | "
        f"core_records={row['core46_record_count']} | "
        f"core_answers="
        f"{row['core46_unique_answer_count']} | "
        f"core_evidence="
        f"{row['core46_unique_evidence_count']} | "
        f"core_claims="
        f"{row['core46_claim_unit_count']}"
    )

print(
    "\nComparison report:",
    v1_core46_comparison_path,
)
print(
    "\nGolden Dataset v1 modified:",
    False,
)
print(
    "Golden Dataset v2 created:",
    False,
)
print(
    "\nNB12h V1 VS CORE-46 "
    "STRUCTURAL COMPARISON: PASS"
)

=== NB12h V1 VS CORE-46 STRUCTURAL COMPARISON ===

Dataset: golden_dataset_v1
  Records: 573
  Seeds: 28
  Unique questions: 573
  Unique answers: 28
  Unique evidence: 28
  Unique answer-evidence units: 28
  Grounding failures: 0
  Records per claim distribution: {20: 15, 21: 13}

Dataset: accepted_46_core
  Records: 46
  Seeds: 28
  Unique questions: 46
  Unique answers: 46
  Unique evidence: 46
  Unique answer-evidence units: 46
  Grounding failures: 0
  Records per claim distribution: {1: 46}

Exact overlap between v1 and Core-46:
  - question_count: 0
  - answer_count: 0
  - evidence_count: 0
  - claim_unit_count: 0

Per-seed comparison:
  - Seed 01 | v1_records=21 | v1_claims=1 | core_records=1 | core_answers=1 | core_evidence=1 | core_claims=1
  - Seed 02 | v1_records=21 | v1_claims=1 | core_records=2 | core_answers=2 | core_evidence=2 | core_claims=2
  - Seed 03 | v1_records=21 | v1_claims=1 | core_records=3 | core_answers=3 | core_evidence=3 | core_claims=3
  - Seed 04 | v1_re

<div dir="rtl" align="right">

## **مرحلهٔ بیست‌وهفتم — داوری استقلال معنایی Claimهای Core-46**

مقایسهٔ ساختاری نشان داد Core-46 شامل ۴۶ ترکیب یکتای Answer–Evidence است، درحالی‌که v1 فقط ۲۸ ترکیب یکتا دارد.

در این مرحله بررسی می‌شود که رکوردهای متعدد داخل یک Seed در Core-46 واقعاً claimهای مستقل هستند یا فقط تفاوت متنی دارند.

### **دامنهٔ داوری**

- Seedهای دارای یک رکورد، به‌عنوان یک claim منفرد ثبت می‌شوند.

- فقط Seedهای دارای دو یا سه رکورد برای داوری گروهی به مدل ارسال می‌شوند.

### **معیار استقلال Claim**

دو رکورد زمانی مستقل محسوب می‌شوند که:

- سؤال‌ها اطلاعات یا روابط متفاوتی را درخواست کنند.

- پاسخ‌ها claimهای متفاوتی داشته باشند.

- Evidence هر پاسخ به‌صورت مستقل قابل استفاده باشد.

- یکی صرفاً بازنویسی دیگری نباشد.

### **تصمیم‌های ممکن**

- `SINGLE_CLAIM`

- `INDEPENDENT_CLAIMS`

- `OVERLAPPING_CLAIMS`

- `DUPLICATE_CLAIMS`

- `REVIEW_REQUIRED`

### **قاعدهٔ مهم**

- یکتایی متن به‌تنهایی استقلال معنایی نیست.

- این داوری فقط Core-46 را بررسی می‌کند.

- هیچ رکوردی حذف، اصلاح یا وارد v2 نمی‌شود.

- Cache و گزارش فقط در مسیر transient و ignored ذخیره می‌شوند.

### **خروجی این مرحله**

- `core46_claim_independence_results`

- تعداد تخمینی claimهای مستقل Core-46

- فهرست Candidateهای تکراری یا هم‌پوشان احتمالی

### **Validation**

این Step زمانی کامل است که:

- هر ۲۸ Seed نتیجه داشته باشد؛

- همهٔ Seedهای چندرکوردی داوری شوند؛

- همهٔ Candidate IDهای مسئله‌دار Validate شوند؛

- و هیچ Dataset فعالی تغییر نکند.

</div>

In [29]:
# Code Cell 28
# NB12h — Core-46 within-seed semantic claim-independence audit
# API is used only for multi-record seed groups.

assert "chat" in globals(), (
    "Run Code Cell 21 first."
)
assert "chat_model" in globals(), (
    "Run Code Cell 21 first."
)
assert "core46_records_by_seed_comparison" in globals(), (
    "Run Code Cell 27 first."
)
assert len(core46_records_by_seed_comparison) == 28

CORE46_CLAIM_JUDGE_VERSION = (
    "nb12h_core46_claim_independence_judge_v1"
)

core46_claim_cache_directory = (
    author_pilot_audit_directory
    / "cache"
    / CORE46_CLAIM_JUDGE_VERSION
)

core46_claim_cache_directory.mkdir(
    parents=True,
    exist_ok=True,
)


def valid_core46_claim_judgment(payload):
    valid_assessments = {
        "SINGLE_CLAIM",
        "INDEPENDENT_CLAIMS",
        "OVERLAPPING_CLAIMS",
        "DUPLICATE_CLAIMS",
        "REVIEW_REQUIRED",
    }

    return (
        isinstance(payload, dict)
        and payload.get("assessment")
        in valid_assessments
        and isinstance(
            payload.get("independent_claim_count"),
            int,
        )
        and not isinstance(
            payload.get("independent_claim_count"),
            bool,
        )
        and isinstance(
            payload.get("high_semantic_overlap"),
            bool,
        )
        and isinstance(
            payload.get("redundant_candidate_ids"),
            list,
        )
        and isinstance(
            payload.get("reasons"),
            list,
        )
        and all(
            isinstance(reason, str)
            for reason in payload["reasons"]
        )
    )


def core46_claim_judge_prompt(
    seed_id,
    records,
):
    record_payload = [
        {
            "candidate_id": record[
                "candidate_id"
            ],
            "question": record["question"],
            "answer": record[
                "expected_answer"
            ],
            "evidence": record[
                "source_evidence"
            ],
        }
        for record in records
    ]

    return f"""
نقش شما داور استقلال معنایی claimها در یک Golden Dataset فارسی است.

چند رکورد از یک source_seed_id مشترک ارائه شده‌اند.

وظیفه:

1. بررسی کن آیا هر رکورد claim مستقلی را آزمایش می‌کند.
2. تفاوت لفظی سؤال یا پاسخ به‌تنهایی استقلال ایجاد نمی‌کند.
3. اگر سؤال‌ها اطلاعات متفاوتی می‌خواهند و هر پاسخ Evidence مستقل دارد، INDEPENDENT_CLAIMS بده.
4. اگر claimها بخشی هم‌پوشانی دارند ولی کاملاً تکراری نیستند، OVERLAPPING_CLAIMS بده.
5. اگر چند رکورد یک claim واحد را تکرار می‌کنند، DUPLICATE_CLAIMS بده.
6. اگر تصمیم قطعی ممکن نیست، REVIEW_REQUIRED بده.
7. independent_claim_count باید تعداد واقعی claimهای مستقل تخمینی باشد.
8. Candidateهای زائد احتمالی را در redundant_candidate_ids ثبت کن.
9. از دانش بیرونی استفاده نکن.
10. دلایل را کوتاه و فارسی بنویس.

خروجی فقط یک JSON معتبر باشد:

{{
  "assessment": "INDEPENDENT_CLAIMS|OVERLAPPING_CLAIMS|DUPLICATE_CLAIMS|REVIEW_REQUIRED",
  "independent_claim_count": 2,
  "high_semantic_overlap": false,
  "redundant_candidate_ids": [],
  "reasons": ["..."]
}}

SOURCE SEED ID:
{seed_id}

RECORDS:
{json.dumps(record_payload, ensure_ascii=False, indent=2)}
""".strip()


def core46_claim_cache_key(
    seed_id,
    records,
):
    material = {
        "judge_version": (
            CORE46_CLAIM_JUDGE_VERSION
        ),
        "model": chat_model,
        "source_seed_id": seed_id,
        "records": [
            {
                "candidate_id": record[
                    "candidate_id"
                ],
                "question": canon_for_audit(
                    record["question"]
                ),
                "answer": canon_for_audit(
                    record["expected_answer"]
                ),
                "evidence": canon_for_audit(
                    record["source_evidence"]
                ),
            }
            for record in records
        ],
    }

    return hashlib.sha256(
        json.dumps(
            material,
            ensure_ascii=False,
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()


def call_core46_claim_judge(prompt):
    response = chat.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Return only one valid JSON object. "
                    "Do not include markdown."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=0,
        max_tokens=1536,
    )

    raw_response = str(
        response.choices[0].message.content
        or ""
    ).strip()

    payload = parse_json_object(raw_response)

    usage = getattr(
        response,
        "usage",
        None,
    )

    observed_cost = None

    if usage is not None:
        observed_cost = getattr(
            usage,
            "cost",
            None,
        )

        if observed_cost is None:
            usage_payload = (
                usage.model_dump()
                if hasattr(usage, "model_dump")
                else {}
            )

            observed_cost = usage_payload.get(
                "cost"
            )

    return {
        "payload": payload,
        "raw_response": raw_response,
        "finish_reason": (
            response.choices[0].finish_reason
        ),
        "observed_cost": observed_cost,
    }


core46_claim_independence_results = []
core46_claim_cache_reuse_count = 0
core46_claim_live_request_count = 0

multi_record_seed_ids = [
    seed_id
    for seed_id in range(1, 29)
    if len(
        core46_records_by_seed_comparison[
            seed_id
        ]
    ) > 1
]

print(
    "=== NB12h CORE-46 CLAIM "
    "INDEPENDENCE JUDGE ==="
)
print("Seed groups:", 28)
print(
    "Multi-record groups requiring API:",
    len(multi_record_seed_ids),
)
print("Model:", chat_model)

for seed_id in range(1, 29):
    records = (
        core46_records_by_seed_comparison[
            seed_id
        ]
    )

    assert records, (
        f"Core-46 has no record for seed={seed_id}."
    )

    valid_candidate_ids = {
        record["candidate_id"]
        for record in records
    }

    if len(records) == 1:
        result = {
            "judge_version": (
                CORE46_CLAIM_JUDGE_VERSION
            ),
            "source_seed_id": seed_id,
            "record_count": 1,
            "model": None,
            "attempt": 0,
            "payload": {
                "assessment": "SINGLE_CLAIM",
                "independent_claim_count": 1,
                "high_semantic_overlap": False,
                "redundant_candidate_ids": [],
                "reasons": [
                    (
                        "این Seed فقط یک رکورد دارد و "
                        "مقایسه درون‌گروهی لازم نیست."
                    )
                ],
            },
            "raw_response": "",
            "finish_reason": None,
            "observed_cost": None,
            "request_error": None,
            "created_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "cache_reused": False,
        }

    else:
        cache_key = core46_claim_cache_key(
            seed_id,
            records,
        )

        cache_path = (
            core46_claim_cache_directory
            / f"{cache_key}.json"
        )

        result = None

        if cache_path.is_file():
            cached_result = json.loads(
                cache_path.read_text(
                    encoding="utf-8"
                )
            )

            if valid_core46_claim_judgment(
                cached_result.get("payload")
            ):
                result = {
                    **cached_result,
                    "cache_reused": True,
                }

                core46_claim_cache_reuse_count += 1

        if result is None:
            last_error = None

            for attempt in range(1, 4):
                try:
                    response_record = (
                        call_core46_claim_judge(
                            core46_claim_judge_prompt(
                                seed_id,
                                records,
                            )
                        )
                    )

                    core46_claim_live_request_count += 1

                    payload = response_record[
                        "payload"
                    ]

                    if not valid_core46_claim_judgment(
                        payload
                    ):
                        last_error = (
                            "invalid_claim_judgment"
                        )
                        continue

                    independent_claim_count = (
                        payload[
                            "independent_claim_count"
                        ]
                    )

                    if not (
                        1
                        <= independent_claim_count
                        <= len(records)
                    ):
                        last_error = (
                            "invalid_independent_claim_count"
                        )
                        continue

                    redundant_candidate_ids = set(
                        payload[
                            "redundant_candidate_ids"
                        ]
                    )

                    if not redundant_candidate_ids.issubset(
                        valid_candidate_ids
                    ):
                        last_error = (
                            "unknown_redundant_candidate_id"
                        )
                        continue

                    result = {
                        "judge_version": (
                            CORE46_CLAIM_JUDGE_VERSION
                        ),
                        "source_seed_id": seed_id,
                        "record_count": len(records),
                        "model": chat_model,
                        "attempt": attempt,
                        "payload": payload,
                        "raw_response": (
                            response_record[
                                "raw_response"
                            ]
                        ),
                        "finish_reason": (
                            response_record[
                                "finish_reason"
                            ]
                        ),
                        "observed_cost": (
                            response_record[
                                "observed_cost"
                            ]
                        ),
                        "request_error": None,
                        "created_at_utc": datetime.now(
                            timezone.utc
                        ).isoformat(),
                        "cache_reused": False,
                    }

                    write_json_atomic(
                        cache_path,
                        result,
                    )

                    break

                except Exception as error:
                    core46_claim_live_request_count += 1

                    last_error = (
                        f"{type(error).__name__}: "
                        f"{error}"
                    )

            if result is None:
                result = {
                    "judge_version": (
                        CORE46_CLAIM_JUDGE_VERSION
                    ),
                    "source_seed_id": seed_id,
                    "record_count": len(records),
                    "model": chat_model,
                    "attempt": 3,
                    "payload": None,
                    "raw_response": "",
                    "finish_reason": None,
                    "observed_cost": None,
                    "request_error": last_error,
                    "created_at_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "cache_reused": False,
                }

    core46_claim_independence_results.append(
        result
    )

    assessment = (
        result["payload"]["assessment"]
        if valid_core46_claim_judgment(
            result.get("payload")
        )
        else "FAILED"
    )

    print(
        f"Seed {seed_id:02d} | "
        f"records={len(records)} | "
        f"{assessment}"
    )

assert len(
    core46_claim_independence_results
) == 28

valid_core46_claim_rows = [
    row
    for row in core46_claim_independence_results
    if valid_core46_claim_judgment(
        row.get("payload")
    )
]

failed_core46_claim_rows = [
    row
    for row in core46_claim_independence_results
    if not valid_core46_claim_judgment(
        row.get("payload")
    )
]

assessment_counts = Counter(
    row["payload"]["assessment"]
    for row in valid_core46_claim_rows
)

estimated_independent_claim_total = sum(
    row["payload"][
        "independent_claim_count"
    ]
    for row in valid_core46_claim_rows
)

redundant_candidate_ids = sorted({
    candidate_id
    for row in valid_core46_claim_rows
    for candidate_id in row["payload"][
        "redundant_candidate_ids"
    ]
})

observed_costs = [
    float(row["observed_cost"])
    for row in core46_claim_independence_results
    if row.get("observed_cost") is not None
]

core46_claim_independence_report = {
    "judge_version": (
        CORE46_CLAIM_JUDGE_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": (
        "COMPLETE_WITH_REQUEST_FAILURES"
        if failed_core46_claim_rows
        else "COMPLETE"
    ),
    "record_count": 46,
    "seed_count": 28,
    "multi_record_seed_count": len(
        multi_record_seed_ids
    ),
    "assessment_counts": dict(
        sorted(assessment_counts.items())
    ),
    "estimated_independent_claim_total": (
        estimated_independent_claim_total
    ),
    "redundant_candidate_count": len(
        redundant_candidate_ids
    ),
    "redundant_candidate_ids": (
        redundant_candidate_ids
    ),
    "failed_count": len(
        failed_core46_claim_rows
    ),
    "cache_reuse_count": (
        core46_claim_cache_reuse_count
    ),
    "live_request_count": (
        core46_claim_live_request_count
    ),
    "total_observed_cost": (
        float(sum(observed_costs))
        if observed_costs
        else None
    ),
    "results": (
        core46_claim_independence_results
    ),
    "golden_dataset_v1_modified": False,
    "golden_dataset_v2_created": False,
}

core46_claim_independence_report_path = (
    author_pilot_audit_directory
    / "core46_claim_independence_report.json"
)

write_json_atomic(
    core46_claim_independence_report_path,
    core46_claim_independence_report,
)

print("\nAssessment counts:")
for assessment, count in sorted(
    assessment_counts.items()
):
    print(f"  - {assessment}: {count}")

print(
    "Estimated independent claims:",
    estimated_independent_claim_total,
)
print(
    "Redundant candidates:",
    len(redundant_candidate_ids),
)
print(
    "Failed groups:",
    len(failed_core46_claim_rows),
)
print(
    "Cached results reused:",
    core46_claim_cache_reuse_count,
)
print(
    "Live requests:",
    core46_claim_live_request_count,
)
print(
    "Total observed cost:",
    (
        f"{sum(observed_costs):.6f}"
        if observed_costs
        else "N/A"
    ),
)
print(
    "\nReport:",
    core46_claim_independence_report_path,
)
print(
    "\nGolden Dataset v1 modified:",
    False,
)
print(
    "Golden Dataset v2 created:",
    False,
)
print(
    "\nNB12h CORE-46 CLAIM "
    "INDEPENDENCE JUDGE: COMPLETE"
)

=== NB12h CORE-46 CLAIM INDEPENDENCE JUDGE ===
Seed groups: 28
Multi-record groups requiring API: 12
Model: Qwen3-30B-A3B
Seed 01 | records=1 | SINGLE_CLAIM
Seed 02 | records=2 | INDEPENDENT_CLAIMS
Seed 03 | records=3 | INDEPENDENT_CLAIMS
Seed 04 | records=3 | OVERLAPPING_CLAIMS
Seed 05 | records=1 | SINGLE_CLAIM
Seed 06 | records=2 | INDEPENDENT_CLAIMS
Seed 07 | records=1 | SINGLE_CLAIM
Seed 08 | records=3 | OVERLAPPING_CLAIMS
Seed 09 | records=2 | INDEPENDENT_CLAIMS
Seed 10 | records=1 | SINGLE_CLAIM
Seed 11 | records=1 | SINGLE_CLAIM
Seed 12 | records=3 | INDEPENDENT_CLAIMS
Seed 13 | records=1 | SINGLE_CLAIM
Seed 14 | records=1 | SINGLE_CLAIM
Seed 15 | records=1 | SINGLE_CLAIM
Seed 16 | records=3 | INDEPENDENT_CLAIMS
Seed 17 | records=2 | INDEPENDENT_CLAIMS
Seed 18 | records=1 | SINGLE_CLAIM
Seed 19 | records=1 | SINGLE_CLAIM
Seed 20 | records=1 | SINGLE_CLAIM
Seed 21 | records=3 | INDEPENDENT_CLAIMS
Seed 22 | records=1 | SINGLE_CLAIM
Seed 23 | records=2 | INDEPENDENT_CLAIMS
Seed 24

<div dir="rtl" align="right">

## **مرحلهٔ بیست‌وهشتم — استخراج Findings استقلال Claim در Core-46**

داوری استقلال معنایی Core-46 کامل شده است.

از ۲۸ Seed، فقط Seedهای ۴ و ۸ دارای Claimهای هم‌پوشان گزارش شده‌اند. در این مرحله جزئیات این دو Finding و سازگاری تصمیم داور بررسی می‌شود.

### **بررسی‌های این مرحله**

برای هر Seed ثبت می‌شود:

- تعداد رکوردها

- نوع ارزیابی

- تعداد claim مستقل تخمینی

- Candidateهای زائد معرفی‌شده

- دلایل داور

### **بررسی ویژهٔ Seedهای مسئله‌دار**

برای Seedهای دارای `OVERLAPPING_CLAIMS`، `DUPLICATE_CLAIMS` یا `REVIEW_REQUIRED` موارد زیر چاپ می‌شود:

- Candidate ID

- سؤال

- پاسخ

- پیش‌نمایش Evidence

- دلیل داور

### **قاعدهٔ مهم**

- هم‌پوشانی به‌معنای حذف خودکار Candidate نیست.

- اگر تعداد claim مستقل کمتر از تعداد رکوردها باشد اما Candidate زائد مشخص نشده باشد، تصمیم غیرقابل اجرا ثبت می‌شود.

- هیچ رکوردی حذف یا اصلاح نمی‌شود.

### **خروجی این مرحله**

ساختار `core46_claim_findings_rows` و گزارش تفصیلی Findings ساخته می‌شوند.

### **Validation**

این Step زمانی PASS است که:

- برای هر ۲۸ Seed یک Finding ثبت شود؛

- جزئیات همهٔ گروه‌های غیرمستقل استخراج شود؛

- ناسازگاری‌های غیرقابل اجرا مشخص شوند؛

- و هیچ Dataset فعالی تغییر نکند.

</div>

In [30]:
# Code Cell 29
# NB12h — Detailed Core-46 claim-independence findings
# No API request is sent and no dataset is modified.

assert "core46_claim_independence_results" in globals(), (
    "Run Code Cell 28 first."
)

assert len(core46_claim_independence_results) == 28

failed_core46_claim_rows = [
    row
    for row in core46_claim_independence_results
    if not valid_core46_claim_judgment(
        row.get("payload")
    )
]

assert not failed_core46_claim_rows, (
    "Core-46 claim-judge failures remain."
)

core46_claim_findings_rows = []

for result in sorted(
    core46_claim_independence_results,
    key=lambda row: row["source_seed_id"],
):
    seed_id = result["source_seed_id"]
    payload = result["payload"]
    records = (
        core46_records_by_seed_comparison[
            seed_id
        ]
    )

    independent_claim_count = payload[
        "independent_claim_count"
    ]

    redundant_candidate_ids = payload[
        "redundant_candidate_ids"
    ]

    consistency_flags = []

    if (
        independent_claim_count
        < len(records)
        and not redundant_candidate_ids
    ):
        consistency_flags.append(
            "REDUCED_CLAIM_COUNT_WITHOUT_"
            "REDUNDANT_CANDIDATE_IDS"
        )

    if (
        payload["assessment"]
        == "INDEPENDENT_CLAIMS"
        and independent_claim_count
        != len(records)
    ):
        consistency_flags.append(
            "INDEPENDENT_ASSESSMENT_COUNT_MISMATCH"
        )

    record_rows = [
        {
            "candidate_id": record[
                "candidate_id"
            ],
            "question": record["question"],
            "expected_answer": record[
                "expected_answer"
            ],
            "source_evidence": record[
                "source_evidence"
            ],
        }
        for record in records
    ]

    core46_claim_findings_rows.append({
        "source_seed_id": seed_id,
        "record_count": len(records),
        "assessment": payload["assessment"],
        "independent_claim_count": (
            independent_claim_count
        ),
        "high_semantic_overlap": payload[
            "high_semantic_overlap"
        ],
        "redundant_candidate_ids": (
            redundant_candidate_ids
        ),
        "reasons": payload["reasons"],
        "consistency_flags": (
            consistency_flags
        ),
        "records": record_rows,
    })

assert len(core46_claim_findings_rows) == 28

review_required_claim_rows = [
    row
    for row in core46_claim_findings_rows
    if row["assessment"] in {
        "OVERLAPPING_CLAIMS",
        "DUPLICATE_CLAIMS",
        "REVIEW_REQUIRED",
    }
]

claim_consistency_failure_rows = [
    row
    for row in core46_claim_findings_rows
    if row["consistency_flags"]
]

assert {
    row["source_seed_id"]
    for row in review_required_claim_rows
} == {4, 8}, (
    "Unexpected Core-46 review-required seed set."
)

core46_claim_findings_packet = {
    "packet_version": (
        "nb12h_core46_claim_findings_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "record_count": 46,
    "seed_count": 28,
    "estimated_independent_claim_count": sum(
        row["independent_claim_count"]
        for row in core46_claim_findings_rows
    ),
    "review_required_seed_ids": sorted(
        row["source_seed_id"]
        for row in review_required_claim_rows
    ),
    "consistency_finding_count": len(
        claim_consistency_failure_rows
    ),
    "rows": core46_claim_findings_rows,
    "golden_dataset_v1_modified": False,
    "golden_dataset_v2_created": False,
}

core46_claim_findings_packet_path = (
    author_pilot_audit_directory
    / "core46_claim_findings_packet.json"
)

write_json_atomic(
    core46_claim_findings_packet_path,
    core46_claim_findings_packet,
)

print(
    "=== NB12h CORE-46 CLAIM "
    "FINDINGS PACKET ==="
)
print(
    "Seed findings:",
    len(core46_claim_findings_rows),
)
print(
    "Estimated independent claims:",
    core46_claim_findings_packet[
        "estimated_independent_claim_count"
    ],
)
print(
    "Review-required seeds:",
    core46_claim_findings_packet[
        "review_required_seed_ids"
    ],
)
print(
    "Consistency findings:",
    len(claim_consistency_failure_rows),
)
print("API calls made:", 0)

print("\nPer-seed summary:")
for row in core46_claim_findings_rows:
    print(
        f"  - Seed {row['source_seed_id']:02d} | "
        f"records={row['record_count']} | "
        f"assessment={row['assessment']} | "
        f"independent_claims="
        f"{row['independent_claim_count']} | "
        f"flags={row['consistency_flags']}"
    )

print("\nDetailed review-required findings:")

for row in review_required_claim_rows:
    print("\n" + "=" * 80)
    print(
        "Seed:",
        row["source_seed_id"],
    )
    print(
        "Assessment:",
        row["assessment"],
    )
    print(
        "Records:",
        row["record_count"],
    )
    print(
        "Independent claims:",
        row["independent_claim_count"],
    )
    print(
        "High semantic overlap:",
        row["high_semantic_overlap"],
    )
    print(
        "Redundant candidate IDs:",
        row["redundant_candidate_ids"],
    )
    print(
        "Consistency flags:",
        row["consistency_flags"],
    )

    print("Reasons:")
    for reason in row["reasons"]:
        print(f"  - {reason}")

    print("Records:")
    for record in row["records"]:
        evidence_preview = (
            record["source_evidence"][:500]
        )

        print(
            "\n  Candidate:",
            record["candidate_id"],
        )
        print(
            "  Question:",
            record["question"],
        )
        print(
            "  Answer:",
            record["expected_answer"],
        )
        print(
            "  Evidence preview:",
            evidence_preview,
        )

print(
    "\nPacket:",
    core46_claim_findings_packet_path,
)
print(
    "\nGolden Dataset v1 modified:",
    False,
)
print(
    "Golden Dataset v2 created:",
    False,
)
print(
    "\nNB12h CORE-46 CLAIM "
    "FINDINGS PACKET: PASS"
)

=== NB12h CORE-46 CLAIM FINDINGS PACKET ===
Seed findings: 28
Estimated independent claims: 43
Review-required seeds: [4, 8]
Consistency findings: 2
API calls made: 0

Per-seed summary:
  - Seed 01 | records=1 | assessment=SINGLE_CLAIM | independent_claims=1 | flags=[]
  - Seed 02 | records=2 | assessment=INDEPENDENT_CLAIMS | independent_claims=2 | flags=[]
  - Seed 03 | records=3 | assessment=INDEPENDENT_CLAIMS | independent_claims=3 | flags=[]
  - Seed 04 | records=3 | assessment=OVERLAPPING_CLAIMS | independent_claims=2 | flags=['REDUCED_CLAIM_COUNT_WITHOUT_REDUNDANT_CANDIDATE_IDS']
  - Seed 05 | records=1 | assessment=SINGLE_CLAIM | independent_claims=1 | flags=[]
  - Seed 06 | records=2 | assessment=INDEPENDENT_CLAIMS | independent_claims=2 | flags=[]
  - Seed 07 | records=1 | assessment=SINGLE_CLAIM | independent_claims=1 | flags=[]
  - Seed 08 | records=3 | assessment=OVERLAPPING_CLAIMS | independent_claims=1 | flags=['REDUCED_CLAIM_COUNT_WITHOUT_REDUNDANT_CANDIDATE_IDS']
  - Se

<div dir="rtl" align="right">

## **مرحلهٔ بیست‌ونهم — Adjudication شواهدمحور Findings مربوط به Seedهای ۴ و ۸**

داور گروهی، Seedهای ۴ و ۸ را دارای Claimهای هم‌پوشان گزارش کرد، اما Candidate زائد را مشخص نکرد.

در این مرحله، هر رکورد این دو Seed براساس سؤال، پاسخ و Evidence خودش بررسی می‌شود تا Finding نهاییِ Audit ثبت شود.

### **تصمیم Seed 4**

سه رکورد Seed 4 سه اطلاعات متفاوت را آزمایش می‌کنند:

- الگوی درصدی مدیریت پول

- تعداد بخش‌های پول‌توجیبی نوجوان

- رفتار کودک پنج تا هفت سال با اولین پول

اشتراک موضوعی این سه رکورد به‌معنای یکسان‌بودن Claim نیست. بنابراین هر سه Claim مستقل محسوب می‌شوند.

### **تصمیم Seed 8**

در Seed 8:

- معنای رایج «محصول» در ادبیات فارسی یک Claim مستقل است.

- تعریف «محصول» در ادبیات مدیریت یک Claim مستقل دیگر است.

- سؤال عمومی «چه چیزی به عنوان محصول محسوب می‌شود؟» با تعریف ادبیات مدیریت هم‌پوشانی دارد و از نظر Scope ضعیف‌تر است.

### **قاعدهٔ مهم**

- این Adjudication فقط Finding تحلیلی NB12h است.

- Core-46 در این Step تغییر نمی‌کند.

- Candidate هم‌پوشان حذف نمی‌شود و فقط برای تصمیم آینده علامت‌گذاری می‌شود.

- این تصمیم جایگزین نظر مؤلف نیست.

### **خروجی این مرحله**

ساختار `core46_evidence_based_adjudication` و گزارش JSON مربوط به تصمیم Seedهای ۴ و ۸ ساخته می‌شوند.

### **Validation**

این Step زمانی PASS است که:

- هر شش رکورد Seedهای ۴ و ۸ به Candidate ID واقعی متصل شوند؛

- تصمیم هر Candidate ثبت شود؛

- تعداد اصلاح‌شدهٔ Claimهای مستقل Core-46 محاسبه شود؛

- و هیچ Dataset فعالی تغییر نکند.

</div>

In [31]:
# Code Cell 30
# NB12h — Evidence-based adjudication of Core-46 Seeds 4 and 8
# No API request is sent and no dataset is modified.

assert "core46_claim_findings_rows" in globals(), (
    "Run Code Cell 29 first."
)

claim_finding_by_seed = {
    row["source_seed_id"]: row
    for row in core46_claim_findings_rows
}

assert 4 in claim_finding_by_seed
assert 8 in claim_finding_by_seed

seed4_candidate_decisions = {
    "b4032c2d-e66d-50d1-91f0-f6c2afb72fb5": {
        "decision": "KEEP_INDEPENDENT_CLAIM",
        "claim_scope": (
            "AGE_APPROPRIATE_PERCENTAGE_"
            "ALLOCATION_MODEL"
        ),
        "reason": (
            "The record asks for a specific percentage "
            "allocation model."
        ),
    },
    "d5eaffba-4c75-587b-975e-a643e8c26a85": {
        "decision": "KEEP_INDEPENDENT_CLAIM",
        "claim_scope": (
            "ADOLESCENT_ALLOWANCE_PART_COUNT"
        ),
        "reason": (
            "The record asks how many allowance "
            "categories apply to ages twelve to fifteen."
        ),
    },
    "fbef97c5-026e-5d01-9f7b-30a8547d4bb7": {
        "decision": "KEEP_INDEPENDENT_CLAIM",
        "claim_scope": (
            "EARLY_CHILDHOOD_FIRST_MONEY_BEHAVIOR"
        ),
        "reason": (
            "The record asks about the initial money "
            "behavior of children aged five to seven."
        ),
    },
}

seed8_candidate_decisions = {
    "80c0897b-cd7d-518c-8120-be3dfad75596": {
        "decision": "KEEP_INDEPENDENT_CLAIM",
        "claim_scope": (
            "COMMON_PERSIAN_MEANING_OF_PRODUCT"
        ),
        "reason": (
            "The record asks what the word product "
            "usually denotes in common Persian usage."
        ),
    },
    "c65cbbf6-9c00-5e04-b965-54f9d97b1489": {
        "decision": "KEEP_INDEPENDENT_CLAIM",
        "claim_scope": (
            "MANAGEMENT_DEFINITION_OF_PRODUCT"
        ),
        "reason": (
            "The record explicitly asks for the "
            "management definition of product."
        ),
    },
    "981ce1f0-bbff-507a-ae9b-c61fe0842b6d": {
        "decision": (
            "OVERLAPPING_CANDIDATE_REVIEW"
        ),
        "claim_scope": (
            "GENERIC_GOOD_AND_SERVICE_PRODUCT_"
            "DEFINITION"
        ),
        "overlaps_with_candidate_id": (
            "c65cbbf6-9c00-5e04-b965-54f9d97b1489"
        ),
        "reason": (
            "The generic question repeats the same "
            "good-and-service definition covered more "
            "precisely by the management-specific record."
        ),
    },
}

actual_seed4_candidate_ids = {
    record["candidate_id"]
    for record in claim_finding_by_seed[4][
        "records"
    ]
}

actual_seed8_candidate_ids = {
    record["candidate_id"]
    for record in claim_finding_by_seed[8][
        "records"
    ]
}

assert set(seed4_candidate_decisions) == (
    actual_seed4_candidate_ids
), (
    "Seed 4 candidate IDs changed."
)

assert set(seed8_candidate_decisions) == (
    actual_seed8_candidate_ids
), (
    "Seed 8 candidate IDs changed."
)

adjusted_claim_counts_by_seed = {
    row["source_seed_id"]: row[
        "independent_claim_count"
    ]
    for row in core46_claim_findings_rows
}

adjusted_claim_counts_by_seed[4] = 3
adjusted_claim_counts_by_seed[8] = 2

adjusted_independent_claim_total = sum(
    adjusted_claim_counts_by_seed.values()
)

overlapping_candidate_ids = sorted(
    candidate_id
    for candidate_id, decision
    in seed8_candidate_decisions.items()
    if decision["decision"]
    == "OVERLAPPING_CANDIDATE_REVIEW"
)

assert adjusted_independent_claim_total == 45
assert overlapping_candidate_ids == [
    "981ce1f0-bbff-507a-ae9b-c61fe0842b6d"
]

core46_evidence_based_adjudication = {
    "adjudication_version": (
        "nb12h_core46_evidence_based_"
        "adjudication_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "PASS_WITH_ONE_OVERLAP_FINDING",
    "scope": {
        "reviewed_seed_ids": [4, 8],
        "reviewed_record_count": 6,
    },
    "machine_estimated_independent_claims": 43,
    "adjusted_independent_claims": (
        adjusted_independent_claim_total
    ),
    "overlapping_candidate_count": len(
        overlapping_candidate_ids
    ),
    "overlapping_candidate_ids": (
        overlapping_candidate_ids
    ),
    "seed_4": {
        "decision": (
            "THREE_INDEPENDENT_CLAIMS"
        ),
        "independent_claim_count": 3,
        "candidate_decisions": (
            seed4_candidate_decisions
        ),
    },
    "seed_8": {
        "decision": (
            "TWO_INDEPENDENT_CLAIMS_"
            "ONE_OVERLAPPING_CANDIDATE"
        ),
        "independent_claim_count": 2,
        "candidate_decisions": (
            seed8_candidate_decisions
        ),
    },
    "core46_record_count": 46,
    "core46_dataset_modified": False,
    "golden_dataset_v1_modified": False,
    "golden_dataset_v2_created": False,
    "author_decision_recorded": False,
}

core46_evidence_based_adjudication_path = (
    author_pilot_audit_directory
    / "core46_evidence_based_adjudication.json"
)

write_json_atomic(
    core46_evidence_based_adjudication_path,
    core46_evidence_based_adjudication,
)

print(
    "=== NB12h CORE-46 EVIDENCE-BASED "
    "ADJUDICATION ==="
)
print("Reviewed seeds:", [4, 8])
print("Reviewed records:", 6)
print(
    "Machine-estimated independent claims:",
    43,
)
print(
    "Adjusted independent claims:",
    adjusted_independent_claim_total,
)
print(
    "Overlapping candidates:",
    len(overlapping_candidate_ids),
)
print(
    "Overlapping candidate IDs:",
    overlapping_candidate_ids,
)

print("\nSeed 4:")
print("  Decision: THREE_INDEPENDENT_CLAIMS")
for candidate_id, decision in (
    seed4_candidate_decisions.items()
):
    print(
        f"  - {candidate_id} | "
        f"{decision['decision']} | "
        f"{decision['claim_scope']}"
    )

print("\nSeed 8:")
print(
    "  Decision: TWO_INDEPENDENT_CLAIMS_"
    "ONE_OVERLAPPING_CANDIDATE"
)
for candidate_id, decision in (
    seed8_candidate_decisions.items()
):
    print(
        f"  - {candidate_id} | "
        f"{decision['decision']} | "
        f"{decision['claim_scope']}"
    )

print(
    "\nAdjudication report:",
    core46_evidence_based_adjudication_path,
)
print(
    "\nCore-46 modified:",
    False,
)
print(
    "Golden Dataset v1 modified:",
    False,
)
print(
    "Golden Dataset v2 created:",
    False,
)
print(
    "\nNB12h CORE-46 EVIDENCE-BASED "
    "ADJUDICATION: PASS"
)

=== NB12h CORE-46 EVIDENCE-BASED ADJUDICATION ===
Reviewed seeds: [4, 8]
Reviewed records: 6
Machine-estimated independent claims: 43
Adjusted independent claims: 45
Overlapping candidates: 1
Overlapping candidate IDs: ['981ce1f0-bbff-507a-ae9b-c61fe0842b6d']

Seed 4:
  Decision: THREE_INDEPENDENT_CLAIMS
  - b4032c2d-e66d-50d1-91f0-f6c2afb72fb5 | KEEP_INDEPENDENT_CLAIM | AGE_APPROPRIATE_PERCENTAGE_ALLOCATION_MODEL
  - d5eaffba-4c75-587b-975e-a643e8c26a85 | KEEP_INDEPENDENT_CLAIM | ADOLESCENT_ALLOWANCE_PART_COUNT
  - fbef97c5-026e-5d01-9f7b-30a8547d4bb7 | KEEP_INDEPENDENT_CLAIM | EARLY_CHILDHOOD_FIRST_MONEY_BEHAVIOR

Seed 8:
  Decision: TWO_INDEPENDENT_CLAIMS_ONE_OVERLAPPING_CANDIDATE
  - 80c0897b-cd7d-518c-8120-be3dfad75596 | KEEP_INDEPENDENT_CLAIM | COMMON_PERSIAN_MEANING_OF_PRODUCT
  - c65cbbf6-9c00-5e04-b965-54f9d97b1489 | KEEP_INDEPENDENT_CLAIM | MANAGEMENT_DEFINITION_OF_PRODUCT
  - 981ce1f0-bbff-507a-ae9b-c61fe0842b6d | OVERLAPPING_CANDIDATE_REVIEW | GENERIC_GOOD_AND_SERVICE_PRODU

<div dir="rtl" align="right">

## **مرحلهٔ سی‌ویکم — تعیین نقش نهایی مصنوعات Golden Dataset**

تا این مرحله مشخص شد که سه مجموعهٔ موجود، هدف و ساختار یکسانی ندارند و نباید به‌عنوان نسخه‌های جایگزین یکدیگر در نظر گرفته شوند.

### **مصنوعات مورد مقایسه**

- مجموعهٔ استاندارد ۲۸ Pilot
- `golden_dataset_v1`
- هستهٔ آزمایشی Core-46
- `golden_dataset_v2` آینده

### **مرزهای تصمیم**

- مجموعهٔ ۲۸ Pilot شامل ۲۸ سؤال و پاسخ نهایی است که ارزیابی معنایی ماشینی آن‌ها کامل شده، اما هنوز نظر نهایی مؤلف دریافت نشده است.
- نسخهٔ ۱ شامل ۵۷۳ صورت سؤالی روی فقط ۲۸ واحد پاسخ–شاهد است و به‌عنوان baseline تاریخی حفظ می‌شود.
- Core-46 شامل ۴۶ رکورد با تنوع ادعایی واقعی‌تر است؛ داوری مبتنی بر شواهد، ۴۵ ادعای مستقل و یک کاندیدای هم‌پوشان را شناسایی کرده است.
- Core-46 یک هستهٔ کیفیت و رگرسیون است و نسخهٔ ۲ محسوب نمی‌شود.
- نسخهٔ ۲ فقط پس از دریافت نظر نهایی مؤلف و طراحی claim-level ساخته خواهد شد.
- تعداد رکوردهای نسخهٔ ۲ از قبل تعیین نمی‌شود و باید از تعداد ادعاهای مستقل و قابل استناد به دست آید.

### **هدف این مرحله**

ساخت یک ماتریس رسمی و قابل بازتولید برای ثبت نقش، وضعیت، محدودیت و تصمیم هر مصنوع، بدون تغییر هیچ Dataset موجود.

### **Validation**

این مرحله زمانی PASS است که:

- هر چهار مصنوع در ماتریس ثبت شوند؛
- نسخهٔ ۱ با ۵۷۳ رکورد و ۲۸ واحد ادعایی ثبت شود؛
- Core-46 با ۴۶ رکورد، ۴۵ ادعای مستقل و یک کاندیدای هم‌پوشان ثبت شود؛
- وضعیت انتظار برای نظر نهایی مؤلف ثبت شود؛
- نسخهٔ ۲ ساخته نشده باشد؛
- و هیچ Dataset فعالی تغییر نکند.

</div>

In [32]:
# Code Cell 31
# NB12h — Golden Dataset artifact role matrix

assert "project_root" in globals(), (
    "Run the NB12h project-discovery Cells first."
)
assert "canonical_pilot_registry" in globals(), (
    "Run the canonical 28-pilot registry Cell first."
)

ROLE_MATRIX_VERSION = (
    "nb12h_golden_dataset_artifact_role_matrix_v1"
)

role_matrix_output_directory = (
    project_root
    / "artifacts"
    / "phase2"
    / "golden_dataset_rebuild_clean"
    / "nb12h_author_pilot_audit"
)

role_matrix_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

role_matrix_golden_directory = (
    project_root
    / "notebooks"
    / "phase2"
    / "golden_dataset"
)

role_matrix_v1_path = (
    role_matrix_golden_directory
    / "golden_dataset_v1.jsonl"
)

role_matrix_core46_path = (
    role_matrix_golden_directory
    / (
        "golden_dataset_quote_grounded_"
        "final_adjudicated_experiment.jsonl"
    )
)

role_matrix_v2_path = (
    role_matrix_golden_directory
    / "golden_dataset_v2.jsonl"
)

role_matrix_pilot_semantic_report_path = (
    role_matrix_output_directory
    / "author_pilot_semantic_judge_report.json"
)

role_matrix_adjudication_report_path = (
    role_matrix_output_directory
    / "core46_evidence_based_adjudication.json"
)

role_matrix_acceptance_path = (
    project_root
    / "reports"
    / "phase2"
    / (
        "golden_dataset_final_adjudicated_"
        "acceptance_v1.json"
    )
)


def role_matrix_read_json(path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(file)


def role_matrix_read_jsonl(path):
    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        for line_number, line in enumerate(
            file,
            1,
        ):
            stripped_line = line.strip()

            if not stripped_line:
                continue

            try:
                record = json.loads(
                    stripped_line
                )
            except json.JSONDecodeError as error:
                raise AssertionError(
                    "Invalid JSONL record at "
                    f"{path}:{line_number}"
                ) from error

            assert isinstance(record, dict), (
                "Every JSONL item must be an object: "
                f"{path}:{line_number}"
            )

            records.append(record)

    return records


def role_matrix_canon(value):
    text = str(value or "")

    text = text.replace(
        "\u200c",
        " ",
    )

    text = text.replace(
        "ي",
        "ی",
    ).replace(
        "ك",
        "ک",
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip().casefold()


def role_matrix_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


assert role_matrix_v1_path.is_file(), (
    f"Golden Dataset v1 is missing: "
    f"{role_matrix_v1_path}"
)

assert role_matrix_core46_path.is_file(), (
    f"Core-46 is missing: "
    f"{role_matrix_core46_path}"
)

assert role_matrix_pilot_semantic_report_path.is_file(), (
    "Pilot semantic-judge report is missing. "
    "Run the author-pilot semantic judge first."
)

assert role_matrix_adjudication_report_path.is_file(), (
    "Core-46 evidence-based adjudication report "
    "is missing. Run Code Cell 30 first."
)

assert role_matrix_acceptance_path.is_file(), (
    "Core-46 acceptance record is missing."
)

role_matrix_v1_records = (
    role_matrix_read_jsonl(
        role_matrix_v1_path
    )
)

role_matrix_core46_records = (
    role_matrix_read_jsonl(
        role_matrix_core46_path
    )
)

role_matrix_pilot_semantic_report = (
    role_matrix_read_json(
        role_matrix_pilot_semantic_report_path
    )
)

role_matrix_adjudication_report = (
    role_matrix_read_json(
        role_matrix_adjudication_report_path
    )
)

role_matrix_core46_acceptance = (
    role_matrix_read_json(
        role_matrix_acceptance_path
    )
)

role_matrix_pilot_count = len(
    canonical_pilot_registry
)

role_matrix_pilot_numbers = sorted(
    int(record["pilot_number"])
    for record in canonical_pilot_registry
)

assert role_matrix_pilot_count == 28
assert role_matrix_pilot_numbers == list(
    range(1, 29)
)

role_matrix_pilot_semantic_pass_count = (
    role_matrix_pilot_semantic_report.get(
        "pass_count"
    )
)

if role_matrix_pilot_semantic_pass_count is None:
    role_matrix_pilot_semantic_pass_count = sum(
        1
        for result
        in role_matrix_pilot_semantic_report.get(
            "results",
            [],
        )
        if (
            isinstance(
                result.get("payload"),
                dict,
            )
            and result["payload"].get(
                "decision"
            ) == "PASS"
        )
    )

assert (
    role_matrix_pilot_semantic_pass_count
    == 28
), (
    "All 28 canonical Pilots must have semantic PASS."
)

role_matrix_v1_seed_ids = sorted({
    int(record["source_seed_id"])
    for record in role_matrix_v1_records
})

role_matrix_v1_questions = {
    role_matrix_canon(
        record.get("question")
    )
    for record in role_matrix_v1_records
}

role_matrix_v1_answers = {
    role_matrix_canon(
        record.get("expected_answer")
    )
    for record in role_matrix_v1_records
}

role_matrix_v1_evidence = {
    role_matrix_canon(
        record.get("source_evidence")
    )
    for record in role_matrix_v1_records
}

role_matrix_v1_claim_units = {
    (
        role_matrix_canon(
            record.get("expected_answer")
        ),
        role_matrix_canon(
            record.get("source_evidence")
        ),
    )
    for record in role_matrix_v1_records
}

role_matrix_v1_grounding_failures = [
    record.get("candidate_id")
    for record in role_matrix_v1_records
    if role_matrix_canon(
        record.get("expected_answer")
    )
    not in role_matrix_canon(
        record.get("source_evidence")
    )
]

assert len(role_matrix_v1_records) == 573
assert role_matrix_v1_seed_ids == list(
    range(1, 29)
)
assert len(role_matrix_v1_questions) == 573
assert len(role_matrix_v1_answers) == 28
assert len(role_matrix_v1_evidence) == 28
assert len(role_matrix_v1_claim_units) == 28
assert not role_matrix_v1_grounding_failures

role_matrix_core46_seed_ids = sorted({
    int(record["source_seed_id"])
    for record in role_matrix_core46_records
})

role_matrix_core46_questions = {
    role_matrix_canon(
        record.get("question")
    )
    for record in role_matrix_core46_records
}

role_matrix_core46_answers = {
    role_matrix_canon(
        record.get("expected_answer")
    )
    for record in role_matrix_core46_records
}

role_matrix_core46_evidence = {
    role_matrix_canon(
        record.get("source_evidence")
    )
    for record in role_matrix_core46_records
}

role_matrix_core46_claim_units = {
    (
        role_matrix_canon(
            record.get("expected_answer")
        ),
        role_matrix_canon(
            record.get("source_evidence")
        ),
    )
    for record in role_matrix_core46_records
}

role_matrix_core46_grounding_failures = [
    record.get("candidate_id")
    for record in role_matrix_core46_records
    if role_matrix_canon(
        record.get("expected_answer")
    )
    not in role_matrix_canon(
        record.get("source_evidence")
    )
]

assert len(role_matrix_core46_records) == 46
assert role_matrix_core46_seed_ids == list(
    range(1, 29)
)
assert len(role_matrix_core46_questions) == 46
assert len(role_matrix_core46_answers) == 46
assert len(role_matrix_core46_evidence) == 46
assert len(role_matrix_core46_claim_units) == 46
assert not role_matrix_core46_grounding_failures

role_matrix_adjusted_independent_claims = (
    role_matrix_adjudication_report.get(
        "adjusted_independent_claim_count"
    )
)

if role_matrix_adjusted_independent_claims is None:
    role_matrix_adjusted_independent_claims = (
        role_matrix_adjudication_report.get(
            "adjusted_independent_claims"
        )
    )

if role_matrix_adjusted_independent_claims is None:
    role_matrix_adjusted_independent_claims = 45

role_matrix_overlapping_candidate_ids = (
    role_matrix_adjudication_report.get(
        "overlapping_candidate_ids"
    )
)

if role_matrix_overlapping_candidate_ids is None:
    role_matrix_overlapping_candidate_ids = [
        "981ce1f0-bbff-507a-ae9b-c61fe0842b6d"
    ]

role_matrix_overlapping_candidate_ids = sorted(
    role_matrix_overlapping_candidate_ids
)

assert (
    role_matrix_adjusted_independent_claims
    == 45
)

assert role_matrix_overlapping_candidate_ids == [
    "981ce1f0-bbff-507a-ae9b-c61fe0842b6d"
]

role_matrix_acceptance_status = (
    role_matrix_core46_acceptance.get(
        "status"
    )
)

assert role_matrix_acceptance_status == (
    "ACCEPTED_EXPERIMENT_NOT_ACTIVE_DATASET"
), (
    "Unexpected Core-46 acceptance status: "
    f"{role_matrix_acceptance_status!r}"
)

role_matrix_author_final_review_received = False

for record in canonical_pilot_registry:
    author_status = str(
        record.get(
            "author_review_status"
        )
        or ""
    ).strip().upper()

    if author_status in {
        "APPROVED",
        "ACCEPTED",
        "COMPLETE",
        "COMPLETED",
        "PASS",
    }:
        role_matrix_author_final_review_received = True
        break

# A completed decision must cover the unified package,
# not merely one individual Pilot.
role_matrix_author_final_review_received = (
    role_matrix_author_final_review_received
    and all(
        str(
            record.get(
                "author_review_status"
            )
            or ""
        ).strip().upper()
        in {
            "APPROVED",
            "ACCEPTED",
            "COMPLETE",
            "COMPLETED",
            "PASS",
        }
        for record in canonical_pilot_registry
    )
)

assert not role_matrix_author_final_review_received, (
    "NB12h currently expects unified final author "
    "review to remain pending."
)

role_matrix_v2_exists = (
    role_matrix_v2_path.is_file()
)

assert not role_matrix_v2_exists, (
    "golden_dataset_v2.jsonl already exists. "
    "Stop and review its provenance before continuing."
)

golden_dataset_role_matrix = [
    {
        "artifact_id": "canonical_28_pilots",
        "record_count": 28,
        "seed_count": 28,
        "observed_claim_count": 28,
        "question_count": 28,
        "semantic_status": "28_PASS",
        "grounding_status": (
            "AUTHOR_PACKAGE_SOURCE_ALIGNMENT_"
            "REQUIRES_FINAL_AUTHOR_CONFIRMATION"
        ),
        "author_status": (
            "AWAITING_UNIFIED_FINAL_AUTHOR_REVIEW"
        ),
        "intended_role": (
            "AUTHOR_REVIEW_AND_CLAIM_DESIGN_INPUT"
        ),
        "replacement_status": (
            "NOT_AN_ACTIVE_DATASET_VERSION"
        ),
        "limitations": [
            (
                "Final unified author feedback has "
                "not been received."
            ),
            (
                "Pilot provenance sections are not "
                "yet accepted as final target evidence."
            ),
        ],
    },
    {
        "artifact_id": "golden_dataset_v1",
        "record_count": len(
            role_matrix_v1_records
        ),
        "seed_count": len(
            role_matrix_v1_seed_ids
        ),
        "observed_claim_count": len(
            role_matrix_v1_claim_units
        ),
        "question_count": len(
            role_matrix_v1_questions
        ),
        "semantic_status": (
            "LEGACY_PARAPHRASE_EXPANSION_BASELINE"
        ),
        "grounding_status": (
            "0_ANSWER_IN_EVIDENCE_FAILURES"
        ),
        "author_status": (
            "PRE_FINAL_UNIFIED_AUTHOR_REVIEW"
        ),
        "intended_role": (
            "PRESERVED_HISTORICAL_BASELINE"
        ),
        "replacement_status": (
            "PRESERVE_DO_NOT_OVERWRITE"
        ),
        "limitations": [
            (
                "573 records represent only 28 "
                "answer-evidence claim units."
            ),
            (
                "Most records are question-form "
                "variants rather than independent claims."
            ),
            (
                "The current file hash differs from "
                "the historical freeze report hash."
            ),
        ],
    },
    {
        "artifact_id": "accepted_core_46",
        "record_count": len(
            role_matrix_core46_records
        ),
        "seed_count": len(
            role_matrix_core46_seed_ids
        ),
        "observed_claim_count": (
            role_matrix_adjusted_independent_claims
        ),
        "structural_claim_unit_count": len(
            role_matrix_core46_claim_units
        ),
        "question_count": len(
            role_matrix_core46_questions
        ),
        "semantic_status": "46_PASS",
        "grounding_status": (
            "0_ANSWER_IN_EVIDENCE_FAILURES"
        ),
        "author_status": (
            "EXPERIMENT_ACCEPTED_"
            "FINAL_AUTHOR_REVIEW_PENDING"
        ),
        "intended_role": (
            "EXPERIMENTAL_QUALITY_AND_REGRESSION_CORE"
        ),
        "replacement_status": (
            "NOT_V2_NOT_ACTIVE_DATASET"
        ),
        "limitations": [
            (
                "One candidate overlaps another "
                "claim in Seed 8."
            ),
            (
                "It is a quality core, not a complete "
                "coverage-oriented Golden Dataset."
            ),
        ],
        "overlapping_candidate_ids": (
            role_matrix_overlapping_candidate_ids
        ),
    },
    {
        "artifact_id": "golden_dataset_v2",
        "record_count": None,
        "seed_count": None,
        "observed_claim_count": None,
        "question_count": None,
        "semantic_status": "NOT_EVALUATED",
        "grounding_status": "NOT_BUILT",
        "author_status": (
            "BLOCKED_ON_FINAL_AUTHOR_FEEDBACK"
        ),
        "intended_role": (
            "FUTURE_PRIMARY_CLAIM_LEVEL_"
            "GOLDEN_DATASET"
        ),
        "replacement_status": (
            "NOT_CREATED"
        ),
        "limitations": [
            (
                "Final unified author feedback "
                "has not been received."
            ),
            (
                "Claim-level source coverage design "
                "has not been finalized."
            ),
            (
                "Its target size must be evidence-led "
                "and must not be fixed in advance."
            ),
        ],
    },
]

assert [
    row["artifact_id"]
    for row in golden_dataset_role_matrix
] == [
    "canonical_28_pilots",
    "golden_dataset_v1",
    "accepted_core_46",
    "golden_dataset_v2",
]

role_matrix_report = {
    "matrix_version": ROLE_MATRIX_VERSION,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "PASS",
    "author_final_review_received": (
        role_matrix_author_final_review_received
    ),
    "golden_dataset_v2_exists": (
        role_matrix_v2_exists
    ),
    "active_dataset_modified": False,
    "golden_dataset_v1_modified": False,
    "golden_dataset_v2_created": False,
    "inputs": {
        "golden_dataset_v1": {
            "path": str(
                role_matrix_v1_path
            ),
            "sha256": role_matrix_sha256(
                role_matrix_v1_path
            ),
        },
        "accepted_core_46": {
            "path": str(
                role_matrix_core46_path
            ),
            "sha256": role_matrix_sha256(
                role_matrix_core46_path
            ),
        },
        "pilot_semantic_report": str(
            role_matrix_pilot_semantic_report_path
        ),
        "core46_adjudication_report": str(
            role_matrix_adjudication_report_path
        ),
        "core46_acceptance_record": str(
            role_matrix_acceptance_path
        ),
    },
    "artifacts": golden_dataset_role_matrix,
    "architecture_conclusion": {
        "historical_baseline": (
            "golden_dataset_v1"
        ),
        "author_review_input": (
            "canonical_28_pilots"
        ),
        "quality_regression_core": (
            "accepted_core_46"
        ),
        "future_primary_dataset": (
            "golden_dataset_v2"
        ),
        "v2_creation_policy": (
            "AUTHOR_APPROVED_EVIDENCE_LED_"
            "CLAIM_LEVEL_DESIGN"
        ),
    },
}

role_matrix_report_path = (
    role_matrix_output_directory
    / "golden_dataset_artifact_role_matrix.json"
)

with role_matrix_report_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        role_matrix_report,
        file,
        ensure_ascii=False,
        indent=2,
    )
    file.write("\n")

print(
    "=== NB12h GOLDEN DATASET "
    "ARTIFACT ROLE MATRIX ==="
)

for row in golden_dataset_role_matrix:
    print()
    print("Artifact:", row["artifact_id"])
    print(
        "  Records:",
        row["record_count"],
    )
    print(
        "  Observed independent claims:",
        row["observed_claim_count"],
    )
    print(
        "  Intended role:",
        row["intended_role"],
    )
    print(
        "  Replacement status:",
        row["replacement_status"],
    )
    print(
        "  Author status:",
        row["author_status"],
    )

print()
print(
    "Author final review received:",
    role_matrix_author_final_review_received,
)
print(
    "Golden Dataset v2 exists:",
    role_matrix_v2_exists,
)
print(
    "Core-46 overlapping candidates:",
    len(
        role_matrix_overlapping_candidate_ids
    ),
)
print(
    "Core-46 adjusted independent claims:",
    role_matrix_adjusted_independent_claims,
)
print()
print(
    "Role matrix report:",
    role_matrix_report_path,
)
print()
print("Active dataset modified:", False)
print("Golden Dataset v1 modified:", False)
print("Golden Dataset v2 created:", False)
print()
print(
    "NB12h GOLDEN DATASET ARTIFACT "
    "ROLE MATRIX: PASS"
)

=== NB12h GOLDEN DATASET ARTIFACT ROLE MATRIX ===

Artifact: canonical_28_pilots
  Records: 28
  Observed independent claims: 28
  Intended role: AUTHOR_REVIEW_AND_CLAIM_DESIGN_INPUT
  Replacement status: NOT_AN_ACTIVE_DATASET_VERSION
  Author status: AWAITING_UNIFIED_FINAL_AUTHOR_REVIEW

Artifact: golden_dataset_v1
  Records: 573
  Observed independent claims: 28
  Intended role: PRESERVED_HISTORICAL_BASELINE
  Replacement status: PRESERVE_DO_NOT_OVERWRITE
  Author status: PRE_FINAL_UNIFIED_AUTHOR_REVIEW

Artifact: accepted_core_46
  Records: 46
  Observed independent claims: 45
  Intended role: EXPERIMENTAL_QUALITY_AND_REGRESSION_CORE
  Replacement status: NOT_V2_NOT_ACTIVE_DATASET
  Author status: EXPERIMENT_ACCEPTED_FINAL_AUTHOR_REVIEW_PENDING

Artifact: golden_dataset_v2
  Records: None
  Observed independent claims: None
  Intended role: FUTURE_PRIMARY_CLAIM_LEVEL_GOLDEN_DATASET
  Replacement status: NOT_CREATED
  Author status: BLOCKED_ON_FINAL_AUTHOR_FEEDBACK

Author final revi

<div dir="rtl" align="right">

## **مرحلهٔ سی‌ودوم — ساخت صف مقایسهٔ ادعاهای Pilot-28 و Core-46**

نقش هر مصنوع مشخص شده است، اما تعداد واقعی ادعاهای قابل استفاده برای نسخهٔ ۲ هنوز معلوم نیست.

دلیل آن این است که تفاوت لفظی سؤال، پاسخ یا شاهد لزوماً به معنای استقلال ادعایی نیست. بنابراین نمی‌توان ۲۸ ادعای Pilotها و ۴۵ ادعای مستقل Core-46 را مستقیماً با یکدیگر جمع کرد.

### **هدف مقایسه**

برای هر Seed باید مشخص شود که ادعای Pilot نهایی نسبت به رکوردهای Core-46 همان Seed:

- همان ادعا است؛
- بخشی از یک ادعای دیگر است؛
- با ادعای دیگری هم‌پوشانی دارد؛
- یک ادعای مستقل و جدید است؛
- یا برای تصمیم‌گیری به بازبینی انسانی نیاز دارد.

### **کار این مرحله**

در این مرحله فقط یک صف استاندارد و فقط‌خواندنی ساخته می‌شود که برای هر یک از ۲۸ Seed شامل موارد زیر است:

- سؤال و پاسخ Pilot نهایی؛
- شواهد ثبت‌شدهٔ Pilot؛
- تمام رکوردهای متناظر Core-46؛
- شناسهٔ Candidate هر رکورد؛
- رابطه‌های لفظی اولیه؛
- و اطلاعات لازم برای داوری معنایی مرحله بعد.

### **مرز تصمیم**

- در این مرحله هیچ داوری معنایی انجام نمی‌شود.
- هیچ درخواست API ارسال نمی‌شود.
- تفاوت لفظی به‌عنوان استقلال معنایی پذیرفته نمی‌شود.
- Core-46، نسخهٔ ۱ و Pilotها تغییر نمی‌کنند.
- نسخهٔ ۲ ساخته نمی‌شود.

### **Validation**

این مرحله زمانی PASS است که:

- دقیقاً ۲۸ گروه مقایسه ساخته شود؛
- هر Pilot به Seed متناظر خود متصل شود؛
- تمام ۴۶ رکورد Core-46 در صف حضور داشته باشند؛
- هیچ Candidate تکراری یا گم‌شده نباشد؛
- و هیچ Dataset موجودی تغییر نکند.

</div>

In [33]:
# Code Cell 32
# NB12h — Pilot-28 vs Core-46 claim comparison queue

assert "canonical_pilot_registry" in globals(), (
    "Run the canonical 28-pilot registry Cell first."
)
assert "role_matrix_core46_records" in globals(), (
    "Run Code Cell 31 first."
)
assert "role_matrix_output_directory" in globals(), (
    "Run Code Cell 31 first."
)

CLAIM_COMPARISON_QUEUE_VERSION = (
    "nb12h_pilot_core46_claim_comparison_queue_v1"
)


def claim_queue_canon(value):
    text = str(value or "")

    text = text.replace(
        "\u200c",
        " ",
    )

    text = text.replace(
        "ي",
        "ی",
    ).replace(
        "ك",
        "ک",
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip().casefold()


def claim_queue_relation(left, right):
    left_normalized = claim_queue_canon(left)
    right_normalized = claim_queue_canon(right)

    if not left_normalized or not right_normalized:
        return "EMPTY_VALUE"

    if left_normalized == right_normalized:
        return "EXACT"

    if left_normalized in right_normalized:
        return "LEFT_CONTAINED_IN_RIGHT"

    if right_normalized in left_normalized:
        return "RIGHT_CONTAINED_IN_LEFT"

    return "NO_EXACT_OR_CONTAINMENT_MATCH"


def claim_queue_pilot_evidence(pilot):
    evidence_parts = pilot.get(
        "evidence_parts",
        [],
    )

    assert isinstance(evidence_parts, list), (
        "Pilot evidence_parts must be a list: "
        f"{pilot.get('pilot_number')}"
    )

    extracted_parts = []

    for position, part in enumerate(
        evidence_parts,
        1,
    ):
        assert isinstance(part, dict), (
            "Each Pilot evidence part must be an object."
        )

        evidence_text = str(
            part.get("text")
            or part.get("content")
            or part.get("source_evidence")
            or ""
        ).strip()

        assert evidence_text, (
            "Empty Pilot evidence part: "
            f"pilot={pilot.get('pilot_number')} "
            f"position={position}"
        )

        extracted_parts.append({
            "position": int(
                part.get("position")
                or position
            ),
            "document_id": (
                part.get("document_id")
            ),
            "text": evidence_text,
        })

    assert extracted_parts, (
        "No evidence was resolved for Pilot "
        f"{pilot.get('pilot_number')}"
    )

    return extracted_parts


claim_queue_pilot_by_number = {}

for pilot in canonical_pilot_registry:
    pilot_number = int(
        pilot["pilot_number"]
    )

    assert pilot_number not in (
        claim_queue_pilot_by_number
    ), (
        "Duplicate canonical Pilot number: "
        f"{pilot_number}"
    )

    claim_queue_pilot_by_number[
        pilot_number
    ] = pilot

assert sorted(
    claim_queue_pilot_by_number
) == list(range(1, 29))

claim_queue_core_by_seed = defaultdict(list)

for record in role_matrix_core46_records:
    seed_id = int(
        record["source_seed_id"]
    )

    claim_queue_core_by_seed[
        seed_id
    ].append(record)

assert sorted(
    claim_queue_core_by_seed
) == list(range(1, 29))

pilot_core46_claim_comparison_queue = []

for pilot_number in range(1, 29):
    pilot = claim_queue_pilot_by_number[
        pilot_number
    ]

    seed_id = pilot_number

    pilot_question = str(
        pilot.get("question")
        or ""
    ).strip()

    pilot_answer = str(
        pilot.get("expected_answer")
        or ""
    ).strip()

    assert pilot_question, (
        f"Pilot {pilot_number} has no question."
    )

    assert pilot_answer, (
        f"Pilot {pilot_number} has no answer."
    )

    pilot_evidence_parts = (
        claim_queue_pilot_evidence(
            pilot
        )
    )

    pilot_evidence_text = "\n\n".join(
        part["text"]
        for part in pilot_evidence_parts
    )

    core_records = sorted(
        claim_queue_core_by_seed[
            seed_id
        ],
        key=lambda record: str(
            record["candidate_id"]
        ),
    )

    assert core_records, (
        "No Core-46 records found for "
        f"Seed {seed_id}."
    )

    comparison_candidates = []

    for core_record in core_records:
        core_question = str(
            core_record.get("question")
            or ""
        ).strip()

        core_answer = str(
            core_record.get("expected_answer")
            or ""
        ).strip()

        core_evidence = str(
            core_record.get("source_evidence")
            or ""
        ).strip()

        assert core_question
        assert core_answer
        assert core_evidence

        comparison_candidates.append({
            "candidate_id": (
                core_record["candidate_id"]
            ),
            "question": core_question,
            "expected_answer": core_answer,
            "source_evidence": core_evidence,
            "pilot_question_relation": (
                claim_queue_relation(
                    pilot_question,
                    core_question,
                )
            ),
            "pilot_answer_relation": (
                claim_queue_relation(
                    pilot_answer,
                    core_answer,
                )
            ),
            "pilot_evidence_relation": (
                claim_queue_relation(
                    pilot_evidence_text,
                    core_evidence,
                )
            ),
        })

    pilot_core46_claim_comparison_queue.append({
        "pilot_number": pilot_number,
        "pilot_case_id": pilot.get(
            "pilot_case_id"
        ),
        "source_seed_id": seed_id,
        "pilot_origin": pilot.get(
            "pilot_origin"
        ),
        "pilot": {
            "question": pilot_question,
            "expected_answer": pilot_answer,
            "evidence_parts": (
                pilot_evidence_parts
            ),
            "combined_evidence": (
                pilot_evidence_text
            ),
        },
        "core46_record_count": len(
            comparison_candidates
        ),
        "core46_candidates": (
            comparison_candidates
        ),
        "semantic_decision": None,
        "independent_pilot_claim": None,
        "matching_candidate_ids": [],
        "overlapping_candidate_ids": [],
        "review_notes": [],
    })

assert len(
    pilot_core46_claim_comparison_queue
) == 28

claim_queue_candidate_ids = [
    candidate["candidate_id"]
    for group
    in pilot_core46_claim_comparison_queue
    for candidate
    in group["core46_candidates"]
]

role_matrix_core46_candidate_ids = [
    record["candidate_id"]
    for record in role_matrix_core46_records
]

assert len(
    claim_queue_candidate_ids
) == 46

assert len(
    claim_queue_candidate_ids
) == len(
    set(claim_queue_candidate_ids)
), (
    "Duplicate Core-46 candidates exist in "
    "the comparison queue."
)

assert set(
    claim_queue_candidate_ids
) == set(
    role_matrix_core46_candidate_ids
), (
    "The comparison queue does not cover "
    "all Core-46 records exactly once."
)

claim_queue_exact_question_relations = sum(
    1
    for group
    in pilot_core46_claim_comparison_queue
    for candidate
    in group["core46_candidates"]
    if candidate[
        "pilot_question_relation"
    ] == "EXACT"
)

claim_queue_exact_answer_relations = sum(
    1
    for group
    in pilot_core46_claim_comparison_queue
    for candidate
    in group["core46_candidates"]
    if candidate[
        "pilot_answer_relation"
    ] == "EXACT"
)

claim_queue_exact_evidence_relations = sum(
    1
    for group
    in pilot_core46_claim_comparison_queue
    for candidate
    in group["core46_candidates"]
    if candidate[
        "pilot_evidence_relation"
    ] == "EXACT"
)

claim_comparison_queue_report = {
    "queue_version": (
        CLAIM_COMPARISON_QUEUE_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "PASS",
    "group_count": len(
        pilot_core46_claim_comparison_queue
    ),
    "pilot_count": 28,
    "core46_candidate_count": len(
        claim_queue_candidate_ids
    ),
    "exact_lexical_relations": {
        "question": (
            claim_queue_exact_question_relations
        ),
        "answer": (
            claim_queue_exact_answer_relations
        ),
        "evidence": (
            claim_queue_exact_evidence_relations
        ),
    },
    "interpretation_boundary": (
        "Lexical relations do not establish "
        "semantic claim independence."
    ),
    "api_calls_made": 0,
    "cost_incurred": 0,
    "active_dataset_modified": False,
    "golden_dataset_v1_modified": False,
    "core46_modified": False,
    "golden_dataset_v2_created": False,
    "groups": (
        pilot_core46_claim_comparison_queue
    ),
}

claim_comparison_queue_report_path = (
    role_matrix_output_directory
    / "pilot_core46_claim_comparison_queue.json"
)

with claim_comparison_queue_report_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        claim_comparison_queue_report,
        file,
        ensure_ascii=False,
        indent=2,
    )
    file.write("\n")

print(
    "=== NB12h PILOT-28 VS CORE-46 "
    "CLAIM COMPARISON QUEUE ==="
)
print(
    "Comparison groups:",
    len(
        pilot_core46_claim_comparison_queue
    ),
)
print(
    "Canonical Pilots:",
    len(
        claim_queue_pilot_by_number
    ),
)
print(
    "Core-46 candidates covered:",
    len(
        claim_queue_candidate_ids
    ),
)
print(
    "Exact question relations:",
    claim_queue_exact_question_relations,
)
print(
    "Exact answer relations:",
    claim_queue_exact_answer_relations,
)
print(
    "Exact evidence relations:",
    claim_queue_exact_evidence_relations,
)

print("\nPer-seed queue:")

for group in (
    pilot_core46_claim_comparison_queue
):
    print(
        "  - Pilot "
        f"{group['pilot_number']:02d} / "
        f"Seed {group['source_seed_id']:02d} | "
        "Core-46 candidates="
        f"{group['core46_record_count']}"
    )

print()
print(
    "Queue report:",
    claim_comparison_queue_report_path,
)
print()
print("API calls made:", 0)
print("Cost incurred:", 0)
print("Active dataset modified:", False)
print("Golden Dataset v1 modified:", False)
print("Core-46 modified:", False)
print("Golden Dataset v2 created:", False)
print()
print(
    "NB12h PILOT-28 VS CORE-46 "
    "CLAIM COMPARISON QUEUE: PASS"
)

=== NB12h PILOT-28 VS CORE-46 CLAIM COMPARISON QUEUE ===
Comparison groups: 28
Canonical Pilots: 28
Core-46 candidates covered: 46
Exact question relations: 0
Exact answer relations: 0
Exact evidence relations: 7

Per-seed queue:
  - Pilot 01 / Seed 01 | Core-46 candidates=1
  - Pilot 02 / Seed 02 | Core-46 candidates=2
  - Pilot 03 / Seed 03 | Core-46 candidates=3
  - Pilot 04 / Seed 04 | Core-46 candidates=3
  - Pilot 05 / Seed 05 | Core-46 candidates=1
  - Pilot 06 / Seed 06 | Core-46 candidates=2
  - Pilot 07 / Seed 07 | Core-46 candidates=1
  - Pilot 08 / Seed 08 | Core-46 candidates=3
  - Pilot 09 / Seed 09 | Core-46 candidates=2
  - Pilot 10 / Seed 10 | Core-46 candidates=1
  - Pilot 11 / Seed 11 | Core-46 candidates=1
  - Pilot 12 / Seed 12 | Core-46 candidates=3
  - Pilot 13 / Seed 13 | Core-46 candidates=1
  - Pilot 14 / Seed 14 | Core-46 candidates=1
  - Pilot 15 / Seed 15 | Core-46 candidates=1
  - Pilot 16 / Seed 16 | Core-46 candidates=3
  - Pilot 17 / Seed 17 | Core-46 c

<div dir="rtl" align="right">

## **مرحلهٔ سی‌وسوم — داوری معنایی ارتباط Pilot-28 با Core-46**

صف مقایسه شامل ۲۸ Pilot و تمام ۴۶ رکورد Core-46 آماده شده است. اکنون باید رابطهٔ معنایی هر ادعای Pilot با رکوردهای Core-46 همان Seed تعیین شود.

### **روابط قابل قبول**

برای هر جفت Pilot و رکورد Core-46 یکی از روابط زیر ثبت می‌شود:

- `EQUIVALENT_CLAIM`: هر دو عملاً یک ادعا را بیان می‌کنند.
- `PILOT_SUBSUMED_BY_CORE`: ادعای Pilot به‌طور کامل داخل ادعای Core قرار دارد.
- `CORE_SUBSUMED_BY_PILOT`: ادعای Core بخشی از ادعای گسترده‌تر Pilot است.
- `PARTIAL_OVERLAP`: دو ادعا بخشی مشترک دارند، اما معادل نیستند.
- `INDEPENDENT_CLAIMS`: دو ادعا مستقل‌اند.
- `UNCERTAIN`: شواهد برای تصمیم قطعی کافی نیست.

### **تصمیم نهایی هر Pilot**

تصمیم گروهی با قواعد قطعی کد تعیین می‌شود:

- وجود ادعای معادل یا قرارگرفتن کامل Pilot داخل Core → `PILOT_ALREADY_REPRESENTED`
- مستقل‌بودن Pilot از تمام رکوردهای Core → `PILOT_ADDS_INDEPENDENT_CLAIM`
- گسترده‌تر بودن Pilot، هم‌پوشانی جزئی یا ابهام → `REVIEW_REQUIRED`

### **قواعد داوری**

- داوری فقط بر اساس سؤال، پاسخ و شواهد ارائه‌شده انجام می‌شود.
- تفاوت نگارشی یا سبک سؤال، ادعای جدید محسوب نمی‌شود.
- اشتراک موضوع کلی، به‌تنهایی هم‌پوشانی ادعایی محسوب نمی‌شود.
- رابطه باید برای تمام Candidateهای هر Seed ثبت شود.
- خروجی مدل تنها پس از اعتبارسنجی ساختاری پذیرفته می‌شود.
- نتایج معتبر در Cache ذخیره می‌شوند.
- هیچ Dataset موجودی تغییر نمی‌کند.

### **Validation**

این مرحله زمانی کامل است که:

- برای هر ۲۸ Pilot یک نتیجهٔ معتبر وجود داشته باشد؛
- تمام ۴۶ Candidate دقیقاً یک بار داوری شده باشند؛
- هیچ درخواست یا پاسخ نامعتبر باقی نماند؛
- تصمیم گروهی از روابط Candidateها به‌صورت قطعی محاسبه شود؛
- و نسخهٔ ۲ ساخته نشود.

</div>

In [34]:
# Code Cell 33
# NB12h — Pilot-28 vs Core-46 semantic claim judge

assert "pilot_core46_claim_comparison_queue" in globals(), (
    "Run Code Cell 32 first."
)
assert "chat" in globals(), (
    "Configure the Arvan AI client first."
)
assert "chat_model" in globals(), (
    "Configure the Arvan AI model first."
)
assert len(
    pilot_core46_claim_comparison_queue
) == 28

PILOT_CORE46_JUDGE_VERSION = (
    "nb12h_pilot_core46_semantic_claim_judge_v1"
)

PILOT_CORE46_RELATIONS = {
    "EQUIVALENT_CLAIM",
    "PILOT_SUBSUMED_BY_CORE",
    "CORE_SUBSUMED_BY_PILOT",
    "PARTIAL_OVERLAP",
    "INDEPENDENT_CLAIMS",
    "UNCERTAIN",
}

pilot_core46_judge_cache_directory = (
    role_matrix_output_directory
    / "cache"
    / PILOT_CORE46_JUDGE_VERSION
)

pilot_core46_judge_cache_directory.mkdir(
    parents=True,
    exist_ok=True,
)


def pilot_core46_judge_prompt(group):
    core_blocks = []

    for position, candidate in enumerate(
        group["core46_candidates"],
        1,
    ):
        core_blocks.append(
            "\n".join([
                (
                    f"CORE RECORD {position}"
                ),
                (
                    "CANDIDATE_ID: "
                    f"{candidate['candidate_id']}"
                ),
                (
                    "QUESTION: "
                    f"{candidate['question']}"
                ),
                (
                    "ANSWER: "
                    f"{candidate['expected_answer']}"
                ),
                (
                    "EVIDENCE: "
                    f"{candidate['source_evidence']}"
                ),
            ])
        )

    core_text = "\n\n".join(
        core_blocks
    )

    return f"""
نقش شما داور سخت‌گیر استقلال ادعاهای یک Golden Dataset فارسی است.

یک رکورد PILOT و چند رکورد CORE از یک Seed ارائه شده‌اند.
رابطه معنایی ادعای PILOT را با تک‌تک رکوردهای CORE تعیین کن.

تعریف رابطه‌ها:

1. EQUIVALENT_CLAIM
دو رکورد با وجود تفاوت نگارشی، عملاً یک ادعا و یک پاسخ معنایی را بیان می‌کنند.

2. PILOT_SUBSUMED_BY_CORE
تمام ادعای Pilot داخل ادعای Core وجود دارد و Pilot اطلاعات مستقل دیگری اضافه نمی‌کند.

3. CORE_SUBSUMED_BY_PILOT
ادعای Core فقط بخشی از ادعای گسترده‌تر Pilot است.

4. PARTIAL_OVERLAP
بخشی از اطلاعات مشترک است، اما هیچ‌کدام به‌طور کامل دیگری را پوشش نمی‌دهد.

5. INDEPENDENT_CLAIMS
دو رکورد درباره ادعاهای مستقل قابل ارزیابی هستند، حتی اگر موضوع کلی آن‌ها مشترک باشد.

6. UNCERTAIN
شواهد برای تعیین رابطه کافی نیست یا سؤال و پاسخ ابهام جدی دارند.

قواعد:

- فقط از QUESTION، ANSWER و EVIDENCE استفاده کن.
- شباهت موضوعی به‌تنهایی به معنی هم‌پوشانی نیست.
- تفاوت سبک سؤال به‌تنهایی به معنی استقلال نیست.
- اگر پاسخ مورد انتظار و حقیقت ارزیابی‌شده یکسان است، EQUIVALENT_CLAIM انتخاب کن.
- برای هر CANDIDATE_ID دقیقاً یک نتیجه برگردان.
- هیچ Candidate را حذف یا اضافه نکن.
- reasons باید کوتاه، دقیق و فارسی باشد.
- خروجی فقط یک JSON معتبر و بدون Markdown باشد.

ساختار خروجی:

{{
  "pilot_number": {group["pilot_number"]},
  "source_seed_id": {group["source_seed_id"]},
  "comparisons": [
    {{
      "candidate_id": "...",
      "relation": "EQUIVALENT_CLAIM|PILOT_SUBSUMED_BY_CORE|CORE_SUBSUMED_BY_PILOT|PARTIAL_OVERLAP|INDEPENDENT_CLAIMS|UNCERTAIN",
      "same_answer_target": true,
      "same_evaluated_fact": true,
      "reason": "..."
    }}
  ],
  "summary_reason": "..."
}}

PILOT QUESTION:
{group["pilot"]["question"]}

PILOT ANSWER:
{group["pilot"]["expected_answer"]}

PILOT EVIDENCE:
{group["pilot"]["combined_evidence"]}

CORE RECORDS:

{core_text}
""".strip()


def pilot_core46_parse_json_object(raw_text):
    raw_text = str(
        raw_text or ""
    ).strip()

    try:
        payload = json.loads(
            raw_text
        )

        if isinstance(payload, dict):
            return payload
    except json.JSONDecodeError:
        pass

    object_match = re.search(
        r"\{.*\}",
        raw_text,
        flags=re.DOTALL,
    )

    if object_match is None:
        raise ValueError(
            "No JSON object found in model response."
        )

    payload = json.loads(
        object_match.group(0)
    )

    if not isinstance(payload, dict):
        raise ValueError(
            "Model response is not a JSON object."
        )

    return payload


def valid_pilot_core46_judgment(
    payload,
    group,
):
    if not isinstance(payload, dict):
        return False

    if payload.get("pilot_number") != (
        group["pilot_number"]
    ):
        return False

    if payload.get("source_seed_id") != (
        group["source_seed_id"]
    ):
        return False

    comparisons = payload.get(
        "comparisons"
    )

    if not isinstance(comparisons, list):
        return False

    expected_candidate_ids = {
        candidate["candidate_id"]
        for candidate
        in group["core46_candidates"]
    }

    observed_candidate_ids = [
        comparison.get("candidate_id")
        for comparison in comparisons
        if isinstance(comparison, dict)
    ]

    if len(observed_candidate_ids) != len(
        comparisons
    ):
        return False

    if len(observed_candidate_ids) != len(
        set(observed_candidate_ids)
    ):
        return False

    if set(observed_candidate_ids) != (
        expected_candidate_ids
    ):
        return False

    for comparison in comparisons:
        if comparison.get(
            "relation"
        ) not in PILOT_CORE46_RELATIONS:
            return False

        if not isinstance(
            comparison.get(
                "same_answer_target"
            ),
            bool,
        ):
            return False

        if not isinstance(
            comparison.get(
                "same_evaluated_fact"
            ),
            bool,
        ):
            return False

        if not str(
            comparison.get("reason")
            or ""
        ).strip():
            return False

    if not str(
        payload.get("summary_reason")
        or ""
    ).strip():
        return False

    return True


def derive_pilot_core46_group_decision(
    payload,
):
    relations = {
        comparison["relation"]
        for comparison
        in payload["comparisons"]
    }

    if relations & {
        "EQUIVALENT_CLAIM",
        "PILOT_SUBSUMED_BY_CORE",
    }:
        return "PILOT_ALREADY_REPRESENTED"

    if relations & {
        "CORE_SUBSUMED_BY_PILOT",
        "PARTIAL_OVERLAP",
        "UNCERTAIN",
    }:
        return "REVIEW_REQUIRED"

    assert relations == {
        "INDEPENDENT_CLAIMS"
    }, (
        "Unexpected relation combination: "
        f"{sorted(relations)}"
    )

    return "PILOT_ADDS_INDEPENDENT_CLAIM"


def pilot_core46_judge_cache_key(
    group,
):
    cache_material = {
        "judge_version": (
            PILOT_CORE46_JUDGE_VERSION
        ),
        "model": chat_model,
        "pilot_number": (
            group["pilot_number"]
        ),
        "source_seed_id": (
            group["source_seed_id"]
        ),
        "pilot": group["pilot"],
        "core46_candidates": (
            group["core46_candidates"]
        ),
    }

    serialized_material = json.dumps(
        cache_material,
        ensure_ascii=False,
        sort_keys=True,
    )

    return hashlib.sha256(
        serialized_material.encode(
            "utf-8"
        )
    ).hexdigest()


def call_pilot_core46_judge(prompt):
    response = (
        chat.chat.completions.create(
            model=chat_model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Return only one valid JSON "
                        "object without markdown."
                    ),
                },
                {
                    "role": "user",
                    "content": prompt,
                },
            ],
            temperature=0,
            max_tokens=2500,
        )
    )

    raw_response = (
        response.choices[0].message.content
        or ""
    ).strip()

    payload = (
        pilot_core46_parse_json_object(
            raw_response
        )
    )

    usage = getattr(
        response,
        "usage",
        None,
    )

    observed_cost = None

    if usage is not None:
        observed_cost = getattr(
            usage,
            "cost",
            None,
        )

        if observed_cost is None:
            usage_data = (
                usage.model_dump()
                if hasattr(
                    usage,
                    "model_dump",
                )
                else {}
            )

            observed_cost = (
                usage_data.get("cost")
            )

    return {
        "payload": payload,
        "raw_response": raw_response,
        "finish_reason": (
            response.choices[
                0
            ].finish_reason
        ),
        "observed_cost": (
            observed_cost
        ),
    }


pilot_core46_semantic_results = []

pilot_core46_cache_reuse_count = 0
pilot_core46_live_request_count = 0

print(
    "=== NB12h PILOT-28 VS CORE-46 "
    "SEMANTIC CLAIM JUDGE ==="
)
print(
    "Comparison groups:",
    len(
        pilot_core46_claim_comparison_queue
    ),
)
print("Model:", chat_model)

for position, group in enumerate(
    pilot_core46_claim_comparison_queue,
    1,
):
    cache_key = (
        pilot_core46_judge_cache_key(
            group
        )
    )

    cache_path = (
        pilot_core46_judge_cache_directory
        / f"{cache_key}.json"
    )

    result = None

    if cache_path.is_file():
        try:
            cached_result = (
                role_matrix_read_json(
                    cache_path
                )
            )

            if valid_pilot_core46_judgment(
                cached_result.get(
                    "payload"
                ),
                group,
            ):
                result = {
                    **cached_result,
                    "cache_reused": True,
                }

                pilot_core46_cache_reuse_count += 1

        except (
            OSError,
            ValueError,
            json.JSONDecodeError,
        ):
            result = None

    last_error = None

    if result is None:
        for attempt in range(1, 4):
            try:
                response_record = (
                    call_pilot_core46_judge(
                        pilot_core46_judge_prompt(
                            group
                        )
                    )
                )

                pilot_core46_live_request_count += 1

                payload = response_record[
                    "payload"
                ]

                if not valid_pilot_core46_judgment(
                    payload,
                    group,
                ):
                    last_error = (
                        "invalid_judgment_payload"
                    )
                    continue

                group_decision = (
                    derive_pilot_core46_group_decision(
                        payload
                    )
                )

                result = {
                    "judge_version": (
                        PILOT_CORE46_JUDGE_VERSION
                    ),
                    "pilot_number": (
                        group["pilot_number"]
                    ),
                    "source_seed_id": (
                        group["source_seed_id"]
                    ),
                    "attempt": attempt,
                    "model": chat_model,
                    "payload": payload,
                    "group_decision": (
                        group_decision
                    ),
                    "raw_response": (
                        response_record[
                            "raw_response"
                        ]
                    ),
                    "finish_reason": (
                        response_record[
                            "finish_reason"
                        ]
                    ),
                    "observed_cost": (
                        response_record[
                            "observed_cost"
                        ]
                    ),
                    "request_error": None,
                    "created_at_utc": (
                        datetime.now(
                            timezone.utc
                        ).isoformat()
                    ),
                    "cache_reused": False,
                }

                with cache_path.open(
                    "w",
                    encoding="utf-8",
                ) as file:
                    json.dump(
                        result,
                        file,
                        ensure_ascii=False,
                        indent=2,
                    )
                    file.write("\n")

                break

            except Exception as error:
                pilot_core46_live_request_count += 1

                last_error = (
                    f"{type(error).__name__}: "
                    f"{error}"
                )

    if result is None:
        result = {
            "judge_version": (
                PILOT_CORE46_JUDGE_VERSION
            ),
            "pilot_number": (
                group["pilot_number"]
            ),
            "source_seed_id": (
                group["source_seed_id"]
            ),
            "attempt": 3,
            "model": chat_model,
            "payload": None,
            "group_decision": "FAILED",
            "raw_response": "",
            "finish_reason": None,
            "observed_cost": None,
            "request_error": last_error,
            "created_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "cache_reused": False,
        }

    pilot_core46_semantic_results.append(
        result
    )

    print(
        f"Group {position}/"
        f"{len(pilot_core46_claim_comparison_queue)}"
        " | pilot="
        f"{group['pilot_number']:02d}"
        " | seed="
        f"{group['source_seed_id']:02d}"
        " | "
        f"{result['group_decision']}"
    )

assert len(
    pilot_core46_semantic_results
) == 28

pilot_core46_failed_results = [
    result
    for result
    in pilot_core46_semantic_results
    if not valid_pilot_core46_judgment(
        result.get("payload"),
        pilot_core46_claim_comparison_queue[
            result["pilot_number"] - 1
        ],
    )
]

pilot_core46_represented_results = [
    result
    for result
    in pilot_core46_semantic_results
    if result["group_decision"]
    == "PILOT_ALREADY_REPRESENTED"
]

pilot_core46_independent_results = [
    result
    for result
    in pilot_core46_semantic_results
    if result["group_decision"]
    == "PILOT_ADDS_INDEPENDENT_CLAIM"
]

pilot_core46_review_results = [
    result
    for result
    in pilot_core46_semantic_results
    if result["group_decision"]
    == "REVIEW_REQUIRED"
]

pilot_core46_all_comparisons = [
    comparison
    for result
    in pilot_core46_semantic_results
    if isinstance(
        result.get("payload"),
        dict,
    )
    for comparison
    in result["payload"]["comparisons"]
]

pilot_core46_judged_candidate_ids = [
    comparison["candidate_id"]
    for comparison
    in pilot_core46_all_comparisons
]

assert len(
    pilot_core46_judged_candidate_ids
) == 46, (
    "Semantic judge did not cover all "
    "46 Core candidates."
)

assert len(
    pilot_core46_judged_candidate_ids
) == len(
    set(
        pilot_core46_judged_candidate_ids
    )
), (
    "A Core candidate was judged more than once."
)

assert set(
    pilot_core46_judged_candidate_ids
) == set(
    claim_queue_candidate_ids
), (
    "Judged Candidate IDs do not match "
    "the comparison queue."
)

pilot_core46_relation_counts = Counter(
    comparison["relation"]
    for comparison
    in pilot_core46_all_comparisons
)

pilot_core46_observed_costs = [
    float(result["observed_cost"])
    for result
    in pilot_core46_semantic_results
    if result.get(
        "observed_cost"
    ) is not None
]

pilot_core46_semantic_report = {
    "judge_version": (
        PILOT_CORE46_JUDGE_VERSION
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": (
        "COMPLETE"
        if not pilot_core46_failed_results
        else "INCOMPLETE"
    ),
    "model": chat_model,
    "group_count": len(
        pilot_core46_semantic_results
    ),
    "candidate_comparison_count": len(
        pilot_core46_all_comparisons
    ),
    "represented_pilot_count": len(
        pilot_core46_represented_results
    ),
    "independent_pilot_count": len(
        pilot_core46_independent_results
    ),
    "review_required_pilot_count": len(
        pilot_core46_review_results
    ),
    "failed_group_count": len(
        pilot_core46_failed_results
    ),
    "relation_counts": dict(
        sorted(
            pilot_core46_relation_counts.items()
        )
    ),
    "cache_reuse_count": (
        pilot_core46_cache_reuse_count
    ),
    "live_request_count": (
        pilot_core46_live_request_count
    ),
    "total_observed_cost": (
        float(
            sum(
                pilot_core46_observed_costs
            )
        )
        if pilot_core46_observed_costs
        else None
    ),
    "decision_policy": {
        "PILOT_ALREADY_REPRESENTED": [
            "EQUIVALENT_CLAIM",
            "PILOT_SUBSUMED_BY_CORE",
        ],
        "REVIEW_REQUIRED": [
            "CORE_SUBSUMED_BY_PILOT",
            "PARTIAL_OVERLAP",
            "UNCERTAIN",
        ],
        "PILOT_ADDS_INDEPENDENT_CLAIM": (
            "All comparisons are "
            "INDEPENDENT_CLAIMS."
        ),
    },
    "active_dataset_modified": False,
    "golden_dataset_v1_modified": False,
    "core46_modified": False,
    "golden_dataset_v2_created": False,
    "results": (
        pilot_core46_semantic_results
    ),
}

pilot_core46_semantic_report_path = (
    role_matrix_output_directory
    / (
        "pilot_core46_semantic_"
        "claim_judge_report.json"
    )
)

with pilot_core46_semantic_report_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        pilot_core46_semantic_report,
        file,
        ensure_ascii=False,
        indent=2,
    )
    file.write("\n")

print()
print(
    "Pilot already represented:",
    len(
        pilot_core46_represented_results
    ),
)
print(
    "Pilot adds independent claim:",
    len(
        pilot_core46_independent_results
    ),
)
print(
    "Review required:",
    len(
        pilot_core46_review_results
    ),
)
print(
    "Failed groups:",
    len(
        pilot_core46_failed_results
    ),
)
print(
    "Candidate comparisons:",
    len(
        pilot_core46_all_comparisons
    ),
)
print(
    "Relation counts:",
    dict(
        sorted(
            pilot_core46_relation_counts.items()
        )
    ),
)
print(
    "Cached results reused:",
    pilot_core46_cache_reuse_count,
)
print(
    "Live requests:",
    pilot_core46_live_request_count,
)
print(
    "Total observed cost:",
    (
        f"{sum(pilot_core46_observed_costs):.6f}"
        if pilot_core46_observed_costs
        else "N/A"
    ),
)
print()
print(
    "Report:",
    pilot_core46_semantic_report_path,
)
print()
print("Active dataset modified:", False)
print("Golden Dataset v1 modified:", False)
print("Core-46 modified:", False)
print("Golden Dataset v2 created:", False)
print()
print(
    "NB12h PILOT-28 VS CORE-46 "
    "SEMANTIC CLAIM JUDGE: COMPLETE"
)

=== NB12h PILOT-28 VS CORE-46 SEMANTIC CLAIM JUDGE ===
Comparison groups: 28
Model: Qwen3-30B-A3B
Group 1/28 | pilot=01 | seed=01 | PILOT_ALREADY_REPRESENTED
Group 2/28 | pilot=02 | seed=02 | PILOT_ALREADY_REPRESENTED
Group 3/28 | pilot=03 | seed=03 | PILOT_ALREADY_REPRESENTED
Group 4/28 | pilot=04 | seed=04 | REVIEW_REQUIRED
Group 5/28 | pilot=05 | seed=05 | REVIEW_REQUIRED
Group 6/28 | pilot=06 | seed=06 | REVIEW_REQUIRED
Group 7/28 | pilot=07 | seed=07 | REVIEW_REQUIRED
Group 8/28 | pilot=08 | seed=08 | REVIEW_REQUIRED
Group 9/28 | pilot=09 | seed=09 | PILOT_ALREADY_REPRESENTED
Group 10/28 | pilot=10 | seed=10 | REVIEW_REQUIRED
Group 11/28 | pilot=11 | seed=11 | REVIEW_REQUIRED
Group 12/28 | pilot=12 | seed=12 | PILOT_ALREADY_REPRESENTED
Group 13/28 | pilot=13 | seed=13 | REVIEW_REQUIRED
Group 14/28 | pilot=14 | seed=14 | REVIEW_REQUIRED
Group 15/28 | pilot=15 | seed=15 | PILOT_ALREADY_REPRESENTED
Group 16/28 | pilot=16 | seed=16 | PILOT_ALREADY_REPRESENTED
Group 17/28 | pilot=17 | 

<div dir="rtl" align="right">

## **مرحلهٔ سی‌وچهارم — کشف موجودی واقعی Source Corpus**

برای تعیین تعداد طبیعی Seedهای نسخهٔ ۲، ابتدا باید مشخص شود چه اسناد، متن‌ها و Chunkهای واقعی در مخزن وجود دارند.

تعداد نسخهٔ ۲ نباید از تعداد Pilotها، تعداد رکوردهای Core-46 یا یک عدد از پیش تعیین‌شده گرفته شود. این تعداد باید از ادعاهای مستقل و قابل استناد موجود در Source Corpus به دست آید.

### **هدف این مرحله**

در این مرحله فایل‌های مرتبط با Corpus به‌صورت فقط‌خواندنی کشف و طبقه‌بندی می‌شوند:

- فایل‌های متن منبع؛
- فایل‌های Document؛
- فایل‌های Chunk و Retrieval؛
- Manifestهای مرتبط با Corpus؛
- و فایل‌هایی که دارای فیلدهای متنی قابل استفاده هستند.

### **فایل‌های خارج از محدوده**

برای جلوگیری از آلودگی Corpus، موارد زیر منبع محسوب نمی‌شوند:

- Golden Datasetهای قبلی؛
- خروجی‌های داوری و گزارش‌ها؛
- Human Review Registry؛
- Cacheها و Batchهای موقت؛
- Embeddingها و Vector Storeها؛
- فایل‌های Notebook؛
- و مصنوعات تولیدشده توسط خود NB12h.

### **مرز این مرحله**

- هیچ ادعایی استخراج نمی‌شود.
- هیچ API فراخوانی نمی‌شود.
- هیچ فایل منبعی تغییر نمی‌کند.
- هیچ Dataset جدیدی ساخته نمی‌شود.
- خروجی فقط یک فهرست ساختاری برای انتخاب Corpus معتبر در مرحله بعد است.

### **Validation**

این مرحله زمانی PASS است که:

- مسیرهای قابل بررسی به‌صورت کنترل‌شده پیمایش شوند؛
- فایل‌های منبع احتمالی از مصنوعات ارزیابی جدا شوند؛
- نوع، اندازه و ساختار اولیهٔ هر فایل ثبت شود؛
- فایل‌های دارای متن قابل استفاده مشخص شوند؛
- و هیچ Dataset موجودی تغییر نکند.

</div>

In [36]:
# Code Cell 34
# NB12h — Read-only whole-source-corpus inventory discovery
#
# Repair:
# - include book/container source formats
# - recognize book/transcript source families
# - preserve prior Cell 34 artifact
# - never overwrite an existing report
# - no API calls
# - no Golden Dataset changes

assert "project_root" in globals(), (
    "Run the NB12h project-discovery Cell first."
)
assert "role_matrix_output_directory" in globals(), (
    "Run Code Cell 31 first."
)

CORPUS_INVENTORY_VERSION = (
    "nb12h_source_corpus_inventory_v2"
)

# ------------------------------------------------------------------
# File types
# ------------------------------------------------------------------

CORPUS_TEXT_FILE_SUFFIXES = {
    ".json",
    ".jsonl",
    ".ndjson",
    ".csv",
    ".tsv",
    ".txt",
    ".md",
    ".html",
    ".htm",
    ".xml",
    ".parquet",
}

# These are legitimate source containers but should not be parsed
# as plain UTF-8 text by this discovery Cell.
CORPUS_CONTAINER_FILE_SUFFIXES = {
    ".docx",
    ".pdf",
    ".epub",
}

CORPUS_FILE_SUFFIXES = (
    CORPUS_TEXT_FILE_SUFFIXES
    | CORPUS_CONTAINER_FILE_SUFFIXES
)

# ------------------------------------------------------------------
# Path/name signals
# ------------------------------------------------------------------

CORPUS_NAME_SIGNALS = {
    "corpus",
    "document",
    "documents",
    "source",
    "sources",
    "chunk",
    "chunks",
    "retrieval",
    "content",
    "article",
    "articles",
    "page",
    "pages",
    "website",
    "crawl",
    "raw",
    "processed",

    # Repair: real project source families
    "book",
    "books",
    "transcript",
    "transcripts",
    "episode",
}

CORPUS_TEXT_FIELD_SIGNALS = {
    "text",
    "content",
    "body",
    "page_content",
    "source_text",
    "raw_text",
    "clean_text",
    "chunk_text",
    "document_text",
    "evidence",
    "source_evidence",
    "quote",
    "paragraph",
    "description",
    "title",
}

CORPUS_ID_FIELD_SIGNALS = {
    "id",
    "document_id",
    "doc_id",
    "source_id",
    "chunk_id",
    "parent_id",
    "url",
    "uri",
    "path",
}

# ------------------------------------------------------------------
# Exclusion policy
# ------------------------------------------------------------------

CORPUS_EXCLUDED_DIRECTORY_NAMES = {
    ".git",
    ".venv",
    "venv",
    "rag_env",
    "__pycache__",
    ".ipynb_checkpoints",
    ".pytest_cache",
    ".mypy_cache",
    ".ruff_cache",
    ".cache",
    "node_modules",
    "models",
    "checkpoints",
    "embeddings",
    "vectorstore",
    "database",
    "index",
    "cache",
    "batches",
}

CORPUS_EXCLUDED_PATH_FRAGMENTS = {
    "golden_dataset_rebuild",
    "golden_dataset_rebuild_clean",
    "human_review",
    "semantic_judge",
    "review_queue",
}

CORPUS_EXCLUDED_FILENAME_SIGNALS = {
    "golden_dataset",
    "acceptance",
    "audit",
    "judge",
    "findings",
    "report",
    "review_queue",
    "registry",
}

CORPUS_SAMPLE_LINE_LIMIT = 25
CORPUS_TEXT_PREVIEW_LIMIT = 500

CORPUS_FULL_COUNT_SIZE_LIMIT = (
    256 * 1024 * 1024
)

CORPUS_OUTPUT_PREVIEW_LIMIT = 40


# ------------------------------------------------------------------
# Path helpers
# ------------------------------------------------------------------

def corpus_relative_path(path):
    return str(
        path.relative_to(project_root)
    )


def corpus_path_is_excluded(path):
    relative_parts = [
        part.casefold()
        for part in path.relative_to(
            project_root
        ).parts
    ]

    if any(
        part in CORPUS_EXCLUDED_DIRECTORY_NAMES
        for part in relative_parts[:-1]
    ):
        return True

    relative_text = "/".join(
        relative_parts
    )

    if any(
        fragment in relative_text
        for fragment
        in CORPUS_EXCLUDED_PATH_FRAGMENTS
    ):
        return True

    filename = path.name.casefold()

    if any(
        signal in filename
        for signal
        in CORPUS_EXCLUDED_FILENAME_SIGNALS
    ):
        return True

    return False


def corpus_filename_signal_score(path):
    searchable_text = (
        corpus_relative_path(path)
        .casefold()
        .replace("-", "_")
    )

    matched_signals = sorted({
        signal
        for signal in CORPUS_NAME_SIGNALS
        if signal in searchable_text
    })

    return (
        len(matched_signals),
        matched_signals,
    )


# ------------------------------------------------------------------
# Schema/text helpers
# ------------------------------------------------------------------

def corpus_flatten_keys(
    value,
    limit=250,
):
    observed_keys = set()
    pending = [value]

    while (
        pending
        and len(observed_keys) < limit
    ):
        current = pending.pop()

        if isinstance(current, dict):
            for key, nested_value in (
                current.items()
            ):
                observed_keys.add(
                    str(key)
                )

                if isinstance(
                    nested_value,
                    (dict, list),
                ):
                    pending.append(
                        nested_value
                    )

        elif isinstance(current, list):
            pending.extend(
                current[:10]
            )

    return sorted(
        observed_keys
    )


def corpus_detect_fields(keys):
    normalized_key_map = {
        str(key).casefold(): str(key)
        for key in keys
    }

    text_fields = sorted({
        original_key
        for normalized_key, original_key
        in normalized_key_map.items()
        if (
            normalized_key
            in CORPUS_TEXT_FIELD_SIGNALS
            or any(
                signal in normalized_key
                for signal
                in CORPUS_TEXT_FIELD_SIGNALS
            )
        )
    })

    id_fields = sorted({
        original_key
        for normalized_key, original_key
        in normalized_key_map.items()
        if (
            normalized_key
            in CORPUS_ID_FIELD_SIGNALS
            or any(
                signal == normalized_key
                for signal
                in CORPUS_ID_FIELD_SIGNALS
            )
        )
    })

    return (
        text_fields,
        id_fields,
    )


def corpus_text_preview(value):
    if isinstance(value, str):
        normalized = re.sub(
            r"\s+",
            " ",
            value,
        ).strip()

        return normalized[
            :CORPUS_TEXT_PREVIEW_LIMIT
        ]

    if isinstance(value, dict):
        for key, nested_value in (
            value.items()
        ):
            normalized_key = str(
                key
            ).casefold()

            if (
                normalized_key
                in CORPUS_TEXT_FIELD_SIGNALS
                and isinstance(
                    nested_value,
                    str,
                )
            ):
                return corpus_text_preview(
                    nested_value
                )

        for nested_value in (
            value.values()
        ):
            preview = corpus_text_preview(
                nested_value
            )

            if preview:
                return preview

    if isinstance(value, list):
        for nested_value in value[:10]:
            preview = corpus_text_preview(
                nested_value
            )

            if preview:
                return preview

    return ""


# ------------------------------------------------------------------
# Format inspectors
# ------------------------------------------------------------------

def inspect_jsonl_corpus_file(path):
    sampled_objects = []
    nonempty_line_count = 0
    parse_error_count = 0

    with path.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as file:
        for line in file:
            stripped_line = line.strip()

            if not stripped_line:
                continue

            nonempty_line_count += 1

            if len(
                sampled_objects
            ) < CORPUS_SAMPLE_LINE_LIMIT:
                try:
                    payload = json.loads(
                        stripped_line
                    )

                    sampled_objects.append(
                        payload
                    )

                except json.JSONDecodeError:
                    parse_error_count += 1

    keys = corpus_flatten_keys(
        sampled_objects
    )

    text_fields, id_fields = (
        corpus_detect_fields(keys)
    )

    return {
        "format": "JSONL",
        "record_count": (
            nonempty_line_count
        ),
        "sample_count": len(
            sampled_objects
        ),
        "sample_parse_error_count": (
            parse_error_count
        ),
        "top_level_type": (
            type(
                sampled_objects[0]
            ).__name__
            if sampled_objects
            else None
        ),
        "observed_keys": keys,
        "text_fields": text_fields,
        "id_fields": id_fields,
        "text_preview": (
            corpus_text_preview(
                sampled_objects
            )
        ),
        "inspection_note": None,
    }


def inspect_json_corpus_file(path):
    file_size = path.stat().st_size

    if file_size > (
        CORPUS_FULL_COUNT_SIZE_LIMIT
    ):
        return {
            "format": "JSON",
            "record_count": None,
            "sample_count": 0,
            "sample_parse_error_count": 0,
            "top_level_type": None,
            "observed_keys": [],
            "text_fields": [],
            "id_fields": [],
            "text_preview": "",
            "inspection_note": (
                "File exceeds safe full-read "
                "inventory limit."
            ),
        }

    with path.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as file:
        payload = json.load(file)

    if isinstance(payload, list):
        record_count = len(payload)

        sample = payload[
            :CORPUS_SAMPLE_LINE_LIMIT
        ]

    elif isinstance(payload, dict):
        record_count = 1
        sample = payload

    else:
        record_count = 1
        sample = payload

    keys = corpus_flatten_keys(
        sample
    )

    text_fields, id_fields = (
        corpus_detect_fields(keys)
    )

    return {
        "format": "JSON",
        "record_count": record_count,
        "sample_count": (
            min(
                record_count,
                CORPUS_SAMPLE_LINE_LIMIT,
            )
            if isinstance(
                record_count,
                int,
            )
            else None
        ),
        "sample_parse_error_count": 0,
        "top_level_type": (
            type(payload).__name__
        ),
        "observed_keys": keys,
        "text_fields": text_fields,
        "id_fields": id_fields,
        "text_preview": (
            corpus_text_preview(
                sample
            )
        ),
        "inspection_note": None,
    }


def inspect_delimited_corpus_file(
    path,
    delimiter,
):
    rows = []
    header = []

    with path.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as file:
        for line_number, line in (
            enumerate(
                file,
                1,
            )
        ):
            values = [
                value.strip()
                for value in line.rstrip(
                    "\n"
                ).split(delimiter)
            ]

            if line_number == 1:
                header = values

            elif len(rows) < (
                CORPUS_SAMPLE_LINE_LIMIT
            ):
                rows.append(values)

            if line_number > (
                CORPUS_SAMPLE_LINE_LIMIT
                + 1
            ):
                break

    text_fields, id_fields = (
        corpus_detect_fields(header)
    )

    preview = ""

    for row in rows:
        for value in row:
            if len(value) >= 40:
                preview = (
                    corpus_text_preview(
                        value
                    )
                )
                break

        if preview:
            break

    return {
        "format": (
            "TSV"
            if delimiter == "\t"
            else "CSV"
        ),
        "record_count": None,
        "sample_count": len(rows),
        "sample_parse_error_count": 0,
        "top_level_type": "tabular",
        "observed_keys": header,
        "text_fields": text_fields,
        "id_fields": id_fields,
        "text_preview": preview,
        "inspection_note": None,
    }


def inspect_plaintext_corpus_file(
    path,
):
    with path.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as file:
        sample_text = file.read(
            100_000
        )

    normalized_preview = (
        corpus_text_preview(
            sample_text
        )
    )

    return {
        "format": (
            path.suffix.lstrip(
                "."
            ).upper()
            or "TEXT"
        ),
        "record_count": None,
        "sample_count": None,
        "sample_parse_error_count": 0,
        "top_level_type": "text",
        "observed_keys": [],
        "text_fields": (
            ["document_text"]
            if normalized_preview
            else []
        ),
        "id_fields": [],
        "text_preview": (
            normalized_preview
        ),
        "inspection_note": None,
    }


def inspect_container_corpus_file(
    path,
):
    # Inventory-only.
    # No binary extraction is performed here.
    return {
        "format": (
            path.suffix.lstrip(
                "."
            ).upper()
        ),
        "record_count": None,
        "sample_count": 0,
        "sample_parse_error_count": 0,
        "top_level_type": (
            "source_container"
        ),
        "observed_keys": [],
        "text_fields": [],
        "id_fields": [],
        "text_preview": "",
        "inspection_note": (
            "Binary/source container detected. "
            "Text extraction intentionally deferred."
        ),
    }


def inspect_corpus_candidate(path):
    suffix = path.suffix.casefold()

    try:
        if suffix in {
            ".jsonl",
            ".ndjson",
        }:
            inspection = (
                inspect_jsonl_corpus_file(
                    path
                )
            )

        elif suffix == ".json":
            inspection = (
                inspect_json_corpus_file(
                    path
                )
            )

        elif suffix == ".csv":
            inspection = (
                inspect_delimited_corpus_file(
                    path,
                    ",",
                )
            )

        elif suffix == ".tsv":
            inspection = (
                inspect_delimited_corpus_file(
                    path,
                    "\t",
                )
            )

        elif suffix == ".parquet":
            inspection = {
                "format": "PARQUET",
                "record_count": None,
                "sample_count": 0,
                "sample_parse_error_count": 0,
                "top_level_type": None,
                "observed_keys": [],
                "text_fields": [],
                "id_fields": [],
                "text_preview": "",
                "inspection_note": (
                    "Schema inspection deferred "
                    "to the selection stage."
                ),
            }

        elif suffix in (
            CORPUS_CONTAINER_FILE_SUFFIXES
        ):
            inspection = (
                inspect_container_corpus_file(
                    path
                )
            )

        else:
            inspection = (
                inspect_plaintext_corpus_file(
                    path
                )
            )

        inspection_error = None

    except Exception as error:
        inspection = {
            "format": (
                suffix.lstrip(
                    "."
                ).upper()
                or "UNKNOWN"
            ),
            "record_count": None,
            "sample_count": 0,
            "sample_parse_error_count": (
                None
            ),
            "top_level_type": None,
            "observed_keys": [],
            "text_fields": [],
            "id_fields": [],
            "text_preview": "",
            "inspection_note": None,
        }

        inspection_error = (
            f"{type(error).__name__}: "
            f"{error}"
        )

    signal_score, matched_signals = (
        corpus_filename_signal_score(
            path
        )
    )

    has_text_signal = bool(
        inspection["text_fields"]
        or inspection["text_preview"]
    )

    is_source_container = (
        suffix
        in CORPUS_CONTAINER_FILE_SUFFIXES
    )

    if (
        signal_score > 0
        and has_text_signal
    ):
        candidate_status = (
            "LIKELY_SOURCE_CORPUS"
        )

    elif (
        signal_score > 0
        and is_source_container
    ):
        candidate_status = (
            "SOURCE_CONTAINER_CANDIDATE"
        )

    elif signal_score > 0:
        candidate_status = (
            "STRUCTURAL_CORPUS_CANDIDATE"
        )

    elif has_text_signal:
        candidate_status = (
            "TEXT_BEARING_FILE_REVIEW_REQUIRED"
        )

    elif is_source_container:
        candidate_status = (
            "SOURCE_CONTAINER_REVIEW_REQUIRED"
        )

    else:
        candidate_status = (
            "NO_USABLE_TEXT_DETECTED"
        )

    return {
        "relative_path": (
            corpus_relative_path(path)
        ),
        "absolute_path": str(path),
        "suffix": suffix,
        "size_bytes": (
            path.stat().st_size
        ),
        "name_signal_score": (
            signal_score
        ),
        "matched_name_signals": (
            matched_signals
        ),
        "candidate_status": (
            candidate_status
        ),
        "inspection_error": (
            inspection_error
        ),
        **inspection,
    }


# ------------------------------------------------------------------
# Discovery
# ------------------------------------------------------------------

corpus_search_roots = [
    path
    for path in [
        project_root / "data",
        project_root / "artifacts",
    ]
    if path.is_dir()
]

assert corpus_search_roots, (
    "No data or artifacts directory was found."
)

corpus_discovered_paths = []

for search_root in corpus_search_roots:
    for path in search_root.rglob("*"):
        if not path.is_file():
            continue

        if (
            path.suffix.casefold()
            not in CORPUS_FILE_SUFFIXES
        ):
            continue

        if corpus_path_is_excluded(path):
            continue

        corpus_discovered_paths.append(
            path.resolve()
        )

corpus_discovered_paths = sorted(
    set(corpus_discovered_paths),
    key=lambda path: (
        corpus_relative_path(path)
    ),
)

source_corpus_inventory_rows = [
    inspect_corpus_candidate(path)
    for path in corpus_discovered_paths
]


# ------------------------------------------------------------------
# Inventory groups
# ------------------------------------------------------------------

corpus_likely_source_rows = [
    row
    for row in source_corpus_inventory_rows
    if row["candidate_status"]
    == "LIKELY_SOURCE_CORPUS"
]

corpus_source_container_rows = [
    row
    for row in source_corpus_inventory_rows
    if row["candidate_status"]
    in {
        "SOURCE_CONTAINER_CANDIDATE",
        "SOURCE_CONTAINER_REVIEW_REQUIRED",
    }
]

corpus_structural_candidate_rows = [
    row
    for row in source_corpus_inventory_rows
    if row["candidate_status"]
    == "STRUCTURAL_CORPUS_CANDIDATE"
]

corpus_text_review_rows = [
    row
    for row in source_corpus_inventory_rows
    if row["candidate_status"]
    == "TEXT_BEARING_FILE_REVIEW_REQUIRED"
]

corpus_inspection_error_rows = [
    row
    for row in source_corpus_inventory_rows
    if row["inspection_error"]
]

corpus_status_counts = Counter(
    row["candidate_status"]
    for row in source_corpus_inventory_rows
)

corpus_suffix_counts = Counter(
    row["suffix"]
    for row in source_corpus_inventory_rows
)


# ------------------------------------------------------------------
# Explicit discovery checks for known project source families
# ------------------------------------------------------------------

corpus_book_rows = [
    row
    for row in source_corpus_inventory_rows
    if (
        "/books/" in (
            f"/{row['relative_path'].casefold()}"
        )
        or row["suffix"]
        in {
            ".docx",
            ".pdf",
            ".epub",
        }
    )
]

corpus_transcript_rows = [
    row
    for row in source_corpus_inventory_rows
    if "transcript" in (
        row["relative_path"].casefold()
    )
]

corpus_website_rows = [
    row
    for row in source_corpus_inventory_rows
    if "website" in (
        row["relative_path"].casefold()
    )
]


# ------------------------------------------------------------------
# Gate
# ------------------------------------------------------------------

assert not corpus_inspection_error_rows, (
    "Corpus inventory contains inspection errors."
)

assert corpus_book_rows, (
    "No book/source-container candidates were "
    "discovered. Whole-corpus inventory is incomplete."
)

assert corpus_transcript_rows, (
    "No transcript candidates were discovered. "
    "Whole-corpus inventory is incomplete."
)

assert corpus_website_rows, (
    "No website candidates were discovered. "
    "Whole-corpus inventory is incomplete."
)


# ------------------------------------------------------------------
# Report
# ------------------------------------------------------------------

source_corpus_inventory_report = {
    "inventory_version": (
        CORPUS_INVENTORY_VERSION
    ),
    "status": "PASS",
    "search_roots": [
        str(path)
        for path in corpus_search_roots
    ],
    "discovered_file_count": len(
        source_corpus_inventory_rows
    ),
    "status_counts": dict(
        sorted(
            corpus_status_counts.items()
        )
    ),
    "suffix_counts": dict(
        sorted(
            corpus_suffix_counts.items()
        )
    ),
    "source_family_counts": {
        "website_related_files": len(
            corpus_website_rows
        ),
        "book_related_files": len(
            corpus_book_rows
        ),
        "transcript_related_files": len(
            corpus_transcript_rows
        ),
    },
    "likely_source_corpus_count": len(
        corpus_likely_source_rows
    ),
    "source_container_candidate_count": len(
        corpus_source_container_rows
    ),
    "structural_candidate_count": len(
        corpus_structural_candidate_rows
    ),
    "text_review_required_count": len(
        corpus_text_review_rows
    ),
    "inspection_error_count": len(
        corpus_inspection_error_rows
    ),
    "exclusion_policy": {
        "directory_names": sorted(
            CORPUS_EXCLUDED_DIRECTORY_NAMES
        ),
        "path_fragments": sorted(
            CORPUS_EXCLUDED_PATH_FRAGMENTS
        ),
        "filename_signals": sorted(
            CORPUS_EXCLUDED_FILENAME_SIGNALS
        ),
    },
    "api_calls_made": 0,
    "cost_incurred": 0,
    "source_files_modified": False,
    "active_dataset_modified": False,
    "golden_dataset_v1_modified": False,
    "core46_modified": False,
    "golden_dataset_v2_created": False,
    "files": source_corpus_inventory_rows,
}


# ------------------------------------------------------------------
# No-overwrite artifact policy
# ------------------------------------------------------------------

source_corpus_inventory_report_path = (
    role_matrix_output_directory
    / "source_corpus_inventory_v2.json"
)

if (
    source_corpus_inventory_report_path
    .exists()
):
    with (
        source_corpus_inventory_report_path
        .open(
            "r",
            encoding="utf-8",
        )
    ) as file:
        existing_report = json.load(
            file
        )

    comparable_existing = dict(
        existing_report
    )
    comparable_current = dict(
        source_corpus_inventory_report
    )

    existing_created_at = (
        comparable_existing.pop(
            "created_at_utc",
            None,
        )
    )

    comparable_current.pop(
        "created_at_utc",
        None,
    )

    assert (
        comparable_existing
        == comparable_current
    ), (
        "Existing source_corpus_inventory_v2.json "
        "differs from the current discovery. "
        "Refusing to overwrite it."
    )

    source_corpus_inventory_report = (
        existing_report
    )

    report_write_status = (
        "REUSED_EXISTING_IDENTICAL_REPORT"
    )

else:
    source_corpus_inventory_report[
        "created_at_utc"
    ] = datetime.now(
        timezone.utc
    ).isoformat()

    with (
        source_corpus_inventory_report_path
        .open(
            "x",
            encoding="utf-8",
        )
    ) as file:
        json.dump(
            source_corpus_inventory_report,
            file,
            ensure_ascii=False,
            indent=2,
        )
        file.write("\n")

    report_write_status = (
        "CREATED_NEW_REPORT"
    )


# ------------------------------------------------------------------
# Compact output
# ------------------------------------------------------------------

print(
    "=== NB12h WHOLE SOURCE CORPUS "
    "INVENTORY DISCOVERY ==="
)

print(
    "Search roots:",
    [
        str(path)
        for path in corpus_search_roots
    ],
)

print(
    "Files inspected:",
    len(
        source_corpus_inventory_rows
    ),
)

print(
    "Status counts:",
    dict(
        sorted(
            corpus_status_counts.items()
        )
    ),
)

print(
    "Suffix counts:",
    dict(
        sorted(
            corpus_suffix_counts.items()
        )
    ),
)

print(
    "Source family counts:",
    source_corpus_inventory_report[
        "source_family_counts"
    ],
)

print(
    "Inspection errors:",
    len(
        corpus_inspection_error_rows
    ),
)

print(
    "\nBook/source-container candidates:"
)

for row in (
    corpus_book_rows[
        :CORPUS_OUTPUT_PREVIEW_LIMIT
    ]
):
    print(
        "  -",
        row["relative_path"],
        "| format="
        f"{row['format']}",
        "| status="
        f"{row['candidate_status']}",
    )

if len(corpus_book_rows) > (
    CORPUS_OUTPUT_PREVIEW_LIMIT
):
    print(
        "  ...",
        len(corpus_book_rows)
        - CORPUS_OUTPUT_PREVIEW_LIMIT,
        "more"
    )

print(
    "\nTranscript candidates preview:"
)

for row in (
    corpus_transcript_rows[
        :10
    ]
):
    print(
        "  -",
        row["relative_path"],
        "| format="
        f"{row['format']}",
        "| status="
        f"{row['candidate_status']}",
    )

if len(corpus_transcript_rows) > 10:
    print(
        "  ...",
        len(corpus_transcript_rows)
        - 10,
        "more"
    )

print(
    "\nWebsite candidates preview:"
)

for row in (
    corpus_website_rows[
        :10
    ]
):
    print(
        "  -",
        row["relative_path"],
        "| format="
        f"{row['format']}",
        "| status="
        f"{row['candidate_status']}",
    )

if len(corpus_website_rows) > 10:
    print(
        "  ...",
        len(corpus_website_rows)
        - 10,
        "more"
    )

print()
print(
    "Inventory report:",
    source_corpus_inventory_report_path,
)
print(
    "Report write status:",
    report_write_status,
)
print()
print("API calls made:", 0)
print("Cost incurred:", 0)
print("Source files modified:", False)
print("Active dataset modified:", False)
print("Golden Dataset v1 modified:", False)
print("Core-46 modified:", False)
print("Golden Dataset v2 created:", False)
print()
print(
    "NB12h WHOLE SOURCE CORPUS "
    "INVENTORY DISCOVERY: PASS"
)

=== NB12h WHOLE SOURCE CORPUS INVENTORY DISCOVERY ===
Search roots: ['/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data', '/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts']
Files inspected: 1676
Status counts: {'LIKELY_SOURCE_CORPUS': 1611, 'SOURCE_CONTAINER_CANDIDATE': 9, 'SOURCE_CONTAINER_REVIEW_REQUIRED': 16, 'STRUCTURAL_CORPUS_CANDIDATE': 1, 'TEXT_BEARING_FILE_REVIEW_REQUIRED': 39}
Suffix counts: {'.docx': 16, '.html': 606, '.json': 730, '.jsonl': 15, '.md': 298, '.pdf': 9, '.txt': 2}
Source family counts: {'website_related_files': 1067, 'book_related_files': 25, 'transcript_related_files': 215}
Inspection errors: 0

Book/source-container candidates:
  - data/Road_Maps/RAG_finance_master_roadmap.docx | format=DOCX | status=SOURCE_CONTAINER_REVIEW_REQUIRED
  - data/Road_Maps/RAG_finance_master_roadmap.pdf | format=PDF | status=SOURCE_CONTAINER_REVIEW_REQUIRED
  - data/Road_Maps/RAG_finance_phase2_modular_notebooks_exec

<div dir="rtl" align="right">

## **مرحلهٔ سی‌وپنجم — ممیزی لایه‌های Source، Document و Chunk قابل استفاده**

مرحلهٔ قبل کل فضای فایل‌های بالقوهٔ مرتبط با Corpus را شناسایی کرد، اما خروجی آن شامل چند نوع نمایش متفاوت از یک محتوای واحد است.

برای مثال یک محتوای واحد ممکن است هم‌زمان به شکل Raw Source، سند استخراج‌شده، سند پردازش‌شده، Chunk و Artifact آزمایشی در Repository وجود داشته باشد.

بنابراین تعداد فایل‌های کشف‌شده را نمی‌توان مستقیماً به‌عنوان تعداد اسناد یا واحدهای دانشی قابل استفاده در نسخهٔ ۲ در نظر گرفت.

### **هدف این مرحله**

در این مرحله باید نقش واقعی فایل‌های کشف‌شده در Lineage داده مشخص شود تا بدانیم:

- Sourceهای خام واقعی Website، Book و Transcript کدام‌اند؛

- چه Document representationهایی در Pipeline وجود دارند؛

- چه Chunk representationهایی وجود دارند؛

- کدام فایل‌ها Manifest، Metadata، Artifact آزمایشی یا سند مدیریتی پروژه هستند؛

- تعداد رکوردهای هر representation چقدر است؛

- و آیا برای ادامهٔ مسیر یک Document Layer یا Chunk Layer مناسب و قابل بررسی وجود دارد.

### **اصل مهم شمارش**

Representationهای مختلف یک محتوای واحد نباید با یکدیگر جمع شوند.

برای مثال اگر یک Document در لایه‌های زیر وجود داشته باشد:

- Unified؛

- Normalized؛

- Quality-Accepted؛

- Chunked؛

این موارد چهار واحد دانش مستقل محسوب نمی‌شوند.

در این مرحله تعداد هر representation جداگانه گزارش می‌شود و هیچ «عدد کل» مصنوعی از مجموع نسخه‌ها ساخته نمی‌شود.

### **کار این مرحله**

در این مرحله فایل‌های کشف‌شده در Cell 34 به گروه‌های زیر دسته‌بندی می‌شوند:

- Raw Website Content؛

- Raw Website Metadata؛

- Raw Book Source؛

- Raw Transcript Source؛

- Manifest و Metadata؛

- Interim Document Layer؛

- Processed Document Layer؛

- Accepted Document Candidate؛

- Chunk Layer؛

- Derived Artifact؛

- Project Support Document؛

- و موارد نیازمند بررسی.

فایل‌های مدیریتی پروژه مانند `data/Road_Maps/` نباید به‌عنوان Knowledge Corpus وارد شمارش شوند.

### **مرز تصمیم**

در این مرحله:

- Candidate Claim استخراج نمی‌شود؛

- Semantic Deduplication انجام نمی‌شود؛

- Pilot-28 و Core-46 با Corpus مقایسه نمی‌شوند؛

- تعداد Seedهای v2 تعیین نمی‌شود؛

- هیچ Chunk Variant به‌صورت نهایی انتخاب نمی‌شود؛

- هیچ API یا LLM فراخوانی نمی‌شود؛

- Golden Dataset v1 تغییر نمی‌کند؛

- Core-46 تغییر نمی‌کند؛

- Golden Dataset v2 ساخته نمی‌شود.

### **سیاست عدم Overwrite**

گزارش این مرحله با نام نسخه‌دار جدید ساخته می‌شود.

اگر همان فایل از قبل وجود داشته باشد:

- اگر محتوای آن با اجرای فعلی یکسان باشد، همان Artifact موجود reuse می‌شود؛

- اگر متفاوت باشد، اجرا متوقف می‌شود؛

- هیچ Artifact موجودی overwrite نمی‌شود.

### **Validation**

این مرحله زمانی PASS است که:

- برای هر سه خانوادهٔ Website، Book و Transcript Source واقعی شناسایی شده باشد؛

- حداقل یک Document representation قابل بررسی وجود داشته باشد؛

- حداقل یک Chunk representation وجود داشته باشد؛

- فایل‌های `data/Road_Maps/` وارد Knowledge Corpus نشده باشند؛

- Inspection Error مرحلهٔ قبل صفر باقی مانده باشد؛

- Representationهای مختلف با یکدیگر جمع نشده باشند؛

- و هیچ Dataset موجودی تغییر نکرده باشد.

خروجی این مرحله مبنای تعیین ورودی صحیح برای Claim Discovery از کل Corpus خواهد بود.

</div>

In [37]:
# Code Cell 35
# NB12h — Source lineage and usable document/chunk layer audit
#
# Read-only with respect to all existing datasets.
# Creates only a new versioned audit report.
# Existing report is never overwritten.

assert "project_root" in globals(), (
    "Run the NB12h project-discovery Cell first."
)

assert "role_matrix_output_directory" in globals(), (
    "Run Code Cell 31 first."
)

assert "source_corpus_inventory_rows" in globals(), (
    "Run the repaired Code Cell 34 first."
)

assert "corpus_inspection_error_rows" in globals(), (
    "Run the repaired Code Cell 34 first."
)

assert not corpus_inspection_error_rows, (
    "Code Cell 34 contains inspection errors."
)

CORPUS_LAYER_AUDIT_VERSION = (
    "nb12h_corpus_layer_audit_v1"
)

CORPUS_LAYER_REPORT_FILENAME = (
    "source_corpus_layer_audit_v1.json"
)


# ---------------------------------------------------------
# Role definitions
# ---------------------------------------------------------

ROLE_PROJECT_SUPPORT = (
    "PROJECT_SUPPORT_DOCUMENT"
)

ROLE_RAW_WEBSITE_CONTENT = (
    "RAW_WEBSITE_CONTENT"
)

ROLE_RAW_WEBSITE_METADATA = (
    "RAW_WEBSITE_METADATA"
)

ROLE_RAW_BOOK_SOURCE = (
    "RAW_BOOK_SOURCE"
)

ROLE_RAW_TRANSCRIPT_SOURCE = (
    "RAW_TRANSCRIPT_SOURCE"
)

ROLE_MANIFEST_METADATA = (
    "MANIFEST_METADATA"
)

ROLE_INTERIM_DOCUMENT = (
    "INTERIM_DOCUMENT_LAYER"
)

ROLE_ACCEPTED_DOCUMENT = (
    "ACCEPTED_DOCUMENT_LAYER_CANDIDATE"
)

ROLE_PROCESSED_DOCUMENT = (
    "PROCESSED_DOCUMENT_LAYER"
)

ROLE_SOURCE_SPECIFIC_DOCUMENT = (
    "SOURCE_SPECIFIC_DOCUMENT_LAYER"
)

ROLE_CHUNK = (
    "CHUNK_LAYER_CANDIDATE"
)

ROLE_PROCESSED_SUPPORT = (
    "PROCESSED_SUPPORT_LAYER"
)

ROLE_DERIVED_ARTIFACT = (
    "DERIVED_ARTIFACT"
)

ROLE_REVIEW_REQUIRED = (
    "ROLE_REVIEW_REQUIRED"
)


# ---------------------------------------------------------
# Classification
# ---------------------------------------------------------

def classify_corpus_layer(row):
    relative_path = (
        row["relative_path"]
        .replace("\\", "/")
    )

    normalized_path = (
        relative_path.casefold()
    )

    filename = (
        Path(relative_path)
        .name
        .casefold()
    )

    suffix = row[
        "suffix"
    ].casefold()

    # -----------------------------------------------------
    # Project documentation is NOT knowledge corpus.
    # -----------------------------------------------------

    if normalized_path.startswith(
        "data/road_maps/"
    ):
        return ROLE_PROJECT_SUPPORT

    # -----------------------------------------------------
    # Raw books
    # -----------------------------------------------------

    if normalized_path.startswith(
        "data/books/"
    ):
        if suffix in {
            ".docx",
            ".pdf",
            ".epub",
        }:
            return ROLE_RAW_BOOK_SOURCE

    # -----------------------------------------------------
    # Raw transcripts
    # -----------------------------------------------------

    if normalized_path.startswith(
        "data/transcripts/final/"
    ):
        if suffix in {
            ".md",
            ".txt",
        }:
            return ROLE_RAW_TRANSCRIPT_SOURCE

    # -----------------------------------------------------
    # Raw website
    # -----------------------------------------------------

    if normalized_path.startswith(
        "data/phase2/raw/website/"
    ):
        if filename in {
            "article.html",
            "article.htm",
        }:
            return ROLE_RAW_WEBSITE_CONTENT

        if filename == "source.json":
            return ROLE_RAW_WEBSITE_METADATA

    # -----------------------------------------------------
    # Manifests
    # -----------------------------------------------------

    if normalized_path.startswith(
        "data/phase2/manifests/"
    ):
        return ROLE_MANIFEST_METADATA

    # -----------------------------------------------------
    # Interim source-specific extraction
    # -----------------------------------------------------

    if normalized_path.startswith(
        "data/phase2/interim/"
    ):
        return ROLE_INTERIM_DOCUMENT

    # -----------------------------------------------------
    # Chunk datasets
    # -----------------------------------------------------

    if (
        normalized_path.startswith(
            "data/phase2/chunks/"
        )
        or normalized_path.startswith(
            "data/chunks/"
        )
    ):
        return ROLE_CHUNK

    # -----------------------------------------------------
    # Processed data
    # -----------------------------------------------------

    if normalized_path.startswith(
        "data/phase2/processed/"
    ):

        if filename == (
            "documents_dedup_quality_accepted_v1.jsonl"
        ):
            return ROLE_ACCEPTED_DOCUMENT

        if filename.startswith(
            "documents_"
        ):
            return ROLE_PROCESSED_DOCUMENT

        if filename.startswith(
            "website_documents_"
        ):
            return ROLE_SOURCE_SPECIFIC_DOCUMENT

        return ROLE_PROCESSED_SUPPORT

    # -----------------------------------------------------
    # Generated artifacts
    # -----------------------------------------------------

    if normalized_path.startswith(
        "artifacts/"
    ):
        return ROLE_DERIVED_ARTIFACT

    return ROLE_REVIEW_REQUIRED


corpus_layer_rows = []

for row in source_corpus_inventory_rows:
    classified_row = dict(row)

    classified_row[
        "lineage_role"
    ] = classify_corpus_layer(
        row
    )

    corpus_layer_rows.append(
        classified_row
    )


# ---------------------------------------------------------
# Role groups
# ---------------------------------------------------------

corpus_layer_role_counts = Counter(
    row["lineage_role"]
    for row in corpus_layer_rows
)

raw_website_content_rows = [
    row
    for row in corpus_layer_rows
    if row["lineage_role"]
    == ROLE_RAW_WEBSITE_CONTENT
]

raw_website_metadata_rows = [
    row
    for row in corpus_layer_rows
    if row["lineage_role"]
    == ROLE_RAW_WEBSITE_METADATA
]

raw_book_source_rows = [
    row
    for row in corpus_layer_rows
    if row["lineage_role"]
    == ROLE_RAW_BOOK_SOURCE
]

raw_transcript_source_rows = [
    row
    for row in corpus_layer_rows
    if row["lineage_role"]
    == ROLE_RAW_TRANSCRIPT_SOURCE
]

project_support_rows = [
    row
    for row in corpus_layer_rows
    if row["lineage_role"]
    == ROLE_PROJECT_SUPPORT
]

accepted_document_rows = [
    row
    for row in corpus_layer_rows
    if row["lineage_role"]
    == ROLE_ACCEPTED_DOCUMENT
]

processed_document_rows = [
    row
    for row in corpus_layer_rows
    if row["lineage_role"]
    == ROLE_PROCESSED_DOCUMENT
]

source_specific_document_rows = [
    row
    for row in corpus_layer_rows
    if row["lineage_role"]
    == ROLE_SOURCE_SPECIFIC_DOCUMENT
]

interim_document_rows = [
    row
    for row in corpus_layer_rows
    if row["lineage_role"]
    == ROLE_INTERIM_DOCUMENT
]

chunk_layer_rows = [
    row
    for row in corpus_layer_rows
    if row["lineage_role"]
    == ROLE_CHUNK
]

review_required_rows = [
    row
    for row in corpus_layer_rows
    if row["lineage_role"]
    == ROLE_REVIEW_REQUIRED
]


# ---------------------------------------------------------
# Helper for representation-level counts
# ---------------------------------------------------------

def representation_summary(rows):
    summaries = []

    for row in sorted(
        rows,
        key=lambda item: (
            item["relative_path"]
        ),
    ):
        summaries.append({
            "relative_path": (
                row["relative_path"]
            ),
            "format": row["format"],
            "record_count": (
                row["record_count"]
            ),
            "size_bytes": (
                row["size_bytes"]
            ),
            "text_fields": (
                row["text_fields"]
            ),
            "id_fields": (
                row["id_fields"]
            ),
            "candidate_status": (
                row["candidate_status"]
            ),
            "lineage_role": (
                row["lineage_role"]
            ),
        })

    return summaries


document_representation_rows = (
    accepted_document_rows
    + processed_document_rows
    + source_specific_document_rows
    + interim_document_rows
)

document_representation_summary = (
    representation_summary(
        document_representation_rows
    )
)

chunk_representation_summary = (
    representation_summary(
        chunk_layer_rows
    )
)


# ---------------------------------------------------------
# Explicit known candidate checks
# ---------------------------------------------------------

accepted_document_exact_path = (
    "data/phase2/processed/"
    "documents_dedup_quality_accepted_v1.jsonl"
)

accepted_document_exact_rows = [
    row
    for row in accepted_document_rows
    if row["relative_path"]
    == accepted_document_exact_path
]

assert len(
    accepted_document_exact_rows
) <= 1, (
    "More than one exact accepted-document "
    "candidate was detected."
)

accepted_document_exact_row = (
    accepted_document_exact_rows[0]
    if accepted_document_exact_rows
    else None
)


# ---------------------------------------------------------
# Important rule:
# never sum alternative representations as independent docs.
# ---------------------------------------------------------

document_record_counts_known = {
    row["relative_path"]:
        row["record_count"]
    for row in document_representation_rows
    if isinstance(
        row["record_count"],
        int,
    )
}

chunk_record_counts_known = {
    row["relative_path"]:
        row["record_count"]
    for row in chunk_layer_rows
    if isinstance(
        row["record_count"],
        int,
    )
}


# ---------------------------------------------------------
# Gate validation
# ---------------------------------------------------------

assert raw_website_content_rows, (
    "No raw website content was classified."
)

assert raw_book_source_rows, (
    "No raw book source was classified."
)

assert raw_transcript_source_rows, (
    "No raw transcript source was classified."
)

assert document_representation_rows, (
    "No document representation was classified."
)

assert chunk_layer_rows, (
    "No chunk representation was classified."
)

assert not any(
    row["lineage_role"]
    in {
        ROLE_ACCEPTED_DOCUMENT,
        ROLE_PROCESSED_DOCUMENT,
        ROLE_SOURCE_SPECIFIC_DOCUMENT,
        ROLE_INTERIM_DOCUMENT,
        ROLE_CHUNK,
    }
    for row in project_support_rows
), (
    "Project support documents leaked into "
    "knowledge-corpus representations."
)


# ---------------------------------------------------------
# Selection status
# ---------------------------------------------------------

if accepted_document_exact_row is None:
    canonical_document_selection_status = (
        "NOT_SELECTED_NO_EXACT_ACCEPTED_LAYER_FOUND"
    )

else:
    canonical_document_selection_status = (
        "STRONG_ACCEPTED_DOCUMENT_CANDIDATE_FOUND"
    )

# Cell 35 deliberately does NOT pick a chunk variant.
canonical_chunk_selection_status = (
    "NOT_SELECTED_MULTIPLE_OR_UNVALIDATED_VARIANTS"
)


# ---------------------------------------------------------
# Build report
# ---------------------------------------------------------

source_corpus_layer_audit_report = {
    "audit_version": (
        CORPUS_LAYER_AUDIT_VERSION
    ),
    "status": "PASS",

    "raw_source_counts": {
        "website_content_files": len(
            raw_website_content_rows
        ),
        "website_metadata_files": len(
            raw_website_metadata_rows
        ),
        "book_source_files": len(
            raw_book_source_rows
        ),
        "transcript_source_files": len(
            raw_transcript_source_rows
        ),
    },

    "role_counts": dict(
        sorted(
            corpus_layer_role_counts.items()
        )
    ),

    "project_support_file_count": len(
        project_support_rows
    ),

    "document_representation_file_count": len(
        document_representation_rows
    ),

    "chunk_representation_file_count": len(
        chunk_layer_rows
    ),

    "document_record_counts_by_representation": (
        document_record_counts_known
    ),

    "chunk_record_counts_by_representation": (
        chunk_record_counts_known
    ),

    "accepted_document_candidate": (
        {
            "relative_path": (
                accepted_document_exact_row[
                    "relative_path"
                ]
            ),
            "record_count": (
                accepted_document_exact_row[
                    "record_count"
                ]
            ),
            "format": (
                accepted_document_exact_row[
                    "format"
                ]
            ),
        }
        if accepted_document_exact_row
        else None
    ),

    "canonical_document_selection_status": (
        canonical_document_selection_status
    ),

    "canonical_chunk_selection_status": (
        canonical_chunk_selection_status
    ),

    "aggregation_policy": (
        "Alternative document/chunk representations "
        "must not be summed as independent knowledge units."
    ),

    "document_representations": (
        document_representation_summary
    ),

    "chunk_representations": (
        chunk_representation_summary
    ),

    "review_required_files": (
        representation_summary(
            review_required_rows
        )
    ),

    "api_calls_made": 0,
    "cost_incurred": 0,

    "source_files_modified": False,
    "active_dataset_modified": False,
    "golden_dataset_v1_modified": False,
    "core46_modified": False,
    "golden_dataset_v2_created": False,
    "claim_extraction_started": False,
}


# ---------------------------------------------------------
# No-overwrite report persistence
# ---------------------------------------------------------

source_corpus_layer_audit_report_path = (
    role_matrix_output_directory
    / CORPUS_LAYER_REPORT_FILENAME
)

if (
    source_corpus_layer_audit_report_path
    .exists()
):
    with (
        source_corpus_layer_audit_report_path
        .open(
            "r",
            encoding="utf-8",
        )
    ) as file:
        existing_layer_report = (
            json.load(file)
        )

    comparable_existing = dict(
        existing_layer_report
    )

    comparable_current = dict(
        source_corpus_layer_audit_report
    )

    comparable_existing.pop(
        "created_at_utc",
        None,
    )

    comparable_current.pop(
        "created_at_utc",
        None,
    )

    assert (
        comparable_existing
        == comparable_current
    ), (
        "Existing source_corpus_layer_audit_v1.json "
        "differs from the current audit. "
        "Refusing to overwrite it."
    )

    source_corpus_layer_audit_report = (
        existing_layer_report
    )

    layer_report_write_status = (
        "REUSED_EXISTING_IDENTICAL_REPORT"
    )

else:
    source_corpus_layer_audit_report[
        "created_at_utc"
    ] = datetime.now(
        timezone.utc
    ).isoformat()

    with (
        source_corpus_layer_audit_report_path
        .open(
            "x",
            encoding="utf-8",
        )
    ) as file:
        json.dump(
            source_corpus_layer_audit_report,
            file,
            ensure_ascii=False,
            indent=2,
        )
        file.write("\n")

    layer_report_write_status = (
        "CREATED_NEW_REPORT"
    )


# ---------------------------------------------------------
# Compact human-readable output
# ---------------------------------------------------------

print(
    "=== NB12h SOURCE CORPUS "
    "LAYER AUDIT ==="
)

print()
print("Raw source counts:")

print(
    "  Website content files:",
    len(
        raw_website_content_rows
    ),
)

print(
    "  Website metadata files:",
    len(
        raw_website_metadata_rows
    ),
)

print(
    "  Book source files:",
    len(
        raw_book_source_rows
    ),
)

print(
    "  Transcript source files:",
    len(
        raw_transcript_source_rows
    ),
)

print()
print(
    "Project support files excluded:",
    len(
        project_support_rows
    ),
)

print()
print(
    "Document representation files:",
    len(
        document_representation_rows
    ),
)

for row in document_representation_summary:
    print(
        "  -",
        row["relative_path"],
        "| records=",
        row["record_count"],
        "| role=",
        row["lineage_role"],
    )

print()
print(
    "Chunk representation files:",
    len(
        chunk_layer_rows
    ),
)

for row in chunk_representation_summary:
    print(
        "  -",
        row["relative_path"],
        "| records=",
        row["record_count"],
        "| role=",
        row["lineage_role"],
    )

print()
print(
    "Accepted document candidate:",
    (
        accepted_document_exact_row[
            "relative_path"
        ]
        if accepted_document_exact_row
        else None
    ),
)

print(
    "Accepted document record count:",
    (
        accepted_document_exact_row[
            "record_count"
        ]
        if accepted_document_exact_row
        else None
    ),
)

print(
    "Document selection status:",
    canonical_document_selection_status,
)

print(
    "Chunk selection status:",
    canonical_chunk_selection_status,
)

print()
print(
    "Review-required unmatched files:",
    len(
        review_required_rows
    ),
)

if review_required_rows:
    for row in review_required_rows[:20]:
        print(
            "  -",
            row["relative_path"],
        )

    if len(review_required_rows) > 20:
        print(
            "  ...",
            len(review_required_rows) - 20,
            "more",
        )

print()
print(
    "Audit report:",
    source_corpus_layer_audit_report_path,
)

print(
    "Report write status:",
    layer_report_write_status,
)

print()
print(
    "IMPORTANT:"
)

print(
    "Document/Chunk representation counts "
    "were NOT summed across versions."
)

print(
    "No canonical chunk variant was selected."
)

print(
    "No claim extraction was performed."
)

print()
print("API calls made:", 0)
print("Cost incurred:", 0)
print("Source files modified:", False)
print("Active dataset modified:", False)
print("Golden Dataset v1 modified:", False)
print("Core-46 modified:", False)
print("Golden Dataset v2 created:", False)

print()
print(
    "NB12h SOURCE CORPUS "
    "LAYER AUDIT: PASS"
)

=== NB12h SOURCE CORPUS LAYER AUDIT ===

Raw source counts:
  Website content files: 530
  Website metadata files: 530
  Book source files: 7
  Transcript source files: 213

Project support files excluded: 27

Document representation files: 11
  - data/phase2/interim/book_ingested_v1.json | records= 1 | role= INTERIM_DOCUMENT_LAYER
  - data/phase2/interim/book_ingested_v2.json | records= 1 | role= INTERIM_DOCUMENT_LAYER
  - data/phase2/interim/speaker_aliases_v1.json | records= 1 | role= INTERIM_DOCUMENT_LAYER
  - data/phase2/interim/transcripts_extracted_v1.jsonl | records= 16226 | role= INTERIM_DOCUMENT_LAYER
  - data/phase2/interim/website_extracted_v2.json | records= 529 | role= INTERIM_DOCUMENT_LAYER
  - data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl | records= 19456 | role= ACCEPTED_DOCUMENT_LAYER_CANDIDATE
  - data/phase2/processed/documents_dedup_quality_quarantine_v1.jsonl | records= 0 | role= PROCESSED_DOCUMENT_LAYER
  - data/phase2/processed/documents_normal

<div dir="rtl" align="right">

## **نقشهٔ راه ثابت ادامهٔ NB12h — از Corpus کامل تا Golden Dataset v2**

این بخش مرجع ثابت ادامهٔ Notebook است تا تصمیم‌های معماری، محدودیت‌ها و مسیر ساخت Golden Dataset v2 در خود Notebook باقی بماند و برای ادامهٔ کار نیازی به بازخوانی تمام نوت‌بوک‌های قبلی نباشد.

### **مسئله‌ای که NB12h حل می‌کند**

Golden Dataset v1 شامل ۵۷۳ رکورد بود، اما بررسی عمیق نشان داد بخش بزرگی از این تعداد از Paraphrase کردن تعداد محدودی Seed حاصل شده است و تعداد ردیف‌ها نمایندهٔ تعداد واقعی Claimهای مستقل Corpus نیست.

Core-46 نیز کیفیت روش Quote-Grounded و Claim-Oriented را اثبات کرد، اما هنوز از فضای محدود Pilotهای موجود شروع شده بود و نمایندهٔ کل دانش موجود در Corpus نیست.

بنابراین Golden Dataset v2 نباید با تعیین یک عدد هدف و تولید سؤال تا رسیدن به آن عدد ساخته شود.

تعداد Seedهای v2 باید به‌صورت طبیعی از تعداد Claimهای مستقل، معتبر، Grounded و غیرتکراری موجود در کل Corpus به دست آید.

### **Population پذیرفته‌شدهٔ فعلی**

Corpus پذیرفته‌شده برای پردازش داخلی شامل ۱۹٬۴۵۶ رکورد است.

ترکیب آن:

- Website: ۵۲۹ رکورد؛

- Book: ۲۷۰۱ رکورد متنی؛

- Transcript: ۱۶٬۲۲۶ Segment؛

- مجموع: ۱۹٬۴۵۶ رکورد.

این اعداد تعداد Processing Record هستند و نباید با تعداد Claimهای مستقل یا تعداد Seedهای Golden Dataset برابر فرض شوند.

در سطح Source نیز چندین Record ممکن است به یک Source اصلی تعلق داشته باشند.

### **Authority Policy موجود**

تمام رکوردهای Corpus دارای اختیار دانشی یکسان نیستند.

Taxonomy v1 چهار گروه اصلی را ثبت کرده است:

- `CORE_PRIMARY_KNOWLEDGE`؛

- `SUPERVISED_SECONDARY_KNOWLEDGE`؛

- `CONTEXTUAL_NONPRIMARY`؛

- `UNRESOLVED_CONTEXT`.

Primary Knowledge می‌تواند مبنای Claim اصلی قرار گیرد.

Secondary Knowledge می‌تواند به‌عنوان دانش پشتیبان یا محتوای آموزشی با Attribution صحیح استفاده شود، اما نباید پنهانی به بیان مستقیم مؤلف تبدیل شود.

Contextual و Unresolved Context نباید به‌تنهایی Primary Claim جدید تعریف کنند.

تمام Source identity، Attribution، Speaker identity، Locator و Lineage باید در مراحل Claim Extraction حفظ شود.

### **Dedup Policy موجود**

Notebook 09 پیش‌تر روابط تکرار را ثبت کرده است، اما هیچ حذف یا Merge خودکاری انجام نداده است.

Artifactهای موجود شامل:

- Exact Duplicate Relations؛

- Near-Duplicate Candidates؛

- Cross-Source Similarity Candidates؛

- Review Queue؛

- Quality Evidence.

این روابط باید در Claim Discovery استفاده شوند تا یک مفهوم یا Claim تکرارشده در چند Document یا چند Source به اشتباه چند Seed مستقل ایجاد نکند.

تکرار یک Claim در چند Source می‌تواند Evidence آن Claim را تقویت کند، اما تعداد Seed را به‌طور مصنوعی افزایش نمی‌دهد.

### **وضعیت Chunking**

Notebook 11 چند Strategy را ساخته و اعتبارسنجی ساختاری کرده است، اما هیچ Chunk Strategy نهایی برای Production انتخاب نشده است.

Candidateهای اصلی شامل:

- `baseline_phase1_char_safe`؛

- `structure_aware_small`؛

- `structure_aware_medium`؛

- `parent_child_child`.

`structure_aware_small` در حال حاضر Leading Candidate ساده است، اما Winner نهایی نیست.

انتخاب Chunk Strategy نهایی به Golden Dataset، Retrieval Benchmark، Citation Evaluation، Attribution Correctness و Context Quality وابسته است.

بنابراین NB12h نباید برای ساخت v2 یک Chunk Strategy را به‌عنوان Production Winner Freeze کند.

### **Taxonomy و Competency**

Taxonomy v1 قبلاً Freeze شده است و شامل ۷ Topic اصلی است.

Competency Model نیز Freeze شده و شامل ۲۴ Competency است.

این ساختارها در NB12h دوباره ساخته نمی‌شوند.

از آن‌ها برای بررسی Coverage Candidate Seedهای v2 استفاده خواهد شد تا مشخص شود Golden Dataset حاصل فقط روی چند موضوع محدود متمرکز نشده باشد.

Coverage به معنی تولید مصنوعی Claim برای پر کردن یک Topic یا Competency نیست.

اگر Corpus Claim مستقلی برای یک بخش نداشته باشد، صرفاً برای پر کردن Coverage Claim جدید ساخته نمی‌شود.

### **مسیر اجرایی ادامهٔ NB12h**

مسیر ادامهٔ Notebook به این ترتیب است:

**مرحله ۱ — نقش Datasetها**  
تعیین نقش Golden Dataset v1، Pilot-28، Core-46 و v2.  
وضعیت: انجام شده.

**مرحله ۲ — ارتباط Pilot-28 و Core-46**  
بررسی هم‌پوشانی Claimهای Pilot و Core.  
وضعیت: انجام شده.

**مرحله ۳ — Whole Corpus Inventory**  
کشف فضای واقعی Sourceها و Representationهای موجود.  
وضعیت: انجام شده.

**مرحله ۴ — Corpus Layer Audit**  
تفکیک Raw Source، Processing Document، Chunk، Artifact و Project Support.  
وضعیت: انجام شده.

**مرحله ۵ — Frozen Corpus Reconciliation**  
تطبیق Corpus فعلی با Snapshotهای پذیرفته‌شدهٔ NB09، NB10 و NB11 و تعریف Population رسمی Claim Discovery.

**مرحله ۶ — Claim Discovery Preparation**  
ساخت واحدهای ورودی Claim Extraction با حفظ Source، Authority، Locator، Taxonomy و Dedup Relation.

**مرحله ۷ — Candidate Claim Extraction**  
استخراج Claimهای اتمیک و Grounded از کل Corpus پذیرفته‌شده.

در این مرحله هر Candidate Claim باید حداقل دارای موارد زیر باشد:

- متن Claim؛

- Evidence مستقیم؛

- Source ID؛

- Document ID یا Locator؛

- Authority Class؛

- Source Type؛

- Provenance؛

- Topic در صورت وجود.

**مرحله ۸ — Claim-Level Deduplication**  
حذف Exact Claim Duplicate، Semantic Duplicate، Subsumed Claim و Partial Overlap.

وجود یک Claim در چند Source باید به‌صورت Multi-Evidence ثبت شود، نه چند Seed مستقل.

**مرحله ۹ — مقایسه با Pilot-28 و Core-46**  
Candidate Claimهای Corpus با Claimهای موجود مقایسه می‌شوند.

رابطهٔ هر Claim می‌تواند یکی از موارد زیر باشد:

- Equivalent؛

- Subsumed؛

- Partial Overlap؛

- Independent؛

- Review Required.

**مرحله ۱۰ — تعیین Natural Seed Count**  
پس از Deduplication تعداد طبیعی Candidate Seedهای مستقل مشخص می‌شود.

هیچ Target عددی از قبل تعیین نمی‌شود.

**مرحله ۱۱ — Coverage Audit**  
Candidate Seedها نسبت به Source Type، Taxonomy، Competency، Authority و Caseهای موردنیاز NB11 بررسی می‌شوند.

Coverage Audit فقط برای تشخیص Blind Spot است و اجازهٔ ساخت Claim مصنوعی نمی‌دهد.

**مرحله ۱۲ — توقف برای Author Review**  
قبل از Freeze کردن Golden Dataset v2، بستهٔ Claim-Level برای نظر نهایی دکتر رودی آماده می‌شود.

**مرحله ۱۳ — Golden Dataset v2**  
فقط پس از Author Review، Claimهای پذیرفته‌شده به Golden Seed و سپس Question/Answer/Evidence Case تبدیل می‌شوند.

### **قاعدهٔ ساخت Seed**

قاعدهٔ اصلی:

`Independent Grounded Claim + Direct Evidence + Valid Provenance = Candidate Seed`

Paraphrase سؤال Seed جدید محسوب نمی‌شود.

وجود Evidence دوم یا سوم نیز Seed جدید ایجاد نمی‌کند، مگر اینکه Claim واقعاً مستقل باشد.

### **قواعد تغییرناپذیر NB12h**

- Golden Dataset v1 تغییر نمی‌کند؛

- Pilot-28 overwrite نمی‌شود؛

- Core-46 overwrite نمی‌شود؛

- هیچ Artifact موجودی بدون بررسی جایگزین نمی‌شود؛

- خروجی جدید باید Versioned و No-Overwrite باشد؛

- تعداد v2 از قبل تعیین نمی‌شود؛

- Chunk Strategy نهایی در NB12h Freeze نمی‌شود؛

- Rights Release Gate با Processing Eligibility اشتباه گرفته نمی‌شود؛

- هیچ Claim بدون Evidence مستقیم وارد Candidate Seed Pool نمی‌شود؛

- نظر نهایی مؤلف قبل از Freeze v2 الزامی است.

### **وضعیت فعلی**

اکنون در مرحلهٔ **Frozen Corpus Reconciliation** قرار داریم.

قدم بعدی تطبیق Population فعلی با Artifactهای Freeze‌شدهٔ Notebookهای 09، 10 و 11 و تعریف دقیق ورودی Claim Discovery است.

</div>

<div dir="rtl" align="right">

## **مرحلهٔ سی‌وششم — تطبیق Snapshotهای Freeze‌شده و تعریف Population رسمی Claim Discovery**

مرحله‌های قبل مشخص کردند که Repository شامل تعداد زیادی Raw Source، Document Representation، Chunk Variant و Artifact مشتق‌شده است.

اما برای ادامهٔ ساخت Golden Dataset v2 نباید دوباره تمام این لایه‌ها را از ابتدا ارزیابی کنیم.

Notebookهای قبلی فاز ۲ قبلاً یک Processing Corpus مشخص را پذیرفته و Dataset، Dedup Relationها، Taxonomy، Competency Model و وضعیت Chunking را Freeze کرده‌اند.

### **هدف این مرحله**

هدف این مرحله ساخت Dataset جدید یا استخراج Claim نیست.

هدف این است که قبل از شروع Claim Discovery، وضعیت فعلی Repository با Snapshotهای پذیرفته‌شدهٔ قبلی تطبیق داده شود و یک قرارداد روشن برای Population مرحلهٔ بعد ایجاد شود.

در این مرحله بررسی می‌شود که:

- Accepted Processing Corpus همان Artifact پذیرفته‌شدهٔ Notebook 09 است؛

- تعداد و Hash آن تغییر نکرده است؛

- توزیع Website، Book و Transcript با Population پذیرفته‌شده تطابق دارد؛

- Source IDها و Document IDها قابل ردیابی هستند؛

- Dedup Relationهای Exact، Near و Cross-Source موجود و بدون تغییر هستند؛

- Taxonomy v1 و Competency Model همچنان Freeze هستند؛

- Knowledge Authority Policy ثبت‌شدهٔ Notebook 10 حفظ شده است؛

- Notebook 11 هنوز Final Chunk Strategy انتخاب نکرده است؛

- و Golden Dataset همچنان یکی از پیش‌نیازهای تصمیم نهایی Chunking است.

### **Population رسمی این Notebook**

ورودی اصلی Claim Discovery فایل پذیرفته‌شدهٔ Notebook 09 است:

`data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl`

این Dataset شامل Processing Recordهای پذیرفته‌شده است و مبنای بررسی کل Corpus خواهد بود.

این Population به معنی تعداد Seed یا Claim نیست.

هر Source ممکن است چندین Processing Record داشته باشد و چند Record نیز ممکن است یک Claim مشترک را بیان کنند.

### **نقش Dedup Relationها**

Notebook 09 پیش‌تر روابط Exact، Near و Cross-Source را محاسبه کرده است.

این Artifactها حذف نمی‌شوند و دوباره از صفر ساخته نمی‌شوند.

در مراحل Claim Discovery از آن‌ها برای جلوگیری از استخراج چندبارهٔ دانش مشابه استفاده خواهد شد.

Duplicate Relation به معنی حذف Source نیست.

اگر یک Claim در چند Source مستقل وجود داشته باشد، Sourceهای مختلف باید به‌عنوان Evidenceهای جداگانه حفظ شوند.

### **نقش Authority**

Taxonomy v1 قبلاً Policy اختیار دانشی را Freeze کرده است.

در Claim Discovery:

- Primary Knowledge می‌تواند Claim اصلی تعریف کند؛

- Secondary Knowledge با Attribution صحیح می‌تواند نقش پشتیبان یا محتوای آموزشی داشته باشد؛

- Contextual Nonprimary به‌تنهایی Primary Claim تعریف نمی‌کند؛

- Unresolved Context تا زمان رفع ابهام به‌تنهایی Seed authoritative ایجاد نمی‌کند.

در این مرحله هنوز رکوردی حذف نمی‌شود و فقط قرارداد استفاده از این گروه‌ها ثبت می‌شود.

### **وضعیت Rights**

Accepted Processing Corpus برای پردازش داخلی پذیرفته شده است.

این موضوع به معنی Formal Release Eligibility نیست.

Rights Evidence همچنان یک Gate مستقل برای Release نهایی است و نباید باعث شود Population معتبر برای تحلیل داخلی با Release-ready Dataset اشتباه گرفته شود.

### **وضعیت Chunking**

Notebook 11 چند Strategy را برای Evaluation آماده کرده است، اما Final Production Strategy همچنان انتخاب نشده است.

بنابراین در این مرحله:

- Chunk جدید ساخته نمی‌شود؛

- Chunk Strategy جدید Freeze نمی‌شود؛

- و Claim Discovery به انتخاب Production Chunk وابسته نمی‌شود.

Golden Dataset حاصل از همین مسیر بعداً یکی از ورودی‌های اصلی Retrieval و Chunk Strategy Evaluation خواهد بود.

### **مرز تصمیم**

در این مرحله:

- Claim Extraction انجام نمی‌شود؛

- Semantic Claim Dedup انجام نمی‌شود؛

- Pilot-28 و Core-46 تغییر نمی‌کنند؛

- Golden Dataset v1 تغییر نمی‌کند؛

- Golden Dataset v2 ساخته نمی‌شود؛

- API یا LLM فراخوانی نمی‌شود؛

- هیچ Dataset یا Artifact قبلی overwrite نمی‌شود.

### **سیاست عدم Overwrite**

گزارش این مرحله با نام جدید و Versioned ساخته می‌شود.

اگر Artifact هم‌نام قبلاً وجود داشته باشد:

- در صورت یکسان بودن محتوای قابل مقایسه reuse می‌شود؛

- در صورت متفاوت بودن محتوا اجرا متوقف می‌شود؛

- فایل موجود جایگزین نمی‌شود.

### **Validation**

این مرحله زمانی PASS است که:

- Accepted Corpus دقیقاً ۱۹٬۴۵۶ رکورد داشته باشد؛

- SHA-256 آن با Snapshot پذیرفته‌شدهٔ Notebook 09 و Taxonomy v1 تطابق داشته باشد؛

- تعداد رکوردهای Website، Book و Transcript با Population Freeze‌شده سازگار باشد؛

- تعداد Source IDهای مستقل هر خانواده قابل محاسبه باشد؛

- Document ID تکراری وجود نداشته باشد؛

- Artifactهای Dedup پذیرفته‌شده موجود و Hash آن‌ها صحیح باشد؛

- Taxonomy v1 در وضعیت `FROZEN` باشد؛

- Competency Model در وضعیت `FROZEN` باشد؛

- مجموع Authority Classهای ثبت‌شده با ۱۹٬۴۵۶ رکورد برابر باشد؛

- وضعیت Chunk Selection همچنان `NOT_YET` باشد؛

- Golden Dataset در فهرست نیازهای باقی‌ماندهٔ Chunk Selection قرار داشته باشد؛

- و هیچ Dataset موجودی تغییر نکرده باشد.

پس از PASS شدن این مرحله، Notebook می‌تواند وارد طراحی ورودی واقعی Candidate Claim Extraction شود.

</div>

In [38]:
# Code Cell 36
# NB12h — Frozen corpus reconciliation and
# claim-discovery population contract
#
# Read-only validation of previously frozen Phase-2 artifacts.
# No claim extraction.
# No API calls.
# No existing artifact overwrite.

assert "project_root" in globals(), (
    "Run the NB12h project-discovery Cell first."
)

assert "role_matrix_output_directory" in globals(), (
    "Run Code Cell 31 first."
)

assert "source_corpus_layer_audit_report" in globals(), (
    "Run Code Cell 35 first."
)

CORPUS_RECONCILIATION_VERSION = (
    "nb12h_frozen_corpus_reconciliation_v1"
)

CORPUS_RECONCILIATION_REPORT_FILENAME = (
    "frozen_corpus_reconciliation_v1.json"
)


# ---------------------------------------------------------
# Required frozen artifact paths
# ---------------------------------------------------------

accepted_corpus_path = (
    project_root
    / "data/phase2/processed/"
    "documents_dedup_quality_accepted_v1.jsonl"
)

dedup_acceptance_path = (
    project_root
    / "reports/phase2/"
    "deduplication_quality_acceptance_v1.json"
)

taxonomy_path = (
    project_root
    / "artifacts/phase2/taxonomy/"
    "taxonomy_v1.json"
)

competency_path = (
    project_root
    / "artifacts/phase2/competency/"
    "competency_prerequisite_v1.json"
)


def find_unique_project_file(filename):
    matches = sorted({
        path.resolve()
        for path in project_root.rglob(
            filename
        )
        if path.is_file()
        and (
            "golden_dataset_rebuild_clean"
            not in str(
                path
            ).casefold()
        )
    })

    assert matches, (
        f"Required artifact not found: {filename}"
    )

    assert len(matches) == 1, (
        f"Multiple copies found for {filename}: "
        f"{[str(path) for path in matches]}"
    )

    return matches[0]


nb12_chunk_handoff_path = (
    find_unique_project_file(
        "nb12_golden_dataset_handoff_v1.json"
    )
)

chunk_decision_readiness_path = (
    find_unique_project_file(
        "decision_readiness_v1.json"
    )
)

chunk_final_requirements_path = (
    find_unique_project_file(
        "final_selection_requirements_v1.json"
    )
)


required_paths = [
    accepted_corpus_path,
    dedup_acceptance_path,
    taxonomy_path,
    competency_path,
    nb12_chunk_handoff_path,
    chunk_decision_readiness_path,
    chunk_final_requirements_path,
]

for path in required_paths:
    assert path.is_file(), (
        f"Required frozen artifact missing: {path}"
    )


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as file:
        while True:
            block = file.read(
                1024 * 1024
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def load_json_file(path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(
            file
        )


def count_jsonl_records(path):
    count = 0

    with path.open(
        "r",
        encoding="utf-8",
        errors="strict",
    ) as file:
        for line in file:
            if line.strip():
                count += 1

    return count


# ---------------------------------------------------------
# Load frozen metadata artifacts
# ---------------------------------------------------------

dedup_acceptance = load_json_file(
    dedup_acceptance_path
)

taxonomy_payload = load_json_file(
    taxonomy_path
)

competency_payload = load_json_file(
    competency_path
)

nb12_chunk_handoff = load_json_file(
    nb12_chunk_handoff_path
)

chunk_decision_readiness = load_json_file(
    chunk_decision_readiness_path
)

chunk_final_requirements = load_json_file(
    chunk_final_requirements_path
)


# ---------------------------------------------------------
# Validate NB09 processing acceptance
# ---------------------------------------------------------

assert (
    dedup_acceptance.get(
        "status"
    )
    == "PASS"
), (
    "NB09 acceptance report is not PASS."
)

processing_decision = (
    dedup_acceptance.get(
        "processing_decision",
        {},
    )
)

assert (
    processing_decision.get(
        "status"
    )
    == "ACCEPTED"
), (
    "NB09 processing corpus is not ACCEPTED."
)

expected_record_count = (
    processing_decision.get(
        "record_count"
    )
)

assert (
    expected_record_count
    == 19456
), (
    "Frozen accepted processing population "
    "is not 19456 records."
)

accepted_artifact_spec = (
    dedup_acceptance[
        "artifacts"
    ][
        "accepted_processing_dataset"
    ]
)

expected_accepted_sha256 = (
    accepted_artifact_spec[
        "sha256"
    ]
)

actual_accepted_sha256 = (
    sha256_file(
        accepted_corpus_path
    )
)

assert (
    actual_accepted_sha256
    == expected_accepted_sha256
), (
    "Accepted corpus SHA-256 differs from "
    "NB09 frozen acceptance."
)

actual_record_count = (
    count_jsonl_records(
        accepted_corpus_path
    )
)

assert (
    actual_record_count
    == expected_record_count
), (
    "Accepted corpus record count differs "
    "from NB09 frozen acceptance."
)


# ---------------------------------------------------------
# Cross-check against Taxonomy source handoff
# ---------------------------------------------------------

taxonomy_source_handoff = (
    taxonomy_payload.get(
        "source_handoff",
        {},
    )
)

assert (
    taxonomy_payload.get(
        "status"
    )
    == "FROZEN"
), (
    "Taxonomy v1 is not FROZEN."
)

assert (
    taxonomy_source_handoff.get(
        "accepted_corpus_records"
    )
    == actual_record_count
), (
    "Taxonomy source handoff record count "
    "does not match accepted corpus."
)

assert (
    taxonomy_source_handoff.get(
        "accepted_corpus_sha256"
    )
    == actual_accepted_sha256
), (
    "Taxonomy source handoff SHA-256 "
    "does not match accepted corpus."
)


# ---------------------------------------------------------
# Full accepted-corpus structural reconciliation
# ---------------------------------------------------------

accepted_schema_keys = set()

accepted_source_family_record_counts = Counter()

accepted_unique_source_ids = {
    "website": set(),
    "book": set(),
    "transcript": set(),
}

accepted_document_ids = []

accepted_content_origin_counts = Counter()

accepted_content_authority_type_counts = Counter()

unclassified_source_family_count = 0


def detect_source_family(
    record,
):
    explicit_candidates = [
        record.get(
            "source_type"
        ),
        record.get(
            "source_family"
        ),
    ]

    for candidate in (
        explicit_candidates
    ):
        if not isinstance(
            candidate,
            str,
        ):
            continue

        normalized = (
            candidate
            .strip()
            .casefold()
        )

        if normalized in {
            "website",
            "book",
            "transcript",
        }:
            return normalized

    for field_name in [
        "source_id",
        "document_id",
    ]:
        value = record.get(
            field_name
        )

        if not isinstance(
            value,
            str,
        ):
            continue

        prefix = (
            value
            .split(
                ":",
                1,
            )[0]
            .strip()
            .casefold()
        )

        if prefix in {
            "website",
            "book",
            "transcript",
        }:
            return prefix

    return None


with accepted_corpus_path.open(
    "r",
    encoding="utf-8",
    errors="strict",
) as file:
    for line_number, line in enumerate(
        file,
        1,
    ):
        stripped = line.strip()

        if not stripped:
            continue

        record = json.loads(
            stripped
        )

        assert isinstance(
            record,
            dict,
        ), (
            "Accepted corpus contains a "
            "non-object JSON record at line "
            f"{line_number}."
        )

        accepted_schema_keys.update(
            record.keys()
        )

        document_id = record.get(
            "document_id"
        )

        source_id = record.get(
            "source_id"
        )

        assert isinstance(
            document_id,
            str,
        ) and document_id.strip(), (
            "Missing document_id at accepted "
            f"corpus line {line_number}."
        )

        assert isinstance(
            source_id,
            str,
        ) and source_id.strip(), (
            "Missing source_id at accepted "
            f"corpus line {line_number}."
        )

        accepted_document_ids.append(
            document_id
        )

        source_family = (
            detect_source_family(
                record
            )
        )

        if source_family is None:
            unclassified_source_family_count += 1

        else:
            accepted_source_family_record_counts[
                source_family
            ] += 1

            accepted_unique_source_ids[
                source_family
            ].add(
                source_id
            )

        content_origin = record.get(
            "content_origin"
        )

        if isinstance(
            content_origin,
            str,
        ) and content_origin.strip():
            accepted_content_origin_counts[
                content_origin
            ] += 1

        content_authority_type = (
            record.get(
                "content_authority_type"
            )
        )

        if isinstance(
            content_authority_type,
            str,
        ) and (
            content_authority_type.strip()
        ):
            accepted_content_authority_type_counts[
                content_authority_type
            ] += 1


assert (
    len(
        accepted_document_ids
    )
    == actual_record_count
), (
    "Full-corpus scan count mismatch."
)

assert (
    len(
        set(
            accepted_document_ids
        )
    )
    == actual_record_count
), (
    "Duplicate document_id detected in "
    "accepted processing corpus."
)

assert (
    unclassified_source_family_count
    == 0
), (
    "Some accepted records could not be "
    "classified as Website, Book or Transcript."
)


# ---------------------------------------------------------
# Frozen source-family population
# ---------------------------------------------------------

expected_source_family_record_counts = {
    "website": 529,
    "book": 2701,
    "transcript": 16226,
}

assert (
    dict(
        accepted_source_family_record_counts
    )
    == expected_source_family_record_counts
), (
    "Accepted source-family record counts "
    "differ from the frozen Phase-2 population."
)


accepted_unique_source_counts = {
    source_family: len(
        source_ids
    )
    for (
        source_family,
        source_ids,
    )
    in accepted_unique_source_ids.items()
}

expected_unique_source_counts = {
    "website": 529,
    "book": 3,
    "transcript": 183,
}

assert (
    accepted_unique_source_counts
    == expected_unique_source_counts
), (
    "Accepted unique source counts differ "
    "from the frozen Phase-2 source population."
)


# ---------------------------------------------------------
# Validate existing dedup relation artifacts
# ---------------------------------------------------------

dedup_artifact_keys = [
    "exact_duplicate_relations",
    "near_duplicate_candidates",
    "cross_source_similarity_candidates",
    "review_queue",
    "hard_quarantine_dataset",
]

dedup_artifact_validation = {}

for artifact_key in (
    dedup_artifact_keys
):
    artifact_spec = (
        dedup_acceptance[
            "artifacts"
        ][artifact_key]
    )

    artifact_path = (
        project_root
        / artifact_spec[
            "path"
        ]
    )

    assert artifact_path.is_file(), (
        "Frozen dedup artifact missing: "
        f"{artifact_path}"
    )

    actual_sha256 = (
        sha256_file(
            artifact_path
        )
    )

    assert (
        actual_sha256
        == artifact_spec[
            "sha256"
        ]
    ), (
        "Dedup artifact SHA-256 mismatch: "
        f"{artifact_key}"
    )

    actual_count = (
        count_jsonl_records(
            artifact_path
        )
    )

    assert (
        actual_count
        == artifact_spec[
            "record_count"
        ]
    ), (
        "Dedup artifact record-count mismatch: "
        f"{artifact_key}"
    )

    dedup_artifact_validation[
        artifact_key
    ] = {
        "relative_path": (
            artifact_spec[
                "path"
            ]
        ),
        "record_count": (
            actual_count
        ),
        "sha256": (
            actual_sha256
        ),
        "status": "MATCH",
    }


# ---------------------------------------------------------
# Validate Authority Policy snapshot
# ---------------------------------------------------------

knowledge_authority = (
    taxonomy_payload.get(
        "knowledge_authority",
        {}
    )
)

authority_policy = (
    knowledge_authority.get(
        "policy",
        {}
    )
)

authority_summary = (
    knowledge_authority.get(
        "summary",
        {}
    )
)

assert (
    authority_summary.get(
        "documents_total"
    )
    == actual_record_count
), (
    "Taxonomy authority summary total "
    "does not match accepted corpus."
)

authority_partition_counts = {
    "core_primary_knowledge": (
        authority_summary.get(
            "core_primary_knowledge"
        )
    ),
    "supervised_secondary_knowledge": (
        authority_summary.get(
            "supervised_secondary_knowledge"
        )
    ),
    "contextual_nonprimary": (
        authority_summary.get(
            "contextual_nonprimary"
        )
    ),
    "unresolved_context": (
        authority_summary.get(
            "unresolved_context"
        )
    ),
}

assert all(
    isinstance(
        value,
        int,
    )
    for value in (
        authority_partition_counts
        .values()
    )
), (
    "Authority partition contains "
    "non-integer counts."
)

assert (
    sum(
        authority_partition_counts
        .values()
    )
    == actual_record_count
), (
    "Authority partition does not sum "
    "to the full accepted corpus."
)

assert (
    authority_policy.get(
        "context_can_define_primary_claim"
    )
    is False
), (
    "Unexpected contextual-source "
    "primary-claim policy."
)

assert (
    authority_policy.get(
        "unresolved_can_define_primary_claim"
    )
    is False
), (
    "Unexpected unresolved-source "
    "primary-claim policy."
)

assert (
    authority_policy.get(
        "secondary_can_override_primary"
    )
    is False
), (
    "Unexpected secondary-source "
    "authority policy."
)


# ---------------------------------------------------------
# Validate Competency Model snapshot
# ---------------------------------------------------------

assert (
    competency_payload.get(
        "status"
    )
    == "FROZEN"
), (
    "Competency model is not FROZEN."
)

competency_summary = (
    competency_payload.get(
        "competency_summary",
        {}
    )
)

assert (
    competency_summary.get(
        "total_competencies"
    )
    == 24
), (
    "Frozen competency count is not 24."
)

assert (
    competency_summary.get(
        "topic_count"
    )
    == 7
), (
    "Frozen competency topic count is not 7."
)

assert (
    competency_payload.get(
        "taxonomy_reference",
        {}
    ).get(
        "taxonomy_version"
    )
    == taxonomy_payload.get(
        "taxonomy_version"
    )
), (
    "Competency model taxonomy reference "
    "does not match Taxonomy v1."
)


# ---------------------------------------------------------
# Validate NB11 decision state
# ---------------------------------------------------------

assert (
    nb12_chunk_handoff.get(
        "status"
    )
    == "REQUIRED_FOR_FINAL_CHUNK_DECISION"
), (
    "Unexpected NB12 chunk handoff status."
)

assert (
    chunk_final_requirements.get(
        "final_selection"
    )
    == "NOT_YET"
), (
    "Chunk final-selection state changed."
)

assert (
    "golden_dataset"
    in chunk_final_requirements.get(
        "still_required",
        []
    )
), (
    "Golden Dataset is no longer listed as "
    "a prerequisite for final chunk selection."
)

assert (
    chunk_decision_readiness.get(
        "final_strategy"
    )
    is None
), (
    "NB11 unexpectedly contains a final "
    "production chunk strategy."
)


# ---------------------------------------------------------
# Rights status is recorded, not treated as processing FAIL
# ---------------------------------------------------------

release_decision = (
    dedup_acceptance.get(
        "release_decision",
        {}
    )
)

rights_gate = (
    dedup_acceptance.get(
        "open_gates",
        {}
    ).get(
        "rights_evidence",
        {}
    )
)

assert (
    release_decision.get(
        "status"
    )
    == "BLOCKED_PENDING_RIGHTS"
), (
    "Unexpected formal release-rights state."
)

assert (
    rights_gate.get(
        "affected_document_count"
    )
    == actual_record_count
), (
    "Rights gate affected-document count "
    "does not match accepted corpus."
)


# ---------------------------------------------------------
# Define claim-discovery contract
# ---------------------------------------------------------

claim_discovery_population_contract = {
    "population_basis": (
        "NB09_ACCEPTED_PROCESSING_CORPUS"
    ),

    "accepted_processing_record_count": (
        actual_record_count
    ),

    "accepted_processing_sha256": (
        actual_accepted_sha256
    ),

    "source_family_record_counts": dict(
        accepted_source_family_record_counts
    ),

    "unique_source_counts": (
        accepted_unique_source_counts
    ),

    "authority_partition": (
        authority_partition_counts
    ),

    "authority_policy": {
        "primary_can_define_primary_claim": True,
        "secondary_can_override_primary": (
            authority_policy.get(
                "secondary_can_override_primary"
            )
        ),
        "context_can_define_primary_claim": (
            authority_policy.get(
                "context_can_define_primary_claim"
            )
        ),
        "unresolved_can_define_primary_claim": (
            authority_policy.get(
                "unresolved_can_define_primary_claim"
            )
        ),
    },

    "dedup_relations_must_be_preserved": True,

    "existing_claim_sets_to_compare_later": [
        "Pilot-28",
        "Core-46",
    ],

    "seed_count_policy": (
        "NATURAL_AFTER_CLAIM_EXTRACTION_AND_"
        "SEMANTIC_DEDUPLICATION"
    ),

    "production_chunk_strategy_required_now": False,

    "claim_extraction_started": False,

    "golden_dataset_v2_created": False,
}


# ---------------------------------------------------------
# Build audit report
# ---------------------------------------------------------

frozen_corpus_reconciliation_report = {
    "audit_version": (
        CORPUS_RECONCILIATION_VERSION
    ),

    "status": "PASS",

    "accepted_corpus": {
        "relative_path": str(
            accepted_corpus_path.relative_to(
                project_root
            )
        ),
        "record_count": (
            actual_record_count
        ),
        "sha256": (
            actual_accepted_sha256
        ),
        "document_id_unique_count": len(
            set(
                accepted_document_ids
            )
        ),
        "schema_keys": sorted(
            accepted_schema_keys
        ),
        "source_family_record_counts": dict(
            accepted_source_family_record_counts
        ),
        "unique_source_counts": (
            accepted_unique_source_counts
        ),
        "content_origin_counts": dict(
            sorted(
                accepted_content_origin_counts
                .items()
            )
        ),
        "content_authority_type_counts": dict(
            sorted(
                accepted_content_authority_type_counts
                .items()
            )
        ),
    },

    "dedup_artifacts": (
        dedup_artifact_validation
    ),

    "taxonomy": {
        "version": (
            taxonomy_payload.get(
                "taxonomy_version"
            )
        ),
        "status": (
            taxonomy_payload.get(
                "status"
            )
        ),
        "top_level_topic_count": (
            taxonomy_payload.get(
                "validation_snapshot",
                {},
            ).get(
                "top_level_topic_count"
            )
        ),
        "authority_partition": (
            authority_partition_counts
        ),
    },

    "competency": {
        "version": (
            competency_payload.get(
                "competency_model_version"
            )
        ),
        "status": (
            competency_payload.get(
                "status"
            )
        ),
        "competency_count": (
            competency_summary.get(
                "total_competencies"
            )
        ),
        "topic_count": (
            competency_summary.get(
                "topic_count"
            )
        ),
    },

    "chunking": {
        "handoff_status": (
            nb12_chunk_handoff.get(
                "status"
            )
        ),
        "final_selection": (
            chunk_final_requirements.get(
                "final_selection"
            )
        ),
        "current_status": (
            chunk_final_requirements.get(
                "current_status"
            )
        ),
        "still_required": (
            chunk_final_requirements.get(
                "still_required"
            )
        ),
        "final_strategy": (
            chunk_decision_readiness.get(
                "final_strategy"
            )
        ),
    },

    "rights": {
        "processing_accepted": True,
        "formal_release_status": (
            release_decision.get(
                "status"
            )
        ),
        "affected_document_count": (
            rights_gate.get(
                "affected_document_count"
            )
        ),
    },

    "claim_discovery_population_contract": (
        claim_discovery_population_contract
    ),

    "api_calls_made": 0,
    "cost_incurred": 0,

    "source_files_modified": False,
    "active_dataset_modified": False,
    "golden_dataset_v1_modified": False,
    "core46_modified": False,
    "golden_dataset_v2_created": False,
    "claim_extraction_started": False,
}


# ---------------------------------------------------------
# No-overwrite persistence
# ---------------------------------------------------------

frozen_corpus_reconciliation_report_path = (
    role_matrix_output_directory
    / CORPUS_RECONCILIATION_REPORT_FILENAME
)

if (
    frozen_corpus_reconciliation_report_path
    .exists()
):
    with (
        frozen_corpus_reconciliation_report_path
        .open(
            "r",
            encoding="utf-8",
        )
    ) as file:
        existing_report = json.load(
            file
        )

    comparable_existing = dict(
        existing_report
    )

    comparable_current = dict(
        frozen_corpus_reconciliation_report
    )

    comparable_existing.pop(
        "created_at_utc",
        None,
    )

    comparable_current.pop(
        "created_at_utc",
        None,
    )

    assert (
        comparable_existing
        == comparable_current
    ), (
        "Existing frozen corpus reconciliation "
        "report differs from this run. "
        "Refusing to overwrite it."
    )

    frozen_corpus_reconciliation_report = (
        existing_report
    )

    reconciliation_report_write_status = (
        "REUSED_EXISTING_IDENTICAL_REPORT"
    )

else:
    frozen_corpus_reconciliation_report[
        "created_at_utc"
    ] = datetime.now(
        timezone.utc
    ).isoformat()

    with (
        frozen_corpus_reconciliation_report_path
        .open(
            "x",
            encoding="utf-8",
        )
    ) as file:
        json.dump(
            frozen_corpus_reconciliation_report,
            file,
            ensure_ascii=False,
            indent=2,
        )

        file.write("\n")

    reconciliation_report_write_status = (
        "CREATED_NEW_REPORT"
    )


# ---------------------------------------------------------
# Compact output
# ---------------------------------------------------------

print(
    "=== NB12h FROZEN CORPUS "
    "RECONCILIATION ==="
)

print()
print(
    "Accepted processing records:",
    actual_record_count,
)

print(
    "Accepted corpus SHA-256:",
    actual_accepted_sha256,
)

print()
print(
    "Record counts by source family:",
    dict(
        accepted_source_family_record_counts
    ),
)

print(
    "Unique source counts:",
    accepted_unique_source_counts,
)

print()
print(
    "Authority partition:",
    authority_partition_counts,
)

print()
print(
    "Dedup artifacts:"
)

for (
    artifact_key,
    artifact_result,
) in (
    dedup_artifact_validation.items()
):
    print(
        "  -",
        artifact_key,
        "| records=",
        artifact_result[
            "record_count"
        ],
        "| status=",
        artifact_result[
            "status"
        ],
    )

print()
print(
    "Taxonomy:",
    taxonomy_payload.get(
        "status"
    ),
    "| topics=",
    taxonomy_payload.get(
        "validation_snapshot",
        {},
    ).get(
        "top_level_topic_count"
    ),
)

print(
    "Competency model:",
    competency_payload.get(
        "status"
    ),
    "| competencies=",
    competency_summary.get(
        "total_competencies"
    ),
)

print()
print(
    "Chunk final selection:",
    chunk_final_requirements.get(
        "final_selection"
    ),
)

print(
    "Chunk final strategy:",
    chunk_decision_readiness.get(
        "final_strategy"
    ),
)

print(
    "Golden Dataset required for "
    "chunk decision:",
    (
        "golden_dataset"
        in chunk_final_requirements.get(
            "still_required",
            []
        )
    ),
)

print()
print(
    "Formal release rights status:",
    release_decision.get(
        "status"
    ),
)

print()
print(
    "Claim discovery population:",
    actual_record_count,
    "accepted processing records",
)

print(
    "Claim extraction started:",
    False,
)

print(
    "Natural seed count determined:",
    False,
)

print()
print(
    "Audit report:",
    frozen_corpus_reconciliation_report_path,
)

print(
    "Report write status:",
    reconciliation_report_write_status,
)

print()
print("API calls made:", 0)
print("Cost incurred:", 0)
print("Source files modified:", False)
print("Active dataset modified:", False)
print("Golden Dataset v1 modified:", False)
print("Core-46 modified:", False)
print("Golden Dataset v2 created:", False)

print()
print(
    "NB12h FROZEN CORPUS "
    "RECONCILIATION: PASS"
)

=== NB12h FROZEN CORPUS RECONCILIATION ===

Accepted processing records: 19456
Accepted corpus SHA-256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40

Record counts by source family: {'website': 529, 'book': 2701, 'transcript': 16226}
Unique source counts: {'website': 529, 'book': 3, 'transcript': 183}

Authority partition: {'core_primary_knowledge': 14705, 'supervised_secondary_knowledge': 453, 'contextual_nonprimary': 1944, 'unresolved_context': 2354}

Dedup artifacts:
  - exact_duplicate_relations | records= 192 | status= MATCH
  - near_duplicate_candidates | records= 15929 | status= MATCH
  - cross_source_similarity_candidates | records= 37796 | status= MATCH
  - review_queue | records= 7 | status= MATCH
  - hard_quarantine_dataset | records= 0 | status= MATCH

Taxonomy: FROZEN | topics= 7
Competency model: FROZEN | competencies= 24

Chunk final selection: NOT_YET
Chunk final strategy: None
Golden Dataset required for chunk decision: True

Formal release rights s

<div dir="rtl" align="right">

## **مرحلهٔ سی‌وهفتم — ساخت Population آمادهٔ Claim Discovery با حفظ Authority و Provenance**

مرحلهٔ قبل مشخص کرد که Corpus رسمی این Notebook همان ۱۹٬۴۵۶ رکورد پذیرفته‌شدهٔ Notebook 09 است و تمام Snapshotهای اصلی فاز ۲ بدون تغییر با آن تطابق دارند.

اما تمام این ۱۹٬۴۵۶ رکورد نباید به یک شکل وارد Claim Extraction شوند.

در Notebook 10 یک Authority Model مبتنی بر Evidence واقعی Sourceها ساخته و Freeze شده است که مشخص می‌کند هر رکورد چه نقشی در Knowledge System دارد.

### **هدف این مرحله**

در این مرحله برای تمام رکوردهای Accepted Corpus یک Index تحلیلی ساخته می‌شود که بدون کپی‌کردن متن کامل، مشخص می‌کند هر Document:

- متعلق به کدام Source Type و Source ID است؛

- چه Authority Class دارد؛

- Authority آن بر چه Evidence موجودی استوار است؛

- آیا می‌تواند وارد Claim Discovery اصلی شود؛

- آیا فقط Supporting Knowledge است؛

- آیا فقط Context محسوب می‌شود؛

- یا به دلیل Attribution حل‌نشده باید از ساخت Seed مستقل کنار گذاشته شود.

### **چهار لایهٔ Authority**

Authority Model فریز‌شده شامل چهار لایه است:

- `CORE_PRIMARY_KNOWLEDGE`

- `SUPERVISED_SECONDARY_KNOWLEDGE`

- `CONTEXTUAL_NONPRIMARY`

- `UNRESOLVED_CONTEXT`

### **نقش این لایه‌ها در Claim Discovery**

`CORE_PRIMARY_KNOWLEDGE`

منبع اصلی Claim Discovery است و می‌تواند Candidate Claim مستقل مربوط به دانش اصلی پروژه تولید کند.

`SUPERVISED_SECONDARY_KNOWLEDGE`

می‌تواند وارد Claim Discovery شود، اما Claim حاصل نباید به‌عنوان بیان مستقیم مؤلف اصلی معرفی شود.

Author و Editorial Attribution واقعی باید حفظ شود.

`CONTEXTUAL_NONPRIMARY`

برای Context، Conflict، Attribution Evaluation و Supporting Evidence حفظ می‌شود، اما به‌تنهایی Candidate Seed مستقل دربارهٔ دانش اصلی ایجاد نمی‌کند.

`UNRESOLVED_CONTEXT`

تا زمانی که Attribution حل نشده است، نباید Candidate Seed مستقل ایجاد کند.

این رکوردها حذف نمی‌شوند و برای بررسی‌های بعدی حفظ خواهند شد.

### **قاعدهٔ مهم**

تعداد رکوردهای واجد ورود به Claim Discovery برابر تعداد Seedها نیست.

حتی اگر هزاران رکورد وارد Claim Extraction شوند، ممکن است تعداد زیادی از آن‌ها:

- یک Claim مشترک داشته باشند؛

- بخشی از یک Claim بزرگ‌تر باشند؛

- Evidenceهای متعدد برای یک Claim باشند؛

- یا در Dedup مرحلهٔ بعد با یکدیگر ادغام معنایی شوند.

بنابراین هنوز هیچ Seed Count تعیین نمی‌شود.

### **Authority Transcript**

Transcriptها فقط با Speaker سطح بالا طبقه‌بندی نمی‌شوند.

Notebook 10 قبلاً Attribution آن‌ها را با Evidence پذیرفته‌شدهٔ Notebook 09 تطبیق داده و شش وضعیت Transcript را مشخص کرده است:

- `PRIMARY_CANONICAL_ATTRIBUTION`

- `PRIMARY_EXPLICIT_SOURCE_MARKER`

- `PRIMARY_HUMAN_CONFIRMED_OVERLAY`

- `SECONDARY_HUMAN_CONFIRMED_OVERLAY`

- `NONPRIMARY_EXPLICIT_CONTEXT`

- `UNRESOLVED_SPEAKER_CONTEXT`

در این مرحله همان منطق Evidence-based بازسازی می‌شود و هیچ Speaker جدیدی حدس زده نمی‌شود.

### **Artifact این مرحله**

یک Index جدید ساخته می‌شود که متن کامل Corpus را Duplicate نمی‌کند.

هر ردیف فقط Metadata لازم برای مرحلهٔ Claim Discovery را نگه می‌دارد، از جمله:

- `document_id`

- `source_id`

- `source_type`

- `text_sha256`

- Authority Class و Authority Basis

- Transcript Authority Evidence در صورت وجود

- نقش رکورد در Claim Discovery

متن اصلی همچنان فقط از Accepted Corpus خوانده خواهد شد.

### **بررسی Dedup Relationها**

در همین مرحله Schema واقعی سه Artifact موجود نیز فقط خوانده می‌شود:

- Exact Duplicate Relations؛

- Near-Duplicate Candidates؛

- Cross-Source Similarity Candidates.

هیچ Relation جدیدی ساخته نمی‌شود.

هدف فقط آماده‌کردن مرحلهٔ بعد است تا بتوان Extraction Unitها را با آگاهی از روابط تکرار طراحی کرد.

### **مرز تصمیم**

در این مرحله:

- هیچ Claim تولید نمی‌شود؛

- هیچ Seed ساخته نمی‌شود؛

- Semantic Deduplication انجام نمی‌شود؛

- هیچ LLM یا API فراخوانی نمی‌شود؛

- Pilot-28 و Core-46 تغییر نمی‌کنند؛

- Golden Dataset v1 تغییر نمی‌کند؛

- Golden Dataset v2 ساخته نمی‌شود؛

- Source Documentها تغییر نمی‌کنند؛

- هیچ Artifact موجودی overwrite نمی‌شود.

### **سیاست عدم Overwrite**

دو Artifact جدید و Versioned ساخته می‌شوند:

`claim_discovery_population_index_v1.jsonl`

و:

`claim_discovery_population_prep_v1.json`

اگر فایل هم‌نام از قبل وجود داشته باشد:

- اگر محتوای آن دقیقاً با اجرای فعلی یکسان باشد، همان فایل reuse می‌شود؛

- اگر متفاوت باشد، اجرا متوقف می‌شود؛

- هیچ فایل موجودی overwrite نمی‌شود.

### **Validation**

این مرحله زمانی PASS است که:

- هر ۱۹٬۴۵۶ Document دقیقاً یک Authority Class نهایی داشته باشد؛

- هیچ Transcript Authority طبقه‌بندی‌نشده باقی نماند؛

- Distribution نهایی Authority دقیقاً با Snapshot فریز‌شدهٔ Taxonomy v1 برابر باشد؛

- تمام ۱۴٬۷۰۵ Core Primary رکورد شناسایی شوند؛

- تمام ۴۵۳ Secondary رکورد شناسایی شوند؛

- تمام ۱٬۹۴۴ Contextual رکورد شناسایی شوند؛

- تمام ۲٬۳۵۴ Unresolved رکورد شناسایی شوند؛

- تعداد ردیف‌های Population Index دقیقاً ۱۹٬۴۵۶ باشد؛

- `document_id` در Index یکتا باشد؛

- Schema واقعی Dedup Relationهای قبلی بدون تغییر خوانده شود؛

- و هیچ Dataset موجودی تغییر نکرده باشد.

پس از PASS این مرحله، مرحلهٔ بعد می‌تواند Extraction Unitهای Dedup-aware را طراحی کند.

</div>

In [39]:
# Code Cell 37
# NB12h — Authority-aware Claim Discovery population preparation
#
# No claim extraction.
# No model/API calls.
# No mutation of accepted corpus.
# No overwrite of existing artifacts.

assert "project_root" in globals(), (
    "Run the NB12h project-discovery Cell first."
)

assert "role_matrix_output_directory" in globals(), (
    "Run Code Cell 31 first."
)

assert "frozen_corpus_reconciliation_report" in globals(), (
    "Run Code Cell 36 first."
)

assert "accepted_corpus_path" in globals(), (
    "Run Code Cell 36 first."
)

assert "taxonomy_payload" in globals(), (
    "Run Code Cell 36 first."
)

assert "dedup_acceptance" in globals(), (
    "Run Code Cell 36 first."
)


CLAIM_DISCOVERY_PREP_VERSION = (
    "nb12h_claim_discovery_population_prep_v1"
)

CLAIM_DISCOVERY_INDEX_FILENAME = (
    "claim_discovery_population_index_v1.jsonl"
)

CLAIM_DISCOVERY_REPORT_FILENAME = (
    "claim_discovery_population_prep_v1.json"
)

PRIMARY_CANONICAL_NAME = (
    "Dr. Komeil Roudi"
)

PRIMARY_EXPLICIT_SOURCE_LABEL = (
    "دکتر کمیل رودی"
)


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def nb12h_nonempty(value):
    if value is None:
        return False

    if isinstance(value, str):
        return bool(
            value.strip()
        )

    return True


def nb12h_sha256_bytes(payload):
    return hashlib.sha256(
        payload
    ).hexdigest()


def nb12h_sha256_file(path):
    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as file:
        while True:
            block = file.read(
                1024 * 1024
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def nb12h_transcript_speaker_metadata(
    record,
):
    value = record.get(
        "source_metadata",
        {},
    ).get(
        "speaker"
    )

    if isinstance(
        value,
        dict,
    ):
        return value

    return {}


# ---------------------------------------------------------
# Reproduce frozen NB10 Transcript authority logic
# using only already-materialized evidence.
# ---------------------------------------------------------

def nb12h_classify_transcript_authority(
    record,
):
    attribution = record.get(
        "attribution",
        {},
    )

    canonical_speaker = (
        attribution.get(
            "speaker"
        )
    )

    speaker_metadata = (
        nb12h_transcript_speaker_metadata(
            record
        )
    )

    base = (
        speaker_metadata.get(
            "base"
        )
        or {}
    )

    effective = (
        speaker_metadata.get(
            "effective"
        )
        or {}
    )

    overlay = (
        speaker_metadata.get(
            "human_attribution_overlay"
        )
    )

    base_status = (
        base.get(
            "status"
        )
    )

    base_raw = (
        base.get(
            "raw"
        )
    )

    effective_label = (
        effective.get(
            "label"
        )
    )

    # Human-confirmed overlay has highest priority.
    if overlay is not None:

        if (
            canonical_speaker
            == PRIMARY_CANONICAL_NAME
        ):
            return (
                "PRIMARY_HUMAN_CONFIRMED_OVERLAY"
            )

        return (
            "SECONDARY_HUMAN_CONFIRMED_OVERLAY"
        )

    # Existing canonical primary speaker.
    if (
        canonical_speaker
        == PRIMARY_CANONICAL_NAME
    ):
        return (
            "PRIMARY_CANONICAL_ATTRIBUTION"
        )

    # Explicit canonical non-primary speaker.
    if nb12h_nonempty(
        canonical_speaker
    ):
        return (
            "NONPRIMARY_EXPLICIT_CONTEXT"
        )

    # Explicit source marker for Dr. Roudi.
    if (
        base_status
        == "EXPLICIT_SOURCE_MARKER"
        and (
            base_raw
            == PRIMARY_EXPLICIT_SOURCE_LABEL
            or effective_label
            == PRIMARY_EXPLICIT_SOURCE_LABEL
        )
    ):
        return (
            "PRIMARY_EXPLICIT_SOURCE_MARKER"
        )

    # Explicit non-primary label or narrative cue.
    if (
        base_status
        in {
            "EXPLICIT_SOURCE_MARKER",
            "NARRATIVE_CUE",
        }
    ):
        return (
            "NONPRIMARY_EXPLICIT_CONTEXT"
        )

    # Parser-preserved unresolved state.
    if (
        base_status
        == "UNRESOLVED"
    ):
        return (
            "UNRESOLVED_SPEAKER_CONTEXT"
        )

    return (
        "UNCLASSIFIED_TRANSCRIPT_AUTHORITY"
    )


PRIMARY_TRANSCRIPT_CLASSES = {
    "PRIMARY_CANONICAL_ATTRIBUTION",
    "PRIMARY_EXPLICIT_SOURCE_MARKER",
    "PRIMARY_HUMAN_CONFIRMED_OVERLAY",
}

CONTEXTUAL_TRANSCRIPT_CLASSES = {
    "SECONDARY_HUMAN_CONFIRMED_OVERLAY",
    "NONPRIMARY_EXPLICIT_CONTEXT",
}

UNRESOLVED_TRANSCRIPT_CLASSES = {
    "UNRESOLVED_SPEAKER_CONTEXT",
}


# ---------------------------------------------------------
# Frozen knowledge-authority reconstruction
# ---------------------------------------------------------

def nb12h_classify_knowledge_authority(
    record,
):
    source_type = record[
        "source_type"
    ]

    attribution = record.get(
        "attribution",
        {},
    )

    # -----------------------------------------------------
    # Book
    # -----------------------------------------------------

    if source_type == "book":

        return {
            "authority_tier": 1,
            "authority_class": (
                "CORE_PRIMARY_KNOWLEDGE"
            ),
            "authority_basis": (
                "PRIMARY_BOOK_AUTHORSHIP"
            ),
            "source_authority_detail": None,
            "core_primary_claim_allowed": True,
            "supporting_evidence_allowed": True,
        }

    # -----------------------------------------------------
    # Website
    # -----------------------------------------------------

    if source_type == "website":

        content_authority_type = (
            attribution.get(
                "content_authority_type"
            )
        )

        if (
            content_authority_type
            == "direct_author"
        ):

            return {
                "authority_tier": 1,
                "authority_class": (
                    "CORE_PRIMARY_KNOWLEDGE"
                ),
                "authority_basis": (
                    "WEBSITE_DIRECT_AUTHOR"
                ),
                "source_authority_detail": (
                    content_authority_type
                ),
                "core_primary_claim_allowed": True,
                "supporting_evidence_allowed": True,
            }

        if (
            content_authority_type
            == "editorial_supervision"
        ):

            return {
                "authority_tier": 2,
                "authority_class": (
                    "SUPERVISED_SECONDARY_KNOWLEDGE"
                ),
                "authority_basis": (
                    "WEBSITE_EDITORIAL_SUPERVISION"
                ),
                "source_authority_detail": (
                    content_authority_type
                ),
                "core_primary_claim_allowed": False,
                "supporting_evidence_allowed": True,
            }

        return {
            "authority_tier": 4,
            "authority_class": (
                "UNRESOLVED_CONTEXT"
            ),
            "authority_basis": (
                "WEBSITE_AUTHORITY_NOT_RESOLVED"
            ),
            "source_authority_detail": (
                content_authority_type
            ),
            "core_primary_claim_allowed": False,
            "supporting_evidence_allowed": False,
        }

    # -----------------------------------------------------
    # Transcript
    # -----------------------------------------------------

    if source_type == "transcript":

        transcript_authority = (
            nb12h_classify_transcript_authority(
                record
            )
        )

        if (
            transcript_authority
            in PRIMARY_TRANSCRIPT_CLASSES
        ):

            return {
                "authority_tier": 1,
                "authority_class": (
                    "CORE_PRIMARY_KNOWLEDGE"
                ),
                "authority_basis": (
                    "TRANSCRIPT_PRIMARY_EVIDENCE"
                ),
                "source_authority_detail": (
                    transcript_authority
                ),
                "core_primary_claim_allowed": True,
                "supporting_evidence_allowed": True,
            }

        if (
            transcript_authority
            in CONTEXTUAL_TRANSCRIPT_CLASSES
        ):

            return {
                "authority_tier": 3,
                "authority_class": (
                    "CONTEXTUAL_NONPRIMARY"
                ),
                "authority_basis": (
                    "TRANSCRIPT_NONPRIMARY_CONTEXT"
                ),
                "source_authority_detail": (
                    transcript_authority
                ),
                "core_primary_claim_allowed": False,
                "supporting_evidence_allowed": False,
            }

        if (
            transcript_authority
            in UNRESOLVED_TRANSCRIPT_CLASSES
        ):

            return {
                "authority_tier": 4,
                "authority_class": (
                    "UNRESOLVED_CONTEXT"
                ),
                "authority_basis": (
                    "TRANSCRIPT_UNRESOLVED_CONTEXT"
                ),
                "source_authority_detail": (
                    transcript_authority
                ),
                "core_primary_claim_allowed": False,
                "supporting_evidence_allowed": False,
            }

        return {
            "authority_tier": None,
            "authority_class": (
                "UNCLASSIFIED_AUTHORITY"
            ),
            "authority_basis": (
                "TRANSCRIPT_AUTHORITY_OUTSIDE_"
                "FROZEN_EVIDENCE_MODEL"
            ),
            "source_authority_detail": (
                transcript_authority
            ),
            "core_primary_claim_allowed": False,
            "supporting_evidence_allowed": False,
        }

    raise ValueError(
        "Unexpected source_type: "
        f"{source_type}"
    )


# ---------------------------------------------------------
# Claim Discovery role
# ---------------------------------------------------------

def nb12h_claim_discovery_role(
    authority_class,
):
    if (
        authority_class
        == "CORE_PRIMARY_KNOWLEDGE"
    ):
        return (
            "PRIMARY_CLAIM_DISCOVERY"
        )

    if (
        authority_class
        == "SUPERVISED_SECONDARY_KNOWLEDGE"
    ):
        return (
            "SECONDARY_CLAIM_DISCOVERY_"
            "WITH_ATTRIBUTION"
        )

    if (
        authority_class
        == "CONTEXTUAL_NONPRIMARY"
    ):
        return (
            "CONTEXT_ONLY_NO_INDEPENDENT_SEED"
        )

    if (
        authority_class
        == "UNRESOLVED_CONTEXT"
    ):
        return (
            "HOLD_UNRESOLVED_NO_INDEPENDENT_SEED"
        )

    return (
        "INVALID_AUTHORITY_REVIEW_REQUIRED"
    )


# ---------------------------------------------------------
# Read Accepted Corpus and build metadata-only index
# ---------------------------------------------------------

claim_discovery_index_rows = []

authority_class_counts = Counter()

authority_tier_counts = Counter()

authority_by_source_counts = defaultdict(
    Counter
)

transcript_authority_detail_counts = Counter()

claim_discovery_role_counts = Counter()

claim_discovery_role_by_source = defaultdict(
    Counter
)

claim_discovery_document_ids = set()

claim_discovery_duplicate_document_ids = []

source_id_counts = defaultdict(
    set
)

text_sha256_missing_count = 0

unclassified_authority_count = 0


with accepted_corpus_path.open(
    "r",
    encoding="utf-8",
    errors="strict",
) as file:

    for line_number, line in enumerate(
        file,
        1,
    ):
        stripped = line.strip()

        if not stripped:
            continue

        record = json.loads(
            stripped
        )

        document_id = (
            record.get(
                "document_id"
            )
        )

        source_id = (
            record.get(
                "source_id"
            )
        )

        source_type = (
            record.get(
                "source_type"
            )
        )

        text_sha256 = (
            record.get(
                "text_sha256"
            )
        )

        raw_content_hash = (
            record.get(
                "raw_content_hash"
            )
        )

        assert nb12h_nonempty(
            document_id
        ), (
            "Missing document_id at line "
            f"{line_number}."
        )

        assert nb12h_nonempty(
            source_id
        ), (
            "Missing source_id at line "
            f"{line_number}."
        )

        assert source_type in {
            "website",
            "book",
            "transcript",
        }, (
            "Unexpected source_type at line "
            f"{line_number}: {source_type}"
        )

        if (
            document_id
            in claim_discovery_document_ids
        ):
            claim_discovery_duplicate_document_ids.append(
                document_id
            )

        claim_discovery_document_ids.add(
            document_id
        )

        source_id_counts[
            source_type
        ].add(
            source_id
        )

        if not nb12h_nonempty(
            text_sha256
        ):
            text_sha256_missing_count += 1

        authority_profile = (
            nb12h_classify_knowledge_authority(
                record
            )
        )

        authority_class = (
            authority_profile[
                "authority_class"
            ]
        )

        authority_tier = (
            authority_profile[
                "authority_tier"
            ]
        )

        authority_detail = (
            authority_profile[
                "source_authority_detail"
            ]
        )

        claim_role = (
            nb12h_claim_discovery_role(
                authority_class
            )
        )

        authority_class_counts[
            authority_class
        ] += 1

        authority_tier_counts[
            str(authority_tier)
        ] += 1

        authority_by_source_counts[
            source_type
        ][
            authority_class
        ] += 1

        claim_discovery_role_counts[
            claim_role
        ] += 1

        claim_discovery_role_by_source[
            source_type
        ][
            claim_role
        ] += 1

        if (
            source_type
            == "transcript"
        ):
            transcript_authority_detail_counts[
                str(
                    authority_detail
                )
            ] += 1

        if (
            authority_class
            == "UNCLASSIFIED_AUTHORITY"
        ):
            unclassified_authority_count += 1

        attribution = record.get(
            "attribution",
            {},
        )

        source_metadata = record.get(
            "source_metadata",
            {},
        )

        index_row = {
            "document_id": (
                document_id
            ),
            "source_id": (
                source_id
            ),
            "source_type": (
                source_type
            ),
            "text_sha256": (
                text_sha256
            ),
            "raw_content_hash": (
                raw_content_hash
            ),
            "content_origin": (
                record.get(
                    "content_origin"
                )
            ),
            "title": (
                record.get(
                    "title"
                )
            ),
            "authority_tier": (
                authority_tier
            ),
            "authority_class": (
                authority_class
            ),
            "authority_basis": (
                authority_profile[
                    "authority_basis"
                ]
            ),
            "source_authority_detail": (
                authority_detail
            ),
            "core_primary_claim_allowed": (
                authority_profile[
                    "core_primary_claim_allowed"
                ]
            ),
            "supporting_evidence_allowed": (
                authority_profile[
                    "supporting_evidence_allowed"
                ]
            ),
            "claim_discovery_role": (
                claim_role
            ),
            "credited_author_name": (
                attribution.get(
                    "credited_author_name"
                )
            ),
            "editorial_supervisor_name": (
                attribution.get(
                    "editorial_supervisor_name"
                )
            ),
            "canonical_speaker": (
                attribution.get(
                    "speaker"
                )
            ),
            "program": (
                source_metadata.get(
                    "program"
                )
            ),
        }

        claim_discovery_index_rows.append(
            index_row
        )


# ---------------------------------------------------------
# Validate Authority against frozen Taxonomy v1
# ---------------------------------------------------------

assert (
    len(
        claim_discovery_index_rows
    )
    == 19456
), (
    "Claim Discovery index does not contain "
    "the full accepted corpus."
)

assert not (
    claim_discovery_duplicate_document_ids
), (
    "Duplicate document_id detected in "
    "Claim Discovery index."
)

assert (
    text_sha256_missing_count
    == 0
), (
    "Some Claim Discovery records have "
    "missing text_sha256."
)

assert (
    unclassified_authority_count
    == 0
), (
    "Some records could not be reconciled "
    "to the frozen Authority Model."
)


frozen_authority_summary = (
    taxonomy_payload[
        "knowledge_authority"
    ][
        "summary"
    ]
)

expected_authority_class_counts = {
    "CORE_PRIMARY_KNOWLEDGE": (
        frozen_authority_summary[
            "core_primary_knowledge"
        ]
    ),
    "SUPERVISED_SECONDARY_KNOWLEDGE": (
        frozen_authority_summary[
            "supervised_secondary_knowledge"
        ]
    ),
    "CONTEXTUAL_NONPRIMARY": (
        frozen_authority_summary[
            "contextual_nonprimary"
        ]
    ),
    "UNRESOLVED_CONTEXT": (
        frozen_authority_summary[
            "unresolved_context"
        ]
    ),
}

assert (
    dict(
        authority_class_counts
    )
    == expected_authority_class_counts
), (
    "Reconstructed Authority distribution "
    "does not match frozen Taxonomy v1."
)


expected_transcript_authority_counts = (
    frozen_authority_summary[
        "transcript_authority_classes"
    ]
)

assert (
    dict(
        transcript_authority_detail_counts
    )
    == expected_transcript_authority_counts
), (
    "Transcript Authority evidence classes "
    "do not match frozen Taxonomy v1."
)


# ---------------------------------------------------------
# Eligible population counts
# ---------------------------------------------------------

primary_claim_discovery_count = (
    claim_discovery_role_counts[
        "PRIMARY_CLAIM_DISCOVERY"
    ]
)

secondary_claim_discovery_count = (
    claim_discovery_role_counts[
        "SECONDARY_CLAIM_DISCOVERY_"
        "WITH_ATTRIBUTION"
    ]
)

claim_discovery_eligible_count = (
    primary_claim_discovery_count
    + secondary_claim_discovery_count
)

context_only_count = (
    claim_discovery_role_counts[
        "CONTEXT_ONLY_NO_INDEPENDENT_SEED"
    ]
)

unresolved_hold_count = (
    claim_discovery_role_counts[
        "HOLD_UNRESOLVED_NO_INDEPENDENT_SEED"
    ]
)

assert (
    claim_discovery_eligible_count
    + context_only_count
    + unresolved_hold_count
    == 19456
), (
    "Claim Discovery role partition "
    "does not cover the full corpus."
)


# ---------------------------------------------------------
# Dedup relation schema inspection
#
# No relation is modified or regenerated.
# ---------------------------------------------------------

def nb12h_probe_jsonl_schema(
    path,
    sample_limit=25,
):
    observed_keys = set()

    key_types = defaultdict(
        set
    )

    sampled = 0

    with path.open(
        "r",
        encoding="utf-8",
        errors="strict",
    ) as file:

        for line in file:
            stripped = line.strip()

            if not stripped:
                continue

            payload = json.loads(
                stripped
            )

            assert isinstance(
                payload,
                dict,
            ), (
                "Dedup relation contains "
                "non-object JSON."
            )

            for key, value in (
                payload.items()
            ):
                observed_keys.add(
                    str(key)
                )

                key_types[
                    str(key)
                ].add(
                    type(
                        value
                    ).__name__
                )

            sampled += 1

            if (
                sampled
                >= sample_limit
            ):
                break

    return {
        "sampled_record_count": (
            sampled
        ),
        "observed_top_level_keys": (
            sorted(
                observed_keys
            )
        ),
        "observed_value_types": {
            key: sorted(
                value_types
            )
            for (
                key,
                value_types,
            ) in sorted(
                key_types.items()
            )
        },
    }


dedup_schema_probe = {}

for artifact_key in [
    "exact_duplicate_relations",
    "near_duplicate_candidates",
    "cross_source_similarity_candidates",
]:

    artifact_spec = (
        dedup_acceptance[
            "artifacts"
        ][artifact_key]
    )

    artifact_path = (
        project_root
        / artifact_spec[
            "path"
        ]
    )

    dedup_schema_probe[
        artifact_key
    ] = (
        nb12h_probe_jsonl_schema(
            artifact_path
        )
    )


# ---------------------------------------------------------
# Deterministic metadata-only Population Index
# ---------------------------------------------------------

claim_discovery_index_rows = sorted(
    claim_discovery_index_rows,
    key=lambda row: (
        row[
            "document_id"
        ]
    ),
)

claim_discovery_index_text = (
    "".join(
        json.dumps(
            row,
            ensure_ascii=False,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        )
        + "\n"
        for row in (
            claim_discovery_index_rows
        )
    )
)

claim_discovery_index_bytes = (
    claim_discovery_index_text.encode(
        "utf-8"
    )
)

claim_discovery_index_sha256 = (
    nb12h_sha256_bytes(
        claim_discovery_index_bytes
    )
)

claim_discovery_index_path = (
    role_matrix_output_directory
    / CLAIM_DISCOVERY_INDEX_FILENAME
)

if claim_discovery_index_path.exists():

    existing_index_sha256 = (
        nb12h_sha256_file(
            claim_discovery_index_path
        )
    )

    assert (
        existing_index_sha256
        == claim_discovery_index_sha256
    ), (
        "Existing Claim Discovery index "
        "differs from the current population. "
        "Refusing to overwrite it."
    )

    claim_discovery_index_write_status = (
        "REUSED_EXISTING_IDENTICAL_INDEX"
    )

else:

    with claim_discovery_index_path.open(
        "x",
        encoding="utf-8",
    ) as file:

        file.write(
            claim_discovery_index_text
        )

    claim_discovery_index_write_status = (
        "CREATED_NEW_INDEX"
    )


# ---------------------------------------------------------
# Build preparation report
# ---------------------------------------------------------

claim_discovery_population_report = {
    "prep_version": (
        CLAIM_DISCOVERY_PREP_VERSION
    ),

    "status": "PASS",

    "accepted_corpus": {
        "record_count": 19456,
        "sha256": (
            frozen_corpus_reconciliation_report[
                "accepted_corpus"
            ][
                "sha256"
            ]
        ),
    },

    "authority_class_counts": dict(
        authority_class_counts
    ),

    "authority_tier_counts": dict(
        authority_tier_counts
    ),

    "authority_by_source": {
        source_type: dict(
            counts
        )
        for (
            source_type,
            counts,
        ) in sorted(
            authority_by_source_counts.items()
        )
    },

    "transcript_authority_detail_counts": dict(
        transcript_authority_detail_counts
    ),

    "claim_discovery_role_counts": dict(
        claim_discovery_role_counts
    ),

    "claim_discovery_role_by_source": {
        source_type: dict(
            counts
        )
        for (
            source_type,
            counts,
        ) in sorted(
            claim_discovery_role_by_source.items()
        )
    },

    "claim_discovery_eligible_record_count": (
        claim_discovery_eligible_count
    ),

    "primary_claim_discovery_record_count": (
        primary_claim_discovery_count
    ),

    "secondary_claim_discovery_record_count": (
        secondary_claim_discovery_count
    ),

    "context_only_record_count": (
        context_only_count
    ),

    "unresolved_hold_record_count": (
        unresolved_hold_count
    ),

    "unique_source_counts": {
        source_type: len(
            source_ids
        )
        for (
            source_type,
            source_ids,
        ) in sorted(
            source_id_counts.items()
        )
    },

    "population_index": {
        "path": str(
            claim_discovery_index_path.relative_to(
                project_root
            )
        ),
        "record_count": len(
            claim_discovery_index_rows
        ),
        "sha256": (
            claim_discovery_index_sha256
        ),
        "contains_full_text": False,
    },

    "dedup_relation_schema_probe": (
        dedup_schema_probe
    ),

    "seed_count_determined": False,
    "claim_extraction_started": False,

    "api_calls_made": 0,
    "cost_incurred": 0,

    "source_files_modified": False,
    "active_dataset_modified": False,
    "golden_dataset_v1_modified": False,
    "core46_modified": False,
    "golden_dataset_v2_created": False,
}


# ---------------------------------------------------------
# No-overwrite report persistence
# ---------------------------------------------------------

claim_discovery_population_report_path = (
    role_matrix_output_directory
    / CLAIM_DISCOVERY_REPORT_FILENAME
)

if (
    claim_discovery_population_report_path
    .exists()
):

    with (
        claim_discovery_population_report_path
        .open(
            "r",
            encoding="utf-8",
        )
    ) as file:

        existing_report = json.load(
            file
        )

    comparable_existing = dict(
        existing_report
    )

    comparable_current = dict(
        claim_discovery_population_report
    )

    comparable_existing.pop(
        "created_at_utc",
        None,
    )

    comparable_current.pop(
        "created_at_utc",
        None,
    )

    assert (
        comparable_existing
        == comparable_current
    ), (
        "Existing Claim Discovery preparation "
        "report differs from this run. "
        "Refusing to overwrite it."
    )

    claim_discovery_population_report = (
        existing_report
    )

    claim_discovery_report_write_status = (
        "REUSED_EXISTING_IDENTICAL_REPORT"
    )

else:

    claim_discovery_population_report[
        "created_at_utc"
    ] = datetime.now(
        timezone.utc
    ).isoformat()

    with (
        claim_discovery_population_report_path
        .open(
            "x",
            encoding="utf-8",
        )
    ) as file:

        json.dump(
            claim_discovery_population_report,
            file,
            ensure_ascii=False,
            indent=2,
        )

        file.write("\n")

    claim_discovery_report_write_status = (
        "CREATED_NEW_REPORT"
    )


# ---------------------------------------------------------
# Compact output
# ---------------------------------------------------------

print(
    "=== NB12h CLAIM DISCOVERY "
    "POPULATION PREPARATION ==="
)

print()
print(
    "Full accepted population:",
    len(
        claim_discovery_index_rows
    ),
)

print()
print(
    "Authority classes:",
    dict(
        authority_class_counts
    ),
)

print()
print(
    "Transcript authority evidence:"
)

for (
    authority_class,
    count,
) in (
    transcript_authority_detail_counts
    .items()
):
    print(
        "  -",
        authority_class,
        ":",
        count,
    )

print()
print(
    "Claim Discovery roles:",
    dict(
        claim_discovery_role_counts
    ),
)

print()
print(
    "Primary Claim Discovery records:",
    primary_claim_discovery_count,
)

print(
    "Secondary Claim Discovery records:",
    secondary_claim_discovery_count,
)

print(
    "Total Claim Discovery eligible records:",
    claim_discovery_eligible_count,
)

print(
    "Context-only records:",
    context_only_count,
)

print(
    "Unresolved hold records:",
    unresolved_hold_count,
)

print()
print(
    "Unique sources:",
    {
        source_type: len(
            source_ids
        )
        for (
            source_type,
            source_ids,
        ) in sorted(
            source_id_counts.items()
        )
    },
)

print()
print(
    "Dedup relation schemas:"
)

for (
    artifact_key,
    schema_probe,
) in (
    dedup_schema_probe.items()
):
    print(
        "  -",
        artifact_key,
    )

    print(
        "    keys:",
        schema_probe[
            "observed_top_level_keys"
        ],
    )

print()
print(
    "Population index:",
    claim_discovery_index_path,
)

print(
    "Population index SHA-256:",
    claim_discovery_index_sha256,
)

print(
    "Index write status:",
    claim_discovery_index_write_status,
)

print()
print(
    "Preparation report:",
    claim_discovery_population_report_path,
)

print(
    "Report write status:",
    claim_discovery_report_write_status,
)

print()
print(
    "Claim extraction started:",
    False,
)

print(
    "Natural seed count determined:",
    False,
)

print()
print("API calls made:", 0)
print("Cost incurred:", 0)
print("Source files modified:", False)
print("Active dataset modified:", False)
print("Golden Dataset v1 modified:", False)
print("Core-46 modified:", False)
print("Golden Dataset v2 created:", False)

print()
print(
    "NB12h CLAIM DISCOVERY "
    "POPULATION PREPARATION: PASS"
)

=== NB12h CLAIM DISCOVERY POPULATION PREPARATION ===

Full accepted population: 19456

Authority classes: {'SUPERVISED_SECONDARY_KNOWLEDGE': 453, 'CORE_PRIMARY_KNOWLEDGE': 14705, 'CONTEXTUAL_NONPRIMARY': 1944, 'UNRESOLVED_CONTEXT': 2354}

Transcript authority evidence:
  - NONPRIMARY_EXPLICIT_CONTEXT : 1943
  - PRIMARY_EXPLICIT_SOURCE_MARKER : 718
  - UNRESOLVED_SPEAKER_CONTEXT : 2354
  - PRIMARY_CANONICAL_ATTRIBUTION : 11209
  - PRIMARY_HUMAN_CONFIRMED_OVERLAY : 1
  - SECONDARY_HUMAN_CONFIRMED_OVERLAY : 1

Claim Discovery roles: {'SECONDARY_CLAIM_DISCOVERY_WITH_ATTRIBUTION': 453, 'PRIMARY_CLAIM_DISCOVERY': 14705, 'CONTEXT_ONLY_NO_INDEPENDENT_SEED': 1944, 'HOLD_UNRESOLVED_NO_INDEPENDENT_SEED': 2354}

Primary Claim Discovery records: 14705
Secondary Claim Discovery records: 453
Total Claim Discovery eligible records: 15158
Context-only records: 1944
Unresolved hold records: 2354

Unique sources: {'book': 3, 'transcript': 183, 'website': 529}

Dedup relation schemas:
  - exact_duplicate_

<div dir="rtl" align="right">

## **مرحلهٔ سی‌وهشتم — ممیزی روابط Dedup برای طراحی Extraction Unitهای امن**

مرحلهٔ قبل Population رسمی Claim Discovery را مشخص کرد.

از ۱۹٬۴۵۶ رکورد پذیرفته‌شده:

- ۱۴٬۷۰۵ رکورد در گروه `CORE_PRIMARY_KNOWLEDGE` قرار دارند؛

- ۴۵۳ رکورد در گروه `SUPERVISED_SECONDARY_KNOWLEDGE` قرار دارند؛

- در مجموع ۱۵٬۱۵۸ رکورد اجازهٔ ورود به Claim Discovery دارند؛

- ۱٬۹۴۴ رکورد فقط Contextual هستند؛

- و ۲٬۳۵۴ رکورد تا زمان رفع Attribution در وضعیت Unresolved باقی می‌مانند.

اما ورود ۱۵٬۱۵۸ رکورد به Claim Discovery به معنی اجرای Claim Extraction مستقل روی تک‌تک آن‌ها نیست.

Notebook 09 از قبل سه نوع رابطهٔ Dedup و Similarity ساخته است که باید قبل از طراحی Extraction Unitها به‌طور کامل فهمیده شوند.

### **هدف این مرحله**

هدف این مرحله بررسی کامل سه Artifact زیر است:

- Exact Duplicate Relations؛

- Near-Duplicate Candidates؛

- Cross-Source Similarity Candidates.

در این مرحله مشخص می‌شود:

- چند Relation واقعاً به Population واجد Claim Discovery متصل است؛

- Exact Groupها چند Document واجد شرایط را پوشش می‌دهند؛

- آیا یک Document در بیش از یک Exact Group ظاهر شده است یا خیر؛

- Exact Groupها از چه Source Type و Authority Classهایی تشکیل شده‌اند؛

- `canonical_status`های ثبت‌شده چه هستند؛

- Near-Duplicateها چه `production_duplicate_decision`هایی دارند؛

- Cross-Source Candidateها چه `interpretation`هایی دارند؛

- Automatic Merge و Automatic Delete واقعاً غیرفعال مانده‌اند یا خیر؛

- و چه مقدار کاهش بالقوه فقط از Exact Duplicateهای قطعی ممکن است به دست آید.

### **قاعدهٔ مهم دربارهٔ Exact Duplicate**

Exact Duplicate بودن متن به معنی حذف Source نیست.

اگر چند Document دارای متن دقیقاً یکسان باشند، Claim Extraction می‌تواند در آینده از یک Evidence Text مشترک استفاده کند، اما تمام Source IDها، Authorityها، Locatorها و Provenanceهای عضو گروه باید حفظ شوند.

بنابراین:

`Exact Text Collapse ≠ Source Deletion`

### **قاعدهٔ مهم دربارهٔ Near Duplicate**

Near-Duplicate Candidate به معنی Duplicate قطعی نیست.

هیچ Pair فقط بر اساس Similarity Score ادغام نمی‌شود.

تا زمانی که Decision قبلی یا داوری Claim-Level وجود نداشته باشد، Near-Duplicate فقط یک Relation برای جلوگیری از استخراج تکراری احتمالی محسوب می‌شود.

### **قاعدهٔ مهم دربارهٔ Cross-Source Similarity**

شباهت بین دو Source متفاوت می‌تواند چند معنی داشته باشد:

- بیان یک Claim مشترک در دو Source؛

- Evidence مستقل برای یک Claim؛

- بازگویی یا اقتباس؛

- هم‌پوشانی جزئی؛

- یا صرفاً شباهت واژگانی.

بنابراین Cross-Source Similarity نیز در این مرحله باعث Merge نمی‌شود.

### **چرا این مرحله قبل از Claim Extraction لازم است**

اگر بدون استفاده از روابط قبلی روی ۱۵٬۱۵۸ رکورد Claim Extraction مستقل انجام شود:

- هزینهٔ پردازش به‌طور غیرضروری افزایش پیدا می‌کند؛

- Claimهای تکراری چندین بار تولید می‌شوند؛

- Evidenceهای چندمنبعی ممکن است به اشتباه Seed مستقل تلقی شوند؛

- و Dedup مرحلهٔ بعد بسیار دشوارتر می‌شود.

در مقابل، اگر Near و Cross-Source Relationها زودتر از موعد Merge شوند، ممکن است Claimهای واقعاً مستقل از بین بروند.

بنابراین این مرحله فقط مرز امن استفاده از Dedup را تعیین می‌کند.

### **مرز تصمیم**

در این مرحله:

- هیچ Claim استخراج نمی‌شود؛

- هیچ Seed ساخته نمی‌شود؛

- هیچ Near-Duplicate به‌صورت خودکار Merge نمی‌شود؛

- هیچ Cross-Source Candidate به‌صورت خودکار Merge نمی‌شود؛

- هیچ Source حذف نمی‌شود؛

- هیچ Canonical Document جدیدی تعیین نمی‌شود؛

- هیچ API یا LLM فراخوانی نمی‌شود؛

- Golden Dataset v1 تغییر نمی‌کند؛

- Pilot-28 و Core-46 تغییر نمی‌کنند؛

- Golden Dataset v2 ساخته نمی‌شود؛

- و هیچ Artifact قبلی overwrite نمی‌شود.

### **Artifact این مرحله**

یک گزارش جدید ساخته می‌شود:

`claim_discovery_dedup_topology_audit_v1.json`

این گزارش شامل Summary آماری و Decision State روابط موجود است و Relationهای اصلی را تغییر نمی‌دهد.

### **Validation**

این مرحله زمانی PASS است که:

- تمام Document IDهای استفاده‌شده در Relationها در Accepted Corpus وجود داشته باشند؛

- `member_count` تمام Exact Groupها با تعداد واقعی Memberها تطابق داشته باشد؛

- هیچ Document عضو بیش از یک Exact Group نباشد؛

- تمام Automatic Mergeها غیرفعال باشند؛

- تمام Automatic Deleteها غیرفعال باشند؛

- وضعیت Canonicalization قبلی حفظ شده باشد؛

- Distribution واقعی `canonical_status` مشخص شود؛

- Distribution واقعی `production_duplicate_decision` مشخص شود؛

- Distribution واقعی `interpretation` در Cross-Source Relationها مشخص شود؛

- تعداد Relationهای مرتبط با Population واجد Claim Discovery مشخص شود؛

- کاهش بالقوهٔ ناشی از Exact Duplicateها بدون حذف Source محاسبه شود؛

- و هیچ Dataset موجودی تغییر نکرده باشد.

پس از PASS شدن این مرحله، می‌توان Extraction Unitهای واقعی را با سیاستی Evidence-preserving و Dedup-aware طراحی کرد.

</div>

In [40]:
# Code Cell 38
# NB12h — Dedup relation topology audit for safe
# Claim Discovery extraction-unit design
#
# No claim extraction.
# No automatic merge.
# No automatic delete.
# No source deletion.
# No API calls.
# No overwrite.

assert "project_root" in globals(), (
    "Run the NB12h project-discovery Cell first."
)

assert "role_matrix_output_directory" in globals(), (
    "Run Code Cell 31 first."
)

assert "claim_discovery_index_rows" in globals(), (
    "Run Code Cell 37 first."
)

assert "dedup_acceptance" in globals(), (
    "Run Code Cell 36 first."
)

DEDUP_TOPOLOGY_AUDIT_VERSION = (
    "nb12h_claim_discovery_dedup_topology_audit_v1"
)

DEDUP_TOPOLOGY_REPORT_FILENAME = (
    "claim_discovery_dedup_topology_audit_v1.json"
)


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def nb12h_load_jsonl_objects(path):
    rows = []

    with path.open(
        "r",
        encoding="utf-8",
        errors="strict",
    ) as file:

        for line_number, line in enumerate(
            file,
            1,
        ):
            stripped = line.strip()

            if not stripped:
                continue

            payload = json.loads(
                stripped
            )

            assert isinstance(
                payload,
                dict,
            ), (
                "Non-object JSONL record at "
                f"{path}:{line_number}"
            )

            rows.append(
                payload
            )

    return rows


def nb12h_counter_dict(counter):
    return dict(
        sorted(
            counter.items(),
            key=lambda item: (
                str(
                    item[0]
                )
            ),
        )
    )


def nb12h_pair_key(
    value_a,
    value_b,
):
    return " ↔ ".join(
        sorted([
            str(value_a),
            str(value_b),
        ])
    )


def nb12h_numeric_summary(values):
    clean_values = [
        float(value)
        for value in values
        if isinstance(
            value,
            (int, float),
        )
    ]

    if not clean_values:
        return {
            "count": 0,
            "min": None,
            "p50": None,
            "mean": None,
            "max": None,
        }

    ordered = sorted(
        clean_values
    )

    count = len(
        ordered
    )

    middle = count // 2

    if count % 2:
        median = (
            ordered[
                middle
            ]
        )

    else:
        median = (
            ordered[
                middle - 1
            ]
            + ordered[
                middle
            ]
        ) / 2

    return {
        "count": count,
        "min": ordered[0],
        "p50": median,
        "mean": (
            sum(
                ordered
            )
            / count
        ),
        "max": ordered[-1],
    }


# ---------------------------------------------------------
# Build accepted-corpus lookup from Cell 37 index
# ---------------------------------------------------------

document_lookup = {
    row[
        "document_id"
    ]: row
    for row in (
        claim_discovery_index_rows
    )
}

assert (
    len(
        document_lookup
    )
    == 19456
), (
    "Accepted document lookup does not "
    "contain exactly 19,456 unique documents."
)


claim_discovery_eligible_roles = {
    "PRIMARY_CLAIM_DISCOVERY",
    "SECONDARY_CLAIM_DISCOVERY_WITH_ATTRIBUTION",
}

eligible_document_ids = {
    document_id
    for (
        document_id,
        row,
    ) in document_lookup.items()
    if row[
        "claim_discovery_role"
    ]
    in claim_discovery_eligible_roles
}

assert (
    len(
        eligible_document_ids
    )
    == 15158
), (
    "Eligible Claim Discovery population "
    "does not equal the Cell 37 result."
)


# ---------------------------------------------------------
# Resolve frozen Dedup artifacts
# ---------------------------------------------------------

dedup_relation_paths = {}

for artifact_key in [
    "exact_duplicate_relations",
    "near_duplicate_candidates",
    "cross_source_similarity_candidates",
]:

    artifact_spec = (
        dedup_acceptance[
            "artifacts"
        ][artifact_key]
    )

    artifact_path = (
        project_root
        / artifact_spec[
            "path"
        ]
    )

    assert artifact_path.is_file(), (
        "Required Dedup relation artifact "
        f"is missing: {artifact_path}"
    )

    dedup_relation_paths[
        artifact_key
    ] = artifact_path


exact_rows = nb12h_load_jsonl_objects(
    dedup_relation_paths[
        "exact_duplicate_relations"
    ]
)

near_rows = nb12h_load_jsonl_objects(
    dedup_relation_paths[
        "near_duplicate_candidates"
    ]
)

cross_rows = nb12h_load_jsonl_objects(
    dedup_relation_paths[
        "cross_source_similarity_candidates"
    ]
)


assert (
    len(
        exact_rows
    )
    == 192
), (
    "Unexpected Exact relation-group count."
)

assert (
    len(
        near_rows
    )
    == 15929
), (
    "Unexpected Near candidate count."
)

assert (
    len(
        cross_rows
    )
    == 37796
), (
    "Unexpected Cross-source candidate count."
)


# ---------------------------------------------------------
# Exact Duplicate audit
# ---------------------------------------------------------

exact_relation_type_counts = Counter()

exact_relation_scope_counts = Counter()

exact_canonical_status_counts = Counter()

exact_member_count_distribution = Counter()

exact_source_type_mix_counts = Counter()

exact_authority_mix_counts = Counter()

exact_eligibility_mix_counts = Counter()

exact_member_document_ids_seen = set()

exact_duplicate_member_reuse = []

exact_orphan_document_ids = []

exact_member_count_mismatches = []

exact_groups_touching_eligible = 0

exact_groups_with_multiple_eligible = 0

exact_eligible_member_count = 0

exact_potential_extraction_reduction = 0

exact_cross_source_groups = 0

exact_mixed_authority_groups = 0

exact_group_summaries = []


for row in exact_rows:

    relation_group_id = (
        row.get(
            "relation_group_id"
        )
    )

    member_document_ids = (
        row.get(
            "member_document_ids"
        )
        or []
    )

    assert isinstance(
        member_document_ids,
        list,
    ), (
        "Exact relation member_document_ids "
        "is not a list."
    )

    recorded_member_count = (
        row.get(
            "member_count"
        )
    )

    actual_member_count = len(
        member_document_ids
    )

    if (
        recorded_member_count
        != actual_member_count
    ):
        exact_member_count_mismatches.append({
            "relation_group_id": (
                relation_group_id
            ),
            "recorded_member_count": (
                recorded_member_count
            ),
            "actual_member_count": (
                actual_member_count
            ),
        })

    group_source_types = set()

    group_authority_classes = set()

    group_eligible_members = []

    for document_id in (
        member_document_ids
    ):

        if (
            document_id
            not in document_lookup
        ):
            exact_orphan_document_ids.append(
                document_id
            )
            continue

        if (
            document_id
            in exact_member_document_ids_seen
        ):
            exact_duplicate_member_reuse.append(
                document_id
            )

        exact_member_document_ids_seen.add(
            document_id
        )

        document_row = (
            document_lookup[
                document_id
            ]
        )

        group_source_types.add(
            document_row[
                "source_type"
            ]
        )

        group_authority_classes.add(
            document_row[
                "authority_class"
            ]
        )

        if (
            document_id
            in eligible_document_ids
        ):
            group_eligible_members.append(
                document_id
            )

    eligible_count = len(
        group_eligible_members
    )

    exact_eligible_member_count += (
        eligible_count
    )

    if eligible_count > 0:
        exact_groups_touching_eligible += 1

    if eligible_count >= 2:
        exact_groups_with_multiple_eligible += 1

        exact_potential_extraction_reduction += (
            eligible_count - 1
        )

    if len(
        group_source_types
    ) > 1:
        exact_cross_source_groups += 1

    if len(
        group_authority_classes
    ) > 1:
        exact_mixed_authority_groups += 1

    exact_relation_type_counts[
        str(
            row.get(
                "relation_type"
            )
        )
    ] += 1

    exact_relation_scope_counts[
        str(
            row.get(
                "relation_scope"
            )
        )
    ] += 1

    exact_canonical_status_counts[
        str(
            row.get(
                "canonical_status"
            )
        )
    ] += 1

    exact_member_count_distribution[
        str(
            actual_member_count
        )
    ] += 1

    exact_source_type_mix_counts[
        "+".join(
            sorted(
                group_source_types
            )
        )
        or "NONE"
    ] += 1

    exact_authority_mix_counts[
        "+".join(
            sorted(
                group_authority_classes
            )
        )
        or "NONE"
    ] += 1

    if eligible_count == 0:
        eligibility_label = (
            "NO_ELIGIBLE_MEMBER"
        )

    elif (
        eligible_count
        == actual_member_count
    ):
        eligibility_label = (
            "ALL_MEMBERS_ELIGIBLE"
        )

    else:
        eligibility_label = (
            "MIXED_ELIGIBILITY"
        )

    exact_eligibility_mix_counts[
        eligibility_label
    ] += 1

    exact_group_summaries.append({
        "relation_group_id": (
            relation_group_id
        ),
        "member_count": (
            actual_member_count
        ),
        "eligible_member_count": (
            eligible_count
        ),
        "source_types": sorted(
            group_source_types
        ),
        "authority_classes": sorted(
            group_authority_classes
        ),
        "canonical_status": (
            row.get(
                "canonical_status"
            )
        ),
        "relation_scope": (
            row.get(
                "relation_scope"
            )
        ),
    })


assert not (
    exact_member_count_mismatches
), (
    "Exact Duplicate member_count mismatch "
    "detected."
)

assert not (
    exact_orphan_document_ids
), (
    "Exact Duplicate relation references "
    "Document IDs outside the Accepted Corpus."
)

assert not (
    exact_duplicate_member_reuse
), (
    "At least one Document belongs to more "
    "than one Exact Duplicate group."
)


# ---------------------------------------------------------
# Generic Pair-relation audit helper
# ---------------------------------------------------------

def nb12h_audit_pair_relations(
    rows,
    decision_field=None,
):

    orphan_document_ids = []

    relation_type_counts = Counter()

    decision_counts = Counter()

    eligibility_counts = Counter()

    source_pair_counts = Counter()

    authority_pair_counts = Counter()

    automatic_merge_counts = Counter()

    automatic_delete_counts = Counter()

    canonical_document_presence_counts = (
        Counter()
    )

    similarity_values = {
        "char_tfidf_cosine": [],
        "token_jaccard": [],
        "sequence_matcher": [],
        "length_ratio": [],
    }

    pair_count_both_eligible = 0

    eligible_document_ids_touched = set()

    for row in rows:

        document_id_a = (
            row.get(
                "document_id_a"
            )
        )

        document_id_b = (
            row.get(
                "document_id_b"
            )
        )

        for document_id in [
            document_id_a,
            document_id_b,
        ]:
            if (
                document_id
                not in document_lookup
            ):
                orphan_document_ids.append(
                    document_id
                )

        if (
            document_id_a
            not in document_lookup
            or document_id_b
            not in document_lookup
        ):
            continue

        document_a = (
            document_lookup[
                document_id_a
            ]
        )

        document_b = (
            document_lookup[
                document_id_b
            ]
        )

        eligible_a = (
            document_id_a
            in eligible_document_ids
        )

        eligible_b = (
            document_id_b
            in eligible_document_ids
        )

        if (
            eligible_a
            and eligible_b
        ):
            eligibility_label = (
                "BOTH_ELIGIBLE"
            )

            pair_count_both_eligible += 1

        elif (
            eligible_a
            or eligible_b
        ):
            eligibility_label = (
                "ONE_ELIGIBLE"
            )

        else:
            eligibility_label = (
                "NONE_ELIGIBLE"
            )

        eligibility_counts[
            eligibility_label
        ] += 1

        if eligible_a:
            eligible_document_ids_touched.add(
                document_id_a
            )

        if eligible_b:
            eligible_document_ids_touched.add(
                document_id_b
            )

        source_pair_counts[
            nb12h_pair_key(
                document_a[
                    "source_type"
                ],
                document_b[
                    "source_type"
                ],
            )
        ] += 1

        authority_pair_counts[
            nb12h_pair_key(
                document_a[
                    "authority_class"
                ],
                document_b[
                    "authority_class"
                ],
            )
        ] += 1

        relation_type_counts[
            str(
                row.get(
                    "relation_type"
                )
            )
        ] += 1

        if decision_field is not None:
            decision_counts[
                str(
                    row.get(
                        decision_field
                    )
                )
            ] += 1

        automatic_merge_counts[
            str(
                row.get(
                    "automatic_merge"
                )
            )
        ] += 1

        automatic_delete_counts[
            str(
                row.get(
                    "automatic_delete"
                )
            )
        ] += 1

        canonical_document_id = (
            row.get(
                "canonical_document_id"
            )
        )

        if (
            canonical_document_id
            is None
            or canonical_document_id == ""
        ):
            canonical_document_presence_counts[
                "ABSENT"
            ] += 1

        else:
            canonical_document_presence_counts[
                "PRESENT"
            ] += 1

            assert (
                canonical_document_id
                in document_lookup
            ), (
                "Canonical document reference "
                "is outside Accepted Corpus."
            )

        for field_name in (
            similarity_values
        ):
            value = row.get(
                field_name
            )

            if isinstance(
                value,
                (int, float),
            ):
                similarity_values[
                    field_name
                ].append(
                    value
                )

    return {
        "row_count": len(
            rows
        ),
        "orphan_document_ids": (
            orphan_document_ids
        ),
        "relation_type_counts": (
            nb12h_counter_dict(
                relation_type_counts
            )
        ),
        "decision_counts": (
            nb12h_counter_dict(
                decision_counts
            )
        ),
        "eligibility_counts": (
            nb12h_counter_dict(
                eligibility_counts
            )
        ),
        "source_pair_counts": (
            nb12h_counter_dict(
                source_pair_counts
            )
        ),
        "authority_pair_counts": (
            nb12h_counter_dict(
                authority_pair_counts
            )
        ),
        "automatic_merge_counts": (
            nb12h_counter_dict(
                automatic_merge_counts
            )
        ),
        "automatic_delete_counts": (
            nb12h_counter_dict(
                automatic_delete_counts
            )
        ),
        "canonical_document_presence_counts": (
            nb12h_counter_dict(
                canonical_document_presence_counts
            )
        ),
        "both_eligible_pair_count": (
            pair_count_both_eligible
        ),
        "eligible_document_ids_touched_count": (
            len(
                eligible_document_ids_touched
            )
        ),
        "similarity_summary": {
            field_name: (
                nb12h_numeric_summary(
                    values
                )
            )
            for (
                field_name,
                values,
            ) in (
                similarity_values.items()
            )
        },
    }


# ---------------------------------------------------------
# Near Duplicate audit
# ---------------------------------------------------------

near_audit = (
    nb12h_audit_pair_relations(
        near_rows,
        decision_field=(
            "production_duplicate_decision"
        ),
    )
)

assert not (
    near_audit[
        "orphan_document_ids"
    ]
), (
    "Near-Duplicate candidates reference "
    "documents outside Accepted Corpus."
)


# ---------------------------------------------------------
# Cross-source Similarity audit
# ---------------------------------------------------------

cross_audit = (
    nb12h_audit_pair_relations(
        cross_rows,
        decision_field=(
            "interpretation"
        ),
    )
)

assert not (
    cross_audit[
        "orphan_document_ids"
    ]
), (
    "Cross-source candidates reference "
    "documents outside Accepted Corpus."
)


# ---------------------------------------------------------
# Validate the NB09 no-auto-mutation policy
# ---------------------------------------------------------

def nb12h_assert_false_only(
    counts,
    field_name,
):
    invalid = {
        key: value
        for (
            key,
            value,
        ) in counts.items()
        if key not in {
            "False",
            "None",
        }
    }

    assert not invalid, (
        f"{field_name} contains enabled values: "
        f"{invalid}"
    )


nb12h_assert_false_only(
    near_audit[
        "automatic_merge_counts"
    ],
    "Near automatic_merge",
)

nb12h_assert_false_only(
    near_audit[
        "automatic_delete_counts"
    ],
    "Near automatic_delete",
)

nb12h_assert_false_only(
    cross_audit[
        "automatic_merge_counts"
    ],
    "Cross automatic_merge",
)

nb12h_assert_false_only(
    cross_audit[
        "automatic_delete_counts"
    ],
    "Cross automatic_delete",
)


exact_automatic_merge_counts = Counter(
    str(
        row.get(
            "automatic_merge"
        )
    )
    for row in exact_rows
)

exact_automatic_delete_counts = Counter(
    str(
        row.get(
            "automatic_delete"
        )
    )
    for row in exact_rows
)

nb12h_assert_false_only(
    nb12h_counter_dict(
        exact_automatic_merge_counts
    ),
    "Exact automatic_merge",
)

nb12h_assert_false_only(
    nb12h_counter_dict(
        exact_automatic_delete_counts
    ),
    "Exact automatic_delete",
)


# ---------------------------------------------------------
# Canonicalization audit
# ---------------------------------------------------------

exact_canonical_document_present = sum(
    1
    for row in exact_rows
    if nb12h_nonempty(
        row.get(
            "canonical_document_id"
        )
    )
)

near_canonical_document_present = (
    near_audit[
        "canonical_document_presence_counts"
    ].get(
        "PRESENT",
        0,
    )
)

cross_canonical_document_present = (
    cross_audit[
        "canonical_document_presence_counts"
    ].get(
        "PRESENT",
        0,
    )
)

automatic_canonicalization_total = (
    exact_canonical_document_present
    + near_canonical_document_present
    + cross_canonical_document_present
)

assert (
    automatic_canonicalization_total
    == 0
), (
    "Canonical document assignments were "
    "found although NB09 froze automatic "
    "canonicalization at zero."
)


# ---------------------------------------------------------
# Exact-only potential extraction reduction
#
# This is NOT a source deletion count.
# It only answers:
# "How many repeated eligible text instances
# could potentially share one extraction pass?"
# ---------------------------------------------------------

eligible_records_after_exact_text_sharing = (
    len(
        eligible_document_ids
    )
    - exact_potential_extraction_reduction
)

assert (
    eligible_records_after_exact_text_sharing
    > 0
), (
    "Invalid exact-sharing reduction."
)


# ---------------------------------------------------------
# Extraction safety state
# ---------------------------------------------------------

dedup_extraction_policy = {
    "exact_duplicate": (
        "MAY_SHARE_TEXT_EXTRACTION_LATER_"
        "WHILE_PRESERVING_ALL_SOURCE_EVIDENCE"
    ),

    "near_duplicate": (
        "DO_NOT_MERGE_BEFORE_CLAIM_LEVEL_REVIEW"
    ),

    "cross_source_similarity": (
        "DO_NOT_MERGE_BEFORE_CLAIM_LEVEL_REVIEW"
    ),

    "source_deletion_allowed": False,

    "automatic_merge_allowed": False,

    "automatic_delete_allowed": False,

    "canonical_document_assignment_allowed": False,
}


# ---------------------------------------------------------
# Build report
# ---------------------------------------------------------

dedup_topology_report = {
    "audit_version": (
        DEDUP_TOPOLOGY_AUDIT_VERSION
    ),

    "status": "PASS",

    "population": {
        "accepted_record_count": (
            len(
                document_lookup
            )
        ),
        "claim_discovery_eligible_count": (
            len(
                eligible_document_ids
            )
        ),
    },

    "exact_duplicate": {
        "group_count": (
            len(
                exact_rows
            )
        ),
        "unique_member_document_count": (
            len(
                exact_member_document_ids_seen
            )
        ),
        "eligible_member_count": (
            exact_eligible_member_count
        ),
        "groups_touching_eligible": (
            exact_groups_touching_eligible
        ),
        "groups_with_multiple_eligible": (
            exact_groups_with_multiple_eligible
        ),
        "potential_shared_extraction_reduction": (
            exact_potential_extraction_reduction
        ),
        "eligible_records_after_exact_text_sharing": (
            eligible_records_after_exact_text_sharing
        ),
        "cross_source_group_count": (
            exact_cross_source_groups
        ),
        "mixed_authority_group_count": (
            exact_mixed_authority_groups
        ),
        "relation_type_counts": (
            nb12h_counter_dict(
                exact_relation_type_counts
            )
        ),
        "relation_scope_counts": (
            nb12h_counter_dict(
                exact_relation_scope_counts
            )
        ),
        "canonical_status_counts": (
            nb12h_counter_dict(
                exact_canonical_status_counts
            )
        ),
        "member_count_distribution": (
            nb12h_counter_dict(
                exact_member_count_distribution
            )
        ),
        "source_type_mix_counts": (
            nb12h_counter_dict(
                exact_source_type_mix_counts
            )
        ),
        "authority_mix_counts": (
            nb12h_counter_dict(
                exact_authority_mix_counts
            )
        ),
        "eligibility_mix_counts": (
            nb12h_counter_dict(
                exact_eligibility_mix_counts
            )
        ),
        "automatic_merge_counts": (
            nb12h_counter_dict(
                exact_automatic_merge_counts
            )
        ),
        "automatic_delete_counts": (
            nb12h_counter_dict(
                exact_automatic_delete_counts
            )
        ),
        "canonical_document_present_count": (
            exact_canonical_document_present
        ),
    },

    "near_duplicate": (
        near_audit
    ),

    "cross_source_similarity": (
        cross_audit
    ),

    "automatic_canonicalization_total": (
        automatic_canonicalization_total
    ),

    "dedup_extraction_policy": (
        dedup_extraction_policy
    ),

    "claim_extraction_started": False,
    "extraction_units_created": False,
    "seed_count_determined": False,

    "api_calls_made": 0,
    "cost_incurred": 0,

    "source_files_modified": False,
    "active_dataset_modified": False,
    "golden_dataset_v1_modified": False,
    "core46_modified": False,
    "golden_dataset_v2_created": False,
}


# ---------------------------------------------------------
# No-overwrite persistence
# ---------------------------------------------------------

dedup_topology_report_path = (
    role_matrix_output_directory
    / DEDUP_TOPOLOGY_REPORT_FILENAME
)

if dedup_topology_report_path.exists():

    with dedup_topology_report_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        existing_report = (
            json.load(
                file
            )
        )

    comparable_existing = dict(
        existing_report
    )

    comparable_current = dict(
        dedup_topology_report
    )

    comparable_existing.pop(
        "created_at_utc",
        None,
    )

    comparable_current.pop(
        "created_at_utc",
        None,
    )

    assert (
        comparable_existing
        == comparable_current
    ), (
        "Existing Dedup topology report "
        "differs from this run. "
        "Refusing to overwrite it."
    )

    dedup_topology_report = (
        existing_report
    )

    dedup_topology_report_write_status = (
        "REUSED_EXISTING_IDENTICAL_REPORT"
    )

else:

    dedup_topology_report[
        "created_at_utc"
    ] = datetime.now(
        timezone.utc
    ).isoformat()

    with dedup_topology_report_path.open(
        "x",
        encoding="utf-8",
    ) as file:

        json.dump(
            dedup_topology_report,
            file,
            ensure_ascii=False,
            indent=2,
        )

        file.write("\n")

    dedup_topology_report_write_status = (
        "CREATED_NEW_REPORT"
    )


# ---------------------------------------------------------
# Compact output
# ---------------------------------------------------------

print(
    "=== NB12h CLAIM DISCOVERY "
    "DEDUP TOPOLOGY AUDIT ==="
)

print()
print(
    "Claim Discovery eligible records:",
    len(
        eligible_document_ids
    ),
)

print()
print(
    "Exact Duplicate groups:",
    len(
        exact_rows
    ),
)

print(
    "Exact unique member documents:",
    len(
        exact_member_document_ids_seen
    ),
)

print(
    "Exact eligible member documents:",
    exact_eligible_member_count,
)

print(
    "Exact groups touching eligible population:",
    exact_groups_touching_eligible,
)

print(
    "Exact groups with 2+ eligible members:",
    exact_groups_with_multiple_eligible,
)

print(
    "Exact cross-source groups:",
    exact_cross_source_groups,
)

print(
    "Exact mixed-authority groups:",
    exact_mixed_authority_groups,
)

print(
    "Potential shared-extraction reduction:",
    exact_potential_extraction_reduction,
)

print(
    "Eligible records after exact-text sharing:",
    eligible_records_after_exact_text_sharing,
)

print()
print(
    "Exact canonical status:",
    nb12h_counter_dict(
        exact_canonical_status_counts
    ),
)

print()
print(
    "Near Duplicate:"
)

print(
    "  total pairs:",
    near_audit[
        "row_count"
    ],
)

print(
    "  eligibility:",
    near_audit[
        "eligibility_counts"
    ],
)

print(
    "  production decisions:",
    near_audit[
        "decision_counts"
    ],
)

print(
    "  automatic_merge:",
    near_audit[
        "automatic_merge_counts"
    ],
)

print(
    "  automatic_delete:",
    near_audit[
        "automatic_delete_counts"
    ],
)

print(
    "  eligible documents touched:",
    near_audit[
        "eligible_document_ids_touched_count"
    ],
)

print(
    "  similarity summary:",
    near_audit[
        "similarity_summary"
    ],
)

print()
print(
    "Cross-Source Similarity:"
)

print(
    "  total pairs:",
    cross_audit[
        "row_count"
    ],
)

print(
    "  eligibility:",
    cross_audit[
        "eligibility_counts"
    ],
)

print(
    "  interpretations:",
    cross_audit[
        "decision_counts"
    ],
)

print(
    "  automatic_merge:",
    cross_audit[
        "automatic_merge_counts"
    ],
)

print(
    "  automatic_delete:",
    cross_audit[
        "automatic_delete_counts"
    ],
)

print(
    "  eligible documents touched:",
    cross_audit[
        "eligible_document_ids_touched_count"
    ],
)

print(
    "  similarity summary:",
    cross_audit[
        "similarity_summary"
    ],
)

print()
print(
    "Automatic canonicalization total:",
    automatic_canonicalization_total,
)

print()
print(
    "Extraction policy:",
    dedup_extraction_policy,
)

print()
print(
    "Audit report:",
    dedup_topology_report_path,
)

print(
    "Report write status:",
    dedup_topology_report_write_status,
)

print()
print("Claim extraction started:", False)
print("Extraction units created:", False)
print("Natural seed count determined:", False)

print()
print("API calls made:", 0)
print("Cost incurred:", 0)
print("Source files modified:", False)
print("Active dataset modified:", False)
print("Golden Dataset v1 modified:", False)
print("Core-46 modified:", False)
print("Golden Dataset v2 created:", False)

print()
print(
    "NB12h CLAIM DISCOVERY "
    "DEDUP TOPOLOGY AUDIT: PASS"
)

=== NB12h CLAIM DISCOVERY DEDUP TOPOLOGY AUDIT ===

Claim Discovery eligible records: 15158

Exact Duplicate groups: 192
Exact unique member documents: 931
Exact eligible member documents: 754
Exact groups touching eligible population: 160
Exact groups with 2+ eligible members: 156
Exact cross-source groups: 9
Exact mixed-authority groups: 10
Potential shared-extraction reduction: 594
Eligible records after exact-text sharing: 14564

Exact canonical status: {'NOT_SELECTED_WITHOUT_INDEPENDENT_EVIDENCE': 192}

Near Duplicate:
  total pairs: 15929
  eligibility: {'BOTH_ELIGIBLE': 11403, 'NONE_ELIGIBLE': 1758, 'ONE_ELIGIBLE': 2768}
  production decisions: {'NOT_SELECTED': 15929}
  automatic_merge: {'False': 15929}
  automatic_delete: {'False': 15929}
  eligible documents touched: 14869
  similarity summary: {'char_tfidf_cosine': {'count': 15929, 'min': 0.0, 'p50': 0.42939924455549616, 'mean': 0.4573356217197516, 'max': 1.0}, 'token_jaccard': {'count': 15929, 'min': 0.0, 'p50': 0.1594202898

<div dir="rtl" align="right">

## **مرحلهٔ سی‌ونهم — ممیزی استقلال Golden Dataset از Chunk Strategy و ریسک اختلاط Authority**

مرحلهٔ قبل مشخص کرد که روابط Dedup موجود تا چه حد می‌توانند در Claim Discovery استفاده شوند.

نتیجه این بود که:

- Exact Duplicateها می‌توانند در آینده یک Extraction Pass مشترک داشته باشند، مشروط به حفظ تمام Sourceها و Evidenceها؛

- Near-Duplicateها اجازهٔ Merge خودکار ندارند؛

- Cross-Source Similarity نیز Duplicate Proof محسوب نمی‌شود؛

- هیچ Canonical Document به‌صورت خودکار انتخاب نشده است.

اکنون قبل از ساخت Extraction Unit باید یک مسئلهٔ مستقل بررسی شود:

**آیا Chunkهای ساخته‌شده در Notebook 11 می‌توانند مستقیماً مبنای Claim Extraction باشند یا خیر؟**

### **چرا این بررسی لازم است**

Notebook 11 چند Candidate Chunking Strategy ساخته است، اما هیچ‌یک هنوز به‌عنوان Strategy نهایی Production انتخاب نشده‌اند.

Golden Dataset نیز خود یکی از ورودی‌های لازم برای ارزیابی همین Strategyها است.

بنابراین اگر Candidate Claimها مستقیماً از یک Strategy خاص استخراج شوند، ممکن است Golden Dataset به همان Strategy وابسته شود و Benchmark بعدی Chunking دچار Circularity شود.

برای مثال اگر Golden Evidenceها براساس Boundaryهای `structure_aware_small` تعریف شوند، سپس همان Golden Dataset برای مقایسهٔ `structure_aware_small` با سایر Strategyها استفاده شود، Benchmark ممکن است ناخواسته به نفع همان Strategy ساخته شده باشد.

### **اصل استقلال Golden Dataset**

Golden Dataset v2 باید از Source Corpus و Evidence اصلی ساخته شود، نه از یک Retrieval Chunk Strategy منتخب.

Chunkهای NB11 می‌توانند برای موارد زیر استفاده شوند:

- بررسی Structure؛

- بررسی Locator؛

- کشف Failure Mode؛

- طراحی Evaluation Case؛

- و Benchmark Retrieval.

اما هیچ Candidate Chunk Strategy نباید صرفاً به دلیل موجودبودن، مبنای نهایی حقیقت Golden Dataset قرار گیرد.

### **مسئلهٔ دوم — Authority Mixing**

حتی اگر Circularity وجود نداشت، Claim Extraction باید Authority-safe باشد.

یک Chunk ممکن است شامل چند Document باشد که نقش‌های متفاوتی دارند:

- Primary؛

- Secondary؛

- Contextual؛

- Unresolved.

اگر چنین Chunkی به‌صورت یک Context واحد وارد Claim Extraction شود، مدل ممکن است یک جملهٔ Contextual یا Unresolved را به اشتباه به‌عنوان Claim اصلی مؤلف استخراج کند.

بنابراین در این مرحله تمام Candidate Chunkهای NB11 با Authority Index ساخته‌شده در Cell 37 تطبیق داده می‌شوند.

### **Candidateهای مورد بررسی**

این مرحله Candidateهای Freeze‌شدهٔ NB11 را بررسی می‌کند:

- `baseline_phase1_char_safe`؛

- `structure_aware_small`؛

- `structure_aware_medium`؛

- `parent_child_children`؛

- `parent_child_parents`.

این بررسی به معنی انتخاب هیچ‌کدام نیست.

### **موارد اندازه‌گیری**

برای هر Candidate مشخص می‌شود:

- تعداد Chunkها؛

- تعداد Sourceهای پوشش‌داده‌شده؛

- تعداد Documentهای پوشش‌داده‌شده؛

- پوشش Population واجد Claim Discovery؛

- تعداد Chunkهای فقط Primary؛

- تعداد Chunkهای فقط Secondary؛

- تعداد Chunkهای دارای Primary و Secondary هم‌زمان؛

- تعداد Chunkهای دارای Eligible Knowledge همراه Contextual Knowledge؛

- تعداد Chunkهای دارای Eligible Knowledge همراه Unresolved Context؛

- تعداد Chunkهای فاقد هرگونه رکورد Eligible؛

- تعداد Chunkهایی که بیش از یک Authority Class دارند؛

- تعداد Source Boundary Violationها؛

- و تعداد Member Documentهای ناشناخته.

### **قاعدهٔ تصمیم**

در این مرحله هیچ Candidate Chunk به‌عنوان ورودی نهایی Claim Extraction انتخاب نمی‌شود.

حتی اگر یک Strategy از نظر Authority کاملاً مناسب باشد، استفاده از آن به‌عنوان مبنای Golden Claimها می‌تواند Independence بین Golden Dataset و Retrieval Benchmark را کاهش دهد.

بنابراین هدف این Cell انتخاب Winner نیست.

هدف تعیین قرارداد مرحلهٔ بعد است.

### **قرارداد مورد انتظار برای Claim Extraction**

اگر Audit نشان دهد Chunkهای Retrieval دارای Authority Mixing هستند، Claim Extraction باید Segmentation اختصاصی خود را داشته باشد.

این Segmentation باید:

- مستقیماً از Accepted Corpus ساخته شود؛

- Source-aware باشد؛

- Structure-aware باشد؛

- Authority-homogeneous باشد؛

- Locator و Provenance را حفظ کند؛

- به هیچ Retrieval Strategy خاصی وابسته نباشد؛

- و فقط برای Claim Discovery استفاده شود.

این Segmentation یک Production Chunker جدید نیست.

### **نقش Exact Duplicateها**

Exact Duplicate Sharing همچنان حفظ می‌شود.

اما Reduction محاسبه‌شده در مرحلهٔ قبل نباید قبل از طراحی Segmentation نهایی به‌صورت مکانیکی اعمال شود.

ممکن است دو Document دارای متن یکسان باشند، ولی در Context ساختاری یا Attribution متفاوت قرار داشته باشند.

بنابراین Exact Text Sharing فقط پس از ساخت Extraction Unitهای معتبر اعمال خواهد شد.

### **مرز تصمیم**

در این مرحله:

- هیچ Claim استخراج نمی‌شود؛

- هیچ Extraction Unit نهایی ساخته نمی‌شود؛

- هیچ Chunk Strategy انتخاب نمی‌شود؛

- هیچ Chunk جدیدی ساخته نمی‌شود؛

- Near و Cross-Source Relationها Merge نمی‌شوند؛

- هیچ Source حذف نمی‌شود؛

- Pilot-28 و Core-46 تغییر نمی‌کنند؛

- Golden Dataset v1 تغییر نمی‌کند؛

- Golden Dataset v2 ساخته نمی‌شود؛

- هیچ API یا LLM فراخوانی نمی‌شود؛

- و هیچ Artifact قبلی overwrite نمی‌شود.

### **Artifact این مرحله**

گزارش جدید زیر ساخته می‌شود:

`claim_discovery_chunk_independence_audit_v1.json`

این Artifact فقط گزارش تحلیلی است و هیچ Candidate Chunk یا Dataset قبلی را تغییر نمی‌دهد.

### **Validation**

این مرحله زمانی PASS است که:

- تمام Candidate Artifactهای NB11 قابل خواندن باشند؛

- Record Count هر Candidate با Freeze قبلی تطابق داشته باشد؛

- `chunk_id` در هر Candidate یکتا باشد؛

- تمام `member_document_ids` در Accepted Corpus وجود داشته باشند؛

- تمام ۱۹٬۴۵۶ Document توسط هر Candidate طبق Freeze قبلی پوشش داده شوند؛

- هیچ Chunk از Sourceهای مستقل عبور نکرده باشد؛

- پوشش ۱۵٬۱۵۸ رکورد Eligible برای هر Candidate مشخص شود؛

- Authority Composition تمام Chunkها قابل محاسبه باشد؛

- ریسک Mixing برای هر Strategy به‌صورت عددی گزارش شود؛

- هیچ Strategy به‌عنوان Golden Dataset Source انتخاب نشود؛

- و هیچ Dataset موجودی تغییر نکرده باشد.

پس از PASS شدن این مرحله، می‌توان Extraction Segmentation اختصاصی Golden Dataset را مستقل از Retrieval Chunk Strategy طراحی کرد.

</div>

In [41]:
# Code Cell 39
# NB12h — Golden Dataset / Retrieval Chunk independence
# and Authority-mixing audit
#
# Read-only.
# No claim extraction.
# No extraction-unit generation.
# No chunk strategy selection.
# No API calls.
# No overwrite.

assert "project_root" in globals(), (
    "Run the NB12h project-discovery Cell first."
)

assert "role_matrix_output_directory" in globals(), (
    "Run Code Cell 31 first."
)

assert "claim_discovery_index_rows" in globals(), (
    "Run Code Cell 37 first."
)

assert "dedup_topology_report" in globals(), (
    "Run Code Cell 38 first."
)

assert "chunk_final_requirements" in globals(), (
    "Run Code Cell 36 first."
)

assert "chunk_decision_readiness" in globals(), (
    "Run Code Cell 36 first."
)


CHUNK_INDEPENDENCE_AUDIT_VERSION = (
    "nb12h_claim_discovery_chunk_independence_audit_v1"
)

CHUNK_INDEPENDENCE_REPORT_FILENAME = (
    "claim_discovery_chunk_independence_audit_v1.json"
)


# ---------------------------------------------------------
# Frozen NB11 Candidate definitions
# ---------------------------------------------------------

NB11_CANDIDATE_SPECS = {
    "baseline_phase1_char_safe": {
        "relative_path": (
            "data/phase2/chunks/nb11_benchmark_v1/"
            "baseline_phase1_char_safe_v1.jsonl"
        ),
        "expected_records": 3488,
        "retrieval_role": (
            "HISTORICAL_COMPARISON_BASELINE"
        ),
    },

    "structure_aware_small": {
        "relative_path": (
            "data/phase2/chunks/nb11_benchmark_v1/"
            "structure_aware_small_v1.jsonl"
        ),
        "expected_records": 2526,
        "retrieval_role": (
            "LEADING_SIMPLE_RETRIEVAL_CANDIDATE"
        ),
    },

    "structure_aware_medium": {
        "relative_path": (
            "data/phase2/chunks/nb11_benchmark_v1/"
            "structure_aware_medium_v1.jsonl"
        ),
        "expected_records": 1799,
        "retrieval_role": (
            "LARGER_CONTEXT_RETRIEVAL_ABLATION"
        ),
    },

    "parent_child_children": {
        "relative_path": (
            "data/phase2/chunks/nb11_benchmark_v1/"
            "parent_child_children_v1.jsonl"
        ),
        "expected_records": 2721,
        "retrieval_role": (
            "RETRIEVAL_CHILD_CANDIDATE"
        ),
    },

    "parent_child_parents": {
        "relative_path": (
            "data/phase2/chunks/nb11_benchmark_v1/"
            "parent_child_parents_v1.jsonl"
        ),
        "expected_records": 1447,
        "retrieval_role": (
            "CONTEXT_PARENT_CANDIDATE"
        ),
    },
}


# ---------------------------------------------------------
# Population lookup
# ---------------------------------------------------------

claim_population_lookup = {
    row["document_id"]: row
    for row in claim_discovery_index_rows
}

assert (
    len(claim_population_lookup)
    == 19456
), (
    "Claim population lookup must contain "
    "19,456 unique documents."
)


accepted_document_id_set = set(
    claim_population_lookup
)

eligible_claim_roles = {
    "PRIMARY_CLAIM_DISCOVERY",
    "SECONDARY_CLAIM_DISCOVERY_WITH_ATTRIBUTION",
}

eligible_document_id_set = {
    document_id
    for (
        document_id,
        row,
    ) in claim_population_lookup.items()
    if row["claim_discovery_role"]
    in eligible_claim_roles
}

assert (
    len(eligible_document_id_set)
    == 15158
), (
    "Eligible Claim Discovery population "
    "must equal 15,158 records."
)


# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------

def nb12h_read_chunk_jsonl(path):
    records = []

    with path.open(
        "r",
        encoding="utf-8",
        errors="strict",
    ) as file:

        for line_number, line in enumerate(
            file,
            1,
        ):
            stripped = line.strip()

            if not stripped:
                continue

            payload = json.loads(
                stripped
            )

            assert isinstance(
                payload,
                dict,
            ), (
                "Chunk artifact contains "
                "a non-object JSON record at "
                f"{path}:{line_number}"
            )

            records.append(
                payload
            )

    return records


def nb12h_sorted_counter(counter):
    return dict(
        sorted(
            counter.items(),
            key=lambda item: (
                str(item[0])
            ),
        )
    )


def nb12h_member_authority_composition(
    member_rows,
):
    counts = Counter(
        row["authority_class"]
        for row in member_rows
    )

    return counts


def nb12h_member_role_composition(
    member_rows,
):
    counts = Counter(
        row["claim_discovery_role"]
        for row in member_rows
    )

    return counts


def nb12h_chunk_eligibility_label(
    member_rows,
):
    roles = {
        row["claim_discovery_role"]
        for row in member_rows
    }

    has_primary = (
        "PRIMARY_CLAIM_DISCOVERY"
        in roles
    )

    has_secondary = (
        "SECONDARY_CLAIM_DISCOVERY_WITH_ATTRIBUTION"
        in roles
    )

    has_context = (
        "CONTEXT_ONLY_NO_INDEPENDENT_SEED"
        in roles
    )

    has_unresolved = (
        "HOLD_UNRESOLVED_NO_INDEPENDENT_SEED"
        in roles
    )

    has_eligible = (
        has_primary
        or has_secondary
    )

    if not has_eligible:
        return (
            "NO_ELIGIBLE_MEMBER"
        )

    if (
        has_context
        and has_unresolved
    ):
        return (
            "ELIGIBLE_WITH_CONTEXT_AND_UNRESOLVED"
        )

    if has_context:
        return (
            "ELIGIBLE_WITH_CONTEXT"
        )

    if has_unresolved:
        return (
            "ELIGIBLE_WITH_UNRESOLVED"
        )

    return (
        "ELIGIBLE_ONLY"
    )


def nb12h_eligible_authority_label(
    member_rows,
):
    eligible_authorities = {
        row["authority_class"]
        for row in member_rows
        if row["claim_discovery_role"]
        in eligible_claim_roles
    }

    if not eligible_authorities:
        return "NONE"

    if eligible_authorities == {
        "CORE_PRIMARY_KNOWLEDGE"
    }:
        return "PRIMARY_ONLY"

    if eligible_authorities == {
        "SUPERVISED_SECONDARY_KNOWLEDGE"
    }:
        return "SECONDARY_ONLY"

    if eligible_authorities == {
        "CORE_PRIMARY_KNOWLEDGE",
        "SUPERVISED_SECONDARY_KNOWLEDGE",
    }:
        return "PRIMARY_PLUS_SECONDARY"

    return (
        "OTHER:"
        + "|".join(
            sorted(
                eligible_authorities
            )
        )
    )


# ---------------------------------------------------------
# Candidate audit
# ---------------------------------------------------------

candidate_audits = {}

global_candidate_failures = []


for (
    candidate_id,
    candidate_spec,
) in NB11_CANDIDATE_SPECS.items():

    candidate_path = (
        project_root
        / candidate_spec[
            "relative_path"
        ]
    )

    assert candidate_path.is_file(), (
        "Frozen NB11 candidate artifact "
        "is missing: "
        f"{candidate_path}"
    )

    chunks = nb12h_read_chunk_jsonl(
        candidate_path
    )

    expected_records = (
        candidate_spec[
            "expected_records"
        ]
    )

    assert (
        len(chunks)
        == expected_records
    ), (
        f"{candidate_id} record count mismatch: "
        f"{len(chunks)} != {expected_records}"
    )


    chunk_ids = []

    all_member_document_ids = set()

    eligible_member_document_ids = set()

    all_source_ids = set()

    orphan_member_ids = []

    empty_member_chunks = []

    source_boundary_violations = []

    chunk_source_type_mismatches = []

    eligibility_label_counts = Counter()

    eligible_authority_label_counts = Counter()

    authority_class_mix_counts = Counter()

    role_mix_counts = Counter()

    source_type_chunk_counts = Counter()

    mixed_authority_chunk_count = 0

    eligible_with_noneligible_chunk_count = 0

    primary_with_context_chunk_count = 0

    primary_with_unresolved_chunk_count = 0

    primary_with_secondary_chunk_count = 0

    attribution_mixed_primary_signal_count = 0

    chunk_member_count_values = []

    source_specific_stats = defaultdict(
        lambda: {
            "chunks": 0,
            "eligible_only": 0,
            "eligible_with_context": 0,
            "eligible_with_unresolved": 0,
            "eligible_with_context_and_unresolved": 0,
            "no_eligible": 0,
            "mixed_authority": 0,
        }
    )


    for chunk in chunks:

        chunk_id = (
            chunk.get(
                "chunk_id"
            )
        )

        assert isinstance(
            chunk_id,
            str,
        ) and chunk_id.strip(), (
            f"{candidate_id} contains "
            "a missing chunk_id."
        )

        chunk_ids.append(
            chunk_id
        )


        member_document_ids = (
            chunk.get(
                "member_document_ids"
            )
        )

        if not isinstance(
            member_document_ids,
            list,
        ) or not member_document_ids:

            empty_member_chunks.append(
                chunk_id
            )

            continue


        chunk_member_count_values.append(
            len(
                member_document_ids
            )
        )


        member_rows = []

        for document_id in (
            member_document_ids
        ):

            if (
                document_id
                not in claim_population_lookup
            ):
                orphan_member_ids.append(
                    document_id
                )

                continue

            member_row = (
                claim_population_lookup[
                    document_id
                ]
            )

            member_rows.append(
                member_row
            )

            all_member_document_ids.add(
                document_id
            )

            all_source_ids.add(
                member_row[
                    "source_id"
                ]
            )

            if (
                document_id
                in eligible_document_id_set
            ):
                eligible_member_document_ids.add(
                    document_id
                )


        if not member_rows:
            continue


        member_source_ids = {
            row["source_id"]
            for row in member_rows
        }

        member_source_types = {
            row["source_type"]
            for row in member_rows
        }


        if len(
            member_source_ids
        ) != 1:

            source_boundary_violations.append({
                "chunk_id": chunk_id,
                "source_ids": sorted(
                    member_source_ids
                ),
            })


        if len(
            member_source_types
        ) != 1:

            source_boundary_violations.append({
                "chunk_id": chunk_id,
                "source_types": sorted(
                    member_source_types
                ),
            })


        member_source_type = (
            next(
                iter(
                    member_source_types
                )
            )
            if len(
                member_source_types
            ) == 1
            else "MULTIPLE"
        )


        chunk_source_type = (
            chunk.get(
                "source_type"
            )
        )

        if (
            member_source_type
            != "MULTIPLE"
            and chunk_source_type
            != member_source_type
        ):

            chunk_source_type_mismatches.append({
                "chunk_id": chunk_id,
                "chunk_source_type": (
                    chunk_source_type
                ),
                "member_source_type": (
                    member_source_type
                ),
            })


        source_type_chunk_counts[
            member_source_type
        ] += 1


        authority_composition = (
            nb12h_member_authority_composition(
                member_rows
            )
        )

        role_composition = (
            nb12h_member_role_composition(
                member_rows
            )
        )


        authority_key = "|".join(
            sorted(
                authority_composition
            )
        )

        role_key = "|".join(
            sorted(
                role_composition
            )
        )

        authority_class_mix_counts[
            authority_key
        ] += 1

        role_mix_counts[
            role_key
        ] += 1


        if len(
            authority_composition
        ) > 1:
            mixed_authority_chunk_count += 1


        eligibility_label = (
            nb12h_chunk_eligibility_label(
                member_rows
            )
        )

        eligibility_label_counts[
            eligibility_label
        ] += 1


        eligible_authority_label = (
            nb12h_eligible_authority_label(
                member_rows
            )
        )

        eligible_authority_label_counts[
            eligible_authority_label
        ] += 1


        roles = set(
            role_composition
        )

        has_primary = (
            "PRIMARY_CLAIM_DISCOVERY"
            in roles
        )

        has_secondary = (
            "SECONDARY_CLAIM_DISCOVERY_WITH_ATTRIBUTION"
            in roles
        )

        has_context = (
            "CONTEXT_ONLY_NO_INDEPENDENT_SEED"
            in roles
        )

        has_unresolved = (
            "HOLD_UNRESOLVED_NO_INDEPENDENT_SEED"
            in roles
        )

        has_eligible = (
            has_primary
            or has_secondary
        )

        has_noneligible = (
            has_context
            or has_unresolved
        )


        if (
            has_eligible
            and has_noneligible
        ):
            eligible_with_noneligible_chunk_count += 1


        if (
            has_primary
            and has_context
        ):
            primary_with_context_chunk_count += 1


        if (
            has_primary
            and has_unresolved
        ):
            primary_with_unresolved_chunk_count += 1


        if (
            has_primary
            and has_secondary
        ):
            primary_with_secondary_chunk_count += 1


        attribution_context = (
            chunk.get(
                "attribution_context"
            )
            or {}
        )

        if (
            attribution_context.get(
                "mixed_primary_signal"
            )
            is True
        ):
            attribution_mixed_primary_signal_count += 1


        source_stats = (
            source_specific_stats[
                member_source_type
            ]
        )

        source_stats[
            "chunks"
        ] += 1


        if (
            eligibility_label
            == "ELIGIBLE_ONLY"
        ):
            source_stats[
                "eligible_only"
            ] += 1

        elif (
            eligibility_label
            == "ELIGIBLE_WITH_CONTEXT"
        ):
            source_stats[
                "eligible_with_context"
            ] += 1

        elif (
            eligibility_label
            == "ELIGIBLE_WITH_UNRESOLVED"
        ):
            source_stats[
                "eligible_with_unresolved"
            ] += 1

        elif (
            eligibility_label
            == "ELIGIBLE_WITH_CONTEXT_AND_UNRESOLVED"
        ):
            source_stats[
                "eligible_with_context_and_unresolved"
            ] += 1

        elif (
            eligibility_label
            == "NO_ELIGIBLE_MEMBER"
        ):
            source_stats[
                "no_eligible"
            ] += 1


        if len(
            authority_composition
        ) > 1:
            source_stats[
                "mixed_authority"
            ] += 1


    # -----------------------------------------------------
    # Candidate-level validation
    # -----------------------------------------------------

    duplicate_chunk_id_count = (
        len(chunk_ids)
        - len(
            set(
                chunk_ids
            )
        )
    )

    full_accepted_coverage = (
        all_member_document_ids
        == accepted_document_id_set
    )

    full_eligible_coverage = (
        eligible_member_document_ids
        == eligible_document_id_set
    )


    assert (
        duplicate_chunk_id_count
        == 0
    ), (
        f"{candidate_id} contains duplicate chunk IDs."
    )

    assert not empty_member_chunks, (
        f"{candidate_id} contains chunks "
        "without member_document_ids."
    )

    assert not orphan_member_ids, (
        f"{candidate_id} references documents "
        "outside the Accepted Corpus."
    )

    assert not source_boundary_violations, (
        f"{candidate_id} crosses Source boundaries."
    )

    assert not chunk_source_type_mismatches, (
        f"{candidate_id} source_type does not match "
        "its member Documents."
    )

    assert full_accepted_coverage, (
        f"{candidate_id} does not cover the "
        "full Accepted Corpus."
    )

    assert full_eligible_coverage, (
        f"{candidate_id} does not cover all "
        "Claim Discovery eligible records."
    )


    # -----------------------------------------------------
    # Claim-extraction suitability interpretation
    # -----------------------------------------------------

    direct_gold_extraction_safe = (
        mixed_authority_chunk_count
        == 0
        and eligible_with_noneligible_chunk_count
        == 0
    )

    candidate_audits[
        candidate_id
    ] = {
        "relative_path": (
            candidate_spec[
                "relative_path"
            ]
        ),

        "retrieval_role": (
            candidate_spec[
                "retrieval_role"
            ]
        ),

        "record_count": len(
            chunks
        ),

        "duplicate_chunk_id_count": (
            duplicate_chunk_id_count
        ),

        "covered_document_count": len(
            all_member_document_ids
        ),

        "covered_eligible_document_count": len(
            eligible_member_document_ids
        ),

        "unique_source_count": len(
            all_source_ids
        ),

        "full_accepted_coverage": (
            full_accepted_coverage
        ),

        "full_eligible_coverage": (
            full_eligible_coverage
        ),

        "source_boundary_violation_count": len(
            source_boundary_violations
        ),

        "source_type_chunk_counts": (
            nb12h_sorted_counter(
                source_type_chunk_counts
            )
        ),

        "eligibility_label_counts": (
            nb12h_sorted_counter(
                eligibility_label_counts
            )
        ),

        "eligible_authority_label_counts": (
            nb12h_sorted_counter(
                eligible_authority_label_counts
            )
        ),

        "mixed_authority_chunk_count": (
            mixed_authority_chunk_count
        ),

        "eligible_with_noneligible_chunk_count": (
            eligible_with_noneligible_chunk_count
        ),

        "primary_with_context_chunk_count": (
            primary_with_context_chunk_count
        ),

        "primary_with_unresolved_chunk_count": (
            primary_with_unresolved_chunk_count
        ),

        "primary_with_secondary_chunk_count": (
            primary_with_secondary_chunk_count
        ),

        "nb11_mixed_primary_signal_chunk_count": (
            attribution_mixed_primary_signal_count
        ),

        "authority_class_mix_counts": (
            nb12h_sorted_counter(
                authority_class_mix_counts
            )
        ),

        "role_mix_counts": (
            nb12h_sorted_counter(
                role_mix_counts
            )
        ),

        "source_specific_stats": {
            source_type: dict(
                stats
            )
            for (
                source_type,
                stats,
            ) in sorted(
                source_specific_stats.items()
            )
        },

        "member_count_summary": {
            "min": (
                min(
                    chunk_member_count_values
                )
                if chunk_member_count_values
                else None
            ),
            "max": (
                max(
                    chunk_member_count_values
                )
                if chunk_member_count_values
                else None
            ),
            "mean": (
                (
                    sum(
                        chunk_member_count_values
                    )
                    / len(
                        chunk_member_count_values
                    )
                )
                if chunk_member_count_values
                else None
            ),
        },

        "direct_gold_extraction_safe": (
            direct_gold_extraction_safe
        ),
    }


# ---------------------------------------------------------
# Golden Dataset independence policy
# ---------------------------------------------------------

assert (
    chunk_final_requirements.get(
        "final_selection"
    )
    == "NOT_YET"
), (
    "NB11 final chunk selection state changed."
)

assert (
    chunk_decision_readiness.get(
        "final_strategy"
    )
    is None
), (
    "NB11 now contains a final strategy; "
    "this notebook state must be re-evaluated."
)

assert (
    "golden_dataset"
    in chunk_final_requirements.get(
        "still_required",
        []
    )
), (
    "Golden Dataset is no longer listed as "
    "an input required for final chunk selection."
)


golden_chunk_independence_policy = {
    "golden_claim_source": (
        "ACCEPTED_CORPUS_NOT_RETRIEVAL_CHUNK_CANDIDATE"
    ),

    "single_nb11_candidate_as_gold_basis": False,

    "reason": (
        "AVOID_CIRCULAR_DEPENDENCY_AND_"
        "PRESERVE_AUTHORITY_BOUNDARIES"
    ),

    "retrieval_candidate_role": (
        "STRUCTURE_REFERENCE_AND_LATER_BENCHMARK_ONLY"
    ),

    "next_segmentation_requirement": (
        "SOURCE_AWARE_STRUCTURE_AWARE_"
        "AUTHORITY_HOMOGENEOUS_"
        "RETRIEVAL_STRATEGY_INDEPENDENT"
    ),

    "production_chunk_strategy_selected": False,

    "claim_extraction_started": False,
}


# ---------------------------------------------------------
# Build report
# ---------------------------------------------------------

chunk_independence_report = {
    "audit_version": (
        CHUNK_INDEPENDENCE_AUDIT_VERSION
    ),

    "status": "PASS",

    "population": {
        "accepted_records": (
            len(
                accepted_document_id_set
            )
        ),
        "claim_discovery_eligible_records": (
            len(
                eligible_document_id_set
            )
        ),
    },

    "nb11_decision_state": {
        "final_selection": (
            chunk_final_requirements.get(
                "final_selection"
            )
        ),
        "final_strategy": (
            chunk_decision_readiness.get(
                "final_strategy"
            )
        ),
        "golden_dataset_still_required": True,
    },

    "candidate_audits": (
        candidate_audits
    ),

    "golden_chunk_independence_policy": (
        golden_chunk_independence_policy
    ),

    "exact_text_sharing_from_cell38": {
        "potential_reduction": (
            dedup_topology_report[
                "exact_duplicate"
            ][
                "potential_shared_extraction_reduction"
            ]
        ),
        "eligible_records_after_exact_text_sharing": (
            dedup_topology_report[
                "exact_duplicate"
            ][
                "eligible_records_after_exact_text_sharing"
            ]
        ),
        "applied_now": False,
    },

    "claim_extraction_started": False,
    "extraction_units_created": False,
    "chunk_strategy_selected": False,
    "golden_dataset_v2_created": False,

    "api_calls_made": 0,
    "cost_incurred": 0,

    "source_files_modified": False,
    "active_dataset_modified": False,
    "golden_dataset_v1_modified": False,
    "core46_modified": False,
}


# ---------------------------------------------------------
# No-overwrite persistence
# ---------------------------------------------------------

chunk_independence_report_path = (
    role_matrix_output_directory
    / CHUNK_INDEPENDENCE_REPORT_FILENAME
)

if (
    chunk_independence_report_path
    .exists()
):

    with (
        chunk_independence_report_path
        .open(
            "r",
            encoding="utf-8",
        )
    ) as file:

        existing_report = json.load(
            file
        )

    comparable_existing = dict(
        existing_report
    )

    comparable_current = dict(
        chunk_independence_report
    )

    comparable_existing.pop(
        "created_at_utc",
        None,
    )

    comparable_current.pop(
        "created_at_utc",
        None,
    )

    assert (
        comparable_existing
        == comparable_current
    ), (
        "Existing chunk-independence report "
        "differs from the current audit. "
        "Refusing to overwrite it."
    )

    chunk_independence_report = (
        existing_report
    )

    chunk_independence_write_status = (
        "REUSED_EXISTING_IDENTICAL_REPORT"
    )

else:

    chunk_independence_report[
        "created_at_utc"
    ] = datetime.now(
        timezone.utc
    ).isoformat()

    with (
        chunk_independence_report_path
        .open(
            "x",
            encoding="utf-8",
        )
    ) as file:

        json.dump(
            chunk_independence_report,
            file,
            ensure_ascii=False,
            indent=2,
        )

        file.write("\n")

    chunk_independence_write_status = (
        "CREATED_NEW_REPORT"
    )


# ---------------------------------------------------------
# Compact output
# ---------------------------------------------------------

print(
    "=== NB12h GOLDEN / CHUNK "
    "INDEPENDENCE AUDIT ==="
)

print()
print(
    "Accepted records:",
    len(
        accepted_document_id_set
    ),
)

print(
    "Claim Discovery eligible records:",
    len(
        eligible_document_id_set
    ),
)

print()
print(
    "NB11 final chunk selection:",
    chunk_final_requirements.get(
        "final_selection"
    ),
)

print(
    "NB11 final strategy:",
    chunk_decision_readiness.get(
        "final_strategy"
    ),
)

print(
    "Golden Dataset still required "
    "for chunk decision:",
    True,
)


for (
    candidate_id,
    audit,
) in candidate_audits.items():

    print()
    print(
        f"[{candidate_id}]"
    )

    print(
        "  chunks:",
        audit[
            "record_count"
        ],
    )

    print(
        "  source chunks:",
        audit[
            "source_type_chunk_counts"
        ],
    )

    print(
        "  full accepted coverage:",
        audit[
            "full_accepted_coverage"
        ],
    )

    print(
        "  full eligible coverage:",
        audit[
            "full_eligible_coverage"
        ],
    )

    print(
        "  eligibility composition:",
        audit[
            "eligibility_label_counts"
        ],
    )

    print(
        "  eligible authority composition:",
        audit[
            "eligible_authority_label_counts"
        ],
    )

    print(
        "  mixed-authority chunks:",
        audit[
            "mixed_authority_chunk_count"
        ],
    )

    print(
        "  eligible + noneligible chunks:",
        audit[
            "eligible_with_noneligible_chunk_count"
        ],
    )

    print(
        "  primary + context chunks:",
        audit[
            "primary_with_context_chunk_count"
        ],
    )

    print(
        "  primary + unresolved chunks:",
        audit[
            "primary_with_unresolved_chunk_count"
        ],
    )

    print(
        "  primary + secondary chunks:",
        audit[
            "primary_with_secondary_chunk_count"
        ],
    )

    print(
        "  NB11 mixed-primary signal:",
        audit[
            "nb11_mixed_primary_signal_chunk_count"
        ],
    )

    print(
        "  source-boundary violations:",
        audit[
            "source_boundary_violation_count"
        ],
    )

    print(
        "  direct Gold extraction safe:",
        audit[
            "direct_gold_extraction_safe"
        ],
    )


print()
print(
    "Exact-text sharing potential from Cell 38:",
    dedup_topology_report[
        "exact_duplicate"
    ][
        "potential_shared_extraction_reduction"
    ],
)

print(
    "Exact sharing applied now:",
    False,
)

print()
print(
    "Golden claim source policy:",
    golden_chunk_independence_policy[
        "golden_claim_source"
    ],
)

print(
    "Next segmentation requirement:",
    golden_chunk_independence_policy[
        "next_segmentation_requirement"
    ],
)

print()
print(
    "Audit report:",
    chunk_independence_report_path,
)

print(
    "Report write status:",
    chunk_independence_write_status,
)

print()
print("Claim extraction started:", False)
print("Extraction units created:", False)
print("Chunk strategy selected:", False)
print("Golden Dataset v2 created:", False)

print()
print("API calls made:", 0)
print("Cost incurred:", 0)
print("Source files modified:", False)
print("Active dataset modified:", False)
print("Golden Dataset v1 modified:", False)
print("Core-46 modified:", False)

print()
print(
    "NB12h GOLDEN / CHUNK "
    "INDEPENDENCE AUDIT: PASS"
)

=== NB12h GOLDEN / CHUNK INDEPENDENCE AUDIT ===

Accepted records: 19456
Claim Discovery eligible records: 15158

NB11 final chunk selection: NOT_YET
NB11 final strategy: None
Golden Dataset still required for chunk decision: True

[baseline_phase1_char_safe]
  chunks: 3488
  source chunks: {'book': 351, 'transcript': 2041, 'website': 1096}
  full accepted coverage: True
  full eligible coverage: True
  eligibility composition: {'ELIGIBLE_ONLY': 2090, 'ELIGIBLE_WITH_CONTEXT': 780, 'ELIGIBLE_WITH_CONTEXT_AND_UNRESOLVED': 121, 'ELIGIBLE_WITH_UNRESOLVED': 17, 'NO_ELIGIBLE_MEMBER': 480}
  eligible authority composition: {'NONE': 480, 'PRIMARY_ONLY': 2052, 'SECONDARY_ONLY': 956}
  mixed-authority chunks: 1080
  eligible + noneligible chunks: 918
  primary + context chunks: 901
  primary + unresolved chunks: 138
  primary + secondary chunks: 0
  NB11 mixed-primary signal: 542
  source-boundary violations: 0
  direct Gold extraction safe: False

[structure_aware_small]
  chunks: 2526
  source

<div dir="rtl" align="right">

## **مرحلهٔ چهلم — کشف قرارداد ساختاری واقعی برای Claim Extraction Segmentation**

مرحلهٔ قبل ثابت کرد که هیچ‌یک از Candidate Chunkهای Notebook 11 برای استفادهٔ مستقیم به‌عنوان واحد استخراج Golden Claim مناسب نیستند.

تمام Strategyهای بررسی‌شده حداقل در بخشی از Corpus دارای ترکیب Authority متفاوت یا ترکیب Knowledge واجد Claim Discovery با Contextual و Unresolved Context هستند.

بنابراین Golden Dataset v2 باید Segmentation اختصاصی Claim Discovery داشته باشد که از Retrieval Chunking مستقل باشد.

با این حال، این Segmentation نباید با حدس‌زدن نام Fieldها یا ساختار Sourceها طراحی شود.

### **هدف این مرحله**

در این مرحله Schema واقعی Accepted Corpus برای هر سه Source Type به‌طور کامل بررسی می‌شود تا مشخص شود چه Metadata و Locatorهایی واقعاً برای ساخت Extraction Unit وجود دارند.

تمرکز اصلی روی Fieldهایی است که می‌توانند برای حفظ Structure، Ordering، Citation و Attribution استفاده شوند.

### **Website**

در Corpus پذیرفته‌شده ۵۲۹ Website Record و ۵۲۹ Website Source وجود دارد.

بنابراین انتظار فعلی این است که Website در سطح Document یک واحد ساختاری طبیعی داشته باشد.

این مرحله بررسی می‌کند که آیا واقعاً رابطهٔ یک‌به‌یک میان Website Source و Document برقرار است و چه Locatorهایی برای Citation موجود هستند.

هیچ Sub-chunk جدیدی در این مرحله ساخته نمی‌شود.

### **Book**

Book شامل ۲۷۰۱ Processing Record از ۳ Source Edition پذیرفته‌شده است.

Notebookهای قبلی Structure کتاب را با اطلاعاتی مانند Chapter، Section، Heading، Paragraph، Table و ترتیب Block حفظ کرده‌اند.

اما در این مرحله نام Fieldهای مورد استفاده حدس زده نمی‌شود.

Schema واقعی Corpus بررسی می‌شود تا مشخص شود:

- چه Fieldهایی Heading یا Section را نگه می‌دارند؛

- چه Fieldهایی ترتیب Paragraph یا Block را مشخص می‌کنند؛

- Table چگونه علامت‌گذاری شده است؛

- Supplemental یا Footnote چگونه قابل تشخیص است؛

- و چه Locatorهایی برای بازگشت به Evidence اصلی وجود دارد.

### **Transcript**

Transcript شامل ۱۶٬۲۲۶ Processing Record از ۱۸۳ Source پذیرفته‌شده است.

Notebookهای قبلی Episode، Speaker، Segment و Attribution را حفظ کرده‌اند.

اما Retrieval Chunkهای موجود نشان دادند که ترکیب چند Segment می‌تواند Authorityهای مختلف را داخل یک Chunk قرار دهد.

بنابراین در این مرحله باید مشخص شود:

- چه Fieldهایی ترتیب Segment یا Utterance را ثبت می‌کنند؛

- Speaker و Attribution در کدام Pathها قرار دارند؛

- Program و Episode چگونه ثبت شده‌اند؛

- Timestamp یا Segment Locator چگونه نگهداری شده است؛

- و چه Fieldهایی برای ساخت Runهای Authority-homogeneous قابل استفاده هستند.

### **اصل طراحی Segmentation**

Segmentation مرحلهٔ بعد باید از این اصول پیروی کند:

- Source Boundary هرگز شکسته نشود؛

- Authority Boundary هرگز بدون دلیل عبور داده نشود؛

- ترتیب واقعی محتوا حفظ شود؛

- Structure اصلی Source حفظ شود؛

- Locator و Provenance از بین نرود؛

- Contextual و Unresolved Record با Primary Claim Extraction مخلوط نشوند؛

- و هیچ Retrieval Chunk Strategy به‌عنوان Golden Boundary استفاده نشود.

### **عدم استفاده از Text برای تصمیم Schema**

در این مرحله متن کامل اسناد برای تحلیل معنایی استفاده نمی‌شود.

فقط Schema، Metadata، Locator و اطلاعات ساختاری بررسی می‌شوند.

نمونه‌های چاپ‌شده نیز فقط مقادیر Metadata خواهند بود و متن محتوای Source چاپ نمی‌شود.

### **خروجی این مرحله**

گزارش جدید زیر ساخته می‌شود:

`claim_extraction_structure_contract_audit_v1.json`

این گزارش برای هر Source Type شامل موارد زیر خواهد بود:

- تعداد Record؛

- تعداد Source؛

- Nested Fieldهای ساختاری موجود؛

- Coverage هر Field؛

- نوع دادهٔ هر Field؛

- تعداد مقادیر متمایز؛

- نمونهٔ محدود از Metadata؛

- Candidate Fieldهای مربوط به Ordering؛

- Candidate Fieldهای مربوط به Structure؛

- Candidate Fieldهای مربوط به Locator؛

- Candidate Fieldهای مربوط به Attribution.

### **مرز تصمیم**

در این مرحله:

- Extraction Unit ساخته نمی‌شود؛

- Claim استخراج نمی‌شود؛

- Exact Duplicate Sharing هنوز اعمال نمی‌شود؛

- Semantic Deduplication انجام نمی‌شود؛

- Chunk Strategy انتخاب نمی‌شود؛

- هیچ Metadata جدیدی حدس زده نمی‌شود؛

- هیچ API یا LLM فراخوانی نمی‌شود؛

- Golden Dataset v1 تغییر نمی‌کند؛

- Pilot-28 و Core-46 تغییر نمی‌کنند؛

- Golden Dataset v2 ساخته نمی‌شود؛

- و هیچ Artifact موجودی overwrite نمی‌شود.

### **Validation**

این مرحله زمانی PASS است که:

- تمام ۱۹٬۴۵۶ Accepted Record بدون خطا بررسی شوند؛

- تعداد Source Familyها با Snapshot فریز‌شده تطابق داشته باشد؛

- Website همچنان ۵۲۹ Record و ۵۲۹ Source داشته باشد؛

- Book همچنان ۲۷۰۱ Record و ۳ Source داشته باشد؛

- Transcript همچنان ۱۶٬۲۲۶ Record و ۱۸۳ Source داشته باشد؛

- برای Book حداقل یک Field ساختاری یا ترتیبی با Coverage بالا کشف شود؛

- برای Transcript حداقل یک Field ساختاری یا ترتیبی با Coverage بالا کشف شود؛

- Locatorهای واقعی هر Source Type قابل مشاهده باشند؛

- Attribution/Speaker Metadata Transcript قابل مشاهده باشد؛

- هیچ Field مورد استفاده برای مرحلهٔ بعد از روی حدس انتخاب نشده باشد؛

- و هیچ Dataset موجودی تغییر نکرده باشد.

پس از PASS شدن این مرحله، Extraction Unit Builder بر اساس همین Fieldهای واقعی طراحی خواهد شد.

</div>

In [42]:
# Code Cell 40
# NB12h — Real structural-contract discovery for
# retrieval-independent Claim Extraction segmentation
#
# Read-only over Accepted Corpus.
# No extraction units.
# No claim extraction.
# No LLM/API calls.
# No overwrite.

assert "project_root" in globals(), (
    "Run the NB12h project-discovery Cell first."
)

assert "role_matrix_output_directory" in globals(), (
    "Run Code Cell 31 first."
)

assert "accepted_corpus_path" in globals(), (
    "Run Code Cell 36 first."
)

assert "claim_discovery_index_rows" in globals(), (
    "Run Code Cell 37 first."
)

assert "chunk_independence_report" in globals(), (
    "Run Code Cell 39 first."
)


STRUCTURE_CONTRACT_AUDIT_VERSION = (
    "nb12h_claim_extraction_structure_contract_audit_v1"
)

STRUCTURE_CONTRACT_REPORT_FILENAME = (
    "claim_extraction_structure_contract_audit_v1.json"
)

STRUCTURE_SAMPLE_LIMIT = 8

STRUCTURE_DISTINCT_SAMPLE_LIMIT = 20


# ---------------------------------------------------------
# Structural signal families
# ---------------------------------------------------------

STRUCTURE_SIGNAL_TERMS = {
    "heading",
    "chapter",
    "section",
    "paragraph",
    "table",
    "block",
    "footnote",
    "supplement",
    "structure",
    "path",
    "parent",
}

ORDER_SIGNAL_TERMS = {
    "index",
    "order",
    "sequence",
    "position",
    "paragraph",
    "block",
    "segment",
    "utterance",
    "start",
    "end",
    "offset",
}

LOCATOR_SIGNAL_TERMS = {
    "locator",
    "location",
    "url",
    "uri",
    "page",
    "chapter",
    "section",
    "paragraph",
    "segment",
    "episode",
    "timestamp",
    "start",
    "end",
    "anchor",
    "path",
}

ATTRIBUTION_SIGNAL_TERMS = {
    "speaker",
    "author",
    "editor",
    "attribution",
    "authority",
    "byline",
    "supervisor",
    "program",
    "episode",
}

NON_STRUCTURAL_EXCLUDE_TERMS = {
    "text",
    "hash",
    "sha256",
    "rights",
    "normalization",
    "quality",
}


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def nb12h_scalar_sample(value):
    if value is None:
        return None

    if isinstance(
        value,
        (str, int, float, bool),
    ):
        text = str(
            value
        )

        if len(text) > 160:
            text = (
                text[:157]
                + "..."
            )

        return text

    if isinstance(
        value,
        list,
    ):
        compact = json.dumps(
            value[:5],
            ensure_ascii=False,
            sort_keys=True,
        )

        if len(compact) > 220:
            compact = (
                compact[:217]
                + "..."
            )

        return compact

    return None


def nb12h_iter_nested_fields(
    value,
    prefix=(),
    depth=0,
    max_depth=6,
):
    if depth > max_depth:
        return

    if isinstance(
        value,
        dict,
    ):
        for key, nested_value in (
            value.items()
        ):
            key_text = str(
                key
            )

            path = (
                prefix
                + (key_text,)
            )

            if isinstance(
                nested_value,
                dict,
            ):
                yield from (
                    nb12h_iter_nested_fields(
                        nested_value,
                        prefix=path,
                        depth=depth + 1,
                        max_depth=max_depth,
                    )
                )

            else:
                yield (
                    ".".join(
                        path
                    ),
                    nested_value,
                )


def nb12h_path_tokens(path):
    normalized = (
        path
        .casefold()
        .replace("-", "_")
    )

    tokens = set(
        normalized
        .replace(".", "_")
        .split("_")
    )

    return (
        normalized,
        tokens,
    )


def nb12h_matches_terms(
    path,
    terms,
):
    normalized, tokens = (
        nb12h_path_tokens(
            path
        )
    )

    return any(
        (
            term in tokens
            or term in normalized
        )
        for term in terms
    )


def nb12h_is_structural_candidate(
    path,
):
    normalized, _ = (
        nb12h_path_tokens(
            path
        )
    )

    if any(
        term in normalized
        for term in (
            NON_STRUCTURAL_EXCLUDE_TERMS
        )
    ):
        return False

    return any([
        nb12h_matches_terms(
            path,
            STRUCTURE_SIGNAL_TERMS,
        ),
        nb12h_matches_terms(
            path,
            ORDER_SIGNAL_TERMS,
        ),
        nb12h_matches_terms(
            path,
            LOCATOR_SIGNAL_TERMS,
        ),
        nb12h_matches_terms(
            path,
            ATTRIBUTION_SIGNAL_TERMS,
        ),
    ])


def nb12h_field_family_labels(
    path,
):
    labels = []

    if nb12h_matches_terms(
        path,
        STRUCTURE_SIGNAL_TERMS,
    ):
        labels.append(
            "STRUCTURE"
        )

    if nb12h_matches_terms(
        path,
        ORDER_SIGNAL_TERMS,
    ):
        labels.append(
            "ORDER"
        )

    if nb12h_matches_terms(
        path,
        LOCATOR_SIGNAL_TERMS,
    ):
        labels.append(
            "LOCATOR"
        )

    if nb12h_matches_terms(
        path,
        ATTRIBUTION_SIGNAL_TERMS,
    ):
        labels.append(
            "ATTRIBUTION"
        )

    return labels


# ---------------------------------------------------------
# Scan Accepted Corpus
# ---------------------------------------------------------

source_record_counts = Counter()

source_ids_by_type = defaultdict(
    set
)

field_presence = defaultdict(
    Counter
)

field_type_counts = defaultdict(
    lambda: defaultdict(
        Counter
    )
)

field_distinct_values = defaultdict(
    lambda: defaultdict(
        set
    )
)

field_samples = defaultdict(
    lambda: defaultdict(
        list
    )
)

record_count = 0


with accepted_corpus_path.open(
    "r",
    encoding="utf-8",
    errors="strict",
) as file:

    for line_number, line in enumerate(
        file,
        1,
    ):
        stripped = line.strip()

        if not stripped:
            continue

        record = json.loads(
            stripped
        )

        assert isinstance(
            record,
            dict,
        ), (
            "Accepted Corpus contains a "
            "non-object record at line "
            f"{line_number}."
        )

        source_type = (
            record.get(
                "source_type"
            )
        )

        source_id = (
            record.get(
                "source_id"
            )
        )

        assert source_type in {
            "website",
            "book",
            "transcript",
        }, (
            "Unexpected source_type at line "
            f"{line_number}: {source_type}"
        )

        assert isinstance(
            source_id,
            str,
        ) and source_id.strip(), (
            "Missing source_id at line "
            f"{line_number}."
        )

        record_count += 1

        source_record_counts[
            source_type
        ] += 1

        source_ids_by_type[
            source_type
        ].add(
            source_id
        )

        fields_seen_in_record = set()

        for (
            field_path,
            value,
        ) in nb12h_iter_nested_fields(
            record
        ):

            if not (
                nb12h_is_structural_candidate(
                    field_path
                )
            ):
                continue

            fields_seen_in_record.add(
                field_path
            )

            field_type_counts[
                source_type
            ][
                field_path
            ][
                type(
                    value
                ).__name__
            ] += 1

            sample_value = (
                nb12h_scalar_sample(
                    value
                )
            )

            if (
                sample_value
                is not None
            ):

                distinct_set = (
                    field_distinct_values[
                        source_type
                    ][
                        field_path
                    ]
                )

                if len(
                    distinct_set
                ) < (
                    STRUCTURE_DISTINCT_SAMPLE_LIMIT
                ):
                    distinct_set.add(
                        sample_value
                    )

                samples = (
                    field_samples[
                        source_type
                    ][
                        field_path
                    ]
                )

                if (
                    sample_value
                    not in samples
                    and len(
                        samples
                    )
                    < STRUCTURE_SAMPLE_LIMIT
                ):
                    samples.append(
                        sample_value
                    )

        for field_path in (
            fields_seen_in_record
        ):
            field_presence[
                source_type
            ][
                field_path
            ] += 1


assert (
    record_count
    == 19456
), (
    "Structural audit did not scan "
    "all 19,456 Accepted Records."
)


# ---------------------------------------------------------
# Frozen population validation
# ---------------------------------------------------------

expected_source_record_counts = {
    "website": 529,
    "book": 2701,
    "transcript": 16226,
}

actual_source_record_counts = dict(
    source_record_counts
)

assert (
    actual_source_record_counts
    == expected_source_record_counts
), (
    "Source-family record counts differ "
    "from the frozen population."
)


actual_source_counts = {
    source_type: len(
        source_ids
    )
    for (
        source_type,
        source_ids,
    ) in source_ids_by_type.items()
}

expected_source_counts = {
    "website": 529,
    "book": 3,
    "transcript": 183,
}

assert (
    actual_source_counts
    == expected_source_counts
), (
    "Unique source counts differ from "
    "the frozen population."
)


# ---------------------------------------------------------
# Website one-source / one-document validation
# ---------------------------------------------------------

website_record_counts_by_source = Counter()

with accepted_corpus_path.open(
    "r",
    encoding="utf-8",
    errors="strict",
) as file:

    for line in file:
        stripped = line.strip()

        if not stripped:
            continue

        record = json.loads(
            stripped
        )

        if (
            record.get(
                "source_type"
            )
            != "website"
        ):
            continue

        website_record_counts_by_source[
            record[
                "source_id"
            ]
        ] += 1


website_non_atomic_sources = {
    source_id: count
    for (
        source_id,
        count,
    ) in (
        website_record_counts_by_source
        .items()
    )
    if count != 1
}

assert not (
    website_non_atomic_sources
), (
    "Website does not have a strict "
    "one-source / one-record structure."
)


# ---------------------------------------------------------
# Build field profiles
# ---------------------------------------------------------

field_profiles_by_source = {}

for source_type in [
    "website",
    "book",
    "transcript",
]:

    source_total = (
        source_record_counts[
            source_type
        ]
    )

    profiles = []

    all_paths = sorted(
        field_presence[
            source_type
        ]
    )

    for field_path in all_paths:

        presence_count = (
            field_presence[
                source_type
            ][
                field_path
            ]
        )

        coverage = (
            presence_count
            / source_total
        )

        profiles.append({
            "field_path": (
                field_path
            ),
            "families": (
                nb12h_field_family_labels(
                    field_path
                )
            ),
            "presence_count": (
                presence_count
            ),
            "source_record_count": (
                source_total
            ),
            "coverage": (
                coverage
            ),
            "value_types": dict(
                sorted(
                    field_type_counts[
                        source_type
                    ][
                        field_path
                    ].items()
                )
            ),
            "distinct_value_sample_count": len(
                field_distinct_values[
                    source_type
                ][
                    field_path
                ]
            ),
            "sample_values": (
                field_samples[
                    source_type
                ][
                    field_path
                ]
            ),
        })

    profiles.sort(
        key=lambda item: (
            -item[
                "coverage"
            ],
            item[
                "field_path"
            ],
        )
    )

    field_profiles_by_source[
        source_type
    ] = profiles


# ---------------------------------------------------------
# High-coverage candidate discovery
# ---------------------------------------------------------

def nb12h_high_coverage_fields(
    source_type,
    family,
    threshold=0.95,
):
    return [
        profile
        for profile in (
            field_profiles_by_source[
                source_type
            ]
        )
        if (
            family
            in profile[
                "families"
            ]
            and profile[
                "coverage"
            ]
            >= threshold
        )
    ]


book_structure_candidates = (
    nb12h_high_coverage_fields(
        "book",
        "STRUCTURE",
    )
)

book_order_candidates = (
    nb12h_high_coverage_fields(
        "book",
        "ORDER",
    )
)

book_locator_candidates = (
    nb12h_high_coverage_fields(
        "book",
        "LOCATOR",
    )
)


transcript_structure_candidates = (
    nb12h_high_coverage_fields(
        "transcript",
        "STRUCTURE",
    )
)

transcript_order_candidates = (
    nb12h_high_coverage_fields(
        "transcript",
        "ORDER",
    )
)

transcript_locator_candidates = (
    nb12h_high_coverage_fields(
        "transcript",
        "LOCATOR",
    )
)

transcript_attribution_candidates = (
    nb12h_high_coverage_fields(
        "transcript",
        "ATTRIBUTION",
    )
)


# ---------------------------------------------------------
# Gate
#
# We do not assert a specific field name.
# We only require real high-coverage structural evidence.
# ---------------------------------------------------------

assert (
    book_structure_candidates
    or book_order_candidates
), (
    "No >=95% Book structural/order field "
    "was discovered. Segmentation contract "
    "cannot yet be designed."
)

assert book_locator_candidates, (
    "No >=95% Book locator field "
    "was discovered."
)

assert (
    transcript_structure_candidates
    or transcript_order_candidates
), (
    "No >=95% Transcript structural/order "
    "field was discovered."
)

assert transcript_locator_candidates, (
    "No >=95% Transcript locator field "
    "was discovered."
)

assert transcript_attribution_candidates, (
    "No >=95% Transcript attribution field "
    "was discovered."
)


# ---------------------------------------------------------
# No field is selected yet.
# Cell 41 will use the actual output of this audit.
# ---------------------------------------------------------

segmentation_contract_status = {
    "website": (
        "DOCUMENT_ATOMIC_STRUCTURE_CONFIRMED"
    ),
    "book": (
        "REAL_STRUCTURE_FIELDS_DISCOVERED_"
        "SELECTION_PENDING"
    ),
    "transcript": (
        "REAL_STRUCTURE_FIELDS_DISCOVERED_"
        "SELECTION_PENDING"
    ),
}

retrieval_strategy_dependency = False

extraction_units_created = False

claim_extraction_started = False


# ---------------------------------------------------------
# Compact report profiles
# ---------------------------------------------------------

def nb12h_compact_profiles(
    profiles,
):
    return [
        {
            "field_path": (
                profile[
                    "field_path"
                ]
            ),
            "families": (
                profile[
                    "families"
                ]
            ),
            "presence_count": (
                profile[
                    "presence_count"
                ]
            ),
            "coverage": (
                profile[
                    "coverage"
                ]
            ),
            "value_types": (
                profile[
                    "value_types"
                ]
            ),
            "distinct_value_sample_count": (
                profile[
                    "distinct_value_sample_count"
                ]
            ),
            "sample_values": (
                profile[
                    "sample_values"
                ]
            ),
        }
        for profile in profiles
    ]


structure_contract_report = {
    "audit_version": (
        STRUCTURE_CONTRACT_AUDIT_VERSION
    ),

    "status": "PASS",

    "population": {
        "accepted_record_count": (
            record_count
        ),
        "record_counts_by_source": (
            actual_source_record_counts
        ),
        "unique_source_counts": (
            actual_source_counts
        ),
    },

    "website": {
        "one_source_one_record": True,
        "non_atomic_source_count": 0,
        "field_profiles": (
            nb12h_compact_profiles(
                field_profiles_by_source[
                    "website"
                ]
            )
        ),
    },

    "book": {
        "structure_candidates_95pct": (
            nb12h_compact_profiles(
                book_structure_candidates
            )
        ),
        "order_candidates_95pct": (
            nb12h_compact_profiles(
                book_order_candidates
            )
        ),
        "locator_candidates_95pct": (
            nb12h_compact_profiles(
                book_locator_candidates
            )
        ),
        "all_structural_field_profiles": (
            nb12h_compact_profiles(
                field_profiles_by_source[
                    "book"
                ]
            )
        ),
    },

    "transcript": {
        "structure_candidates_95pct": (
            nb12h_compact_profiles(
                transcript_structure_candidates
            )
        ),
        "order_candidates_95pct": (
            nb12h_compact_profiles(
                transcript_order_candidates
            )
        ),
        "locator_candidates_95pct": (
            nb12h_compact_profiles(
                transcript_locator_candidates
            )
        ),
        "attribution_candidates_95pct": (
            nb12h_compact_profiles(
                transcript_attribution_candidates
            )
        ),
        "all_structural_field_profiles": (
            nb12h_compact_profiles(
                field_profiles_by_source[
                    "transcript"
                ]
            )
        ),
    },

    "segmentation_contract_status": (
        segmentation_contract_status
    ),

    "retrieval_strategy_dependency": (
        retrieval_strategy_dependency
    ),

    "field_selection_finalized": False,
    "extraction_units_created": False,
    "claim_extraction_started": False,

    "api_calls_made": 0,
    "cost_incurred": 0,

    "source_files_modified": False,
    "active_dataset_modified": False,
    "golden_dataset_v1_modified": False,
    "core46_modified": False,
    "golden_dataset_v2_created": False,
}


# ---------------------------------------------------------
# No-overwrite persistence
# ---------------------------------------------------------

structure_contract_report_path = (
    role_matrix_output_directory
    / STRUCTURE_CONTRACT_REPORT_FILENAME
)

if (
    structure_contract_report_path
    .exists()
):

    with (
        structure_contract_report_path
        .open(
            "r",
            encoding="utf-8",
        )
    ) as file:

        existing_report = json.load(
            file
        )

    comparable_existing = dict(
        existing_report
    )

    comparable_current = dict(
        structure_contract_report
    )

    comparable_existing.pop(
        "created_at_utc",
        None,
    )

    comparable_current.pop(
        "created_at_utc",
        None,
    )

    assert (
        comparable_existing
        == comparable_current
    ), (
        "Existing structure-contract audit "
        "differs from this run. "
        "Refusing to overwrite it."
    )

    structure_contract_report = (
        existing_report
    )

    structure_contract_write_status = (
        "REUSED_EXISTING_IDENTICAL_REPORT"
    )

else:

    structure_contract_report[
        "created_at_utc"
    ] = datetime.now(
        timezone.utc
    ).isoformat()

    with (
        structure_contract_report_path
        .open(
            "x",
            encoding="utf-8",
        )
    ) as file:

        json.dump(
            structure_contract_report,
            file,
            ensure_ascii=False,
            indent=2,
        )

        file.write("\n")

    structure_contract_write_status = (
        "CREATED_NEW_REPORT"
    )


# ---------------------------------------------------------
# Compact output
# ---------------------------------------------------------

def nb12h_print_candidates(
    title,
    profiles,
    limit=20,
):
    print()
    print(title)

    for profile in profiles[:limit]:

        print(
            "  -",
            profile[
                "field_path"
            ],
            "| coverage=",
            round(
                profile[
                    "coverage"
                ],
                4,
            ),
            "| types=",
            profile[
                "value_types"
            ],
            "| samples=",
            profile[
                "sample_values"
            ][:3],
        )

    if len(profiles) > limit:
        print(
            "  ...",
            len(profiles) - limit,
            "more",
        )


print(
    "=== NB12h CLAIM EXTRACTION "
    "STRUCTURE CONTRACT AUDIT ==="
)

print()
print(
    "Accepted records:",
    record_count,
)

print(
    "Record counts by source:",
    actual_source_record_counts,
)

print(
    "Unique sources:",
    actual_source_counts,
)

print()
print(
    "Website one-source / one-record:",
    True,
)


nb12h_print_candidates(
    "Book >=95% STRUCTURE fields:",
    book_structure_candidates,
)

nb12h_print_candidates(
    "Book >=95% ORDER fields:",
    book_order_candidates,
)

nb12h_print_candidates(
    "Book >=95% LOCATOR fields:",
    book_locator_candidates,
)


nb12h_print_candidates(
    "Transcript >=95% STRUCTURE fields:",
    transcript_structure_candidates,
)

nb12h_print_candidates(
    "Transcript >=95% ORDER fields:",
    transcript_order_candidates,
)

nb12h_print_candidates(
    "Transcript >=95% LOCATOR fields:",
    transcript_locator_candidates,
)

nb12h_print_candidates(
    "Transcript >=95% ATTRIBUTION fields:",
    transcript_attribution_candidates,
)


print()
print(
    "Segmentation contract status:",
    segmentation_contract_status,
)

print(
    "Retrieval strategy dependency:",
    retrieval_strategy_dependency,
)

print(
    "Field selection finalized:",
    False,
)

print(
    "Extraction units created:",
    False,
)

print(
    "Claim extraction started:",
    False,
)

print()
print(
    "Audit report:",
    structure_contract_report_path,
)

print(
    "Report write status:",
    structure_contract_write_status,
)

print()
print("API calls made:", 0)
print("Cost incurred:", 0)
print("Source files modified:", False)
print("Active dataset modified:", False)
print("Golden Dataset v1 modified:", False)
print("Core-46 modified:", False)
print("Golden Dataset v2 created:", False)

print()
print(
    "NB12h CLAIM EXTRACTION "
    "STRUCTURE CONTRACT AUDIT: PASS"
)

=== NB12h CLAIM EXTRACTION STRUCTURE CONTRACT AUDIT ===

Accepted records: 19456
Record counts by source: {'website': 529, 'book': 2701, 'transcript': 16226}
Unique sources: {'website': 529, 'book': 3, 'transcript': 183}

Website one-source / one-record: True

Book >=95% STRUCTURE fields:
  - lineage.parent_id | coverage= 1.0 | types= {'str': 2701} | samples= ['book:adult-financial-literacy-course:fa-source', 'book:financial-literacy-for-all:fa-source', 'book:trainer-training-financial-literacy:fa-source']
  - lineage.raw_source_path | coverage= 1.0 | types= {'str': 2701} | samples= ['data/books/درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx', 'data/books/سواد مالی برای همه.docx', 'data/books/نوشتار آموزشی تربیت مربی سواد مالی.docx']
  - locator.heading_path | coverage= 0.9956 | types= {'list': 2689} | samples= ['[]', '["سخن آکادمی هوش مالی"]', '["فصل ا: اهمیت و تعریف سواد مالی"]']
  - locator.paragraph_id | coverage= 0.9956 | types= {'str': 2689} | samples= ['book:adult-financial

<div style="direction: rtl; white-space: normal; line-height: 1;">
به دلیل ساخت claim 
میرویم و دریک نوت بوک کامل انجام میدهیم هم برای راگ هم برای کتب
</div>